# Эксперимент 1: расширение словаря токенизатора

**Статус: no_go**

## Гипотеза

Если добавить в токенизатор новые "целые слова" — часто встречающиеся
русские слова, которые сейчас режутся на несколько токенов — вопрос
будет описываться меньшим числом токенов, что должно сократить время
обработки входа (prefill).


In [2]:
%pip install -q "transformers>=4.46" tokenizers



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Метод

- До 4096 новых токенов-кандидатов (`AddedToken`), выбранных по частоте
  на внешнем корпусе (10 000 обучающих документов Wikipedia RU)
- Старые BPE-объединения и ID токенизатора не изменяются
- Критерий перехода к GPU-проверке (закреплён до измерений):
  сокращение суммарного числа токенов ≥5% на dev-выборке Wikipedia,
  отсутствие роста числа токенов на 70 реальных MCQ-промптах
  (validation), замедление кодирования не более чем в 1.5 раза

Список кандидатов (слово, частота на train, старые ID) — результат уже
проведённого отбора; он встроен ниже как данные, чтобы не тянуть заново
весь Wikipedia-корпус. Токенизатор — настоящий, загружается живьём.


In [3]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3.8-27B"
MODEL_REVISION = "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0"

original = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
print(f"Оригинальный словарь: {len(original)} токенов")


Оригинальный словарь: 248077 токенов


In [4]:
import json

# 4096 кандидатов новых токенов — результат отбора по частоте на
# train-корпусе (10 000 документов Wikipedia RU), встроены как данные.
NEW_TOKENS_JSON = '''[{"text":"году","train_count":20126,"old_ids":[63046,148660],"score":20126,"new_id":248077},{"text":"Примечания","train_count":4459,"old_ids":[51844,15630,53887,36916],"score":13377,"new_id":248078},{"text":"был","train_count":11933,"old_ids":[148812,2941],"score":11933,"new_id":248079},{"text":"также","train_count":8591,"old_ids":[149338,56090],"score":8591,"new_id":248080},{"text":"после","train_count":4266,"old_ids":[7754,22078,148284],"score":8532,"new_id":248081},{"text":"Ссылки","train_count":4167,"old_ids":[18727,63469,16256],"score":8334,"new_id":248082},{"text":"России","train_count":3789,"old_ids":[32392,149318,44633],"score":7578,"new_id":248083},{"text":"СССР","train_count":3559,"old_ids":[152611,18727,32392],"score":7118,"new_id":248084},{"text":"Население","train_count":1970,"old_ids":[19582,17417,11777,16669],"score":5910,"new_id":248085},{"text":"После","train_count":2901,"old_ids":[16362,22078,148284],"score":5802,"new_id":248086},{"text":"человек","train_count":2767,"old_ids":[5831,11777,155189],"score":5534,"new_id":248087},{"text":"была","train_count":5524,"old_ids":[148812,29354],"score":5524,"new_id":248088},{"text":"время","train_count":5474,"old_ids":[5301,156579],"score":5474,"new_id":248089},{"text":"История","train_count":2710,"old_ids":[29189,152950,37835],"score":5420,"new_id":248090},{"text":"были","train_count":5102,"old_ids":[148812,10750],"score":5102,"new_id":248091},{"text":"было","train_count":5021,"old_ids":[148812,37483],"score":5021,"new_id":248092},{"text":"Литература","train_count":1541,"old_ids":[61757,9080,7358,154182],"score":4623,"new_id":248093},{"text":"декабря","train_count":2169,"old_ids":[95525,12751,151439],"score":4338,"new_id":248094},{"text":"деревня","train_count":2152,"old_ids":[152138,148380,37855],"score":4304,"new_id":248095},{"text":"области","train_count":4103,"old_ids":[148280,162730],"score":4103,"new_id":248096},{"text":"человека","train_count":1271,"old_ids":[5831,11777,150937,12751],"score":3813,"new_id":248097},{"text":"группы","train_count":1785,"old_ids":[14942,83147,4403],"score":3570,"new_id":248098},{"text":"несколько","train_count":1752,"old_ids":[149904,45527,150535],"score":3504,"new_id":248099},{"text":"университета","train_count":1159,"old_ids":[3652,150439,152898,149005],"score":3477,"new_id":248100},{"text":"футболист","train_count":1149,"old_ids":[18768,36183,157759,6389],"score":3447,"new_id":248101},{"text":"территории","train_count":1712,"old_ids":[55055,149454,152597],"score":3424,"new_id":248102},{"text":"января","train_count":1694,"old_ids":[4096,156345,4096],"score":3388,"new_id":248103},{"text":"составе","train_count":1647,"old_ids":[2175,150838,1450],"score":3294,"new_id":248104},{"text":"Биография","train_count":1593,"old_ids":[207347,1403,181923],"score":3186,"new_id":248105},{"text":"затем","train_count":1572,"old_ids":[8688,7721,148281],"score":3144,"new_id":248106},{"text":"занимает","train_count":1017,"old_ids":[8688,13603,6240,26046],"score":3051,"new_id":248107},{"text":"районе","train_count":3010,"old_ids":[180093,148917],"score":3010,"new_id":248108},{"text":"американский","train_count":1494,"old_ids":[1424,155899,149800],"score":2988,"new_id":248109},{"text":"получил","train_count":1492,"old_ids":[148634,27448,61057],"score":2984,"new_id":248110},{"text":"который","train_count":2982,"old_ids":[151549,33477],"score":2982,"new_id":248111},{"text":"района","train_count":2952,"old_ids":[180093,71221],"score":2952,"new_id":248112},{"text":"США","train_count":2948,"old_ids":[18727,159433],"score":2948,"new_id":248113},{"text":"всех","train_count":1466,"old_ids":[5301,2175,93834],"score":2932,"new_id":248114},{"text":"Великобритании","train_count":586,"old_ids":[15729,58503,45527,9629,149454,149091],"score":2930,"new_id":248115},{"text":"является","train_count":2874,"old_ids":[4096,153190],"score":2874,"new_id":248116},{"text":"Германии","train_count":1436,"old_ids":[240053,6240,149091],"score":2872,"new_id":248117},{"text":"города","train_count":2866,"old_ids":[162121,38183],"score":2866,"new_id":248118},{"text":"однако","train_count":1430,"old_ids":[148277,13785,45527],"score":2860,"new_id":248119},{"text":"двух","train_count":1405,"old_ids":[6733,5301,149576],"score":2810,"new_id":248120},{"text":"составляет","train_count":1403,"old_ids":[2175,150838,151730],"score":2806,"new_id":248121},{"text":"века","train_count":2805,"old_ids":[148537,12751],"score":2805,"new_id":248122},{"text":"Однако","train_count":1399,"old_ids":[19736,152209,45527],"score":2798,"new_id":248123},{"text":"жизни","train_count":1396,"old_ids":[16467,39427,21918],"score":2792,"new_id":248124},{"text":"Владимир","train_count":913,"old_ids":[15729,148606,15630,74990],"score":2739,"new_id":248125},{"text":"сентября","train_count":1369,"old_ids":[2175,156113,151439],"score":2738,"new_id":248126},{"text":"августа","train_count":1368,"old_ids":[148292,14942,154271],"score":2736,"new_id":248127},{"text":"апреля","train_count":1368,"old_ids":[1424,148489,41580],"score":2736,"new_id":248128},{"text":"вместе","train_count":1350,"old_ids":[5301,150436,1450],"score":2700,"new_id":248129},{"text":"советский","train_count":2696,"old_ids":[227012,149800],"score":2696,"new_id":248130},{"text":"состав","train_count":2693,"old_ids":[2175,150838],"score":2693,"new_id":248131},{"text":"только","train_count":2657,"old_ids":[23858,150535],"score":2657,"new_id":248132},{"text":"Российской","train_count":1302,"old_ids":[32392,150405,149193],"score":2604,"new_id":248133},{"text":"своей","train_count":1302,"old_ids":[2175,159117,11796],"score":2604,"new_id":248134},{"text":"февраля","train_count":1296,"old_ids":[18768,154003,13302],"score":2592,"new_id":248135},{"text":"деятель","train_count":1293,"old_ids":[95525,86128,24623],"score":2586,"new_id":248136},{"text":"посёлок","train_count":862,"old_ids":[7754,22078,42565,171144],"score":2586,"new_id":248137},{"text":"всего","train_count":1290,"old_ids":[5301,2175,148702],"score":2580,"new_id":248138},{"text":"имеет","train_count":1288,"old_ids":[15630,1450,7921],"score":2576,"new_id":248139},{"text":"генерал","train_count":1287,"old_ids":[170802,7358,15481],"score":2574,"new_id":248140},{"text":"режиссёр","train_count":643,"old_ids":[148279,16467,13113,2175,156204],"score":2572,"new_id":248141},{"text":"Федерации","train_count":852,"old_ids":[52337,148276,7358,51788],"score":2556,"new_id":248142},{"text":"войны","train_count":2554,"old_ids":[159035,12014],"score":2554,"new_id":248143},{"text":"годах","train_count":2545,"old_ids":[163542,10161],"score":2545,"new_id":248144},{"text":"ноября","train_count":1268,"old_ids":[13292,4096,151439],"score":2536,"new_id":248145},{"text":"округа","train_count":1268,"old_ids":[148518,62215,1424],"score":2536,"new_id":248146},{"text":"населения","train_count":1258,"old_ids":[13785,189921,17941],"score":2516,"new_id":248147},{"text":"связи","train_count":1221,"old_ids":[2175,163720,1739],"score":2442,"new_id":248148},{"text":"период","train_count":1216,"old_ids":[7754,149098,148277],"score":2432,"new_id":248149},{"text":"находится","train_count":1189,"old_ids":[13785,148334,92907],"score":2378,"new_id":248150},{"text":"Занимает","train_count":766,"old_ids":[34279,13603,6240,26046],"score":2298,"new_id":248151},{"text":"этого","train_count":2247,"old_ids":[65131,171664],"score":2247,"new_id":248152},{"text":"между","train_count":2215,"old_ids":[148446,150043],"score":2215,"new_id":248153},{"text":"своего","train_count":1100,"old_ids":[2175,159117,63046],"score":2200,"new_id":248154},{"text":"результате","train_count":1092,"old_ids":[150564,62709,152657],"score":2184,"new_id":248155},{"text":"Согласно","train_count":726,"old_ids":[18727,149095,149827,13292],"score":2178,"new_id":248156},{"text":"Населённые","train_count":541,"old_ids":[19582,17417,11777,42565,155130],"score":2164,"new_id":248157},{"text":"государственный","train_count":720,"old_ids":[14942,22078,150968,159759],"score":2160,"new_id":248158},{"text":"область","train_count":1074,"old_ids":[148280,29354,33348],"score":2148,"new_id":248159},{"text":"Отечественной","train_count":526,"old_ids":[19736,48824,5831,150198,148317],"score":2104,"new_id":248160},{"text":"Официальный","train_count":1050,"old_ids":[19736,149243,152027],"score":2100,"new_id":248161},{"text":"случае","train_count":686,"old_ids":[38612,27448,1424,1450],"score":2058,"new_id":248162},{"text":"страница","train_count":1017,"old_ids":[48941,45144,1424],"score":2034,"new_id":248163},{"text":"российский","train_count":2021,"old_ids":[184874,149800],"score":2021,"new_id":248164},{"text":"актриса","train_count":668,"old_ids":[148318,153298,13113,1424],"score":2004,"new_id":248165},{"text":"участие","train_count":1001,"old_ids":[27448,80378,47552],"score":2002,"new_id":248166},{"text":"этом","train_count":1999,"old_ids":[65131,149127],"score":1999,"new_id":248167},{"text":"Официальная","train_count":991,"old_ids":[19736,149243,231448],"score":1982,"new_id":248168},{"text":"группа","train_count":989,"old_ids":[14942,83147,1424],"score":1978,"new_id":248169},{"text":"своих","train_count":966,"old_ids":[2175,148331,64392],"score":1932,"new_id":248170},{"text":"один","train_count":1909,"old_ids":[1403,67958],"score":1909,"new_id":248171},{"text":"место","train_count":1906,"old_ids":[150436,1403],"score":1906,"new_id":248172},{"text":"чемпионата","train_count":632,"old_ids":[155356,148686,16806,40710],"score":1896,"new_id":248173},{"text":"когда","train_count":1890,"old_ids":[45527,150921],"score":1890,"new_id":248174},{"text":"Иванович","train_count":629,"old_ids":[29189,149137,148275,31889],"score":1887,"new_id":248175},{"text":"губернии","train_count":627,"old_ids":[14942,61324,7358,155016],"score":1881,"new_id":248176},{"text":"профессор","train_count":627,"old_ids":[149490,18768,90202,8789],"score":1881,"new_id":248177},{"text":"деятельности","train_count":626,"old_ids":[95525,86128,24623,65360],"score":1878,"new_id":248178},{"text":"часть","train_count":1837,"old_ids":[148659,33348],"score":1837,"new_id":248179},{"text":"название","train_count":1836,"old_ids":[13785,202403],"score":1836,"new_id":248180},{"text":"актёр","train_count":917,"old_ids":[148318,1729,156204],"score":1834,"new_id":248181},{"text":"около","train_count":1801,"old_ids":[1403,159322],"score":1801,"new_id":248182},{"text":"степени","train_count":900,"old_ids":[6389,148445,9396],"score":1800,"new_id":248183},{"text":"Игроки","train_count":899,"old_ids":[29189,177501,16256],"score":1798,"new_id":248184},{"text":"театра","train_count":895,"old_ids":[48824,7721,21441],"score":1790,"new_id":248185},{"text":"через","train_count":1786,"old_ids":[151735,148377],"score":1786,"new_id":248186},{"text":"Михаил","train_count":595,"old_ids":[36736,64392,1424,61057],"score":1785,"new_id":248187},{"text":"смерти","train_count":882,"old_ids":[2175,60909,21797],"score":1764,"new_id":248188},{"text":"фамилия","train_count":865,"old_ids":[18768,150190,150095],"score":1730,"new_id":248189},{"text":"империи","train_count":864,"old_ids":[15630,149208,44633],"score":1728,"new_id":248190},{"text":"Москве","train_count":859,"old_ids":[212095,4640,148537],"score":1718,"new_id":248191},{"text":"которых","train_count":1710,"old_ids":[151549,60038],"score":1710,"new_id":248192},{"text":"которой","train_count":1707,"old_ids":[151549,148293],"score":1707,"new_id":248193},{"text":"деятельность","train_count":565,"old_ids":[95525,86128,24623,58245],"score":1695,"new_id":248194},{"text":"государства","train_count":562,"old_ids":[14942,22078,150968,89771],"score":1686,"new_id":248195},{"text":"других","train_count":1678,"old_ids":[192548,64392],"score":1678,"new_id":248196},{"text":"состоит","train_count":839,"old_ids":[2175,151290,9080],"score":1678,"new_id":248197},{"text":"Гмина","train_count":1673,"old_ids":[35884,149717],"score":1673,"new_id":248198},{"text":"административная","train_count":328,"old_ids":[149099,150704,32884,77779,26125,42308],"score":1640,"new_id":248199},{"text":"ещё","train_count":1632,"old_ids":[82557,42565],"score":1632,"new_id":248200},{"text":"британский","train_count":542,"old_ids":[9629,149454,7105,149800],"score":1626,"new_id":248201},{"text":"наиболее","train_count":805,"old_ids":[13785,151223,152156],"score":1610,"new_id":248202},{"text":"Англии","train_count":535,"old_ids":[34960,2040,151203,1739],"score":1605,"new_id":248203},{"text":"чтобы","train_count":1597,"old_ids":[160051,148812],"score":1597,"new_id":248204},{"text":"район","train_count":1581,"old_ids":[180093,16806],"score":1581,"new_id":248205},{"text":"культуры","train_count":785,"old_ids":[231736,3652,36747],"score":1570,"new_id":248206},{"text":"качестве","train_count":1554,"old_ids":[12751,198458],"score":1554,"new_id":248207},{"text":"данным","train_count":771,"old_ids":[6733,26269,6240],"score":1542,"new_id":248208},{"text":"Известные","train_count":770,"old_ids":[193863,149562,41546],"score":1540,"new_id":248209},{"text":"команды","train_count":770,"old_ids":[148558,72884,4403],"score":1540,"new_id":248210},{"text":"свои","train_count":768,"old_ids":[2175,148331,1739],"score":1536,"new_id":248211},{"text":"Достопримечательности","train_count":254,"old_ids":[23355,151290,148489,15630,53887,57868,65360],"score":1524,"new_id":248212},{"text":"имени","train_count":1519,"old_ids":[15630,9396],"score":1519,"new_id":248213},{"text":"основном","train_count":758,"old_ids":[22078,148366,148524],"score":1516,"new_id":248214},{"text":"этих","train_count":756,"old_ids":[65131,21797,10161],"score":1512,"new_id":248215},{"text":"Кроме","train_count":754,"old_ids":[25497,148283,148446],"score":1508,"new_id":248216},{"text":"Города","train_count":750,"old_ids":[35884,148367,38183],"score":1500,"new_id":248217},{"text":"поселения","train_count":499,"old_ids":[7754,22078,11777,17941],"score":1497,"new_id":248218},{"text":"которого","train_count":1493,"old_ids":[151549,148449],"score":1493,"new_id":248219},{"text":"которая","train_count":1488,"old_ids":[151549,34226],"score":1488,"new_id":248220},{"text":"быть","train_count":1482,"old_ids":[148812,27851],"score":1482,"new_id":248221},{"text":"годов","train_count":1482,"old_ids":[63046,148716],"score":1482,"new_id":248222},{"text":"управления","train_count":741,"old_ids":[47174,148381,59595],"score":1482,"new_id":248223},{"text":"пункты","train_count":727,"old_ids":[7754,38629,82942],"score":1454,"new_id":248224},{"text":"есть","train_count":1445,"old_ids":[1450,33348],"score":1445,"new_id":248225},{"text":"Андрей","train_count":481,"old_ids":[34960,2040,60659,148298],"score":1443,"new_id":248226},{"text":"Николай","train_count":719,"old_ids":[19582,151907,152487],"score":1438,"new_id":248227},{"text":"марта","train_count":1438,"old_ids":[173719,42074],"score":1438,"new_id":248228},{"text":"поскольку","train_count":478,"old_ids":[7754,22078,45527,153542],"score":1434,"new_id":248229},{"text":"сезона","train_count":715,"old_ids":[2175,148377,71221],"score":1430,"new_id":248230},{"text":"июля","train_count":1418,"old_ids":[163855,13302],"score":1418,"new_id":248231},{"text":"зависимости","train_count":353,"old_ids":[8688,148292,13113,15630,81839],"score":1412,"new_id":248232},{"text":"используются","train_count":352,"old_ids":[13113,183192,8688,149460,19937],"score":1408,"new_id":248233},{"text":"различных","train_count":702,"old_ids":[148464,87018,42240],"score":1404,"new_id":248234},{"text":"Политическая","train_count":350,"old_ids":[16362,148294,9080,77436,150621],"score":1400,"new_id":248235},{"text":"названием","train_count":700,"old_ids":[13785,8688,159535],"score":1400,"new_id":248236},{"text":"Республики","train_count":698,"old_ids":[32392,151869,16256],"score":1396,"new_id":248237},{"text":"Австрии","train_count":696,"old_ids":[34960,5301,215753],"score":1392,"new_id":248238},{"text":"времени","train_count":1389,"old_ids":[199366,152168],"score":1389,"new_id":248239},{"text":"тысяч","train_count":693,"old_ids":[82942,19937,5831],"score":1386,"new_id":248240},{"text":"июня","train_count":1385,"old_ids":[163855,37855],"score":1385,"new_id":248241},{"text":"используется","train_count":690,"old_ids":[13113,183192,185250],"score":1380,"new_id":248242},{"text":"всё","train_count":687,"old_ids":[5301,2175,42565],"score":1374,"new_id":248243},{"text":"административно","train_count":342,"old_ids":[149099,150704,32884,77779,167596],"score":1368,"new_id":248244},{"text":"октября","train_count":1367,"old_ids":[148518,159338],"score":1367,"new_id":248245},{"text":"клуба","train_count":683,"old_ids":[4640,152887,1424],"score":1366,"new_id":248246},{"text":"некоторых","train_count":681,"old_ids":[76069,151549,60038],"score":1362,"new_id":248247},{"text":"французский","train_count":454,"old_ids":[18768,33437,163227,149800],"score":1362,"new_id":248248},{"text":"назначен","train_count":679,"old_ids":[13785,149340,5092],"score":1358,"new_id":248249},{"text":"своё","train_count":675,"old_ids":[2175,148331,42565],"score":1350,"new_id":248250},{"text":"академии","train_count":670,"old_ids":[1424,154408,159999],"score":1340,"new_id":248251},{"text":"английский","train_count":444,"old_ids":[7105,151203,11796,149800],"score":1332,"new_id":248252},{"text":"РСФСР","train_count":660,"old_ids":[205332,220449,32392],"score":1320,"new_id":248253},{"text":"метрополитена","train_count":329,"old_ids":[148557,148283,148634,9080,48623],"score":1316,"new_id":248254},{"text":"работы","train_count":1313,"old_ids":[149100,4403],"score":1313,"new_id":248255},{"text":"середине","train_count":437,"old_ids":[2175,148579,67958,1450],"score":1311,"new_id":248256},{"text":"представителей","train_count":654,"old_ids":[177917,148448,150471],"score":1308,"new_id":248257},{"text":"нескольких","train_count":653,"old_ids":[149904,193929,148797],"score":1306,"new_id":248258},{"text":"Второй","train_count":652,"old_ids":[15729,35458,148293],"score":1304,"new_id":248259},{"text":"истории","train_count":1304,"old_ids":[32884,152597],"score":1304,"new_id":248260},{"text":"государственного","train_count":433,"old_ids":[14942,22078,150968,172012],"score":1299,"new_id":248261},{"text":"происходит","train_count":649,"old_ids":[149490,13113,150294],"score":1298,"new_id":248262},{"text":"позже","train_count":647,"old_ids":[7754,148303,56090],"score":1294,"new_id":248263},{"text":"поступил","train_count":429,"old_ids":[7754,23514,47174,61057],"score":1287,"new_id":248264},{"text":"Европы","train_count":643,"old_ids":[53241,149552,150258],"score":1286,"new_id":248265},{"text":"конце","train_count":1286,"old_ids":[150270,149053],"score":1286,"new_id":248266},{"text":"этой","train_count":1283,"old_ids":[65131,170627],"score":1283,"new_id":248267},{"text":"песни","train_count":635,"old_ids":[7754,21785,21918],"score":1270,"new_id":248268},{"text":"позднее","train_count":423,"old_ids":[7754,148303,6733,149499],"score":1269,"new_id":248269},{"text":"результатам","train_count":423,"old_ids":[150564,62709,40710,6240],"score":1269,"new_id":248270},{"text":"себя","train_count":1269,"old_ids":[148938,170202],"score":1269,"new_id":248271},{"text":"двумя","train_count":422,"old_ids":[6733,5301,37815,4096],"score":1266,"new_id":248272},{"text":"согласно","train_count":420,"old_ids":[148454,14942,149827,13292],"score":1260,"new_id":248273},{"text":"режиссёра","train_count":251,"old_ids":[148279,16467,13113,2175,42565,21441],"score":1255,"new_id":248274},{"text":"директор","train_count":417,"old_ids":[6733,74990,148301,35458],"score":1251,"new_id":248275},{"text":"первый","train_count":1248,"old_ids":[149208,159061],"score":1248,"new_id":248276},{"text":"площадь","train_count":1247,"old_ids":[170421,166595],"score":1247,"new_id":248277},{"text":"алфавиту","train_count":415,"old_ids":[15481,18768,148292,149452],"score":1245,"new_id":248278},{"text":"Бургомистр","train_count":311,"old_ids":[58297,53555,160259,32884,2115],"score":1244,"new_id":248279},{"text":"Москвы","train_count":622,"old_ids":[212095,4640,148705],"score":1244,"new_id":248280},{"text":"университет","train_count":414,"old_ids":[3652,150439,152898,7921],"score":1242,"new_id":248281},{"text":"жизнь","train_count":620,"old_ids":[16467,39427,149497],"score":1240,"new_id":248282},{"text":"немецкий","train_count":619,"old_ids":[76069,148446,203809],"score":1238,"new_id":248283},{"text":"развития","train_count":619,"old_ids":[148464,149734,37835],"score":1238,"new_id":248284},{"text":"Александрович","train_count":411,"old_ids":[34960,221097,148275,31889],"score":1233,"new_id":248285},{"text":"Фильмы","train_count":616,"old_ids":[52337,64257,148706],"score":1232,"new_id":248286},{"text":"художник","train_count":615,"old_ids":[157951,149244,67494],"score":1230,"new_id":248287},{"text":"имеют","train_count":614,"old_ids":[15630,1450,66694],"score":1228,"new_id":248288},{"text":"Николаевич","train_count":409,"old_ids":[19582,151907,29354,198326],"score":1227,"new_id":248289},{"text":"Административный","train_count":244,"old_ids":[210138,150704,32884,77779,26125,32401],"score":1220,"new_id":248290},{"text":"Заслуженный","train_count":243,"old_ids":[34279,17417,2941,54025,5092,32401],"score":1215,"new_id":248291},{"text":"помощью","train_count":606,"old_ids":[151201,88260,84920],"score":1212,"new_id":248292},{"text":"принял","train_count":404,"old_ids":[148489,18372,4096,2941],"score":1212,"new_id":248293},{"text":"Александр","train_count":1203,"old_ids":[34960,221097],"score":1203,"new_id":248294},{"text":"окончания","train_count":401,"old_ids":[148518,16806,5831,36916],"score":1203,"new_id":248295},{"text":"завод","train_count":601,"old_ids":[8688,148292,148277],"score":1202,"new_id":248296},{"text":"Социалистического","train_count":300,"old_ids":[173518,14666,45679,6389,158638],"score":1200,"new_id":248297},{"text":"университете","train_count":400,"old_ids":[3652,150439,152898,149155],"score":1200,"new_id":248298},{"text":"Санкт","train_count":1198,"old_ids":[245722,150094],"score":1198,"new_id":248299},{"text":"большой","train_count":596,"old_ids":[9629,148701,148293],"score":1192,"new_id":248300},{"text":"Италии","train_count":591,"old_ids":[29189,42074,160195],"score":1182,"new_id":248301},{"text":"политический","train_count":393,"old_ids":[148634,9080,77436,149537],"score":1179,"new_id":248302},{"text":"административному","train_count":235,"old_ids":[149099,150704,32884,77779,26125,149891],"score":1175,"new_id":248303},{"text":"условиях","train_count":391,"old_ids":[41635,148569,37835,10161],"score":1173,"new_id":248304},{"text":"альбома","train_count":585,"old_ids":[36771,148421,66050],"score":1170,"new_id":248305},{"text":"свою","train_count":1167,"old_ids":[2175,187779],"score":1167,"new_id":248306},{"text":"Белоруссии","train_count":290,"old_ids":[58297,11777,8789,41635,151883],"score":1160,"new_id":248307},{"text":"село","train_count":1157,"old_ids":[2175,150195],"score":1157,"new_id":248308},{"text":"трёх","train_count":577,"old_ids":[153298,42565,10161],"score":1154,"new_id":248309},{"text":"наук","train_count":1151,"old_ids":[13785,63855],"score":1151,"new_id":248310},{"text":"второй","train_count":1146,"old_ids":[167396,148293],"score":1146,"new_id":248311},{"text":"расположен","train_count":573,"old_ids":[148761,220495,148942],"score":1146,"new_id":248312},{"text":"Кубка","train_count":570,"old_ids":[25497,61324,12751],"score":1140,"new_id":248313},{"text":"стала","train_count":1139,"old_ids":[6389,92647],"score":1139,"new_id":248314},{"text":"География","train_count":569,"old_ids":[35884,151709,181923],"score":1138,"new_id":248315},{"text":"окончил","train_count":567,"old_ids":[148518,16806,161907],"score":1134,"new_id":248316},{"text":"чемпионате","train_count":377,"old_ids":[155356,148686,16806,152657],"score":1131,"new_id":248317},{"text":"Московской","train_count":565,"old_ids":[212095,148336,149193],"score":1130,"new_id":248318},{"text":"Родился","train_count":564,"old_ids":[32392,148277,154838],"score":1128,"new_id":248319},{"text":"другие","train_count":1128,"old_ids":[192548,47552],"score":1128,"new_id":248320},{"text":"Административное","train_count":225,"old_ids":[210138,150704,32884,77779,26125,45480],"score":1125,"new_id":248321},{"text":"собой","train_count":1125,"old_ids":[169983,148293],"score":1125,"new_id":248322},{"text":"сборной","train_count":562,"old_ids":[2175,148913,148317],"score":1124,"new_id":248323},{"text":"Александра","train_count":559,"old_ids":[34960,199291,21441],"score":1118,"new_id":248324},{"text":"одной","train_count":1118,"old_ids":[148277,148317],"score":1118,"new_id":248325},{"text":"института","train_count":558,"old_ids":[18372,158693,152644],"score":1116,"new_id":248326},{"text":"своим","train_count":557,"old_ids":[2175,148331,15630],"score":1114,"new_id":248327},{"text":"создания","train_count":557,"old_ids":[2175,148303,158529],"score":1114,"new_id":248328},{"text":"фильм","train_count":1114,"old_ids":[165171,6240],"score":1114,"new_id":248329},{"text":"Несмотря","train_count":369,"old_ids":[19582,21785,93942,4096],"score":1107,"new_id":248330},{"text":"федеральной","train_count":368,"old_ids":[18768,148276,7358,151291],"score":1104,"new_id":248331},{"text":"среди","train_count":1103,"old_ids":[2175,149520],"score":1103,"new_id":248332},{"text":"позволяет","train_count":367,"old_ids":[7754,148303,148331,151730],"score":1101,"new_id":248333},{"text":"армии","train_count":1100,"old_ids":[16763,159999],"score":1100,"new_id":248334},{"text":"реки","train_count":1099,"old_ids":[148279,16256],"score":1099,"new_id":248335},{"text":"получила","train_count":549,"old_ids":[148634,27448,150174],"score":1098,"new_id":248336},{"text":"являлся","train_count":366,"old_ids":[4096,5301,13302,151397],"score":1098,"new_id":248337},{"text":"преимущественно","train_count":365,"old_ids":[149790,217543,5092,13292],"score":1095,"new_id":248338},{"text":"короля","train_count":541,"old_ids":[4640,148367,13302],"score":1082,"new_id":248339},{"text":"этот","train_count":1082,"old_ids":[184088,12666],"score":1082,"new_id":248340},{"text":"дальнейшем","train_count":359,"old_ids":[6733,36771,148820,152333],"score":1077,"new_id":248341},{"text":"заслуженный","train_count":215,"old_ids":[8688,17417,2941,54025,5092,32401],"score":1075,"new_id":248342},{"text":"Советского","train_count":534,"old_ids":[173518,45405,149280],"score":1068,"new_id":248343},{"text":"образом","train_count":1068,"old_ids":[149586,148288],"score":1068,"new_id":248344},{"text":"Также","train_count":1066,"old_ids":[217720,56090],"score":1066,"new_id":248345},{"text":"благодаря","train_count":533,"old_ids":[218359,154039,4096],"score":1066,"new_id":248346},{"text":"Великой","train_count":531,"old_ids":[15729,58503,148554],"score":1062,"new_id":248347},{"text":"искусства","train_count":530,"old_ids":[13113,150352,89771],"score":1060,"new_id":248348},{"text":"Василий","train_count":352,"old_ids":[15729,17417,149037,11796],"score":1056,"new_id":248349},{"text":"роль","train_count":1054,"old_ids":[148283,27112],"score":1054,"new_id":248350},{"text":"посёлка","train_count":351,"old_ids":[7754,22078,162292,12751],"score":1053,"new_id":248351},{"text":"участвовал","train_count":525,"old_ids":[27448,1424,199375],"score":1050,"new_id":248352},{"text":"Владимирович","train_count":261,"old_ids":[15729,148606,15630,149591,31889],"score":1044,"new_id":248353},{"text":"очень","train_count":1040,"old_ids":[36103,49603],"score":1040,"new_id":248354},{"text":"заслуги","train_count":259,"old_ids":[8688,17417,2941,91206,1739],"score":1036,"new_id":248355},{"text":"августе","train_count":345,"old_ids":[148292,14942,58475,1450],"score":1035,"new_id":248356},{"text":"системы","train_count":1030,"old_ids":[167641,159066],"score":1030,"new_id":248357},{"text":"Владимира","train_count":343,"old_ids":[15729,148606,148623,21441],"score":1029,"new_id":248358},{"text":"одним","train_count":1028,"old_ids":[148277,150067],"score":1028,"new_id":248359},{"text":"стали","train_count":1028,"old_ids":[6389,45679],"score":1028,"new_id":248360},{"text":"контракт","train_count":342,"old_ids":[4640,92914,21441,150094],"score":1026,"new_id":248361},{"text":"Польше","train_count":512,"old_ids":[16362,148319,56916],"score":1024,"new_id":248362},{"text":"использования","train_count":340,"old_ids":[13113,183192,148675,36916],"score":1020,"new_id":248363},{"text":"поселение","train_count":340,"old_ids":[7754,22078,11777,16669],"score":1020,"new_id":248364},{"text":"политик","train_count":508,"old_ids":[148634,9080,36473],"score":1016,"new_id":248365},{"text":"гмины","train_count":507,"old_ids":[14942,61628,12014],"score":1014,"new_id":248366},{"text":"население","train_count":507,"old_ids":[13785,189921,16669],"score":1014,"new_id":248367},{"text":"провинции","train_count":505,"old_ids":[161800,18372,54201],"score":1010,"new_id":248368},{"text":"большинство","train_count":336,"old_ids":[9629,148701,18372,148593],"score":1008,"new_id":248369},{"text":"решение","train_count":504,"old_ids":[2115,44432,16669],"score":1008,"new_id":248370},{"text":"епископ","train_count":335,"old_ids":[1450,28696,45527,7754],"score":1005,"new_id":248371},{"text":"имя","train_count":1005,"old_ids":[15630,4096],"score":1005,"new_id":248372},{"text":"котором","train_count":1005,"old_ids":[151549,148288],"score":1005,"new_id":248373},{"text":"Швейцарии","train_count":251,"old_ids":[151311,156211,10064,16763,44633],"score":1004,"new_id":248374},{"text":"компании","train_count":1001,"old_ids":[148558,164420],"score":1001,"new_id":248375},{"text":"Похороненные","train_count":200,"old_ids":[16362,154822,8789,16806,5092,41546],"score":1000,"new_id":248376},{"text":"занимался","train_count":250,"old_ids":[8688,13603,6240,15481,19937],"score":1000,"new_id":248377},{"text":"Олимпийских","train_count":499,"old_ids":[19736,204086,150584],"score":998,"new_id":248378},{"text":"происхождения","train_count":498,"old_ids":[149490,13113,154490],"score":996,"new_id":248379},{"text":"главным","train_count":497,"old_ids":[149809,5301,148540],"score":994,"new_id":248380},{"text":"руководством","train_count":497,"old_ids":[19160,150463,152225],"score":994,"new_id":248381},{"text":"кладбище","train_count":496,"old_ids":[149450,148687,150006],"score":992,"new_id":248382},{"text":"Украины","train_count":991,"old_ids":[208317,152711],"score":991,"new_id":248383},{"text":"Франции","train_count":990,"old_ids":[221647,54201],"score":990,"new_id":248384},{"text":"русский","train_count":990,"old_ids":[152528,149800],"score":990,"new_id":248385},{"text":"могут","train_count":986,"old_ids":[148324,153168],"score":986,"new_id":248386},{"text":"Армении","train_count":492,"old_ids":[199593,6240,80228],"score":984,"new_id":248387},{"text":"книги","train_count":491,"old_ids":[4640,21918,148388],"score":982,"new_id":248388},{"text":"Выпускники","train_count":490,"old_ids":[44093,159859,149665],"score":980,"new_id":248389},{"text":"Государственной","train_count":326,"old_ids":[35884,22078,150968,171994],"score":978,"new_id":248390},{"text":"архитектор","train_count":326,"old_ids":[16763,10161,25135,151676],"score":978,"new_id":248391},{"text":"Михайлович","train_count":325,"old_ids":[36736,64392,148390,246235],"score":975,"new_id":248392},{"text":"журналист","train_count":325,"old_ids":[16467,53555,150981,6389],"score":975,"new_id":248393},{"text":"включая","train_count":486,"old_ids":[5301,49689,34226],"score":972,"new_id":248394},{"text":"возрасте","train_count":486,"old_ids":[150348,21441,151382],"score":972,"new_id":248395},{"text":"Васильевич","train_count":323,"old_ids":[15729,17417,64257,198326],"score":969,"new_id":248396},{"text":"названия","train_count":484,"old_ids":[13785,8688,155024],"score":968,"new_id":248397},{"text":"лишь","train_count":967,"old_ids":[10750,155081],"score":967,"new_id":248398},{"text":"Его","train_count":966,"old_ids":[53241,63046],"score":966,"new_id":248399},{"text":"Святого","train_count":483,"old_ids":[18727,152139,171664],"score":966,"new_id":248400},{"text":"возможность","train_count":483,"old_ids":[150348,149014,58245],"score":966,"new_id":248401},{"text":"литературы","train_count":322,"old_ids":[159329,7358,149959,36747],"score":966,"new_id":248402},{"text":"переписи","train_count":483,"old_ids":[159357,28696,1739],"score":966,"new_id":248403},{"text":"сезоне","train_count":483,"old_ids":[2175,148377,148917],"score":966,"new_id":248404},{"text":"Евгений","train_count":320,"old_ids":[53241,5301,14942,59318],"score":960,"new_id":248405},{"text":"населённых","train_count":320,"old_ids":[13785,189921,42565,170741],"score":960,"new_id":248406},{"text":"член","train_count":960,"old_ids":[5831,58361],"score":960,"new_id":248407},{"text":"единица","train_count":478,"old_ids":[152021,1739,83895],"score":956,"new_id":248408},{"text":"Орден","train_count":477,"old_ids":[19736,2115,149637],"score":954,"new_id":248409},{"text":"работал","train_count":953,"old_ids":[149100,15481],"score":953,"new_id":248410},{"text":"архитектуры","train_count":238,"old_ids":[16763,10161,25135,4640,153281],"score":952,"new_id":248411},{"text":"церкви","train_count":951,"old_ids":[180269,155974],"score":951,"new_id":248412},{"text":"педагог","train_count":475,"old_ids":[148728,154155,14942],"score":950,"new_id":248413},{"text":"американская","train_count":472,"old_ids":[1424,155899,152141],"score":944,"new_id":248414},{"text":"целью","train_count":470,"old_ids":[10064,24623,11577],"score":940,"new_id":248415},{"text":"Восточной","train_count":313,"old_ids":[15729,23514,36103,148317],"score":939,"new_id":248416},{"text":"даже","train_count":939,"old_ids":[38183,56090],"score":939,"new_id":248417},{"text":"особенно","train_count":469,"old_ids":[149034,5092,13292],"score":938,"new_id":248418},{"text":"встречается","train_count":311,"old_ids":[5301,48941,53887,64903],"score":933,"new_id":248419},{"text":"страны","train_count":927,"old_ids":[155690,4403],"score":927,"new_id":248420},{"text":"упоминается","train_count":307,"old_ids":[47174,1403,149717,48046],"score":921,"new_id":248421},{"text":"очередь","train_count":460,"old_ids":[36103,152555,4669],"score":920,"new_id":248422},{"text":"войска","train_count":459,"old_ids":[159035,2175,12751],"score":918,"new_id":248423},{"text":"премии","train_count":913,"old_ids":[149790,159999],"score":913,"new_id":248424},{"text":"удалось","train_count":456,"old_ids":[34723,15481,95306],"score":912,"new_id":248425},{"text":"Алексей","train_count":454,"old_ids":[34960,151205,148298],"score":908,"new_id":248426},{"text":"первой","train_count":907,"old_ids":[149208,159035],"score":907,"new_id":248427},{"text":"впервые","train_count":905,"old_ids":[5301,162206],"score":905,"new_id":248428},{"text":"институт","train_count":452,"old_ids":[18372,158693,36183],"score":904,"new_id":248429},{"text":"корпуса","train_count":450,"old_ids":[153589,149447,1424],"score":900,"new_id":248430},{"text":"статус","train_count":448,"old_ids":[6389,7721,41635],"score":896,"new_id":248431},{"text":"корреспондент","train_count":179,"old_ids":[153589,2115,149097,16806,6733,18110],"score":895,"new_id":248432},{"text":"построен","train_count":298,"old_ids":[7754,23514,148283,5092],"score":894,"new_id":248433},{"text":"футбольный","train_count":297,"old_ids":[18768,36183,170946,32401],"score":891,"new_id":248434},{"text":"административного","train_count":178,"old_ids":[149099,150704,32884,77779,26125,148325],"score":890,"new_id":248435},{"text":"членом","train_count":445,"old_ids":[5831,58361,148288],"score":890,"new_id":248436},{"text":"развитие","train_count":444,"old_ids":[148464,149734,47552],"score":888,"new_id":248437},{"text":"являются","train_count":887,"old_ids":[4096,155537],"score":887,"new_id":248438},{"text":"титул","train_count":443,"old_ids":[1729,149452,2941],"score":886,"new_id":248439},{"text":"именно","train_count":440,"old_ids":[15630,5092,13292],"score":880,"new_id":248440},{"text":"Родившиеся","train_count":219,"old_ids":[32392,148277,26125,150366,19937],"score":876,"new_id":248441},{"text":"записи","train_count":438,"old_ids":[148397,28696,1739],"score":876,"new_id":248442},{"text":"типа","train_count":871,"old_ids":[21797,148414],"score":871,"new_id":248443},{"text":"журнала","train_count":435,"old_ids":[16467,53555,156045],"score":870,"new_id":248444},{"text":"кино","train_count":869,"old_ids":[16256,13292],"score":869,"new_id":248445},{"text":"партии","train_count":869,"old_ids":[151963,159681],"score":869,"new_id":248446},{"text":"артист","train_count":433,"old_ids":[16763,21797,6389],"score":866,"new_id":248447},{"text":"музыки","train_count":433,"old_ids":[148405,148707,16256],"score":866,"new_id":248448},{"text":"материалов","train_count":214,"old_ids":[6240,7721,149098,15481,148275],"score":856,"new_id":248449},{"text":"Персоналии","train_count":426,"old_ids":[95169,152590,160195],"score":852,"new_id":248450},{"text":"входит","train_count":852,"old_ids":[5301,150294],"score":852,"new_id":248451},{"text":"деревни","train_count":426,"old_ids":[152138,148380,21918],"score":852,"new_id":248452},{"text":"театр","train_count":425,"old_ids":[48824,7721,2115],"score":850,"new_id":248453},{"text":"коммуны","train_count":848,"old_ids":[210786,12014],"score":848,"new_id":248454},{"text":"проект","train_count":423,"old_ids":[148489,42317,150094],"score":846,"new_id":248455},{"text":"Похоронен","train_count":211,"old_ids":[16362,154822,8789,16806,5092],"score":844,"new_id":248456},{"text":"восточной","train_count":281,"old_ids":[5301,23514,36103,148317],"score":843,"new_id":248457},{"text":"государств","train_count":280,"old_ids":[14942,22078,150968,19587],"score":840,"new_id":248458},{"text":"музыкант","train_count":280,"old_ids":[148405,148707,151011,1729],"score":840,"new_id":248459},{"text":"странах","train_count":420,"old_ids":[155690,1424,10161],"score":840,"new_id":248460},{"text":"веке","train_count":839,"old_ids":[148537,50800],"score":839,"new_id":248461},{"text":"обычно","train_count":839,"old_ids":[158689,159037],"score":839,"new_id":248462},{"text":"помощи","train_count":419,"old_ids":[151201,88260,1739],"score":838,"new_id":248463},{"text":"сайте","train_count":838,"old_ids":[2175,153032],"score":838,"new_id":248464},{"text":"историк","train_count":418,"old_ids":[32884,8789,36473],"score":836,"new_id":248465},{"text":"непосредственно","train_count":209,"old_ids":[76069,7754,22078,148420,155195],"score":836,"new_id":248466},{"text":"защиты","train_count":278,"old_ids":[148397,13739,9080,4403],"score":834,"new_id":248467},{"text":"стороны","train_count":834,"old_ids":[176981,12014],"score":834,"new_id":248468},{"text":"Список","train_count":416,"old_ids":[18727,28696,148518],"score":832,"new_id":248469},{"text":"применяется","train_count":208,"old_ids":[148489,15630,5092,4096,48046],"score":832,"new_id":248470},{"text":"лейтенант","train_count":277,"old_ids":[152134,150947,7105,1729],"score":831,"new_id":248471},{"text":"Затем","train_count":414,"old_ids":[34279,7721,148281],"score":828,"new_id":248472},{"text":"Европе","train_count":413,"old_ids":[53241,149552,148728],"score":826,"new_id":248473},{"text":"Союза","train_count":823,"old_ids":[173518,154635],"score":823,"new_id":248474},{"text":"Джон","train_count":411,"old_ids":[23355,16467,16806],"score":822,"new_id":248475},{"text":"земле","train_count":821,"old_ids":[154161,148284],"score":821,"new_id":248476},{"text":"количество","train_count":821,"old_ids":[45527,161317],"score":821,"new_id":248477},{"text":"мировой","train_count":820,"old_ids":[61628,156527],"score":820,"new_id":248478},{"text":"считается","train_count":410,"old_ids":[2175,87148,64903],"score":820,"new_id":248479},{"text":"Виктор","train_count":408,"old_ids":[15729,36473,35458],"score":816,"new_id":248480},{"text":"железной","train_count":408,"old_ids":[16467,77969,199502],"score":816,"new_id":248481},{"text":"православной","train_count":272,"old_ids":[148607,22078,150800,148317],"score":816,"new_id":248482},{"text":"называют","train_count":407,"old_ids":[13785,8688,152011],"score":814,"new_id":248483},{"text":"песен","train_count":407,"old_ids":[7754,21785,5092],"score":814,"new_id":248484},{"text":"Австралии","train_count":271,"old_ids":[34960,5301,149389,160195],"score":813,"new_id":248485},{"text":"композитор","train_count":406,"old_ids":[166349,148303,158559],"score":812,"new_id":248486},{"text":"председатель","train_count":406,"old_ids":[177917,152824,57868],"score":812,"new_id":248487},{"text":"феврале","train_count":270,"old_ids":[18768,148380,2115,150188],"score":810,"new_id":248488},{"text":"площади","train_count":404,"old_ids":[170421,159031,148467],"score":808,"new_id":248489},{"text":"протяжении","train_count":404,"old_ids":[148489,156442,159605],"score":808,"new_id":248490},{"text":"участник","train_count":404,"old_ids":[27448,80378,67494],"score":808,"new_id":248491},{"text":"клуб","train_count":806,"old_ids":[4640,152887],"score":806,"new_id":248492},{"text":"состоянию","train_count":403,"old_ids":[2175,152544,159038],"score":806,"new_id":248493},{"text":"семье","train_count":805,"old_ids":[159707,151583],"score":805,"new_id":248494},{"text":"литературе","train_count":267,"old_ids":[159329,7358,150849,1450],"score":801,"new_id":248495},{"text":"настоящее","train_count":800,"old_ids":[168712,172027],"score":800,"new_id":248496},{"text":"относится","train_count":400,"old_ids":[152928,2175,92907],"score":800,"new_id":248497},{"text":"поверхности","train_count":400,"old_ids":[148428,150652,65360],"score":800,"new_id":248498},{"text":"одного","train_count":799,"old_ids":[148277,148325],"score":799,"new_id":248499},{"text":"Борис","train_count":399,"old_ids":[58297,8789,13113],"score":798,"new_id":248500},{"text":"семейства","train_count":398,"old_ids":[159707,148298,89771],"score":796,"new_id":248501},{"text":"Золотой","train_count":265,"old_ids":[34279,148294,12666,148293],"score":795,"new_id":248502},{"text":"линии","train_count":793,"old_ids":[10750,155016],"score":793,"new_id":248503},{"text":"менее","train_count":792,"old_ids":[36621,47745],"score":792,"new_id":248504},{"text":"хорошо","train_count":396,"old_ids":[10161,148367,155041],"score":792,"new_id":248505},{"text":"административный","train_count":158,"old_ids":[149099,150704,32884,77779,26125,32401],"score":790,"new_id":248506},{"text":"писатель","train_count":789,"old_ids":[28696,57868],"score":789,"new_id":248507},{"text":"футболу","train_count":262,"old_ids":[18768,36183,168460,3652],"score":786,"new_id":248508},{"text":"ситуация","train_count":392,"old_ids":[2175,149452,53890],"score":784,"new_id":248509},{"text":"Северной","train_count":391,"old_ids":[181302,46576,148317],"score":782,"new_id":248510},{"text":"Японии","train_count":391,"old_ids":[82440,154767,44633],"score":782,"new_id":248511},{"text":"языка","train_count":782,"old_ids":[214432,12751],"score":782,"new_id":248512},{"text":"группу","train_count":390,"old_ids":[14942,83147,3652],"score":780,"new_id":248513},{"text":"находился","train_count":389,"old_ids":[13785,148334,154838],"score":778,"new_id":248514},{"text":"государственной","train_count":259,"old_ids":[14942,22078,150968,171994],"score":777,"new_id":248515},{"text":"почти","train_count":776,"old_ids":[152988,21797],"score":776,"new_id":248516},{"text":"председателем","train_count":194,"old_ids":[177917,152824,7721,77969,6240],"score":776,"new_id":248517},{"text":"независимости","train_count":155,"old_ids":[76069,8688,148292,13113,15630,81839],"score":775,"new_id":248518},{"text":"свой","train_count":775,"old_ids":[2175,159035],"score":775,"new_id":248519},{"text":"альбом","train_count":772,"old_ids":[36771,173910],"score":772,"new_id":248520},{"text":"промышленности","train_count":386,"old_ids":[149490,153886,65360],"score":772,"new_id":248521},{"text":"Великобритания","train_count":154,"old_ids":[15729,58503,45527,9629,149454,36916],"score":770,"new_id":248522},{"text":"украинский","train_count":385,"old_ids":[241656,18372,149800],"score":770,"new_id":248523},{"text":"Герой","train_count":769,"old_ids":[240053,148293],"score":769,"new_id":248524},{"text":"губернатор","train_count":192,"old_ids":[14942,61324,7358,2040,55389],"score":768,"new_id":248525},{"text":"перешёл","train_count":384,"old_ids":[149208,44432,162292],"score":768,"new_id":248526},{"text":"язык","train_count":767,"old_ids":[214432,4640],"score":767,"new_id":248527},{"text":"Входит","train_count":766,"old_ids":[15729,150294],"score":766,"new_id":248528},{"text":"соответствии","train_count":383,"old_ids":[148454,150076,152394],"score":766,"new_id":248529},{"text":"сооружения","train_count":255,"old_ids":[148454,8789,54025,17941],"score":765,"new_id":248530},{"text":"Соседние","train_count":254,"old_ids":[18727,22078,148276,150528],"score":762,"new_id":248531},{"text":"дивизии","train_count":381,"old_ids":[150394,39427,44633],"score":762,"new_id":248532},{"text":"западе","train_count":381,"old_ids":[148397,148414,95525],"score":762,"new_id":248533},{"text":"тыс","train_count":761,"old_ids":[82942,2175],"score":761,"new_id":248534},{"text":"учёный","train_count":380,"old_ids":[27448,42565,32401],"score":760,"new_id":248535},{"text":"Дмитрий","train_count":378,"old_ids":[23355,157255,155163],"score":756,"new_id":248536},{"text":"Испании","train_count":378,"old_ids":[29189,31608,149091],"score":756,"new_id":248537},{"text":"растений","train_count":378,"old_ids":[21441,6389,59318],"score":756,"new_id":248538},{"text":"языке","train_count":756,"old_ids":[214432,50800],"score":756,"new_id":248539},{"text":"места","train_count":755,"old_ids":[150436,1424],"score":755,"new_id":248540},{"text":"начал","train_count":755,"old_ids":[29196,15481],"score":755,"new_id":248541},{"text":"число","train_count":755,"old_ids":[153683,37483],"score":755,"new_id":248542},{"text":"революции","train_count":377,"old_ids":[148279,157308,54201],"score":754,"new_id":248543},{"text":"построена","train_count":251,"old_ids":[7754,23514,148283,48623],"score":753,"new_id":248544},{"text":"долларов","train_count":375,"old_ids":[159172,152637,148275],"score":750,"new_id":248545},{"text":"инициативе","train_count":150,"old_ids":[18372,1739,14666,7721,26125,1450],"score":750,"new_id":248546},{"text":"широко","train_count":374,"old_ids":[28420,148283,45527],"score":748,"new_id":248547},{"text":"постепенно","train_count":249,"old_ids":[7754,23514,157515,13292],"score":747,"new_id":248548},{"text":"посёлке","train_count":249,"old_ids":[7754,22078,162292,50800],"score":747,"new_id":248549},{"text":"завода","train_count":373,"old_ids":[8688,148292,150028],"score":746,"new_id":248550},{"text":"периода","train_count":373,"old_ids":[7754,149098,150028],"score":746,"new_id":248551},{"text":"Галерея","train_count":248,"old_ids":[35884,15481,148579,4096],"score":744,"new_id":248552},{"text":"музыканты","train_count":248,"old_ids":[148405,148707,151011,82942],"score":744,"new_id":248553},{"text":"принимал","train_count":248,"old_ids":[148489,18372,15630,15481],"score":744,"new_id":248554},{"text":"Академии","train_count":371,"old_ids":[34960,154408,159999],"score":742,"new_id":248555},{"text":"секретарь","train_count":247,"old_ids":[241651,166196,16763,4669],"score":741,"new_id":248556},{"text":"кинорежиссёр","train_count":148,"old_ids":[178738,150194,16467,13113,2175,156204],"score":740,"new_id":248557},{"text":"министр","train_count":370,"old_ids":[150704,32884,2115],"score":740,"new_id":248558},{"text":"Награды","train_count":369,"old_ids":[81573,148510,149055],"score":738,"new_id":248559},{"text":"президента","train_count":368,"old_ids":[149790,8688,163395],"score":736,"new_id":248560},{"text":"виде","train_count":735,"old_ids":[152833,1450],"score":735,"new_id":248561},{"text":"энциклопедия","train_count":147,"old_ids":[156454,14666,151639,7754,148276,37835],"score":735,"new_id":248562},{"text":"декабре","train_count":367,"old_ids":[95525,12751,151838],"score":734,"new_id":248563},{"text":"серии","train_count":734,"old_ids":[159541,44633],"score":734,"new_id":248564},{"text":"течение","train_count":734,"old_ids":[48824,155038],"score":734,"new_id":248565},{"text":"СДПА","train_count":366,"old_ids":[18727,23355,203141],"score":732,"new_id":248566},{"text":"вместо","train_count":366,"old_ids":[5301,150436,1403],"score":732,"new_id":248567},{"text":"используют","train_count":244,"old_ids":[13113,183192,8688,149460],"score":732,"new_id":248568},{"text":"больше","train_count":731,"old_ids":[170946,56916],"score":731,"new_id":248569},{"text":"относительно","train_count":365,"old_ids":[152928,2175,151269],"score":730,"new_id":248570},{"text":"сильно","train_count":365,"old_ids":[2175,64257,13292],"score":730,"new_id":248571},{"text":"называется","train_count":364,"old_ids":[13785,8688,152017],"score":728,"new_id":248572},{"text":"процесс","train_count":364,"old_ids":[149490,10064,90202],"score":728,"new_id":248573},{"text":"сценарист","train_count":242,"old_ids":[2175,149147,16763,32884],"score":726,"new_id":248574},{"text":"факультет","train_count":363,"old_ids":[152236,231736,7921],"score":726,"new_id":248575},{"text":"Этот","train_count":362,"old_ids":[89030,1729,12666],"score":724,"new_id":248576},{"text":"Пётр","train_count":361,"old_ids":[16362,150396,2115],"score":722,"new_id":248577},{"text":"январе","train_count":361,"old_ids":[4096,156345,1450],"score":722,"new_id":248578},{"text":"создании","train_count":240,"old_ids":[2175,148303,6733,149091],"score":720,"new_id":248579},{"text":"спустя","train_count":360,"old_ids":[31608,58475,4096],"score":720,"new_id":248580},{"text":"хотя","train_count":719,"old_ids":[10161,156442],"score":719,"new_id":248581},{"text":"список","train_count":359,"old_ids":[2175,28696,148518],"score":718,"new_id":248582},{"text":"влияние","train_count":358,"old_ids":[5301,150095,150528],"score":716,"new_id":248583},{"text":"Австрия","train_count":238,"old_ids":[34960,5301,153342,4096],"score":714,"new_id":248584},{"text":"Михаила","train_count":238,"old_ids":[36736,64392,1424,150174],"score":714,"new_id":248585},{"text":"включает","train_count":357,"old_ids":[5301,49689,26046],"score":714,"new_id":248586},{"text":"которое","train_count":714,"old_ids":[151549,42317],"score":714,"new_id":248587},{"text":"проблемы","train_count":238,"old_ids":[148489,148280,151032,4403],"score":714,"new_id":248588},{"text":"сентябре","train_count":357,"old_ids":[2175,156113,151838],"score":714,"new_id":248589},{"text":"следующие","train_count":357,"old_ids":[149161,68608,47552],"score":714,"new_id":248590},{"text":"первого","train_count":713,"old_ids":[149208,151394],"score":713,"new_id":248591},{"text":"честь","train_count":713,"old_ids":[149188,33348],"score":713,"new_id":248592},{"text":"Польши","train_count":356,"old_ids":[16362,148319,28420],"score":712,"new_id":248593},{"text":"архиепископ","train_count":178,"old_ids":[16763,229751,28696,45527,7754],"score":712,"new_id":248594},{"text":"Классификация","train_count":142,"old_ids":[25497,2941,27473,59627,71658,68335],"score":710,"new_id":248595},{"text":"базе","train_count":355,"old_ids":[9629,148646,1450],"score":710,"new_id":248596},{"text":"церковь","train_count":709,"old_ids":[180269,163687],"score":709,"new_id":248597},{"text":"Анатолий","train_count":236,"old_ids":[34960,2040,149900,159497],"score":708,"new_id":248598},{"text":"русская","train_count":354,"old_ids":[19160,149663,150621],"score":708,"new_id":248599},{"text":"использовать","train_count":353,"old_ids":[13113,183192,163138],"score":706,"new_id":248600},{"text":"организации","train_count":705,"old_ids":[167304,51788],"score":705,"new_id":248601},{"text":"Верховного","train_count":352,"old_ids":[238301,152335,148325],"score":704,"new_id":248602},{"text":"Президиума","train_count":176,"old_ids":[51844,148377,45587,1739,149352],"score":704,"new_id":248603},{"text":"Финляндии","train_count":176,"old_ids":[52337,18372,13302,150559,44633],"score":704,"new_id":248604},{"text":"выступал","train_count":352,"old_ids":[148705,148757,15481],"score":704,"new_id":248605},{"text":"каждый","train_count":352,"old_ids":[12751,148343,33477],"score":704,"new_id":248606},{"text":"сезон","train_count":352,"old_ids":[2175,148377,16806],"score":704,"new_id":248607},{"text":"сельсовета","train_count":352,"old_ids":[226181,148454,200154],"score":704,"new_id":248608},{"text":"членов","train_count":352,"old_ids":[5831,58361,148275],"score":704,"new_id":248609},{"text":"железнодорожная","train_count":175,"old_ids":[16467,77969,8688,166815,173848],"score":700,"new_id":248610},{"text":"чемпионом","train_count":233,"old_ids":[155356,148686,1403,148524],"score":699,"new_id":248611},{"text":"Сюжет","train_count":349,"old_ids":[18727,11577,159032],"score":698,"new_id":248612},{"text":"своём","train_count":349,"old_ids":[2175,148331,151666],"score":698,"new_id":248613},{"text":"людей","train_count":697,"old_ids":[35587,149232],"score":697,"new_id":248614},{"text":"содержит","train_count":232,"old_ids":[2175,148277,55638,9080],"score":696,"new_id":248615},{"text":"крупнейших","train_count":231,"old_ids":[4640,149568,148820,150047],"score":693,"new_id":248616},{"text":"случаях","train_count":231,"old_ids":[38612,27448,34226,10161],"score":693,"new_id":248617},{"text":"власти","train_count":692,"old_ids":[160966,52598],"score":692,"new_id":248618},{"text":"Нью","train_count":691,"old_ids":[19582,84920],"score":691,"new_id":248619},{"text":"школы","train_count":691,"old_ids":[160291,149383],"score":691,"new_id":248620},{"text":"небольшой","train_count":230,"old_ids":[76069,9629,148701,148293],"score":690,"new_id":248621},{"text":"необходимо","train_count":230,"old_ids":[76069,148280,148334,149076],"score":690,"new_id":248622},{"text":"появилась","train_count":230,"old_ids":[7754,58255,151631,152993],"score":690,"new_id":248623},{"text":"продюсер","train_count":230,"old_ids":[148489,148277,11577,159541],"score":690,"new_id":248624},{"text":"состава","train_count":344,"old_ids":[2175,151196,90219],"score":688,"new_id":248625},{"text":"монастырь","train_count":343,"old_ids":[179659,151407,149213],"score":686,"new_id":248626},{"text":"поселении","train_count":227,"old_ids":[7754,22078,11777,80228],"score":681,"new_id":248627},{"text":"частности","train_count":681,"old_ids":[150813,65360],"score":681,"new_id":248628},{"text":"группе","train_count":340,"old_ids":[14942,83147,1450],"score":680,"new_id":248629},{"text":"искусств","train_count":340,"old_ids":[13113,150352,19587],"score":680,"new_id":248630},{"text":"северо","train_count":679,"old_ids":[148938,187942],"score":679,"new_id":248631},{"text":"Площадь","train_count":339,"old_ids":[16362,37483,166595],"score":678,"new_id":248632},{"text":"книга","train_count":339,"old_ids":[4640,21918,148369],"score":678,"new_id":248633},{"text":"оказался","train_count":226,"old_ids":[1403,148330,15481,19937],"score":678,"new_id":248634},{"text":"детей","train_count":674,"old_ids":[150017,148298],"score":674,"new_id":248635},{"text":"многих","train_count":674,"old_ids":[178416,172532],"score":674,"new_id":248636},{"text":"Ленинградской","train_count":224,"old_ids":[61757,9396,2040,167220],"score":672,"new_id":248637},{"text":"находятся","train_count":336,"old_ids":[13785,150782,19937],"score":672,"new_id":248638},{"text":"окончании","train_count":224,"old_ids":[148518,16806,5831,149091],"score":672,"new_id":248639},{"text":"совместно","train_count":336,"old_ids":[149297,150436,13292],"score":672,"new_id":248640},{"text":"здесь","train_count":670,"old_ids":[8688,170901],"score":670,"new_id":248641},{"text":"основных","train_count":335,"old_ids":[22078,148366,42240],"score":670,"new_id":248642},{"text":"млн","train_count":669,"old_ids":[220083,2040],"score":669,"new_id":248643},{"text":"принадлежит","train_count":223,"old_ids":[148895,149578,148284,152735],"score":669,"new_id":248644},{"text":"бывший","train_count":333,"old_ids":[9629,33422,151642],"score":666,"new_id":248645},{"text":"миллионов","train_count":222,"old_ids":[6240,61057,10750,152740],"score":666,"new_id":248646},{"text":"обороны","train_count":332,"old_ids":[148280,148367,12014],"score":664,"new_id":248647},{"text":"фамилии","train_count":332,"old_ids":[18768,150190,160195],"score":664,"new_id":248648},{"text":"занимал","train_count":221,"old_ids":[8688,13603,6240,15481],"score":663,"new_id":248649},{"text":"платформы","train_count":221,"old_ids":[7754,75774,54724,4403],"score":663,"new_id":248650},{"text":"Наиболее","train_count":331,"old_ids":[81573,151223,152156],"score":662,"new_id":248651},{"text":"будет","train_count":661,"old_ids":[156543,7921],"score":661,"new_id":248652},{"text":"Большой","train_count":330,"old_ids":[58297,148701,148293],"score":660,"new_id":248653},{"text":"внутренних","train_count":220,"old_ids":[5301,154175,148516,149648],"score":660,"new_id":248654},{"text":"позволило","train_count":165,"old_ids":[7754,148303,148331,10750,37483],"score":660,"new_id":248655},{"text":"Сергей","train_count":658,"old_ids":[209616,160936],"score":658,"new_id":248656},{"text":"дворов","train_count":329,"old_ids":[6733,149086,148275],"score":658,"new_id":248657},{"text":"жителей","train_count":658,"old_ids":[16467,150471],"score":658,"new_id":248658},{"text":"месте","train_count":657,"old_ids":[150436,1450],"score":657,"new_id":248659},{"text":"движения","train_count":656,"old_ids":[155218,150282],"score":656,"new_id":248660},{"text":"заместителем","train_count":164,"old_ids":[201250,34981,9080,77969,6240],"score":656,"new_id":248661},{"text":"журнале","train_count":218,"old_ids":[16467,53555,13785,148284],"score":654,"new_id":248662},{"text":"западной","train_count":327,"old_ids":[148397,148414,172332],"score":654,"new_id":248663},{"text":"расположена","train_count":327,"old_ids":[148761,220495,164991],"score":654,"new_id":248664},{"text":"чемпион","train_count":327,"old_ids":[155356,148686,16806],"score":654,"new_id":248665},{"text":"юго","train_count":654,"old_ids":[11577,63046],"score":654,"new_id":248666},{"text":"железнодорожной","train_count":163,"old_ids":[16467,77969,8688,166815,159484],"score":652,"new_id":248667},{"text":"книге","train_count":326,"old_ids":[4640,21918,149068],"score":652,"new_id":248668},{"text":"получили","train_count":326,"old_ids":[148634,27448,149037],"score":652,"new_id":248669},{"text":"улицы","train_count":326,"old_ids":[3652,63890,4403],"score":652,"new_id":248670},{"text":"деревне","train_count":325,"old_ids":[152138,148380,76069],"score":650,"new_id":248671},{"text":"Иван","train_count":649,"old_ids":[29189,149137],"score":649,"new_id":248672},{"text":"родился","train_count":649,"old_ids":[149003,154838],"score":649,"new_id":248673},{"text":"Венгрии","train_count":216,"old_ids":[15729,5092,14942,159079],"score":648,"new_id":248674},{"text":"Джованни","train_count":162,"old_ids":[23355,16467,148275,7105,21918],"score":648,"new_id":248675},{"text":"вновь","train_count":648,"old_ids":[5301,163667],"score":648,"new_id":248676},{"text":"городского","train_count":648,"old_ids":[152689,149280],"score":648,"new_id":248677},{"text":"информации","train_count":324,"old_ids":[18372,54724,51788],"score":648,"new_id":248678},{"text":"полностью","train_count":648,"old_ids":[148634,150616],"score":648,"new_id":248679},{"text":"Центр","train_count":323,"old_ids":[89602,18110,2115],"score":646,"new_id":248680},{"text":"театре","train_count":323,"old_ids":[48824,7721,148279],"score":646,"new_id":248681},{"text":"Режиссёр","train_count":161,"old_ids":[175129,16467,13113,2175,156204],"score":644,"new_id":248682},{"text":"Труда","train_count":322,"old_ids":[32868,19160,38183],"score":644,"new_id":248683},{"text":"означает","train_count":322,"old_ids":[148303,29196,26046],"score":644,"new_id":248684},{"text":"тот","train_count":643,"old_ids":[1729,12666],"score":643,"new_id":248685},{"text":"Григорий","train_count":214,"old_ids":[35884,29119,152974,42387],"score":642,"new_id":248686},{"text":"Чемпион","train_count":214,"old_ids":[70345,148281,148686,16806],"score":642,"new_id":248687},{"text":"современной","train_count":321,"old_ids":[149297,148801,148317],"score":642,"new_id":248688},{"text":"уровня","train_count":321,"old_ids":[53555,148275,37855],"score":642,"new_id":248689},{"text":"философии","train_count":214,"old_ids":[18768,61057,152406,221151],"score":642,"new_id":248690},{"text":"апреле","train_count":320,"old_ids":[1424,148489,77969],"score":640,"new_id":248691},{"text":"компания","train_count":640,"old_ids":[166349,36916],"score":640,"new_id":248692},{"text":"училище","train_count":320,"old_ids":[27448,149037,150006],"score":640,"new_id":248693},{"text":"Благодаря","train_count":213,"old_ids":[58297,29354,154039,4096],"score":639,"new_id":248694},{"text":"Совета","train_count":639,"old_ids":[173518,200154],"score":639,"new_id":248695},{"text":"директора","train_count":213,"old_ids":[6733,74990,148301,150587],"score":639,"new_id":248696},{"text":"ноябре","train_count":319,"old_ids":[13292,4096,151838],"score":638,"new_id":248697},{"text":"основе","train_count":638,"old_ids":[22078,153300],"score":638,"new_id":248698},{"text":"существует","train_count":319,"old_ids":[2175,82577,69632],"score":638,"new_id":248699},{"text":"расположены","train_count":317,"old_ids":[148761,220495,163352],"score":634,"new_id":248700},{"text":"Сергеевич","train_count":211,"old_ids":[209616,14942,47745,159380],"score":633,"new_id":248701},{"text":"большое","train_count":316,"old_ids":[9629,148701,42317],"score":632,"new_id":248702},{"text":"научно","train_count":316,"old_ids":[13785,27448,13292],"score":632,"new_id":248703},{"text":"основан","train_count":315,"old_ids":[22078,148366,7105],"score":630,"new_id":248704},{"text":"процессе","train_count":210,"old_ids":[149490,10064,90202,1450],"score":630,"new_id":248705},{"text":"составляла","train_count":210,"old_ids":[2175,150838,13302,29354],"score":630,"new_id":248706},{"text":"одна","train_count":628,"old_ids":[148277,13785],"score":628,"new_id":248707},{"text":"заводе","train_count":209,"old_ids":[8688,148292,1403,95525],"score":627,"new_id":248708},{"text":"Демография","train_count":313,"old_ids":[23355,151438,181923],"score":626,"new_id":248709},{"text":"острове","train_count":313,"old_ids":[23514,148327,1450],"score":626,"new_id":248710},{"text":"студии","train_count":313,"old_ids":[6389,34723,44633],"score":626,"new_id":248711},{"text":"таких","train_count":625,"old_ids":[42074,148797],"score":625,"new_id":248712},{"text":"Бразилии","train_count":208,"old_ids":[58297,148464,149037,1739],"score":624,"new_id":248713},{"text":"Топоним","train_count":208,"old_ids":[32868,148561,16806,15630],"score":624,"new_id":248714},{"text":"арестован","train_count":208,"old_ids":[16763,34981,148275,7105],"score":624,"new_id":248715},{"text":"книг","train_count":312,"old_ids":[4640,21918,14942],"score":624,"new_id":248716},{"text":"Игорь","train_count":311,"old_ids":[29189,152974,4669],"score":622,"new_id":248717},{"text":"именем","train_count":311,"old_ids":[15630,5092,148281],"score":622,"new_id":248718},{"text":"драматург","train_count":207,"old_ids":[151486,6240,150849,14942],"score":621,"new_id":248719},{"text":"административным","train_count":124,"old_ids":[149099,150704,32884,77779,26125,148540],"score":620,"new_id":248720},{"text":"версии","train_count":620,"old_ids":[46576,151883],"score":620,"new_id":248721},{"text":"игр","train_count":620,"old_ids":[66951,2115],"score":620,"new_id":248722},{"text":"начальник","train_count":310,"old_ids":[29196,36771,67494],"score":620,"new_id":248723},{"text":"КПСС","train_count":309,"old_ids":[25497,16362,152611],"score":618,"new_id":248724},{"text":"соответственно","train_count":309,"old_ids":[148454,150076,155195],"score":618,"new_id":248725},{"text":"Америки","train_count":308,"old_ids":[34960,60909,151278],"score":616,"new_id":248726},{"text":"Андреевич","train_count":154,"old_ids":[34960,2040,60659,47745,159380],"score":616,"new_id":248727},{"text":"использовался","train_count":154,"old_ids":[13113,183192,148675,15481,19937],"score":616,"new_id":248728},{"text":"модели","train_count":308,"old_ids":[6240,148277,58503],"score":616,"new_id":248729},{"text":"система","train_count":616,"old_ids":[167641,149721],"score":616,"new_id":248730},{"text":"выпущен","train_count":205,"old_ids":[148705,7754,40794,5092],"score":615,"new_id":248731},{"text":"переводчик","train_count":205,"old_ids":[149208,148380,148277,159796],"score":615,"new_id":248732},{"text":"Позже","train_count":307,"old_ids":[16362,148303,56090],"score":614,"new_id":248733},{"text":"иностранных","train_count":307,"old_ids":[158231,48941,151990],"score":614,"new_id":248734},{"text":"итальянский","train_count":307,"old_ids":[9080,157230,176321],"score":614,"new_id":248735},{"text":"река","train_count":614,"old_ids":[148279,12751],"score":614,"new_id":248736},{"text":"встречаются","train_count":204,"old_ids":[5301,48941,53887,152998],"score":612,"new_id":248737},{"text":"животных","train_count":306,"old_ids":[151303,12666,42240],"score":612,"new_id":248738},{"text":"постройки","train_count":204,"old_ids":[7754,23514,170149,16256],"score":612,"new_id":248739},{"text":"довольно","train_count":305,"old_ids":[148716,148319,13292],"score":610,"new_id":248740},{"text":"переехал","train_count":203,"old_ids":[149208,47745,10161,15481],"score":609,"new_id":248741},{"text":"подразделяется","train_count":203,"old_ids":[155908,148464,153501,48046],"score":609,"new_id":248742},{"text":"территорию","train_count":203,"old_ids":[55055,149454,150140,11577],"score":609,"new_id":248743},{"text":"успешно","train_count":203,"old_ids":[3652,31608,44432,13292],"score":609,"new_id":248744},{"text":"безопасности","train_count":304,"old_ids":[161258,151672,65360],"score":608,"new_id":248745},{"text":"консерватории","train_count":152,"old_ids":[150270,159541,5301,55389,44633],"score":608,"new_id":248746},{"text":"работу","train_count":607,"old_ids":[149100,3652],"score":607,"new_id":248747},{"text":"иногда","train_count":606,"old_ids":[1739,155874],"score":606,"new_id":248748},{"text":"институте","train_count":202,"old_ids":[18372,158693,36183,1450],"score":606,"new_id":248749},{"text":"составляло","train_count":202,"old_ids":[2175,150838,13302,37483],"score":606,"new_id":248750},{"text":"флота","train_count":303,"old_ids":[18768,2941,150475],"score":606,"new_id":248751},{"text":"общества","train_count":605,"old_ids":[149350,150167],"score":605,"new_id":248752},{"text":"занял","train_count":302,"old_ids":[193134,4096,2941],"score":604,"new_id":248753},{"text":"имеется","train_count":302,"old_ids":[15630,1450,48046],"score":604,"new_id":248754},{"text":"территория","train_count":302,"old_ids":[55055,149454,154617],"score":604,"new_id":248755},{"text":"мире","train_count":603,"old_ids":[61628,148279],"score":603,"new_id":248756},{"text":"острова","train_count":603,"old_ids":[23514,169887],"score":603,"new_id":248757},{"text":"вариант","train_count":301,"old_ids":[155281,7105,1729],"score":602,"new_id":248758},{"text":"крупных","train_count":301,"old_ids":[4640,149568,42240],"score":602,"new_id":248759},{"text":"автомобилей","train_count":150,"old_ids":[148292,149127,148280,61057,148298],"score":600,"new_id":248760},{"text":"английском","train_count":200,"old_ids":[7105,151203,11796,153195],"score":600,"new_id":248761},{"text":"математик","train_count":150,"old_ids":[6240,7721,148281,7721,36473],"score":600,"new_id":248762},{"text":"многочисленные","train_count":150,"old_ids":[178416,14942,36103,13113,164071],"score":600,"new_id":248763},{"text":"отец","train_count":600,"old_ids":[149744,10064],"score":600,"new_id":248764},{"text":"премию","train_count":300,"old_ids":[149790,61628,11577],"score":600,"new_id":248765},{"text":"приводит","train_count":200,"old_ids":[148489,26125,148277,9080],"score":600,"new_id":248766},{"text":"Фёдорович","train_count":199,"old_ids":[52337,42565,186408,31889],"score":597,"new_id":248767},{"text":"государство","train_count":199,"old_ids":[14942,22078,150968,148593],"score":597,"new_id":248768},{"text":"момент","train_count":597,"old_ids":[148324,89315],"score":597,"new_id":248769},{"text":"компанией","train_count":298,"old_ids":[166349,25767,11796],"score":596,"new_id":248770},{"text":"началась","train_count":298,"old_ids":[29196,15481,152993],"score":596,"new_id":248771},{"text":"операции","train_count":298,"old_ids":[148561,7358,51788],"score":596,"new_id":248772},{"text":"Отец","train_count":297,"old_ids":[19736,48824,10064],"score":594,"new_id":248773},{"text":"ранее","train_count":594,"old_ids":[33437,47745],"score":594,"new_id":248774},{"text":"французского","train_count":198,"old_ids":[18768,33437,163227,149280],"score":594,"new_id":248775},{"text":"стало","train_count":593,"old_ids":[6389,150182],"score":593,"new_id":248776},{"text":"белорусский","train_count":148,"old_ids":[9629,11777,8789,41635,149800],"score":592,"new_id":248777},{"text":"возглавлял","train_count":148,"old_ids":[150348,149809,5301,13302,2941],"score":592,"new_id":248778},{"text":"процесса","train_count":197,"old_ids":[149490,10064,90202,1424],"score":591,"new_id":248779},{"text":"команда","train_count":590,"old_ids":[148558,158021],"score":590,"new_id":248780},{"text":"Алексеевич","train_count":196,"old_ids":[34960,151205,47745,159380],"score":588,"new_id":248781},{"text":"Боксёры","train_count":147,"old_ids":[58297,148518,2175,42565,36747],"score":588,"new_id":248782},{"text":"Павел","train_count":294,"old_ids":[16362,148292,11777],"score":588,"new_id":248783},{"text":"карьеру","train_count":294,"old_ids":[12751,194211,3652],"score":588,"new_id":248784},{"text":"командованием","train_count":196,"old_ids":[148558,72884,150187,6240],"score":588,"new_id":248785},{"text":"самолёта","train_count":196,"old_ids":[160002,148294,42565,42074],"score":588,"new_id":248786},{"text":"создана","train_count":196,"old_ids":[2175,148303,6733,148841],"score":588,"new_id":248787},{"text":"эти","train_count":587,"old_ids":[65131,21797],"score":587,"new_id":248788},{"text":"Группа","train_count":293,"old_ids":[35884,83147,1424],"score":586,"new_id":248789},{"text":"императора","train_count":293,"old_ids":[15630,149208,168640],"score":586,"new_id":248790},{"text":"материалы","train_count":146,"old_ids":[6240,7721,149098,15481,4403],"score":584,"new_id":248791},{"text":"появились","train_count":292,"old_ids":[7754,58255,236525],"score":584,"new_id":248792},{"text":"моря","train_count":582,"old_ids":[6240,158121],"score":582,"new_id":248793},{"text":"награждён","train_count":291,"old_ids":[168847,148343,185709],"score":582,"new_id":248794},{"text":"самых","train_count":581,"old_ids":[160002,60038],"score":581,"new_id":248795},{"text":"Григорьевич","train_count":145,"old_ids":[35884,29119,152974,158525,31889],"score":580,"new_id":248796},{"text":"матче","train_count":290,"old_ids":[6240,7721,149188],"score":580,"new_id":248797},{"text":"оружия","train_count":290,"old_ids":[8789,54025,37835],"score":580,"new_id":248798},{"text":"памяти","train_count":290,"old_ids":[148414,79142,21797],"score":580,"new_id":248799},{"text":"приток","train_count":290,"old_ids":[148489,9080,148518],"score":580,"new_id":248800},{"text":"вооружение","train_count":193,"old_ids":[148331,8789,54025,16669],"score":579,"new_id":248801},{"text":"директором","train_count":193,"old_ids":[6733,74990,148301,175101],"score":579,"new_id":248802},{"text":"театральный","train_count":193,"old_ids":[48824,7721,2115,152027],"score":579,"new_id":248803},{"text":"ныне","train_count":578,"old_ids":[12014,76069],"score":578,"new_id":248804},{"text":"себе","train_count":578,"old_ids":[148938,149056],"score":578,"new_id":248805},{"text":"дебютировал","train_count":192,"old_ids":[159789,11577,21797,177702],"score":576,"new_id":248806},{"text":"объединения","train_count":192,"old_ids":[148280,32482,152021,17941],"score":576,"new_id":248807},{"text":"видов","train_count":575,"old_ids":[152833,148275],"score":575,"new_id":248808},{"text":"земли","train_count":575,"old_ids":[154161,10750],"score":575,"new_id":248809},{"text":"создан","train_count":287,"old_ids":[2175,148303,172895],"score":574,"new_id":248810},{"text":"художественный","train_count":191,"old_ids":[157951,149244,150198,32401],"score":573,"new_id":248811},{"text":"газеты","train_count":286,"old_ids":[148369,8688,150453],"score":572,"new_id":248812},{"text":"основной","train_count":286,"old_ids":[22078,148366,148317],"score":572,"new_id":248813},{"text":"пунктов","train_count":286,"old_ids":[7754,38629,148779],"score":572,"new_id":248814},{"text":"Семья","train_count":285,"old_ids":[18727,148281,149885],"score":570,"new_id":248815},{"text":"государственных","train_count":190,"old_ids":[14942,22078,150968,159545],"score":570,"new_id":248816},{"text":"лауреат","train_count":285,"old_ids":[29354,201647,7721],"score":570,"new_id":248817},{"text":"режиссёром","train_count":114,"old_ids":[148279,16467,13113,2175,42565,149633],"score":570,"new_id":248818},{"text":"совета","train_count":570,"old_ids":[148454,200154],"score":570,"new_id":248819},{"text":"станция","train_count":569,"old_ids":[49224,68335],"score":569,"new_id":248820},{"text":"песня","train_count":284,"old_ids":[7754,21785,37855],"score":568,"new_id":248821},{"text":"заместитель","train_count":283,"old_ids":[201250,34981,53062],"score":566,"new_id":248822},{"text":"озеро","train_count":282,"old_ids":[148303,7358,1403],"score":564,"new_id":248823},{"text":"многие","train_count":562,"old_ids":[178416,153277],"score":562,"new_id":248824},{"text":"научных","train_count":281,"old_ids":[13785,27448,42240],"score":562,"new_id":248825},{"text":"несмотря","train_count":281,"old_ids":[149904,93942,4096],"score":562,"new_id":248826},{"text":"факультета","train_count":281,"old_ids":[152236,231736,149005],"score":562,"new_id":248827},{"text":"Позднее","train_count":187,"old_ids":[16362,148303,6733,149499],"score":561,"new_id":248828},{"text":"французской","train_count":187,"old_ids":[18768,33437,163227,149193],"score":561,"new_id":248829},{"text":"своему","train_count":280,"old_ids":[2175,159117,148405],"score":560,"new_id":248830},{"text":"Статистика","train_count":186,"old_ids":[71000,7721,32884,71658],"score":558,"new_id":248831},{"text":"использовались","train_count":186,"old_ids":[13113,183192,148675,158605],"score":558,"new_id":248832},{"text":"которую","train_count":558,"old_ids":[151549,31953],"score":558,"new_id":248833},{"text":"тогда","train_count":558,"old_ids":[23858,150921],"score":558,"new_id":248834},{"text":"дороги","train_count":557,"old_ids":[152578,148388],"score":557,"new_id":248835},{"text":"носители","train_count":557,"old_ids":[161905,153009],"score":557,"new_id":248836},{"text":"целом","train_count":278,"old_ids":[10064,11777,148288],"score":556,"new_id":248837},{"text":"Поскольку","train_count":185,"old_ids":[16362,22078,45527,153542],"score":555,"new_id":248838},{"text":"приблизительно","train_count":185,"old_ids":[148895,9629,149466,151269],"score":555,"new_id":248839},{"text":"специалист","train_count":185,"old_ids":[155036,14666,45679,6389],"score":555,"new_id":248840},{"text":"команду","train_count":277,"old_ids":[148558,72884,3652],"score":554,"new_id":248841},{"text":"половине","train_count":277,"old_ids":[220495,5301,149961],"score":554,"new_id":248842},{"text":"пункт","train_count":277,"old_ids":[7754,38629,1729],"score":554,"new_id":248843},{"text":"Валерий","train_count":184,"old_ids":[15729,15481,7358,42387],"score":552,"new_id":248844},{"text":"многочисленных","train_count":138,"old_ids":[178416,14942,36103,13113,182575],"score":552,"new_id":248845},{"text":"фестиваля","train_count":138,"old_ids":[18768,34981,26125,1424,13302],"score":552,"new_id":248846},{"text":"производства","train_count":551,"old_ids":[161612,221196],"score":551,"new_id":248847},{"text":"категории","train_count":275,"old_ids":[150843,148645,152597],"score":550,"new_id":248848},{"text":"использованием","train_count":183,"old_ids":[13113,183192,148675,152952],"score":549,"new_id":248849},{"text":"ордена","train_count":549,"old_ids":[8789,197347],"score":549,"new_id":248850},{"text":"профессора","train_count":183,"old_ids":[149490,18768,90202,148676],"score":549,"new_id":248851},{"text":"самолётов","train_count":183,"old_ids":[160002,148294,42565,148779],"score":549,"new_id":248852},{"text":"автор","train_count":548,"old_ids":[148292,35458],"score":548,"new_id":248853},{"text":"инженер","train_count":274,"old_ids":[18372,148942,7358],"score":548,"new_id":248854},{"text":"историю","train_count":274,"old_ids":[32884,150140,11577],"score":548,"new_id":248855},{"text":"чемпионат","train_count":274,"old_ids":[155356,148686,232067],"score":548,"new_id":248856},{"text":"Индии","train_count":273,"old_ids":[29189,150559,44633],"score":546,"new_id":248857},{"text":"изменения","train_count":273,"old_ids":[39427,36621,17941],"score":546,"new_id":248858},{"text":"комиссии","train_count":273,"old_ids":[148558,13113,151883],"score":546,"new_id":248859},{"text":"первым","train_count":546,"old_ids":[149208,159147],"score":546,"new_id":248860},{"text":"Ленина","train_count":272,"old_ids":[61757,9396,13785],"score":544,"new_id":248861},{"text":"конца","train_count":544,"old_ids":[150270,83895],"score":544,"new_id":248862},{"text":"официально","train_count":272,"old_ids":[1403,149243,149749],"score":544,"new_id":248863},{"text":"проведения","train_count":272,"old_ids":[161800,148276,17941],"score":544,"new_id":248864},{"text":"сыграл","train_count":272,"old_ids":[54330,154731,15481],"score":544,"new_id":248865},{"text":"неоднократно","train_count":181,"old_ids":[76069,148277,150992,158397],"score":543,"new_id":248866},{"text":"фильме","train_count":541,"old_ids":[165171,148446],"score":541,"new_id":248867},{"text":"дипломат","train_count":180,"old_ids":[6733,45768,150954,7721],"score":540,"new_id":248868},{"text":"свободы","train_count":135,"old_ids":[2175,5301,148280,148277,4403],"score":540,"new_id":248869},{"text":"Первый","train_count":539,"old_ids":[95169,159061],"score":539,"new_id":248870},{"text":"некоторые","train_count":538,"old_ids":[76069,178717],"score":538,"new_id":248871},{"text":"округу","train_count":269,"old_ids":[148518,62215,3652],"score":538,"new_id":248872},{"text":"английского","train_count":179,"old_ids":[7105,151203,11796,149280],"score":537,"new_id":248873},{"text":"Они","train_count":536,"old_ids":[19736,21918],"score":536,"new_id":248874},{"text":"дочери","train_count":268,"old_ids":[6733,36103,149098],"score":536,"new_id":248875},{"text":"князя","train_count":268,"old_ids":[4640,37855,152175],"score":536,"new_id":248876},{"text":"своими","train_count":268,"old_ids":[2175,148331,148623],"score":536,"new_id":248877},{"text":"управление","train_count":268,"old_ids":[47174,148381,53921],"score":536,"new_id":248878},{"text":"такие","train_count":535,"old_ids":[42074,149012],"score":535,"new_id":248879},{"text":"вынужден","train_count":178,"old_ids":[148705,2040,54025,149637],"score":534,"new_id":248880},{"text":"границы","train_count":267,"old_ids":[14942,91872,4403],"score":534,"new_id":248881},{"text":"зависит","train_count":178,"old_ids":[8688,148292,13113,9080],"score":534,"new_id":248882},{"text":"применение","train_count":178,"old_ids":[148489,15630,5092,16669],"score":534,"new_id":248883},{"text":"словам","train_count":267,"old_ids":[2175,160162,6240],"score":534,"new_id":248884},{"text":"чел","train_count":534,"old_ids":[5831,11777],"score":534,"new_id":248885},{"text":"исследований","train_count":266,"old_ids":[13113,149161,163808],"score":532,"new_id":248886},{"text":"началось","train_count":266,"old_ids":[29196,15481,95306],"score":532,"new_id":248887},{"text":"Большинство","train_count":177,"old_ids":[58297,148701,18372,148593],"score":531,"new_id":248888},{"text":"переименован","train_count":177,"old_ids":[159357,15630,69729,149137],"score":531,"new_id":248889},{"text":"Экономика","train_count":265,"old_ids":[89030,150662,12751],"score":530,"new_id":248890},{"text":"выборах","train_count":265,"old_ids":[148705,153414,10161],"score":530,"new_id":248891},{"text":"Народный","train_count":176,"old_ids":[19582,16763,148277,32401],"score":528,"new_id":248892},{"text":"берегу","train_count":528,"old_ids":[159941,148994],"score":528,"new_id":248893},{"text":"возможности","train_count":264,"old_ids":[150348,149014,65360],"score":528,"new_id":248894},{"text":"окончательно","train_count":176,"old_ids":[148518,16806,5831,150807],"score":528,"new_id":248895},{"text":"произошло","train_count":264,"old_ids":[149490,152476,160554],"score":528,"new_id":248896},{"text":"руководителем","train_count":132,"old_ids":[19160,150463,9080,77969,6240],"score":528,"new_id":248897},{"text":"созыва","train_count":176,"old_ids":[2175,148303,33422,1424],"score":528,"new_id":248898},{"text":"династии","train_count":263,"old_ids":[153462,6389,44633],"score":526,"new_id":248899},{"text":"монастыря","train_count":263,"old_ids":[179659,151407,148939],"score":526,"new_id":248900},{"text":"полковник","train_count":263,"old_ids":[148634,148336,67494],"score":526,"new_id":248901},{"text":"Вильгельм","train_count":175,"old_ids":[15729,64257,225639,6240],"score":525,"new_id":248902},{"text":"права","train_count":525,"old_ids":[150013,90219],"score":525,"new_id":248903},{"text":"другой","train_count":524,"old_ids":[192548,148293],"score":524,"new_id":248904},{"text":"которым","train_count":524,"old_ids":[151549,149957],"score":524,"new_id":248905},{"text":"независимость","train_count":131,"old_ids":[76069,8688,148292,13113,161757],"score":524,"new_id":248906},{"text":"современного","train_count":262,"old_ids":[149297,148801,148325],"score":524,"new_id":248907},{"text":"Был","train_count":523,"old_ids":[58297,165666],"score":523,"new_id":248908},{"text":"фильма","train_count":523,"old_ids":[165171,66050],"score":523,"new_id":248909},{"text":"Москву","train_count":261,"old_ids":[212095,4640,149628],"score":522,"new_id":248910},{"text":"европейских","train_count":174,"old_ids":[148380,148283,152679,150584],"score":522,"new_id":248911},{"text":"Западной","train_count":260,"old_ids":[172194,148414,172332],"score":520,"new_id":248912},{"text":"Ленинградского","train_count":130,"old_ids":[61757,9396,2040,153987,149280],"score":520,"new_id":248913},{"text":"вдоль","train_count":260,"old_ids":[5301,6733,148319],"score":520,"new_id":248914},{"text":"журнал","train_count":260,"old_ids":[16467,53555,155371],"score":520,"new_id":248915},{"text":"написал","train_count":260,"old_ids":[13785,28696,15481],"score":520,"new_id":248916},{"text":"царь","train_count":260,"old_ids":[10064,16763,4669],"score":520,"new_id":248917},{"text":"Она","train_count":519,"old_ids":[19736,13785],"score":519,"new_id":248918},{"text":"начинается","train_count":259,"old_ids":[29196,66353,48046],"score":518,"new_id":248919},{"text":"руководитель","train_count":259,"old_ids":[19160,150463,53062],"score":518,"new_id":248920},{"text":"уровне","train_count":259,"old_ids":[53555,148275,76069],"score":518,"new_id":248921},{"text":"хоккеист","train_count":259,"old_ids":[148491,201229,32884],"score":518,"new_id":248922},{"text":"Борисович","train_count":129,"old_ids":[58297,8789,13113,148275,31889],"score":516,"new_id":248923},{"text":"Всего","train_count":258,"old_ids":[15729,2175,148702],"score":516,"new_id":248924},{"text":"вплоть","train_count":258,"old_ids":[5301,150469,167048],"score":516,"new_id":248925},{"text":"группой","train_count":257,"old_ids":[14942,83147,148293],"score":514,"new_id":248926},{"text":"растения","train_count":257,"old_ids":[21441,6389,17941],"score":514,"new_id":248927},{"text":"Министерства","train_count":171,"old_ids":[36736,18372,154507,89771],"score":513,"new_id":248928},{"text":"Саксония","train_count":171,"old_ids":[18727,148318,177567,37835],"score":513,"new_id":248929},{"text":"побережье","train_count":171,"old_ids":[7754,148280,148579,204769],"score":513,"new_id":248930},{"text":"начинает","train_count":256,"old_ids":[29196,66353,7921],"score":512,"new_id":248931},{"text":"построенные","train_count":128,"old_ids":[7754,23514,148283,5092,41546],"score":512,"new_id":248932},{"text":"принято","train_count":256,"old_ids":[148489,18372,186048],"score":512,"new_id":248933},{"text":"профессором","train_count":128,"old_ids":[149490,18768,90202,8789,148288],"score":512,"new_id":248934},{"text":"Окончил","train_count":255,"old_ids":[19736,150270,161907],"score":510,"new_id":248935},{"text":"Чемпионат","train_count":170,"old_ids":[70345,148281,148686,232067],"score":510,"new_id":248936},{"text":"модель","train_count":255,"old_ids":[6240,148277,24623],"score":510,"new_id":248937},{"text":"обязанности","train_count":170,"old_ids":[148280,60860,7105,65360],"score":510,"new_id":248938},{"text":"отличие","train_count":510,"old_ids":[12666,173563],"score":510,"new_id":248939},{"text":"впоследствии","train_count":509,"old_ids":[5301,212703],"score":509,"new_id":248940},{"text":"представляет","train_count":509,"old_ids":[177917,155203],"score":509,"new_id":248941},{"text":"Николая","train_count":254,"old_ids":[19582,151907,175551],"score":508,"new_id":248942},{"text":"много","train_count":508,"old_ids":[6240,148325],"score":508,"new_id":248943},{"text":"неё","train_count":508,"old_ids":[76069,42565],"score":508,"new_id":248944},{"text":"орденом","train_count":254,"old_ids":[8789,194750,6240],"score":508,"new_id":248945},{"text":"памятник","train_count":254,"old_ids":[148414,79142,186469],"score":508,"new_id":248946},{"text":"предприниматель","train_count":127,"old_ids":[177917,148489,18372,15630,57868],"score":508,"new_id":248947},{"text":"обозначения","train_count":169,"old_ids":[148280,148303,29196,17941],"score":507,"new_id":248948},{"text":"тренер","train_count":507,"old_ids":[155339,7358],"score":507,"new_id":248949},{"text":"Район","train_count":253,"old_ids":[32392,148390,16806],"score":506,"new_id":248950},{"text":"использование","train_count":253,"old_ids":[13113,183192,166506],"score":506,"new_id":248951},{"text":"председателя","train_count":253,"old_ids":[177917,152824,152000],"score":506,"new_id":248952},{"text":"республики","train_count":253,"old_ids":[2115,151869,16256],"score":506,"new_id":248953},{"text":"значения","train_count":505,"old_ids":[149340,17941],"score":505,"new_id":248954},{"text":"Джордж","train_count":168,"old_ids":[23355,16467,8789,151828],"score":504,"new_id":248955},{"text":"Культура","train_count":168,"old_ids":[25497,62709,3652,21441],"score":504,"new_id":248956},{"text":"Роберт","train_count":252,"old_ids":[32392,148280,153922],"score":504,"new_id":248957},{"text":"договора","train_count":168,"old_ids":[6733,148449,5301,148676],"score":504,"new_id":248958},{"text":"заведующий","train_count":126,"old_ids":[8688,148292,148276,68608,42387],"score":504,"new_id":248959},{"text":"создание","train_count":252,"old_ids":[2175,148303,158351],"score":504,"new_id":248960},{"text":"эксплуатации","train_count":126,"old_ids":[192809,31608,148990,7721,51788],"score":504,"new_id":248961},{"text":"гитара","train_count":251,"old_ids":[14942,9080,148622],"score":502,"new_id":248962},{"text":"некоторое","train_count":251,"old_ids":[76069,151549,42317],"score":502,"new_id":248963},{"text":"правило","train_count":502,"old_ids":[155367,37483],"score":502,"new_id":248964},{"text":"Гатчинского","train_count":167,"old_ids":[35884,7721,155812,149280],"score":501,"new_id":248965},{"text":"Института","train_count":250,"old_ids":[161484,158693,152644],"score":500,"new_id":248966},{"text":"вышел","train_count":500,"old_ids":[148705,152664],"score":500,"new_id":248967},{"text":"никогда","train_count":250,"old_ids":[21918,45527,150921],"score":500,"new_id":248968},{"text":"округе","train_count":250,"old_ids":[148518,62215,1450],"score":500,"new_id":248969},{"text":"улице","train_count":250,"old_ids":[3652,63890,1450],"score":500,"new_id":248970},{"text":"Бавария","train_count":166,"old_ids":[58297,148292,16763,37835],"score":498,"new_id":248971},{"text":"Центральной","train_count":166,"old_ids":[89602,18110,2115,151291],"score":498,"new_id":248972},{"text":"дирижёр","train_count":166,"old_ids":[148467,29119,16467,156204],"score":498,"new_id":248973},{"text":"состоялся","train_count":249,"old_ids":[2175,152544,151397],"score":498,"new_id":248974},{"text":"Кубок","train_count":248,"old_ids":[25497,61324,148518],"score":496,"new_id":248975},{"text":"Торпедо","train_count":124,"old_ids":[32868,8789,7754,148276,1403],"score":496,"new_id":248976},{"text":"депутат","train_count":248,"old_ids":[95525,198375,7721],"score":496,"new_id":248977},{"text":"получить","train_count":248,"old_ids":[148634,27448,17256],"score":496,"new_id":248978},{"text":"состоялась","train_count":248,"old_ids":[2175,152544,155063],"score":496,"new_id":248979},{"text":"распространение","train_count":165,"old_ids":[148761,155611,33437,16669],"score":495,"new_id":248980},{"text":"слова","train_count":495,"old_ids":[2175,160162],"score":495,"new_id":248981},{"text":"матчей","train_count":247,"old_ids":[6240,7721,152371],"score":494,"new_id":248982},{"text":"северной","train_count":247,"old_ids":[148938,46576,148317],"score":494,"new_id":248983},{"text":"данных","train_count":493,"old_ids":[6733,151990],"score":493,"new_id":248984},{"text":"АНП","train_count":492,"old_ids":[161159,16362],"score":492,"new_id":248985},{"text":"Джеймс","train_count":246,"old_ids":[23355,153703,233559],"score":492,"new_id":248986},{"text":"войск","train_count":492,"old_ids":[159035,62847],"score":492,"new_id":248987},{"text":"Помимо","train_count":245,"old_ids":[16362,148288,149076],"score":490,"new_id":248988},{"text":"матери","train_count":245,"old_ids":[6240,7721,149098],"score":490,"new_id":248989},{"text":"решения","train_count":245,"old_ids":[2115,44432,17941],"score":490,"new_id":248990},{"text":"эпохи","train_count":245,"old_ids":[65131,148428,150931],"score":490,"new_id":248991},{"text":"русской","train_count":489,"old_ids":[152528,149193],"score":489,"new_id":248992},{"text":"вокал","train_count":244,"old_ids":[148331,12751,2941],"score":488,"new_id":248993},{"text":"выиграл","train_count":244,"old_ids":[148705,66951,197362],"score":488,"new_id":248994},{"text":"капитан","train_count":244,"old_ids":[12751,150478,7105],"score":488,"new_id":248995},{"text":"коммуна","train_count":487,"old_ids":[210786,13785],"score":487,"new_id":248996},{"text":"Иоанна","train_count":162,"old_ids":[29189,1403,7105,13785],"score":486,"new_id":248997},{"text":"причём","train_count":243,"old_ids":[148895,5831,151666],"score":486,"new_id":248998},{"text":"участок","train_count":243,"old_ids":[27448,80378,148518],"score":486,"new_id":248999},{"text":"пути","train_count":485,"old_ids":[148729,21797],"score":485,"new_id":249000},{"text":"четыре","train_count":485,"old_ids":[243429,148279],"score":485,"new_id":249001},{"text":"края","train_count":484,"old_ids":[148998,4096],"score":484,"new_id":249002},{"text":"муниципальное","train_count":121,"old_ids":[148405,151744,7754,36771,45480],"score":484,"new_id":249003},{"text":"память","train_count":242,"old_ids":[148414,79142,27851],"score":484,"new_id":249004},{"text":"центром","train_count":484,"old_ids":[151971,149633],"score":484,"new_id":249005},{"text":"большинстве","train_count":161,"old_ids":[9629,148701,18372,151418],"score":483,"new_id":249006},{"text":"Константин","train_count":241,"old_ids":[157349,49224,150779],"score":482,"new_id":249007},{"text":"воеводства","train_count":482,"old_ids":[159117,221196],"score":482,"new_id":249008},{"text":"другими","train_count":482,"old_ids":[192548,148623],"score":482,"new_id":249009},{"text":"открыт","train_count":241,"old_ids":[12666,91201,1729],"score":482,"new_id":249010},{"text":"частично","train_count":241,"old_ids":[150813,31889,13292],"score":482,"new_id":249011},{"text":"Государственного","train_count":160,"old_ids":[35884,22078,150968,172012],"score":480,"new_id":249012},{"text":"отца","train_count":480,"old_ids":[12666,83895],"score":480,"new_id":249013},{"text":"участников","train_count":239,"old_ids":[27448,80378,149668],"score":478,"new_id":249014},{"text":"заниматься","train_count":159,"old_ids":[8688,13603,6240,94309],"score":477,"new_id":249015},{"text":"проспект","train_count":159,"old_ids":[160383,7754,148301,1729],"score":477,"new_id":249016},{"text":"содержание","train_count":159,"old_ids":[2175,148277,55638,25767],"score":477,"new_id":249017},{"text":"матч","train_count":238,"old_ids":[6240,7721,5831],"score":476,"new_id":249018},{"text":"появился","train_count":238,"old_ids":[7754,58255,218711],"score":476,"new_id":249019},{"text":"Святой","train_count":237,"old_ids":[18727,152139,170627],"score":474,"new_id":249020},{"text":"Формулы","train_count":158,"old_ids":[52337,30206,3652,149383],"score":474,"new_id":249021},{"text":"карьера","train_count":237,"old_ids":[12751,149213,87658],"score":474,"new_id":249022},{"text":"муниципальный","train_count":158,"old_ids":[148405,151744,7754,152027],"score":474,"new_id":249023},{"text":"поэтому","train_count":474,"old_ids":[148428,155084],"score":474,"new_id":249024},{"text":"приняли","train_count":158,"old_ids":[148489,18372,4096,10750],"score":474,"new_id":249025},{"text":"турнира","train_count":237,"old_ids":[152238,21918,21441],"score":474,"new_id":249026},{"text":"большую","train_count":236,"old_ids":[9629,148701,31953],"score":472,"new_id":249027},{"text":"каждого","train_count":236,"old_ids":[12751,148343,148449],"score":472,"new_id":249028},{"text":"политической","train_count":236,"old_ids":[148634,9080,158585],"score":472,"new_id":249029},{"text":"призёр","train_count":236,"old_ids":[148895,8688,156204],"score":472,"new_id":249030},{"text":"упоминание","train_count":157,"old_ids":[47174,148288,18372,25767],"score":471,"new_id":249031},{"text":"Анны","train_count":235,"old_ids":[34960,2040,12014],"score":470,"new_id":249032},{"text":"дочь","train_count":470,"old_ids":[6733,157006],"score":470,"new_id":249033},{"text":"победу","train_count":235,"old_ids":[7754,148280,150419],"score":470,"new_id":249034},{"text":"ролях","train_count":235,"old_ids":[148283,13302,10161],"score":470,"new_id":249035},{"text":"современных","train_count":235,"old_ids":[149297,148801,42240],"score":470,"new_id":249036},{"text":"энциклопедии","train_count":94,"old_ids":[156454,14666,151639,7754,148276,44633],"score":470,"new_id":249037},{"text":"большая","train_count":234,"old_ids":[9629,148701,34226],"score":468,"new_id":249038},{"text":"весьма","train_count":234,"old_ids":[5301,88434,66050],"score":468,"new_id":249039},{"text":"живописи","train_count":156,"old_ids":[151303,1403,28696,1739],"score":468,"new_id":249040},{"text":"использовалась","train_count":117,"old_ids":[13113,183192,148675,15481,152993],"score":468,"new_id":249041},{"text":"эксплуатацию","train_count":117,"old_ids":[192809,31608,148990,7721,150451],"score":468,"new_id":249042},{"text":"эпоху","train_count":156,"old_ids":[65131,148428,10161,3652],"score":468,"new_id":249043},{"text":"разных","train_count":466,"old_ids":[148464,42240],"score":466,"new_id":249044},{"text":"состояние","train_count":233,"old_ids":[2175,152544,150528],"score":466,"new_id":249045},{"text":"становится","train_count":466,"old_ids":[148722,92907],"score":466,"new_id":249046},{"text":"технологии","train_count":233,"old_ids":[157971,152910,44633],"score":466,"new_id":249047},{"text":"Октябрьской","train_count":155,"old_ids":[19736,4640,161498,149193],"score":465,"new_id":249048},{"text":"Подчиняется","train_count":155,"old_ids":[174249,155812,4096,48046],"score":465,"new_id":249049},{"text":"километрах","train_count":155,"old_ids":[16256,37483,148557,172199],"score":465,"new_id":249050},{"text":"Хоккеисты","train_count":116,"old_ids":[150853,148518,50800,32884,4403],"score":464,"new_id":249051},{"text":"документах","train_count":116,"old_ids":[148551,34083,6240,84706,10161],"score":464,"new_id":249052},{"text":"командир","train_count":232,"old_ids":[148558,72884,74990],"score":464,"new_id":249053},{"text":"поддержку","train_count":232,"old_ids":[155908,148605,34083],"score":464,"new_id":249054},{"text":"рамках","train_count":464,"old_ids":[172422,150577],"score":464,"new_id":249055},{"text":"торговли","train_count":232,"old_ids":[35458,148762,10750],"score":464,"new_id":249056},{"text":"военный","train_count":463,"old_ids":[224606,32401],"score":463,"new_id":249057},{"text":"поэт","train_count":463,"old_ids":[148428,184088],"score":463,"new_id":249058},{"text":"частью","train_count":463,"old_ids":[148659,155150],"score":463,"new_id":249059},{"text":"Московского","train_count":462,"old_ids":[212095,220104],"score":462,"new_id":249060},{"text":"Участники","train_count":231,"old_ids":[39243,150813,149665],"score":462,"new_id":249061},{"text":"генерала","train_count":231,"old_ids":[170802,7358,92647],"score":462,"new_id":249062},{"text":"собрания","train_count":231,"old_ids":[2175,149962,36916],"score":462,"new_id":249063},{"text":"уезда","train_count":462,"old_ids":[3652,157279],"score":462,"new_id":249064},{"text":"царя","train_count":231,"old_ids":[10064,16763,4096],"score":462,"new_id":249065},{"text":"Другие","train_count":230,"old_ids":[23355,62215,47552],"score":460,"new_id":249066},{"text":"Швеции","train_count":230,"old_ids":[151311,148537,54201],"score":460,"new_id":249067},{"text":"католической","train_count":115,"old_ids":[4640,149900,87018,21785,148554],"score":460,"new_id":249068},{"text":"музыку","train_count":230,"old_ids":[148405,148707,34083],"score":460,"new_id":249069},{"text":"началом","train_count":230,"old_ids":[29196,15481,148288],"score":460,"new_id":249070},{"text":"Екатерина","train_count":153,"old_ids":[53241,150843,7358,66353],"score":459,"new_id":249071},{"text":"использоваться","train_count":153,"old_ids":[13113,183192,148675,94309],"score":459,"new_id":249072},{"text":"НХЛ","train_count":229,"old_ids":[19582,150853,61757],"score":458,"new_id":249073},{"text":"Нижняя","train_count":229,"old_ids":[19582,151932,151713],"score":458,"new_id":249074},{"text":"шведский","train_count":229,"old_ids":[11802,148700,149800],"score":458,"new_id":249075},{"text":"школа","train_count":458,"old_ids":[11802,159527],"score":458,"new_id":249076},{"text":"всей","train_count":457,"old_ids":[5301,174220],"score":457,"new_id":249077},{"text":"Деревня","train_count":152,"old_ids":[23355,7358,148380,37855],"score":456,"new_id":249078},{"text":"Национальной","train_count":228,"old_ids":[19582,158048,159075],"score":456,"new_id":249079},{"text":"Центрального","train_count":152,"old_ids":[89602,18110,2115,151302],"score":456,"new_id":249080},{"text":"называлась","train_count":152,"old_ids":[13785,152953,15481,152993],"score":456,"new_id":249081},{"text":"обеспечивает","train_count":114,"old_ids":[148280,149097,53887,26125,26046],"score":456,"new_id":249082},{"text":"островов","train_count":228,"old_ids":[23514,148327,148275],"score":456,"new_id":249083},{"text":"побережья","train_count":152,"old_ids":[7754,148280,148579,210830],"score":456,"new_id":249084},{"text":"представители","train_count":228,"old_ids":[177917,148448,153009],"score":456,"new_id":249085},{"text":"применяются","train_count":114,"old_ids":[148489,15630,5092,4096,149124],"score":456,"new_id":249086},{"text":"уровень","train_count":228,"old_ids":[53555,148275,49603],"score":456,"new_id":249087},{"text":"фактически","train_count":228,"old_ids":[18768,148318,155916],"score":456,"new_id":249088},{"text":"должен","train_count":455,"old_ids":[159172,148942],"score":455,"new_id":249089},{"text":"Большая","train_count":227,"old_ids":[58297,148701,34226],"score":454,"new_id":249090},{"text":"Название","train_count":454,"old_ids":[81573,202403],"score":454,"new_id":249091},{"text":"создать","train_count":227,"old_ids":[2175,148303,156156],"score":454,"new_id":249092},{"text":"принимает","train_count":151,"old_ids":[148489,18372,15630,26046],"score":453,"new_id":249093},{"text":"Церковь","train_count":226,"old_ids":[89602,7358,163687],"score":452,"new_id":249094},{"text":"армянский","train_count":226,"old_ids":[16763,167664,149800],"score":452,"new_id":249095},{"text":"обучения","train_count":226,"old_ids":[148280,27448,17941],"score":452,"new_id":249096},{"text":"помощь","train_count":226,"old_ids":[151201,88260,4669],"score":452,"new_id":249097},{"text":"псевдонимом","train_count":113,"old_ids":[7754,215194,182503,15630,148288],"score":452,"new_id":249098},{"text":"сравнению","train_count":226,"old_ids":[2175,148381,155731],"score":452,"new_id":249099},{"text":"Португалии","train_count":90,"old_ids":[16362,8789,85208,14942,45679,1739],"score":450,"new_id":249100},{"text":"каждой","train_count":225,"old_ids":[12751,148343,148293],"score":450,"new_id":249101},{"text":"мужчин","train_count":225,"old_ids":[6240,54025,155812],"score":450,"new_id":249102},{"text":"недалеко","train_count":150,"old_ids":[76069,39596,1450,45527],"score":450,"new_id":249103},{"text":"рабочих","train_count":225,"old_ids":[21441,151293,64392],"score":450,"new_id":249104},{"text":"соединения","train_count":150,"old_ids":[2175,42317,67958,17941],"score":450,"new_id":249105},{"text":"уровнем","train_count":225,"old_ids":[53555,148275,152217],"score":450,"new_id":249106},{"text":"русского","train_count":449,"old_ids":[152528,149280],"score":449,"new_id":249107},{"text":"начиная","train_count":224,"old_ids":[29196,1739,42308],"score":448,"new_id":249108},{"text":"песню","train_count":224,"old_ids":[7754,21785,155131],"score":448,"new_id":249109},{"text":"получения","train_count":224,"old_ids":[148634,27448,17941],"score":448,"new_id":249110},{"text":"связано","train_count":224,"old_ids":[2175,163720,149600],"score":448,"new_id":249111},{"text":"следующем","train_count":224,"old_ids":[149161,68608,148281],"score":448,"new_id":249112},{"text":"участия","train_count":224,"old_ids":[27448,80378,37835],"score":448,"new_id":249113},{"text":"Дмитриевич","train_count":149,"old_ids":[23355,157255,188601,31889],"score":447,"new_id":249114},{"text":"Кафедра","train_count":149,"old_ids":[172753,18768,148276,21441],"score":447,"new_id":249115},{"text":"Начиная","train_count":149,"old_ids":[19582,47474,1739,42308],"score":447,"new_id":249116},{"text":"кислоты","train_count":149,"old_ids":[4640,13113,154713,4403],"score":447,"new_id":249117},{"text":"отказался","train_count":149,"old_ids":[12666,148330,15481,19937],"score":447,"new_id":249118},{"text":"практически","train_count":447,"old_ids":[229238,169866],"score":447,"new_id":249119},{"text":"соревнования","train_count":149,"old_ids":[154399,148380,148366,36916],"score":447,"new_id":249120},{"text":"участием","train_count":149,"old_ids":[27448,80378,47552,6240],"score":447,"new_id":249121},{"text":"языков","train_count":447,"old_ids":[214432,148336],"score":447,"new_id":249122},{"text":"армию","train_count":223,"old_ids":[16763,61628,11577],"score":446,"new_id":249123},{"text":"итогам","train_count":223,"old_ids":[9080,149095,148357],"score":446,"new_id":249124},{"text":"международных","train_count":223,"old_ids":[148446,151896,42240],"score":446,"new_id":249125},{"text":"образован","train_count":223,"old_ids":[149586,148275,7105],"score":446,"new_id":249126},{"text":"выборов","train_count":445,"old_ids":[148705,194213],"score":445,"new_id":249127},{"text":"имел","train_count":445,"old_ids":[15630,11777],"score":445,"new_id":249128},{"text":"таким","train_count":445,"old_ids":[149338,15630],"score":445,"new_id":249129},{"text":"Германия","train_count":222,"old_ids":[240053,6240,36916],"score":444,"new_id":249130},{"text":"Леонид","train_count":222,"old_ids":[190021,16806,45587],"score":444,"new_id":249131},{"text":"Первой","train_count":444,"old_ids":[95169,159035],"score":444,"new_id":249132},{"text":"национального","train_count":222,"old_ids":[13785,152208,155045],"score":444,"new_id":249133},{"text":"национальной","train_count":222,"old_ids":[13785,152208,159075],"score":444,"new_id":249134},{"text":"общественный","train_count":222,"old_ids":[149350,150198,32401],"score":444,"new_id":249135},{"text":"формы","train_count":443,"old_ids":[54724,4403],"score":443,"new_id":249136},{"text":"Земли","train_count":221,"old_ids":[34279,148281,10750],"score":442,"new_id":249137},{"text":"Источники","train_count":221,"old_ids":[29189,161308,149665],"score":442,"new_id":249138},{"text":"Обладатель","train_count":221,"old_ids":[70749,148606,57868],"score":442,"new_id":249139},{"text":"середины","train_count":221,"old_ids":[159541,148969,12014],"score":442,"new_id":249140},{"text":"создал","train_count":221,"old_ids":[2175,148303,39596],"score":442,"new_id":249141},{"text":"губерния","train_count":147,"old_ids":[14942,61324,7358,155012],"score":441,"new_id":249142},{"text":"окружающей","train_count":147,"old_ids":[148518,19160,148458,153351],"score":441,"new_id":249143},{"text":"случаев","train_count":147,"old_ids":[38612,27448,1424,148380],"score":441,"new_id":249144},{"text":"Вооружённых","train_count":110,"old_ids":[173647,8789,54025,42565,170741],"score":440,"new_id":249145},{"text":"выступил","train_count":220,"old_ids":[148705,148757,61057],"score":440,"new_id":249146},{"text":"кроме","train_count":440,"old_ids":[151472,148446],"score":440,"new_id":249147},{"text":"насчитывалось","train_count":110,"old_ids":[222958,87148,33422,15481,95306],"score":440,"new_id":249148},{"text":"начало","train_count":440,"old_ids":[29196,150182],"score":440,"new_id":249149},{"text":"основана","train_count":220,"old_ids":[22078,148366,148841],"score":440,"new_id":249150},{"text":"отдельных","train_count":220,"old_ids":[12666,159194,42240],"score":440,"new_id":249151},{"text":"победителем","train_count":88,"old_ids":[7754,148280,148276,9080,77969,6240],"score":440,"new_id":249152},{"text":"самым","train_count":440,"old_ids":[160002,149957],"score":440,"new_id":249153},{"text":"улица","train_count":440,"old_ids":[3652,168314],"score":440,"new_id":249154},{"text":"немецкого","train_count":219,"old_ids":[76069,218942,149075],"score":438,"new_id":249155},{"text":"определяется","train_count":146,"old_ids":[148561,148420,41580,48046],"score":438,"new_id":249156},{"text":"следующий","train_count":219,"old_ids":[149161,68608,42387],"score":438,"new_id":249157},{"text":"состоянии","train_count":219,"old_ids":[2175,152544,155016],"score":438,"new_id":249158},{"text":"философ","train_count":146,"old_ids":[18768,61057,152406,18768],"score":438,"new_id":249159},{"text":"элементов","train_count":219,"old_ids":[167360,89315,148275],"score":438,"new_id":249160},{"text":"востоке","train_count":436,"old_ids":[194731,50800],"score":436,"new_id":249161},{"text":"князь","train_count":218,"old_ids":[4640,37855,171730],"score":436,"new_id":249162},{"text":"направлении","train_count":218,"old_ids":[13785,148607,152709],"score":436,"new_id":249163},{"text":"военачальник","train_count":145,"old_ids":[224606,47474,36771,67494],"score":435,"new_id":249164},{"text":"Фёдор","train_count":217,"old_ids":[52337,42565,163347],"score":434,"new_id":249165},{"text":"вероятно","train_count":217,"old_ids":[46576,150810,13292],"score":434,"new_id":249166},{"text":"музыкальный","train_count":217,"old_ids":[148405,168062,159069],"score":434,"new_id":249167},{"text":"обеспечения","train_count":217,"old_ids":[148280,149097,154783],"score":434,"new_id":249168},{"text":"первую","train_count":434,"old_ids":[149208,170755],"score":434,"new_id":249169},{"text":"политики","train_count":217,"old_ids":[148634,9080,151278],"score":434,"new_id":249170},{"text":"фронта","train_count":217,"old_ids":[18768,2115,150833],"score":434,"new_id":249171},{"text":"первая","train_count":433,"old_ids":[149208,155044],"score":433,"new_id":249172},{"text":"Совет","train_count":432,"old_ids":[173518,45405],"score":432,"new_id":249173},{"text":"Фильмография","train_count":144,"old_ids":[52337,64257,148324,181923],"score":432,"new_id":249174},{"text":"большого","train_count":216,"old_ids":[9629,148701,148449],"score":432,"new_id":249175},{"text":"вооружённых","train_count":108,"old_ids":[148331,8789,54025,42565,170741],"score":432,"new_id":249176},{"text":"двигателя","train_count":144,"old_ids":[6733,5301,66951,152000],"score":432,"new_id":249177},{"text":"диссертацию","train_count":144,"old_ids":[216775,2175,153922,150451],"score":432,"new_id":249178},{"text":"успехом","train_count":144,"old_ids":[3652,31608,93834,148288],"score":432,"new_id":249179},{"text":"вскоре","train_count":430,"old_ids":[5301,197641],"score":430,"new_id":249180},{"text":"пришлось","train_count":215,"old_ids":[148895,11802,153174],"score":430,"new_id":249181},{"text":"села","train_count":430,"old_ids":[2175,150456],"score":430,"new_id":249182},{"text":"элементы","train_count":215,"old_ids":[167360,89315,4403],"score":430,"new_id":249183},{"text":"Фридрих","train_count":143,"old_ids":[52337,29119,186983,10161],"score":429,"new_id":249184},{"text":"архитектора","train_count":143,"old_ids":[16763,10161,25135,160510],"score":429,"new_id":249185},{"text":"дней","train_count":429,"old_ids":[6733,148820],"score":429,"new_id":249186},{"text":"командиром","train_count":143,"old_ids":[148558,72884,74990,148288],"score":429,"new_id":249187},{"text":"учебных","train_count":143,"old_ids":[27448,1450,9629,42240],"score":429,"new_id":249188},{"text":"Вместе","train_count":214,"old_ids":[15729,150436,1450],"score":428,"new_id":249189},{"text":"борьбы","train_count":214,"old_ids":[148913,4669,148812],"score":428,"new_id":249190},{"text":"математики","train_count":107,"old_ids":[6240,7721,148281,149165,16256],"score":428,"new_id":249191},{"text":"основании","train_count":214,"old_ids":[22078,148366,149091],"score":428,"new_id":249192},{"text":"постоянно","train_count":214,"old_ids":[7754,152544,171142],"score":428,"new_id":249193},{"text":"Киев","train_count":213,"old_ids":[25497,47552,5301],"score":426,"new_id":249194},{"text":"Первоначально","train_count":213,"old_ids":[95169,148331,167193],"score":426,"new_id":249195},{"text":"гражданской","train_count":213,"old_ids":[148510,222837,149193],"score":426,"new_id":249196},{"text":"материала","train_count":142,"old_ids":[6240,7721,149098,92647],"score":426,"new_id":249197},{"text":"порядка","train_count":213,"old_ids":[153624,93629,12751],"score":426,"new_id":249198},{"text":"скончался","train_count":142,"old_ids":[62847,16806,198597,19937],"score":426,"new_id":249199},{"text":"Сельские","train_count":212,"old_ids":[18727,24623,153218],"score":424,"new_id":249200},{"text":"Среди","train_count":424,"old_ids":[18727,149520],"score":424,"new_id":249201},{"text":"бывшего","train_count":212,"old_ids":[9629,33422,150083],"score":424,"new_id":249202},{"text":"третий","train_count":212,"old_ids":[153298,7921,42387],"score":424,"new_id":249203},{"text":"Генриха","train_count":141,"old_ids":[35884,5092,29119,150532],"score":423,"new_id":249204},{"text":"городской","train_count":422,"old_ids":[152689,149193],"score":422,"new_id":249205},{"text":"остров","train_count":422,"old_ids":[23514,148327],"score":422,"new_id":249206},{"text":"традиции","train_count":211,"old_ids":[150246,148467,54201],"score":422,"new_id":249207},{"text":"этим","train_count":422,"old_ids":[65131,162151],"score":422,"new_id":249208},{"text":"Город","train_count":421,"old_ids":[35884,149345],"score":421,"new_id":249209},{"text":"должность","train_count":421,"old_ids":[159172,156767],"score":421,"new_id":249210},{"text":"Беларусь","train_count":105,"old_ids":[58297,11777,16763,41635,4669],"score":420,"new_id":249211},{"text":"Верхняя","train_count":210,"old_ids":[238301,10161,151713],"score":420,"new_id":249212},{"text":"Париже","train_count":210,"old_ids":[16362,148699,56090],"score":420,"new_id":249213},{"text":"Энциклопедия","train_count":84,"old_ids":[89030,159489,151639,7754,148276,37835],"score":420,"new_id":249214},{"text":"администрации","train_count":210,"old_ids":[149099,152379,54201],"score":420,"new_id":249215},{"text":"артиллерии","train_count":140,"old_ids":[16763,21797,165328,44633],"score":420,"new_id":249216},{"text":"главный","train_count":420,"old_ids":[149809,203446],"score":420,"new_id":249217},{"text":"женился","train_count":210,"old_ids":[16467,9396,151397],"score":420,"new_id":249218},{"text":"оркестра","train_count":140,"old_ids":[8789,4640,34981,21441],"score":420,"new_id":249219},{"text":"первом","train_count":420,"old_ids":[149208,170668],"score":420,"new_id":249220},{"text":"проекта","train_count":420,"old_ids":[162947,42074],"score":420,"new_id":249221},{"text":"произведён","train_count":210,"old_ids":[161612,148700,185709],"score":420,"new_id":249222},{"text":"сделал","train_count":210,"old_ids":[2175,55986,15481],"score":420,"new_id":249223},{"text":"человеком","train_count":140,"old_ids":[5831,11777,150937,148558],"score":420,"new_id":249224},{"text":"Через","train_count":419,"old_ids":[214764,148377],"score":419,"new_id":249225},{"text":"Лауреаты","train_count":209,"old_ids":[196219,201647,149351],"score":418,"new_id":249226},{"text":"Происхождение","train_count":209,"old_ids":[159806,13113,193219],"score":418,"new_id":249227},{"text":"американского","train_count":209,"old_ids":[1424,155899,149280],"score":418,"new_id":249228},{"text":"высокой","train_count":209,"old_ids":[148705,148454,148554],"score":418,"new_id":249229},{"text":"местах","train_count":209,"old_ids":[150436,1424,10161],"score":418,"new_id":249230},{"text":"шоссе","train_count":209,"old_ids":[11802,22078,148938],"score":418,"new_id":249231},{"text":"Генрих","train_count":139,"old_ids":[35884,5092,29119,10161],"score":417,"new_id":249232},{"text":"Чемпионы","train_count":139,"old_ids":[70345,148281,148686,154479],"score":417,"new_id":249233},{"text":"классификации","train_count":139,"old_ids":[182718,59627,71658,54201],"score":417,"new_id":249234},{"text":"населённые","train_count":139,"old_ids":[13785,189921,42565,155130],"score":417,"new_id":249235},{"text":"площадью","train_count":139,"old_ids":[170421,13739,149099,84920],"score":417,"new_id":249236},{"text":"построено","train_count":139,"old_ids":[7754,23514,148283,69729],"score":417,"new_id":249237},{"text":"органов","train_count":208,"old_ids":[8789,153702,148275],"score":416,"new_id":249238},{"text":"открыта","train_count":208,"old_ids":[12666,91201,42074],"score":416,"new_id":249239},{"text":"постановщик","train_count":104,"old_ids":[7754,23514,7105,148275,161782],"score":416,"new_id":249240},{"text":"принят","train_count":208,"old_ids":[148489,18372,86128],"score":416,"new_id":249241},{"text":"сельского","train_count":416,"old_ids":[226181,149280],"score":416,"new_id":249242},{"text":"школе","train_count":416,"old_ids":[11802,157779],"score":416,"new_id":249243},{"text":"Петропавловка","train_count":83,"old_ids":[154192,148283,7754,148292,148569,12751],"score":415,"new_id":249244},{"text":"примерно","train_count":415,"old_ids":[155147,13292],"score":415,"new_id":249245},{"text":"второго","train_count":414,"old_ids":[167396,148449],"score":414,"new_id":249246},{"text":"кинофестиваля","train_count":69,"old_ids":[16256,13292,18768,34981,26125,1424,13302],"score":414,"new_id":249247},{"text":"королевства","train_count":138,"old_ids":[4640,148367,151186,89771],"score":414,"new_id":249248},{"text":"президентом","train_count":207,"old_ids":[149790,235531,148288],"score":414,"new_id":249249},{"text":"применения","train_count":138,"old_ids":[148489,15630,5092,17941],"score":414,"new_id":249250},{"text":"температуры","train_count":138,"old_ids":[170744,149208,149959,36747],"score":414,"new_id":249251},{"text":"общины","train_count":413,"old_ids":[148280,182004],"score":413,"new_id":249252},{"text":"Степанович","train_count":103,"old_ids":[18727,48824,181841,148275,31889],"score":412,"new_id":249253},{"text":"брат","train_count":412,"old_ids":[9629,77779],"score":412,"new_id":249254},{"text":"губернатора","train_count":103,"old_ids":[14942,61324,7358,2040,168640],"score":412,"new_id":249255},{"text":"исторических","train_count":206,"old_ids":[32884,8789,151308],"score":412,"new_id":249256},{"text":"литературовед","train_count":103,"old_ids":[159329,7358,150849,148275,148276],"score":412,"new_id":249257},{"text":"мать","train_count":412,"old_ids":[6240,17460],"score":412,"new_id":249258},{"text":"степень","train_count":206,"old_ids":[6389,148445,49603],"score":412,"new_id":249259},{"text":"школу","train_count":412,"old_ids":[160291,148990],"score":412,"new_id":249260},{"text":"Баварии","train_count":137,"old_ids":[58297,148292,16763,44633],"score":411,"new_id":249261},{"text":"Депутаты","train_count":137,"old_ids":[23355,148445,36183,149351],"score":411,"new_id":249262},{"text":"Киевского","train_count":137,"old_ids":[25497,47552,5301,149280],"score":411,"new_id":249263},{"text":"Ричард","train_count":137,"old_ids":[32392,31889,16763,6733],"score":411,"new_id":249264},{"text":"Футбольные","train_count":137,"old_ids":[52337,36183,170946,41546],"score":411,"new_id":249265},{"text":"вернулся","train_count":411,"old_ids":[46576,163976],"score":411,"new_id":249266},{"text":"переведён","train_count":137,"old_ids":[149208,148380,148276,185709],"score":411,"new_id":249267},{"text":"победил","train_count":137,"old_ids":[7754,148280,148276,61057],"score":411,"new_id":249268},{"text":"процессов","train_count":137,"old_ids":[149490,10064,90202,148275],"score":411,"new_id":249269},{"text":"Георгий","train_count":205,"old_ids":[228209,158568,42387],"score":410,"new_id":249270},{"text":"Национального","train_count":205,"old_ids":[19582,158048,155045],"score":410,"new_id":249271},{"text":"вступил","train_count":205,"old_ids":[5301,148757,61057],"score":410,"new_id":249272},{"text":"Вячеслав","train_count":136,"old_ids":[15729,4096,148614,150800],"score":408,"new_id":249273},{"text":"Георгия","train_count":204,"old_ids":[228209,8789,156685],"score":408,"new_id":249274},{"text":"Краснодарского","train_count":102,"old_ids":[25497,148761,13292,152148,149280],"score":408,"new_id":249275},{"text":"Ленинграде","train_count":102,"old_ids":[61757,9396,2040,148510,95525],"score":408,"new_id":249276},{"text":"Саксонии","train_count":136,"old_ids":[18727,148318,177567,44633],"score":408,"new_id":249277},{"text":"бразильский","train_count":136,"old_ids":[9629,148464,64257,149800],"score":408,"new_id":249278},{"text":"будучи","train_count":204,"old_ids":[156543,27448,1739],"score":408,"new_id":249279},{"text":"железнодорожных","train_count":102,"old_ids":[16467,77969,8688,166815,155228],"score":408,"new_id":249280},{"text":"начальника","train_count":204,"old_ids":[29196,36771,84480],"score":408,"new_id":249281},{"text":"похоронен","train_count":102,"old_ids":[148428,10161,8789,16806,5092],"score":408,"new_id":249282},{"text":"присоединился","train_count":102,"old_ids":[148489,13113,42317,67958,154838],"score":408,"new_id":249283},{"text":"силы","train_count":408,"old_ids":[177419,4403],"score":408,"new_id":249284},{"text":"Гмины","train_count":203,"old_ids":[35884,61628,12014],"score":406,"new_id":249285},{"text":"Российская","train_count":203,"old_ids":[32392,150405,152141],"score":406,"new_id":249286},{"text":"большим","train_count":203,"old_ids":[9629,148701,15630],"score":406,"new_id":249287},{"text":"империя","train_count":203,"old_ids":[15630,149208,37835],"score":406,"new_id":249288},{"text":"исключением","train_count":203,"old_ids":[13113,49689,149353],"score":406,"new_id":249289},{"text":"связь","train_count":203,"old_ids":[2175,163720,4669],"score":406,"new_id":249290},{"text":"Результаты","train_count":135,"old_ids":[32392,148377,62709,149351],"score":405,"new_id":249291},{"text":"двигателем","train_count":81,"old_ids":[6733,5301,66951,7721,77969,6240],"score":405,"new_id":249292},{"text":"издательство","train_count":135,"old_ids":[39427,6733,57868,148593],"score":405,"new_id":249293},{"text":"использовали","train_count":135,"old_ids":[13113,183192,148675,45679],"score":405,"new_id":249294},{"text":"оборудования","train_count":135,"old_ids":[148280,8789,34723,149349],"score":405,"new_id":249295},{"text":"продолжал","train_count":135,"old_ids":[148489,148277,148294,153002],"score":405,"new_id":249296},{"text":"Китая","train_count":202,"old_ids":[25497,9080,34226],"score":404,"new_id":249297},{"text":"Многие","train_count":202,"old_ids":[36736,13292,153277],"score":404,"new_id":249298},{"text":"Римской","train_count":202,"old_ids":[32392,15630,149193],"score":404,"new_id":249299},{"text":"времён","train_count":202,"old_ids":[5301,187973,185709],"score":404,"new_id":249300},{"text":"войне","train_count":403,"old_ids":[159035,76069],"score":403,"new_id":249301},{"text":"Шотландии","train_count":134,"old_ids":[151311,12666,200641,44633],"score":402,"new_id":249302},{"text":"Эдуард","train_count":134,"old_ids":[89030,148660,16763,6733],"score":402,"new_id":249303},{"text":"победы","train_count":201,"old_ids":[7754,148280,166844],"score":402,"new_id":249304},{"text":"привело","train_count":201,"old_ids":[148489,26125,150195],"score":402,"new_id":249305},{"text":"созданы","train_count":134,"old_ids":[2175,148303,6733,150165],"score":402,"new_id":249306},{"text":"воеводство","train_count":401,"old_ids":[159117,163406],"score":401,"new_id":249307},{"text":"системе","train_count":401,"old_ids":[167641,150181],"score":401,"new_id":249308},{"text":"находилась","train_count":200,"old_ids":[13785,217731,152993],"score":400,"new_id":249309},{"text":"редактор","train_count":200,"old_ids":[148279,38183,151676],"score":400,"new_id":249310},{"text":"Антонио","train_count":133,"old_ids":[34960,155092,149585,1403],"score":399,"new_id":249311},{"text":"Деятельность","train_count":133,"old_ids":[185698,86128,24623,58245],"score":399,"new_id":249312},{"text":"Ленинград","train_count":133,"old_ids":[61757,9396,2040,153987],"score":399,"new_id":249313},{"text":"австралийский","train_count":133,"old_ids":[148292,149389,159497,149800],"score":399,"new_id":249314},{"text":"бассейна","train_count":133,"old_ids":[9629,27473,148298,13785],"score":399,"new_id":249315},{"text":"двигатель","train_count":133,"old_ids":[6733,5301,66951,57868],"score":399,"new_id":249316},{"text":"занимают","train_count":133,"old_ids":[8688,13603,66050,66694],"score":399,"new_id":249317},{"text":"необходимости","train_count":133,"old_ids":[76069,148280,227708,81839],"score":399,"new_id":249318},{"text":"построены","train_count":133,"old_ids":[7754,23514,148283,75896],"score":399,"new_id":249319},{"text":"Париж","train_count":199,"old_ids":[16362,148699,16467],"score":398,"new_id":249320},{"text":"восстания","train_count":199,"old_ids":[163934,6389,36916],"score":398,"new_id":249321},{"text":"высшей","train_count":199,"old_ids":[148705,2175,150281],"score":398,"new_id":249322},{"text":"здание","train_count":398,"old_ids":[8688,158351],"score":398,"new_id":249323},{"text":"пределах","train_count":199,"old_ids":[149790,151400,10161],"score":398,"new_id":249324},{"text":"проблем","train_count":199,"old_ids":[148489,148280,151032],"score":398,"new_id":249325},{"text":"производство","train_count":398,"old_ids":[161612,163406],"score":398,"new_id":249326},{"text":"Вильгельма","train_count":132,"old_ids":[15729,64257,225639,66050],"score":396,"new_id":249327},{"text":"Казахстана","train_count":198,"old_ids":[172753,152999,162200],"score":396,"new_id":249328},{"text":"Посёлок","train_count":132,"old_ids":[16362,22078,42565,171144],"score":396,"new_id":249329},{"text":"Премия","train_count":198,"old_ids":[51844,148281,37835],"score":396,"new_id":249330},{"text":"авиации","train_count":198,"old_ids":[148292,1739,51788],"score":396,"new_id":249331},{"text":"вследствие","train_count":198,"old_ids":[5301,149161,149168],"score":396,"new_id":249332},{"text":"звёзд","train_count":132,"old_ids":[8688,5301,42565,171158],"score":396,"new_id":249333},{"text":"карьеры","train_count":198,"old_ids":[12751,149213,172094],"score":396,"new_id":249334},{"text":"количества","train_count":198,"old_ids":[45527,87018,150167],"score":396,"new_id":249335},{"text":"назван","train_count":198,"old_ids":[13785,8688,149137],"score":396,"new_id":249336},{"text":"параллельно","train_count":99,"old_ids":[151963,15481,2941,24623,13292],"score":396,"new_id":249337},{"text":"познакомился","train_count":132,"old_ids":[7754,158289,148558,154838],"score":396,"new_id":249338},{"text":"продолжил","train_count":132,"old_ids":[148489,148277,148294,208239],"score":396,"new_id":249339},{"text":"Азии","train_count":395,"old_ids":[34960,155585],"score":395,"new_id":249340},{"text":"азербайджанский","train_count":79,"old_ids":[148646,151292,148390,151828,7105,149800],"score":395,"new_id":249341},{"text":"российская","train_count":395,"old_ids":[184874,152141],"score":395,"new_id":249342},{"text":"Член","train_count":394,"old_ids":[70345,58361],"score":394,"new_id":249343},{"text":"Эстонии","train_count":197,"old_ids":[89030,211593,44633],"score":394,"new_id":249344},{"text":"Юрий","train_count":394,"old_ids":[168959,155163],"score":394,"new_id":249345},{"text":"корабля","train_count":197,"old_ids":[228309,9629,13302],"score":394,"new_id":249346},{"text":"основания","train_count":197,"old_ids":[22078,148366,36916],"score":394,"new_id":249347},{"text":"Альбомы","train_count":131,"old_ids":[34960,27112,173910,4403],"score":393,"new_id":249348},{"text":"Викторович","train_count":131,"old_ids":[15729,36473,155610,31889],"score":393,"new_id":249349},{"text":"Горького","train_count":131,"old_ids":[35884,8789,4669,149075],"score":393,"new_id":249350},{"text":"завоевал","train_count":131,"old_ids":[8688,148292,42317,151019],"score":393,"new_id":249351},{"text":"поселений","train_count":131,"old_ids":[7754,22078,11777,59318],"score":393,"new_id":249352},{"text":"существенно","train_count":131,"old_ids":[2175,82577,5092,13292],"score":393,"new_id":249353},{"text":"Императорского","train_count":98,"old_ids":[29189,6240,149208,55389,149280],"score":392,"new_id":249354},{"text":"Нобелевской","train_count":98,"old_ids":[19582,148280,77969,5301,149193],"score":392,"new_id":249355},{"text":"Последний","train_count":196,"old_ids":[16362,158537,149623],"score":392,"new_id":249356},{"text":"Россию","train_count":196,"old_ids":[32392,22078,157928],"score":392,"new_id":249357},{"text":"западу","train_count":196,"old_ids":[148397,148414,148660],"score":392,"new_id":249358},{"text":"император","train_count":196,"old_ids":[15630,149208,55389],"score":392,"new_id":249359},{"text":"осуществляется","train_count":196,"old_ids":[22078,82577,191535],"score":392,"new_id":249360},{"text":"раунде","train_count":196,"old_ids":[21441,33065,95525],"score":392,"new_id":249361},{"text":"Калифорния","train_count":130,"old_ids":[25497,45679,151385,155012],"score":390,"new_id":249362},{"text":"границах","train_count":130,"old_ids":[14942,91872,1424,10161],"score":390,"new_id":249363},{"text":"канадский","train_count":195,"old_ids":[179944,6733,149800],"score":390,"new_id":249364},{"text":"крупные","train_count":195,"old_ids":[4640,149568,41546],"score":390,"new_id":249365},{"text":"немецкой","train_count":195,"old_ids":[76069,218942,148554],"score":390,"new_id":249366},{"text":"одновременно","train_count":390,"old_ids":[148277,161523],"score":390,"new_id":249367},{"text":"первоначально","train_count":195,"old_ids":[149208,148331,167193],"score":390,"new_id":249368},{"text":"подписал","train_count":195,"old_ids":[155908,28696,15481],"score":390,"new_id":249369},{"text":"семьи","train_count":390,"old_ids":[159707,156610],"score":390,"new_id":249370},{"text":"сохранились","train_count":130,"old_ids":[148454,10161,149926,150938],"score":390,"new_id":249371},{"text":"установлен","train_count":195,"old_ids":[3652,148722,58361],"score":390,"new_id":249372},{"text":"строительства","train_count":389,"old_ids":[148943,158269],"score":389,"new_id":249373},{"text":"команде","train_count":194,"old_ids":[148558,72884,1450],"score":388,"new_id":249374},{"text":"покинул","train_count":194,"old_ids":[148428,16256,177447],"score":388,"new_id":249375},{"text":"успех","train_count":194,"old_ids":[3652,31608,93834],"score":388,"new_id":249376},{"text":"исторический","train_count":129,"old_ids":[32884,8789,77436,149537],"score":387,"new_id":249377},{"text":"множество","train_count":387,"old_ids":[178416,242702],"score":387,"new_id":249378},{"text":"оказалась","train_count":129,"old_ids":[1403,148330,15481,152993],"score":387,"new_id":249379},{"text":"Африки","train_count":193,"old_ids":[34960,162684,16256],"score":386,"new_id":249380},{"text":"Роман","train_count":193,"old_ids":[32392,148288,7105],"score":386,"new_id":249381},{"text":"данный","train_count":193,"old_ids":[6733,26269,11796],"score":386,"new_id":249382},{"text":"изображения","train_count":193,"old_ids":[39427,150065,150282],"score":386,"new_id":249383},{"text":"объектов","train_count":193,"old_ids":[148280,152743,148779],"score":386,"new_id":249384},{"text":"президент","train_count":386,"old_ids":[149790,235531],"score":386,"new_id":249385},{"text":"сестра","train_count":193,"old_ids":[2175,34981,21441],"score":386,"new_id":249386},{"text":"Некоторые","train_count":385,"old_ids":[60508,178717],"score":385,"new_id":249387},{"text":"Описание","train_count":385,"old_ids":[19736,90397],"score":385,"new_id":249388},{"text":"городов","train_count":385,"old_ids":[152689,148275],"score":385,"new_id":249389},{"text":"достаточно","train_count":385,"old_ids":[230530,153361],"score":385,"new_id":249390},{"text":"провёл","train_count":385,"old_ids":[161800,162292],"score":385,"new_id":249391},{"text":"Израиля","train_count":128,"old_ids":[29189,156494,1739,13302],"score":384,"new_id":249392},{"text":"Книга","train_count":192,"old_ids":[25497,21918,148369],"score":384,"new_id":249393},{"text":"возглавил","train_count":192,"old_ids":[150348,149809,151631],"score":384,"new_id":249394},{"text":"евреев","train_count":192,"old_ids":[148380,174005,5301],"score":384,"new_id":249395},{"text":"занимается","train_count":128,"old_ids":[8688,13603,6240,64903],"score":384,"new_id":249396},{"text":"командовал","train_count":128,"old_ids":[148558,72884,148275,15481],"score":384,"new_id":249397},{"text":"располагается","train_count":192,"old_ids":[217455,151286,64903],"score":384,"new_id":249398},{"text":"программы","train_count":383,"old_ids":[224012,148706],"score":383,"new_id":249399},{"text":"всегда","train_count":382,"old_ids":[5301,167669],"score":382,"new_id":249400},{"text":"любовь","train_count":191,"old_ids":[35587,155489,4669],"score":382,"new_id":249401},{"text":"мае","train_count":382,"old_ids":[66050,1450],"score":382,"new_id":249402},{"text":"оружие","train_count":191,"old_ids":[8789,54025,47552],"score":382,"new_id":249403},{"text":"сборник","train_count":191,"old_ids":[2175,148913,67494],"score":382,"new_id":249404},{"text":"епископа","train_count":127,"old_ids":[1450,28696,45527,148414],"score":381,"new_id":249405},{"text":"звезды","train_count":127,"old_ids":[8688,5301,148377,149055],"score":381,"new_id":249406},{"text":"известен","train_count":381,"old_ids":[244718,5092],"score":381,"new_id":249407},{"text":"Анхальт","train_count":95,"old_ids":[34960,2040,10161,36771,1729],"score":380,"new_id":249408},{"text":"Белоруссия","train_count":95,"old_ids":[58297,11777,8789,41635,155511],"score":380,"new_id":249409},{"text":"голосов","train_count":190,"old_ids":[229476,22078,148275],"score":380,"new_id":249410},{"text":"округ","train_count":380,"old_ids":[148518,62215],"score":380,"new_id":249411},{"text":"получает","train_count":190,"old_ids":[148634,27448,26046],"score":380,"new_id":249412},{"text":"появляется","train_count":190,"old_ids":[7754,58255,153190],"score":380,"new_id":249413},{"text":"созданный","train_count":95,"old_ids":[2175,148303,6733,26269,11796],"score":380,"new_id":249414},{"text":"упразднён","train_count":95,"old_ids":[47174,148464,6733,2040,185709],"score":380,"new_id":249415},{"text":"игрок","train_count":379,"old_ids":[66951,155110],"score":379,"new_id":249416},{"text":"король","train_count":379,"old_ids":[153589,148319],"score":379,"new_id":249417},{"text":"Америке","train_count":189,"old_ids":[34960,155898,1450],"score":378,"new_id":249418},{"text":"Генерал","train_count":126,"old_ids":[35884,5092,7358,15481],"score":378,"new_id":249419},{"text":"Красной","train_count":189,"old_ids":[25497,148761,148317],"score":378,"new_id":249420},{"text":"Распространение","train_count":126,"old_ids":[225381,155611,33437,16669],"score":378,"new_id":249421},{"text":"восток","train_count":189,"old_ids":[5301,23514,148518],"score":378,"new_id":249422},{"text":"другим","train_count":378,"old_ids":[192548,15630],"score":378,"new_id":249423},{"text":"епархии","train_count":189,"old_ids":[148445,16763,218943],"score":378,"new_id":249424},{"text":"населённый","train_count":126,"old_ids":[13785,189921,42565,171096],"score":378,"new_id":249425},{"text":"условия","train_count":189,"old_ids":[41635,148569,37835],"score":378,"new_id":249426},{"text":"эпизод","train_count":126,"old_ids":[65131,7754,39427,148277],"score":378,"new_id":249427},{"text":"Первая","train_count":377,"old_ids":[95169,155044],"score":377,"new_id":249428},{"text":"море","train_count":377,"old_ids":[6240,150194],"score":377,"new_id":249429},{"text":"Богородицы","train_count":94,"old_ids":[58297,149095,149345,1739,155055],"score":376,"new_id":249430},{"text":"Греции","train_count":188,"old_ids":[35884,148279,54201],"score":376,"new_id":249431},{"text":"Италия","train_count":188,"old_ids":[29189,42074,150095],"score":376,"new_id":249432},{"text":"Преподаватели","train_count":188,"old_ids":[192416,161978,190978],"score":376,"new_id":249433},{"text":"самоуправления","train_count":94,"old_ids":[148629,148324,47174,148381,59595],"score":376,"new_id":249434},{"text":"главного","train_count":375,"old_ids":[149809,193490],"score":375,"new_id":249435},{"text":"переехала","train_count":125,"old_ids":[149208,47745,10161,92647],"score":375,"new_id":249436},{"text":"присвоено","train_count":125,"old_ids":[148489,13113,148331,69729],"score":375,"new_id":249437},{"text":"специальности","train_count":125,"old_ids":[155036,14666,36771,65360],"score":375,"new_id":249438},{"text":"Питер","train_count":187,"old_ids":[16362,9080,7358],"score":374,"new_id":249439},{"text":"доктор","train_count":374,"old_ids":[171228,35458],"score":374,"new_id":249440},{"text":"здания","train_count":374,"old_ids":[8688,158529],"score":374,"new_id":249441},{"text":"науки","train_count":374,"old_ids":[216424,16256],"score":374,"new_id":249442},{"text":"печати","train_count":187,"old_ids":[7754,53887,149165],"score":374,"new_id":249443},{"text":"порядке","train_count":187,"old_ids":[153624,93629,50800],"score":374,"new_id":249444},{"text":"службы","train_count":374,"old_ids":[150431,148812],"score":374,"new_id":249445},{"text":"температура","train_count":187,"old_ids":[170744,149208,154182],"score":374,"new_id":249446},{"text":"Применение","train_count":124,"old_ids":[51844,15630,5092,16669],"score":372,"new_id":249447},{"text":"аппарата","train_count":124,"old_ids":[1424,74359,16763,40710],"score":372,"new_id":249448},{"text":"кафедры","train_count":124,"old_ids":[12751,18768,148276,36747],"score":372,"new_id":249449},{"text":"клетки","train_count":186,"old_ids":[4640,149550,16256],"score":372,"new_id":249450},{"text":"материал","train_count":124,"old_ids":[6240,7721,149098,15481],"score":372,"new_id":249451},{"text":"муниципального","train_count":124,"old_ids":[148405,151744,7754,151302],"score":372,"new_id":249452},{"text":"называются","train_count":124,"old_ids":[13785,8688,152011,19937],"score":372,"new_id":249453},{"text":"основал","train_count":186,"old_ids":[22078,148366,15481],"score":372,"new_id":249454},{"text":"рассказывает","train_count":124,"old_ids":[2115,27473,148330,151991],"score":372,"new_id":249455},{"text":"родилась","train_count":186,"old_ids":[149003,61057,152993],"score":372,"new_id":249456},{"text":"слово","train_count":372,"old_ids":[169948,148331],"score":372,"new_id":249457},{"text":"художественной","train_count":124,"old_ids":[157951,149244,150198,148317],"score":372,"new_id":249458},{"text":"чемпионатах","train_count":93,"old_ids":[155356,148686,16806,40710,10161],"score":372,"new_id":249459},{"text":"Мария","train_count":371,"old_ids":[225978,37835],"score":371,"new_id":249460},{"text":"роман","train_count":371,"old_ids":[149633,7105],"score":371,"new_id":249461},{"text":"Эта","train_count":370,"old_ids":[89030,42074],"score":370,"new_id":249462},{"text":"машин","train_count":185,"old_ids":[6240,48989,18372],"score":370,"new_id":249463},{"text":"руководство","train_count":185,"old_ids":[19160,150463,148593],"score":370,"new_id":249464},{"text":"самолёт","train_count":185,"old_ids":[160002,148294,150396],"score":370,"new_id":249465},{"text":"селе","train_count":370,"old_ids":[2175,77969],"score":370,"new_id":249466},{"text":"счётом","train_count":185,"old_ids":[2175,180003,149127],"score":370,"new_id":249467},{"text":"училища","train_count":185,"old_ids":[27448,149037,159031],"score":370,"new_id":249468},{"text":"эту","train_count":370,"old_ids":[65131,85208],"score":370,"new_id":249469},{"text":"вокруг","train_count":369,"old_ids":[159496,62215],"score":369,"new_id":249470},{"text":"защитник","train_count":123,"old_ids":[148397,13739,9080,67494],"score":369,"new_id":249471},{"text":"использовал","train_count":123,"old_ids":[13113,183192,148675,15481],"score":369,"new_id":249472},{"text":"музыкантов","train_count":123,"old_ids":[148405,148707,151011,148779],"score":369,"new_id":249473},{"text":"специалистов","train_count":123,"old_ids":[155036,14666,45679,153305],"score":369,"new_id":249474},{"text":"умер","train_count":369,"old_ids":[37815,7358],"score":369,"new_id":249475},{"text":"фортепиано","train_count":123,"old_ids":[151385,48824,148686,149600],"score":369,"new_id":249476},{"text":"Спартак","train_count":184,"old_ids":[18727,151963,149338],"score":368,"new_id":249477},{"text":"Транспорт","train_count":184,"old_ids":[32868,33437,155295],"score":368,"new_id":249478},{"text":"называли","train_count":184,"old_ids":[13785,152953,45679],"score":368,"new_id":249479},{"text":"соревнованиях","train_count":92,"old_ids":[154399,148380,148366,36916,10161],"score":368,"new_id":249480},{"text":"Таким","train_count":367,"old_ids":[217720,15630],"score":367,"new_id":249481},{"text":"играл","train_count":367,"old_ids":[66951,197362],"score":367,"new_id":249482},{"text":"повят","train_count":367,"old_ids":[150772,86128],"score":367,"new_id":249483},{"text":"различные","train_count":367,"old_ids":[148464,213351],"score":367,"new_id":249484},{"text":"Болгарии","train_count":122,"old_ids":[58297,148294,151937,44633],"score":366,"new_id":249485},{"text":"Ирландии","train_count":122,"old_ids":[29189,2115,200641,44633],"score":366,"new_id":249486},{"text":"Норвегии","train_count":122,"old_ids":[19582,8789,148537,151229],"score":366,"new_id":249487},{"text":"Церкви","train_count":183,"old_ids":[89602,7358,155974],"score":366,"new_id":249488},{"text":"будущего","train_count":183,"old_ids":[156543,40794,148702],"score":366,"new_id":249489},{"text":"имеются","train_count":183,"old_ids":[15630,1450,149124],"score":366,"new_id":249490},{"text":"княжества","train_count":122,"old_ids":[4640,37855,16467,150167],"score":366,"new_id":249491},{"text":"наименование","train_count":122,"old_ids":[13785,15630,69729,150544],"score":366,"new_id":249492},{"text":"опубликовал","train_count":122,"old_ids":[148561,149090,148336,15481],"score":366,"new_id":249493},{"text":"строительство","train_count":366,"old_ids":[148943,158663],"score":366,"new_id":249494},{"text":"художника","train_count":183,"old_ids":[157951,149244,84480],"score":366,"new_id":249495},{"text":"Тем","train_count":365,"old_ids":[32868,148281],"score":365,"new_id":249496},{"text":"Беларуси","train_count":91,"old_ids":[58297,11777,16763,41635,1739],"score":364,"new_id":249497},{"text":"Станислава","train_count":91,"old_ids":[71000,7105,13113,29354,90219],"score":364,"new_id":249498},{"text":"Югославии","train_count":91,"old_ids":[168959,14942,22078,150800,44633],"score":364,"new_id":249499},{"text":"играх","train_count":364,"old_ids":[154524,10161],"score":364,"new_id":249500},{"text":"неожиданно","train_count":91,"old_ids":[76069,149244,45587,7105,13292],"score":364,"new_id":249501},{"text":"природы","train_count":182,"old_ids":[148895,149003,4403],"score":364,"new_id":249502},{"text":"реформы","train_count":182,"old_ids":[148279,54724,4403],"score":364,"new_id":249503},{"text":"состоял","train_count":182,"old_ids":[2175,152544,2941],"score":364,"new_id":249504},{"text":"Алексея","train_count":121,"old_ids":[34960,148470,148938,4096],"score":363,"new_id":249505},{"text":"Британской","train_count":121,"old_ids":[58297,149454,7105,149193],"score":363,"new_id":249506},{"text":"Гран","train_count":363,"old_ids":[35884,33437],"score":363,"new_id":249507},{"text":"Почётные","train_count":121,"old_ids":[16362,36103,150396,41546],"score":363,"new_id":249508},{"text":"депутатом","train_count":121,"old_ids":[95525,198375,7721,148288],"score":363,"new_id":249509},{"text":"термин","train_count":363,"old_ids":[55055,150704],"score":363,"new_id":249510},{"text":"фестивале","train_count":121,"old_ids":[18768,34981,26125,150188],"score":363,"new_id":249511},{"text":"Национальный","train_count":181,"old_ids":[19582,158048,159069],"score":362,"new_id":249512},{"text":"Обычно","train_count":181,"old_ids":[70749,4403,159037],"score":362,"new_id":249513},{"text":"выходит","train_count":181,"old_ids":[155037,148277,9080],"score":362,"new_id":249514},{"text":"гвардии","train_count":181,"old_ids":[14942,148972,159712],"score":362,"new_id":249515},{"text":"начальником","train_count":181,"old_ids":[29196,36771,159659],"score":362,"new_id":249516},{"text":"возможно","train_count":361,"old_ids":[150348,153165],"score":361,"new_id":249517},{"text":"пять","train_count":361,"old_ids":[7754,90638],"score":361,"new_id":249518},{"text":"Летом","train_count":180,"old_ids":[61757,7921,148288],"score":360,"new_id":249519},{"text":"артистка","train_count":180,"old_ids":[16763,21797,179178],"score":360,"new_id":249520},{"text":"борьбе","train_count":180,"old_ids":[148913,4669,149056],"score":360,"new_id":249521},{"text":"деятельностью","train_count":120,"old_ids":[95525,86128,24623,150616],"score":360,"new_id":249522},{"text":"драматический","train_count":90,"old_ids":[151486,6240,7721,77436,149537],"score":360,"new_id":249523},{"text":"окрестностях","train_count":120,"old_ids":[148518,2115,34981,202246],"score":360,"new_id":249524},{"text":"приняла","train_count":120,"old_ids":[148489,18372,4096,29354],"score":360,"new_id":249525},{"text":"рядом","train_count":360,"old_ids":[151446,148288],"score":360,"new_id":249526},{"text":"свойства","train_count":180,"old_ids":[2175,159035,89771],"score":360,"new_id":249527},{"text":"фестиваль","train_count":120,"old_ids":[18768,34981,26125,36771],"score":360,"new_id":249528},{"text":"экземпляров","train_count":120,"old_ids":[192809,154161,162948,148327],"score":360,"new_id":249529},{"text":"главе","train_count":359,"old_ids":[149809,148537],"score":359,"new_id":249530},{"text":"избран","train_count":359,"old_ids":[39427,174188],"score":359,"new_id":249531},{"text":"люди","train_count":359,"old_ids":[35587,148467],"score":359,"new_id":249532},{"text":"Великого","train_count":179,"old_ids":[15729,58503,149075],"score":358,"new_id":249533},{"text":"Медаль","train_count":179,"old_ids":[36736,148276,36771],"score":358,"new_id":249534},{"text":"Первые","train_count":358,"old_ids":[95169,153156],"score":358,"new_id":249535},{"text":"Фамилия","train_count":179,"old_ids":[52337,150190,150095],"score":358,"new_id":249536},{"text":"идеи","train_count":179,"old_ids":[45587,1450,1739],"score":358,"new_id":249537},{"text":"испанский","train_count":179,"old_ids":[13113,181841,149800],"score":358,"new_id":249538},{"text":"персонаж","train_count":179,"old_ids":[149208,152590,16467],"score":358,"new_id":249539},{"text":"Художники","train_count":119,"old_ids":[150853,34723,149244,149665],"score":357,"new_id":249540},{"text":"британской","train_count":119,"old_ids":[9629,149454,7105,149193],"score":357,"new_id":249541},{"text":"живописец","train_count":119,"old_ids":[151303,1403,28696,154800],"score":357,"new_id":249542},{"text":"отношения","train_count":357,"old_ids":[152928,149656],"score":357,"new_id":249543},{"text":"офицеров","train_count":119,"old_ids":[1403,148594,10064,158732],"score":357,"new_id":249544},{"text":"созданию","train_count":119,"old_ids":[2175,148303,6733,87252],"score":357,"new_id":249545},{"text":"Павла","train_count":178,"old_ids":[16362,148292,29354],"score":356,"new_id":249546},{"text":"Русской","train_count":356,"old_ids":[221073,149193],"score":356,"new_id":249547},{"text":"объявил","train_count":178,"old_ids":[148280,156545,151631],"score":356,"new_id":249548},{"text":"половины","train_count":178,"old_ids":[220495,148341,12014],"score":356,"new_id":249549},{"text":"работе","train_count":356,"old_ids":[149100,1450],"score":356,"new_id":249550},{"text":"районов","train_count":356,"old_ids":[180093,152740],"score":356,"new_id":249551},{"text":"сериала","train_count":178,"old_ids":[2175,149098,92647],"score":356,"new_id":249552},{"text":"территориальное","train_count":89,"old_ids":[55055,149454,150140,36771,45480],"score":356,"new_id":249553},{"text":"одном","train_count":355,"old_ids":[148277,148524],"score":355,"new_id":249554},{"text":"самого","train_count":355,"old_ids":[148629,160860],"score":355,"new_id":249555},{"text":"Существует","train_count":177,"old_ids":[18727,82577,69632],"score":354,"new_id":249556},{"text":"актёра","train_count":118,"old_ids":[148318,1729,42565,21441],"score":354,"new_id":249557},{"text":"большинства","train_count":177,"old_ids":[9629,148701,240966],"score":354,"new_id":249558},{"text":"дворец","train_count":177,"old_ids":[6733,149086,154800],"score":354,"new_id":249559},{"text":"находились","train_count":177,"old_ids":[13785,205240,148685],"score":354,"new_id":249560},{"text":"основным","train_count":177,"old_ids":[22078,148366,148540],"score":354,"new_id":249561},{"text":"пассажиров","train_count":118,"old_ids":[7754,27473,149096,149591],"score":354,"new_id":249562},{"text":"рассматривается","train_count":59,"old_ids":[2115,27473,6240,7721,2115,26125,64903],"score":354,"new_id":249563},{"text":"снова","train_count":354,"old_ids":[2175,153213],"score":354,"new_id":249564},{"text":"являлась","train_count":118,"old_ids":[4096,5301,13302,155063],"score":354,"new_id":249565},{"text":"являлись","train_count":118,"old_ids":[4096,5301,13302,150938],"score":354,"new_id":249566},{"text":"реке","train_count":353,"old_ids":[148279,50800],"score":353,"new_id":249567},{"text":"Воронежской","train_count":88,"old_ids":[15729,8789,148917,16467,149193],"score":352,"new_id":249568},{"text":"Станция","train_count":176,"old_ids":[71000,7105,68335],"score":352,"new_id":249569},{"text":"Фильм","train_count":176,"old_ids":[52337,64257,6240],"score":352,"new_id":249570},{"text":"воздуха","train_count":176,"old_ids":[150348,148660,150532],"score":352,"new_id":249571},{"text":"композиции","train_count":176,"old_ids":[148558,161339,54201],"score":352,"new_id":249572},{"text":"конкурса","train_count":176,"old_ids":[150270,150183,148629],"score":352,"new_id":249573},{"text":"сразу","train_count":352,"old_ids":[2175,155252],"score":352,"new_id":249574},{"text":"центре","train_count":352,"old_ids":[151971,148279],"score":352,"new_id":249575},{"text":"Будучи","train_count":117,"old_ids":[58297,34723,27448,1739],"score":351,"new_id":249576},{"text":"Роберта","train_count":117,"old_ids":[32392,148280,7358,42074],"score":351,"new_id":249577},{"text":"директоров","train_count":117,"old_ids":[6733,74990,148301,155610],"score":351,"new_id":249578},{"text":"назывался","train_count":117,"old_ids":[13785,152953,15481,19937],"score":351,"new_id":249579},{"text":"псевдоним","train_count":117,"old_ids":[7754,215194,182503,15630],"score":351,"new_id":249580},{"text":"русских","train_count":351,"old_ids":[152528,150584],"score":351,"new_id":249581},{"text":"футбола","train_count":117,"old_ids":[18768,36183,148421,29354],"score":351,"new_id":249582},{"text":"высоте","train_count":175,"old_ids":[148705,2175,149744],"score":350,"new_id":249583},{"text":"исключительно","train_count":175,"old_ids":[13113,49689,151269],"score":350,"new_id":249584},{"text":"корпус","train_count":350,"old_ids":[153589,149447],"score":350,"new_id":249585},{"text":"писательница","train_count":175,"old_ids":[28696,57868,153313],"score":350,"new_id":249586},{"text":"преподавал","train_count":175,"old_ids":[149790,161978,151019],"score":350,"new_id":249587},{"text":"рабочий","train_count":175,"old_ids":[21441,151293,42387],"score":350,"new_id":249588},{"text":"состоялось","train_count":175,"old_ids":[2175,152544,153174],"score":350,"new_id":249589},{"text":"такой","train_count":349,"old_ids":[42074,148554],"score":349,"new_id":249590},{"text":"Буревестник","train_count":87,"old_ids":[58297,53555,148380,34981,67494],"score":348,"new_id":249591},{"text":"Канады","train_count":174,"old_ids":[25497,148841,149055],"score":348,"new_id":249592},{"text":"Уильям","train_count":174,"old_ids":[39243,64257,150686],"score":348,"new_id":249593},{"text":"баскетболист","train_count":87,"old_ids":[9629,17417,152417,157759,6389],"score":348,"new_id":249594},{"text":"внутренней","train_count":116,"old_ids":[5301,154175,148516,148820],"score":348,"new_id":249595},{"text":"должны","train_count":348,"old_ids":[159172,176125],"score":348,"new_id":249596},{"text":"километров","train_count":174,"old_ids":[16256,37483,163371],"score":348,"new_id":249597},{"text":"клубы","train_count":174,"old_ids":[4640,152887,4403],"score":348,"new_id":249598},{"text":"книгу","train_count":174,"old_ids":[4640,21918,148994],"score":348,"new_id":249599},{"text":"крупный","train_count":174,"old_ids":[4640,149568,32401],"score":348,"new_id":249600},{"text":"построенный","train_count":87,"old_ids":[7754,23514,148283,5092,32401],"score":348,"new_id":249601},{"text":"распространения","train_count":116,"old_ids":[148761,155611,33437,17941],"score":348,"new_id":249602},{"text":"шахматист","train_count":87,"old_ids":[150011,10161,6240,7721,32884],"score":348,"new_id":249603},{"text":"Члены","train_count":347,"old_ids":[70345,154641],"score":347,"new_id":249604},{"text":"известный","train_count":347,"old_ids":[244718,32401],"score":347,"new_id":249605},{"text":"млрд","train_count":347,"old_ids":[220083,159881],"score":347,"new_id":249606},{"text":"вопросам","train_count":173,"old_ids":[148331,160383,148357],"score":346,"new_id":249607},{"text":"вышла","train_count":346,"old_ids":[148705,153870],"score":346,"new_id":249608},{"text":"должности","train_count":346,"old_ids":[159172,153925],"score":346,"new_id":249609},{"text":"заняла","train_count":173,"old_ids":[193134,4096,29354],"score":346,"new_id":249610},{"text":"которому","train_count":346,"old_ids":[151549,149169],"score":346,"new_id":249611},{"text":"мнению","train_count":346,"old_ids":[6240,155731],"score":346,"new_id":249612},{"text":"полка","train_count":346,"old_ids":[148634,12751],"score":346,"new_id":249613},{"text":"расположено","train_count":173,"old_ids":[148761,220495,196782],"score":346,"new_id":249614},{"text":"составляют","train_count":173,"old_ids":[2175,150838,152984],"score":346,"new_id":249615},{"text":"функции","train_count":346,"old_ids":[210385,1739],"score":346,"new_id":249616},{"text":"одноимённой","train_count":115,"old_ids":[164551,15630,42565,170968],"score":345,"new_id":249617},{"text":"появилось","train_count":115,"old_ids":[7754,58255,151631,95306],"score":345,"new_id":249618},{"text":"сына","train_count":345,"old_ids":[54330,13785],"score":345,"new_id":249619},{"text":"Ломоносова","train_count":86,"old_ids":[61757,148288,1403,13292,180298],"score":344,"new_id":249620},{"text":"всему","train_count":172,"old_ids":[5301,2175,149581],"score":344,"new_id":249621},{"text":"гимназии","train_count":172,"old_ids":[185715,13785,155585],"score":344,"new_id":249622},{"text":"золотой","train_count":172,"old_ids":[227597,12666,148293],"score":344,"new_id":249623},{"text":"кампании","train_count":172,"old_ids":[12751,153260,149091],"score":344,"new_id":249624},{"text":"месяцев","train_count":344,"old_ids":[164065,152242],"score":344,"new_id":249625},{"text":"предыдущих","train_count":86,"old_ids":[177917,4403,6733,40794,64392],"score":344,"new_id":249626},{"text":"средств","train_count":344,"old_ids":[174301,19587],"score":344,"new_id":249627},{"text":"численность","train_count":172,"old_ids":[153683,58361,58245],"score":344,"new_id":249628},{"text":"Академические","train_count":114,"old_ids":[34960,154408,6240,168329],"score":342,"new_id":249629},{"text":"Генерального","train_count":114,"old_ids":[35884,5092,7358,151302],"score":342,"new_id":249630},{"text":"Лауреат","train_count":171,"old_ids":[196219,201647,7721],"score":342,"new_id":249631},{"text":"Молдавии","train_count":114,"old_ids":[36736,148294,38183,210740],"score":342,"new_id":249632},{"text":"белые","train_count":171,"old_ids":[9629,11777,48933],"score":342,"new_id":249633},{"text":"дворца","train_count":171,"old_ids":[6733,149086,83895],"score":342,"new_id":249634},{"text":"документов","train_count":114,"old_ids":[148551,34083,89315,148275],"score":342,"new_id":249635},{"text":"документы","train_count":114,"old_ids":[148551,34083,89315,4403],"score":342,"new_id":249636},{"text":"игроков","train_count":171,"old_ids":[66951,148283,148336],"score":342,"new_id":249637},{"text":"назначения","train_count":171,"old_ids":[13785,149340,17941],"score":342,"new_id":249638},{"text":"определения","train_count":171,"old_ids":[148561,148279,154426],"score":342,"new_id":249639},{"text":"открытия","train_count":171,"old_ids":[12666,91201,152149],"score":342,"new_id":249640},{"text":"положение","train_count":342,"old_ids":[220495,150307],"score":342,"new_id":249641},{"text":"происхождение","train_count":171,"old_ids":[149490,13113,193219],"score":342,"new_id":249642},{"text":"путь","train_count":342,"old_ids":[7754,79860],"score":342,"new_id":249643},{"text":"скульптор","train_count":171,"old_ids":[2175,168317,35458],"score":342,"new_id":249644},{"text":"правительства","train_count":341,"old_ids":[148607,158269],"score":341,"new_id":249645},{"text":"Андреевское","train_count":68,"old_ids":[34960,2040,60659,47745,5301,170391],"score":340,"new_id":249646},{"text":"академик","train_count":170,"old_ids":[1424,154408,201286],"score":340,"new_id":249647},{"text":"американской","train_count":170,"old_ids":[1424,155899,149193],"score":340,"new_id":249648},{"text":"армией","train_count":170,"old_ids":[16763,6240,158718],"score":340,"new_id":249649},{"text":"железнодорожного","train_count":85,"old_ids":[16467,77969,8688,166815,155552],"score":340,"new_id":249650},{"text":"нужно","train_count":170,"old_ids":[2040,54025,13292],"score":340,"new_id":249651},{"text":"обучение","train_count":170,"old_ids":[148280,27448,16669],"score":340,"new_id":249652},{"text":"основные","train_count":170,"old_ids":[22078,148366,41546],"score":340,"new_id":249653},{"text":"сообщения","train_count":170,"old_ids":[148454,149350,17941],"score":340,"new_id":249654},{"text":"Берлине","train_count":113,"old_ids":[58297,7358,10750,76069],"score":339,"new_id":249655},{"text":"Государственный","train_count":113,"old_ids":[35884,22078,150968,159759],"score":339,"new_id":249656},{"text":"Скончался","train_count":113,"old_ids":[18727,150270,198597,19937],"score":339,"new_id":249657},{"text":"Яковлевич","train_count":113,"old_ids":[82440,148336,151186,31889],"score":339,"new_id":249658},{"text":"лиги","train_count":339,"old_ids":[10750,148388],"score":339,"new_id":249659},{"text":"стороне","train_count":339,"old_ids":[152950,148917],"score":339,"new_id":249660},{"text":"Большого","train_count":169,"old_ids":[58297,148701,148449],"score":338,"new_id":249661},{"text":"Имеет","train_count":169,"old_ids":[29189,148446,7921],"score":338,"new_id":249662},{"text":"марте","train_count":338,"old_ids":[173719,48824],"score":338,"new_id":249663},{"text":"община","train_count":338,"old_ids":[149350,66353],"score":338,"new_id":249664},{"text":"статьи","train_count":169,"old_ids":[6389,17460,1739],"score":338,"new_id":249665},{"text":"стране","train_count":338,"old_ids":[155690,1450],"score":338,"new_id":249666},{"text":"топоним","train_count":169,"old_ids":[23858,154767,15630],"score":338,"new_id":249667},{"text":"характеристики","train_count":169,"old_ids":[229103,149463,156911],"score":338,"new_id":249668},{"text":"женщин","train_count":337,"old_ids":[148942,198323],"score":337,"new_id":249669},{"text":"исследования","train_count":337,"old_ids":[13113,196037],"score":337,"new_id":249670},{"text":"история","train_count":337,"old_ids":[32884,154617],"score":337,"new_id":249671},{"text":"Евгения","train_count":112,"old_ids":[53241,5301,14942,17941],"score":336,"new_id":249672},{"text":"Золотая","train_count":112,"old_ids":[34279,148294,12666,34226],"score":336,"new_id":249673},{"text":"Иоанн","train_count":112,"old_ids":[29189,1403,7105,2040],"score":336,"new_id":249674},{"text":"кафедрой","train_count":112,"old_ids":[12751,151398,60659,148293],"score":336,"new_id":249675},{"text":"матча","train_count":168,"old_ids":[6240,7721,148659],"score":336,"new_id":249676},{"text":"неподалёку","train_count":84,"old_ids":[76069,148428,39596,42565,34083],"score":336,"new_id":249677},{"text":"быстро","train_count":335,"old_ids":[148812,148943],"score":335,"new_id":249678},{"text":"имели","train_count":335,"old_ids":[15630,58503],"score":335,"new_id":249679},{"text":"общей","train_count":335,"old_ids":[149350,148298],"score":335,"new_id":249680},{"text":"первое","train_count":335,"old_ids":[149208,159117],"score":335,"new_id":249681},{"text":"Петра","train_count":334,"old_ids":[154192,21441],"score":334,"new_id":249682},{"text":"Сайт","train_count":167,"old_ids":[18727,148390,1729],"score":334,"new_id":249683},{"text":"кавалер","train_count":167,"old_ids":[12751,151019,7358],"score":334,"new_id":249684},{"text":"конкурсе","train_count":167,"old_ids":[150270,150183,148938],"score":334,"new_id":249685},{"text":"октябре","train_count":334,"old_ids":[148518,165358],"score":334,"new_id":249686},{"text":"перевод","train_count":167,"old_ids":[149208,148380,148277],"score":334,"new_id":249687},{"text":"хутор","train_count":167,"old_ids":[10161,3652,35458],"score":334,"new_id":249688},{"text":"Формула","train_count":111,"old_ids":[52337,30206,3652,29354],"score":333,"new_id":249689},{"text":"взаимодействия","train_count":111,"old_ids":[5301,148397,149076,236199],"score":333,"new_id":249690},{"text":"защитил","train_count":111,"old_ids":[148397,13739,9080,61057],"score":333,"new_id":249691},{"text":"объединение","train_count":111,"old_ids":[148280,32482,152021,16669],"score":333,"new_id":249692},{"text":"посредством","train_count":111,"old_ids":[7754,22078,148420,152225],"score":333,"new_id":249693},{"text":"путём","train_count":333,"old_ids":[198375,151666],"score":333,"new_id":249694},{"text":"рассказов","train_count":111,"old_ids":[2115,27473,148330,148275],"score":333,"new_id":249695},{"text":"футбольной","train_count":111,"old_ids":[18768,36183,170946,148317],"score":333,"new_id":249696},{"text":"ярмарочная","train_count":111,"old_ids":[163466,173719,36103,42308],"score":333,"new_id":249697},{"text":"Андреевка","train_count":83,"old_ids":[34960,2040,60659,47745,172637],"score":332,"new_id":249698},{"text":"Нидерландов","train_count":83,"old_ids":[19582,45587,7358,200641,148275],"score":332,"new_id":249699},{"text":"Нидерланды","train_count":83,"old_ids":[19582,45587,7358,200641,4403],"score":332,"new_id":249700},{"text":"Советов","train_count":166,"old_ids":[173518,45405,148275],"score":332,"new_id":249701},{"text":"движение","train_count":332,"old_ids":[155218,150307],"score":332,"new_id":249702},{"text":"нефти","train_count":166,"old_ids":[76069,18768,21797],"score":332,"new_id":249703},{"text":"попал","train_count":166,"old_ids":[148428,7754,15481],"score":332,"new_id":249704},{"text":"проживали","train_count":166,"old_ids":[149490,151303,45679],"score":332,"new_id":249705},{"text":"профессиональный","train_count":166,"old_ids":[149490,165373,159069],"score":332,"new_id":249706},{"text":"словарь","train_count":166,"old_ids":[154703,16763,4669],"score":332,"new_id":249707},{"text":"существования","train_count":166,"old_ids":[2175,154659,155024],"score":332,"new_id":249708},{"text":"версия","train_count":331,"old_ids":[46576,155511],"score":331,"new_id":249709},{"text":"последний","train_count":331,"old_ids":[167014,149623],"score":331,"new_id":249710},{"text":"Бориса","train_count":110,"old_ids":[58297,8789,13113,1424],"score":330,"new_id":249711},{"text":"Университета","train_count":110,"old_ids":[39243,150439,152898,149005],"score":330,"new_id":249712},{"text":"академию","train_count":110,"old_ids":[1424,154408,61628,11577],"score":330,"new_id":249713},{"text":"боксёр","train_count":165,"old_ids":[148421,170821,156204],"score":330,"new_id":249714},{"text":"изучения","train_count":165,"old_ids":[39427,27448,17941],"score":330,"new_id":249715},{"text":"интервью","train_count":165,"old_ids":[18372,55055,162497],"score":330,"new_id":249716},{"text":"комитета","train_count":330,"old_ids":[148558,166907],"score":330,"new_id":249717},{"text":"оркестр","train_count":110,"old_ids":[8789,4640,34981,2115],"score":330,"new_id":249718},{"text":"официальном","train_count":110,"old_ids":[1403,149243,36771,148524],"score":330,"new_id":249719},{"text":"принимать","train_count":110,"old_ids":[148489,18372,15630,17460],"score":330,"new_id":249720},{"text":"систем","train_count":330,"old_ids":[167641,148281],"score":330,"new_id":249721},{"text":"числа","train_count":330,"old_ids":[153683,29354],"score":330,"new_id":249722},{"text":"комплекс","train_count":329,"old_ids":[166349,151205],"score":329,"new_id":249723},{"text":"партия","train_count":329,"old_ids":[151963,152149],"score":329,"new_id":249724},{"text":"рождения","train_count":329,"old_ids":[148283,149880],"score":329,"new_id":249725},{"text":"Джозеф","train_count":82,"old_ids":[23355,16467,148303,1450,18768],"score":328,"new_id":249726},{"text":"Международного","train_count":164,"old_ids":[181834,151896,148325],"score":328,"new_id":249727},{"text":"Турции","train_count":164,"old_ids":[32868,53555,54201],"score":328,"new_id":249728},{"text":"Харьковской","train_count":82,"old_ids":[150853,16763,4669,148336,149193],"score":328,"new_id":249729},{"text":"виды","train_count":328,"old_ids":[152833,4403],"score":328,"new_id":249730},{"text":"отправился","train_count":164,"old_ids":[12666,148607,154838],"score":328,"new_id":249731},{"text":"постановлением","train_count":82,"old_ids":[7754,23514,7105,148275,157264],"score":328,"new_id":249732},{"text":"возвращается","train_count":109,"old_ids":[150348,5301,50022,64903],"score":327,"new_id":249733},{"text":"используя","train_count":109,"old_ids":[13113,183192,148995,4096],"score":327,"new_id":249734},{"text":"матерью","train_count":109,"old_ids":[6240,7721,7358,84920],"score":327,"new_id":249735},{"text":"одну","train_count":327,"old_ids":[148277,148776],"score":327,"new_id":249736},{"text":"пока","train_count":327,"old_ids":[148428,12751],"score":327,"new_id":249737},{"text":"Риме","train_count":163,"old_ids":[32392,15630,1450],"score":326,"new_id":249738},{"text":"Румынии","train_count":163,"old_ids":[32392,168823,155016],"score":326,"new_id":249739},{"text":"бюро","train_count":163,"old_ids":[9629,11577,148283],"score":326,"new_id":249740},{"text":"заявил","train_count":163,"old_ids":[8688,34226,151631],"score":326,"new_id":249741},{"text":"клубов","train_count":163,"old_ids":[4640,152887,148275],"score":326,"new_id":249742},{"text":"объекта","train_count":163,"old_ids":[148280,152743,42074],"score":326,"new_id":249743},{"text":"основатель","train_count":163,"old_ids":[22078,148366,57868],"score":326,"new_id":249744},{"text":"Упразднённые","train_count":65,"old_ids":[39243,7754,148464,6733,194972,155130],"score":325,"new_id":249745},{"text":"административных","train_count":65,"old_ids":[149099,150704,32884,77779,26125,42240],"score":325,"new_id":249746},{"text":"теории","train_count":325,"old_ids":[48824,152597],"score":325,"new_id":249747},{"text":"Азербайджана","train_count":81,"old_ids":[34960,158401,148390,151828,148841],"score":324,"new_id":249748},{"text":"Виталий","train_count":108,"old_ids":[15729,9080,45679,11796],"score":324,"new_id":249749},{"text":"Семёнович","train_count":108,"old_ids":[18727,148281,42565,192611],"score":324,"new_id":249750},{"text":"Южной","train_count":324,"old_ids":[168959,159484],"score":324,"new_id":249751},{"text":"английская","train_count":108,"old_ids":[7105,151203,11796,152141],"score":324,"new_id":249752},{"text":"выиграла","train_count":108,"old_ids":[148705,66951,2115,92647],"score":324,"new_id":249753},{"text":"гражданин","train_count":108,"old_ids":[148510,148343,13603,2040],"score":324,"new_id":249754},{"text":"математических","train_count":81,"old_ids":[6240,7721,148281,7721,151308],"score":324,"new_id":249755},{"text":"особенности","train_count":162,"old_ids":[149034,5092,65360],"score":324,"new_id":249756},{"text":"произведения","train_count":324,"old_ids":[161612,151536],"score":324,"new_id":249757},{"text":"пункта","train_count":162,"old_ids":[7754,38629,42074],"score":324,"new_id":249758},{"text":"совершил","train_count":162,"old_ids":[148454,46576,194788],"score":324,"new_id":249759},{"text":"сторону","train_count":324,"old_ids":[152950,151281],"score":324,"new_id":249760},{"text":"Институт","train_count":161,"old_ids":[161484,158693,36183],"score":322,"new_id":249761},{"text":"городских","train_count":322,"old_ids":[152689,150584],"score":322,"new_id":249762},{"text":"договор","train_count":161,"old_ids":[6733,148449,149086],"score":322,"new_id":249763},{"text":"крайне","train_count":161,"old_ids":[148998,11796,76069],"score":322,"new_id":249764},{"text":"сборную","train_count":161,"old_ids":[2175,148913,73781],"score":322,"new_id":249765},{"text":"четырёх","train_count":322,"old_ids":[243429,198071],"score":322,"new_id":249766},{"text":"штат","train_count":322,"old_ids":[184984,7721],"score":322,"new_id":249767},{"text":"Египта","train_count":107,"old_ids":[53241,14942,45768,42074],"score":321,"new_id":249768},{"text":"Колёня","train_count":107,"old_ids":[25497,148294,42565,37855],"score":321,"new_id":249769},{"text":"вооружения","train_count":107,"old_ids":[148331,8789,54025,17941],"score":321,"new_id":249770},{"text":"оно","train_count":321,"old_ids":[1403,13292],"score":321,"new_id":249771},{"text":"поездов","train_count":107,"old_ids":[7754,42317,8688,148716],"score":321,"new_id":249772},{"text":"территориальная","train_count":107,"old_ids":[55055,149454,150140,231448],"score":321,"new_id":249773},{"text":"Советской","train_count":160,"old_ids":[173518,45405,149193],"score":320,"new_id":249774},{"text":"Чехословакии","train_count":64,"old_ids":[70345,93834,22078,160162,16256,1739],"score":320,"new_id":249775},{"text":"владения","train_count":160,"old_ids":[5301,148606,17941],"score":320,"new_id":249776},{"text":"всю","train_count":320,"old_ids":[5301,164502],"score":320,"new_id":249777},{"text":"операций","train_count":160,"old_ids":[148561,7358,154512],"score":320,"new_id":249778},{"text":"офицер","train_count":160,"old_ids":[1403,148594,180269],"score":320,"new_id":249779},{"text":"подряд","train_count":160,"old_ids":[148428,60659,93629],"score":320,"new_id":249780},{"text":"самой","train_count":320,"old_ids":[160002,148293],"score":320,"new_id":249781},{"text":"судов","train_count":160,"old_ids":[2175,34723,148275],"score":320,"new_id":249782},{"text":"часов","train_count":320,"old_ids":[155183,148275],"score":320,"new_id":249783},{"text":"экономических","train_count":160,"old_ids":[65131,150192,151308],"score":320,"new_id":249784},{"text":"война","train_count":319,"old_ids":[159035,13785],"score":319,"new_id":249785},{"text":"Андрея","train_count":106,"old_ids":[34960,2040,156329,4096],"score":318,"new_id":249786},{"text":"Наталья","train_count":159,"old_ids":[19582,7721,157230],"score":318,"new_id":249787},{"text":"Почётный","train_count":106,"old_ids":[16362,36103,150396,32401],"score":318,"new_id":249788},{"text":"Северный","train_count":159,"old_ids":[181302,46576,32401],"score":318,"new_id":249789},{"text":"британская","train_count":106,"old_ids":[9629,149454,7105,152141],"score":318,"new_id":249790},{"text":"герцог","train_count":159,"old_ids":[162127,10064,149095],"score":318,"new_id":249791},{"text":"значительно","train_count":318,"old_ids":[149340,151269],"score":318,"new_id":249792},{"text":"компьютерных","train_count":106,"old_ids":[166349,84920,55055,42240],"score":318,"new_id":249793},{"text":"музея","train_count":159,"old_ids":[148405,152165,4096],"score":318,"new_id":249794},{"text":"начали","train_count":318,"old_ids":[29196,45679],"score":318,"new_id":249795},{"text":"распространены","train_count":106,"old_ids":[148761,155611,33437,75896],"score":318,"new_id":249796},{"text":"сигналов","train_count":106,"old_ids":[2175,66951,13785,148569],"score":318,"new_id":249797},{"text":"экономического","train_count":159,"old_ids":[65131,150192,158638],"score":318,"new_id":249798},{"text":"служил","train_count":317,"old_ids":[150431,61057],"score":317,"new_id":249799},{"text":"Климат","train_count":158,"old_ids":[25497,160827,7721],"score":316,"new_id":249800},{"text":"Лондоне","train_count":158,"old_ids":[61757,16806,226397],"score":316,"new_id":249801},{"text":"Один","train_count":316,"old_ids":[19736,67958],"score":316,"new_id":249802},{"text":"американских","train_count":158,"old_ids":[1424,155899,150584],"score":316,"new_id":249803},{"text":"высшего","train_count":158,"old_ids":[148705,2175,150083],"score":316,"new_id":249804},{"text":"мог","train_count":316,"old_ids":[148324,14942],"score":316,"new_id":249805},{"text":"обладает","train_count":158,"old_ids":[148280,148606,26046],"score":316,"new_id":249806},{"text":"поверхность","train_count":158,"old_ids":[148428,150652,58245],"score":316,"new_id":249807},{"text":"полузащитник","train_count":79,"old_ids":[164724,148397,13739,9080,67494],"score":316,"new_id":249808},{"text":"Осенью","train_count":105,"old_ids":[19736,2175,49603,11577],"score":315,"new_id":249809},{"text":"Серебряный","train_count":105,"old_ids":[18727,148579,151439,32401],"score":315,"new_id":249810},{"text":"озера","train_count":315,"old_ids":[148303,87658],"score":315,"new_id":249811},{"text":"семья","train_count":315,"old_ids":[159707,149885],"score":315,"new_id":249812},{"text":"службу","train_count":315,"old_ids":[150431,148853],"score":315,"new_id":249813},{"text":"столетия","train_count":105,"old_ids":[6389,148294,7921,37835],"score":315,"new_id":249814},{"text":"Джона","train_count":157,"old_ids":[23355,16467,71221],"score":314,"new_id":249815},{"text":"Дэвид","train_count":157,"old_ids":[23355,65131,152833],"score":314,"new_id":249816},{"text":"Нижней","train_count":157,"old_ids":[19582,151932,148820],"score":314,"new_id":249817},{"text":"Председатель","train_count":157,"old_ids":[214767,152824,57868],"score":314,"new_id":249818},{"text":"Северного","train_count":157,"old_ids":[181302,46576,148325],"score":314,"new_id":249819},{"text":"Эти","train_count":314,"old_ids":[89030,21797],"score":314,"new_id":249820},{"text":"внешней","train_count":157,"old_ids":[5301,157540,148820],"score":314,"new_id":249821},{"text":"депутатов","train_count":157,"old_ids":[95525,198375,152968],"score":314,"new_id":249822},{"text":"научный","train_count":157,"old_ids":[13785,27448,32401],"score":314,"new_id":249823},{"text":"небольших","train_count":157,"old_ids":[76069,170946,150047],"score":314,"new_id":249824},{"text":"объекты","train_count":157,"old_ids":[148280,152743,82942],"score":314,"new_id":249825},{"text":"остался","train_count":157,"old_ids":[23514,15481,19937],"score":314,"new_id":249826},{"text":"советская","train_count":314,"old_ids":[227012,152141],"score":314,"new_id":249827},{"text":"соответствует","train_count":157,"old_ids":[148454,150076,151005],"score":314,"new_id":249828},{"text":"студентов","train_count":157,"old_ids":[6389,34723,151282],"score":314,"new_id":249829},{"text":"телевидения","train_count":157,"old_ids":[159593,152833,17941],"score":314,"new_id":249830},{"text":"установки","train_count":157,"old_ids":[3652,148722,16256],"score":314,"new_id":249831},{"text":"вида","train_count":313,"old_ids":[148341,38183],"score":313,"new_id":249832},{"text":"могли","train_count":313,"old_ids":[148324,151203],"score":313,"new_id":249833},{"text":"счёт","train_count":313,"old_ids":[2175,163436],"score":313,"new_id":249834},{"text":"Бывшие","train_count":156,"old_ids":[58297,33422,150366],"score":312,"new_id":249835},{"text":"Президента","train_count":156,"old_ids":[51844,148377,163395],"score":312,"new_id":249836},{"text":"Художник","train_count":104,"old_ids":[150853,34723,149244,67494],"score":312,"new_id":249837},{"text":"автомобиля","train_count":104,"old_ids":[148292,149127,151015,13302],"score":312,"new_id":249838},{"text":"влиянием","train_count":156,"old_ids":[5301,150095,169952],"score":312,"new_id":249839},{"text":"использовании","train_count":104,"old_ids":[13113,183192,148675,149091],"score":312,"new_id":249840},{"text":"кандидата","train_count":104,"old_ids":[4640,72884,45587,40710],"score":312,"new_id":249841},{"text":"кислорода","train_count":78,"old_ids":[4640,13113,2941,148367,38183],"score":312,"new_id":249842},{"text":"медалью","train_count":156,"old_ids":[235410,36771,11577],"score":312,"new_id":249843},{"text":"проходит","train_count":312,"old_ids":[149490,150294],"score":312,"new_id":249844},{"text":"сражении","train_count":156,"old_ids":[2175,21441,159605],"score":312,"new_id":249845},{"text":"физики","train_count":156,"old_ids":[18768,39427,151278],"score":312,"new_id":249846},{"text":"чемпиона","train_count":156,"old_ids":[155356,148686,71221],"score":312,"new_id":249847},{"text":"чемпионов","train_count":156,"old_ids":[155356,148686,152740],"score":312,"new_id":249848},{"text":"просто","train_count":311,"old_ids":[148489,151290],"score":311,"new_id":249849},{"text":"Вскоре","train_count":310,"old_ids":[15729,197641],"score":310,"new_id":249850},{"text":"Жизнь","train_count":155,"old_ids":[154849,39427,149497],"score":310,"new_id":249851},{"text":"Йорк","train_count":310,"old_ids":[213830,4640],"score":310,"new_id":249852},{"text":"Параллельно","train_count":62,"old_ids":[16362,16763,15481,2941,24623,13292],"score":310,"new_id":249853},{"text":"Республика","train_count":155,"old_ids":[32392,151869,12751],"score":310,"new_id":249854},{"text":"Средняя","train_count":155,"old_ids":[18727,148420,151713],"score":310,"new_id":249855},{"text":"УЕФА","train_count":155,"old_ids":[39243,53241,229259],"score":310,"new_id":249856},{"text":"военных","train_count":310,"old_ids":[224606,42240],"score":310,"new_id":249857},{"text":"двора","train_count":155,"old_ids":[6733,5301,148676],"score":310,"new_id":249858},{"text":"мексиканский","train_count":62,"old_ids":[6240,148301,2175,36473,7105,149800],"score":310,"new_id":249859},{"text":"номинации","train_count":155,"old_ids":[13292,149717,54201],"score":310,"new_id":249860},{"text":"округов","train_count":155,"old_ids":[148518,62215,148275],"score":310,"new_id":249861},{"text":"отличается","train_count":155,"old_ids":[12666,87018,64903],"score":310,"new_id":249862},{"text":"представитель","train_count":155,"old_ids":[177917,148448,53062],"score":310,"new_id":249863},{"text":"физик","train_count":155,"old_ids":[18768,39427,36473],"score":310,"new_id":249864},{"text":"экспедиции","train_count":155,"old_ids":[192809,228709,54201],"score":310,"new_id":249865},{"text":"Территория","train_count":103,"old_ids":[32868,7358,149454,154617],"score":309,"new_id":249866},{"text":"австрийский","train_count":103,"old_ids":[148292,153342,11796,149800],"score":309,"new_id":249867},{"text":"выпущена","train_count":103,"old_ids":[148705,7754,40794,48623],"score":309,"new_id":249868},{"text":"июле","train_count":309,"old_ids":[163855,148284],"score":309,"new_id":249869},{"text":"процессы","train_count":103,"old_ids":[149490,10064,21785,54330],"score":309,"new_id":249870},{"text":"пулемёта","train_count":103,"old_ids":[148729,151032,42565,42074],"score":309,"new_id":249871},{"text":"сначала","train_count":309,"old_ids":[2175,156541],"score":309,"new_id":249872},{"text":"экономики","train_count":309,"old_ids":[162366,16256],"score":309,"new_id":249873},{"text":"Вашингтон","train_count":77,"old_ids":[15729,48989,18372,14942,173049],"score":308,"new_id":249874},{"text":"Киеве","train_count":154,"old_ids":[25497,47552,148537],"score":308,"new_id":249875},{"text":"Ольга","train_count":154,"old_ids":[19736,27112,148369],"score":308,"new_id":249876},{"text":"башни","train_count":154,"old_ids":[9629,48989,21918],"score":308,"new_id":249877},{"text":"игре","train_count":308,"old_ids":[66951,148279],"score":308,"new_id":249878},{"text":"королём","train_count":154,"old_ids":[153589,148294,151666],"score":308,"new_id":249879},{"text":"поддержки","train_count":154,"old_ids":[155908,148605,16256],"score":308,"new_id":249880},{"text":"результаты","train_count":154,"old_ids":[150564,62709,149351],"score":308,"new_id":249881},{"text":"северу","train_count":154,"old_ids":[148938,46576,3652],"score":308,"new_id":249882},{"text":"строительстве","train_count":154,"old_ids":[148943,53062,151418],"score":308,"new_id":249883},{"text":"технологий","train_count":154,"old_ids":[157971,152910,42387],"score":308,"new_id":249884},{"text":"Гражданской","train_count":153,"old_ids":[243873,222837,149193],"score":306,"new_id":249885},{"text":"Египет","train_count":102,"old_ids":[53241,14942,45768,7921],"score":306,"new_id":249886},{"text":"Международный","train_count":153,"old_ids":[181834,151896,32401],"score":306,"new_id":249887},{"text":"Нижегородской","train_count":102,"old_ids":[188973,56090,152689,149193],"score":306,"new_id":249888},{"text":"Образование","train_count":153,"old_ids":[70749,148464,150187],"score":306,"new_id":249889},{"text":"Работал","train_count":153,"old_ids":[173031,148453,15481],"score":306,"new_id":249890},{"text":"Станислав","train_count":102,"old_ids":[71000,7105,13113,150800],"score":306,"new_id":249891},{"text":"Театр","train_count":153,"old_ids":[177471,7721,2115],"score":306,"new_id":249892},{"text":"Харьков","train_count":102,"old_ids":[150853,16763,4669,148336],"score":306,"new_id":249893},{"text":"Хорватии","train_count":102,"old_ids":[150853,8789,150156,44633],"score":306,"new_id":249894},{"text":"восточных","train_count":102,"old_ids":[5301,23514,36103,42240],"score":306,"new_id":249895},{"text":"зачастую","train_count":102,"old_ids":[8688,47474,80378,31953],"score":306,"new_id":249896},{"text":"инструментов","train_count":102,"old_ids":[18372,150379,89315,148275],"score":306,"new_id":249897},{"text":"интерес","train_count":153,"old_ids":[18372,55055,21785],"score":306,"new_id":249898},{"text":"младший","train_count":153,"old_ids":[6240,148606,151642],"score":306,"new_id":249899},{"text":"насчитывала","train_count":102,"old_ids":[222958,87148,33422,92647],"score":306,"new_id":249900},{"text":"одноимённого","train_count":102,"old_ids":[164551,15630,42565,171935],"score":306,"new_id":249901},{"text":"отдельные","train_count":153,"old_ids":[12666,159194,41546],"score":306,"new_id":249902},{"text":"политических","train_count":153,"old_ids":[148634,9080,151308],"score":306,"new_id":249903},{"text":"построить","train_count":102,"old_ids":[7754,23514,148283,17256],"score":306,"new_id":249904},{"text":"содержания","train_count":102,"old_ids":[2175,148277,55638,36916],"score":306,"new_id":249905},{"text":"соединений","train_count":102,"old_ids":[2175,42317,67958,59318],"score":306,"new_id":249906},{"text":"специально","train_count":153,"old_ids":[155036,14666,149749],"score":306,"new_id":249907},{"text":"существование","train_count":153,"old_ids":[2175,154659,150544],"score":306,"new_id":249908},{"text":"сцене","train_count":153,"old_ids":[2175,149147,1450],"score":306,"new_id":249909},{"text":"произведений","train_count":305,"old_ids":[161612,183214],"score":305,"new_id":249910},{"text":"фильмы","train_count":305,"old_ids":[165171,148706],"score":305,"new_id":249911},{"text":"хозяйства","train_count":305,"old_ids":[158370,89771],"score":305,"new_id":249912},{"text":"Альбом","train_count":152,"old_ids":[34960,27112,173910],"score":304,"new_id":249913},{"text":"Индонезии","train_count":76,"old_ids":[29189,150559,16806,148377,44633],"score":304,"new_id":249914},{"text":"Книги","train_count":152,"old_ids":[25497,21918,148388],"score":304,"new_id":249915},{"text":"Например","train_count":304,"old_ids":[81573,155147],"score":304,"new_id":249916},{"text":"Палаты","train_count":152,"old_ids":[16362,15481,149351],"score":304,"new_id":249917},{"text":"Посполитой","train_count":76,"old_ids":[16362,22078,148634,9080,148293],"score":304,"new_id":249918},{"text":"альбомов","train_count":152,"old_ids":[36771,173910,148275],"score":304,"new_id":249919},{"text":"вокзала","train_count":152,"old_ids":[159496,8688,92647],"score":304,"new_id":249920},{"text":"побратимы","train_count":76,"old_ids":[7754,148280,77779,15630,4403],"score":304,"new_id":249921},{"text":"управлению","train_count":152,"old_ids":[47174,148381,160315],"score":304,"new_id":249922},{"text":"юге","train_count":304,"old_ids":[11577,149068],"score":304,"new_id":249923},{"text":"Одессе","train_count":101,"old_ids":[19736,6733,90202,1450],"score":303,"new_id":249924},{"text":"военного","train_count":303,"old_ids":[224606,148325],"score":303,"new_id":249925},{"text":"греческого","train_count":101,"old_ids":[154731,53887,21785,149075],"score":303,"new_id":249926},{"text":"губерний","train_count":101,"old_ids":[14942,61324,7358,149623],"score":303,"new_id":249927},{"text":"называемый","train_count":101,"old_ids":[13785,152953,153056,33477],"score":303,"new_id":249928},{"text":"оказалось","train_count":101,"old_ids":[1403,148330,15481,95306],"score":303,"new_id":249929},{"text":"оркестром","train_count":101,"old_ids":[8789,4640,34981,149633],"score":303,"new_id":249930},{"text":"принадлежал","train_count":101,"old_ids":[148895,149578,148284,153002],"score":303,"new_id":249931},{"text":"случайно","train_count":101,"old_ids":[38612,27448,148390,13292],"score":303,"new_id":249932},{"text":"Генри","train_count":151,"old_ids":[35884,5092,29119],"score":302,"new_id":249933},{"text":"Майкл","train_count":151,"old_ids":[36736,148390,152967],"score":302,"new_id":249934},{"text":"Тренеры","train_count":151,"old_ids":[32868,148516,172094],"score":302,"new_id":249935},{"text":"анализа","train_count":151,"old_ids":[7105,68813,1424],"score":302,"new_id":249936},{"text":"единиц","train_count":151,"old_ids":[152021,1739,10064],"score":302,"new_id":249937},{"text":"майор","train_count":302,"old_ids":[191881,8789],"score":302,"new_id":249938},{"text":"переулок","train_count":151,"old_ids":[159357,3652,171144],"score":302,"new_id":249939},{"text":"породы","train_count":151,"old_ids":[7754,149345,4403],"score":302,"new_id":249940},{"text":"сотрудник","train_count":151,"old_ids":[217181,149035,67494],"score":302,"new_id":249941},{"text":"июне","train_count":301,"old_ids":[163855,76069],"score":301,"new_id":249942},{"text":"Геннадий","train_count":100,"old_ids":[35884,5092,149578,42387],"score":300,"new_id":249943},{"text":"Давид","train_count":150,"old_ids":[23355,148292,45587],"score":300,"new_id":249944},{"text":"Ильинское","train_count":100,"old_ids":[29189,27112,18372,170391],"score":300,"new_id":249945},{"text":"Китае","train_count":150,"old_ids":[25497,148648,1450],"score":300,"new_id":249946},{"text":"Компьютерные","train_count":75,"old_ids":[191297,7754,84920,55055,41546],"score":300,"new_id":249947},{"text":"Николаевна","train_count":100,"old_ids":[19582,151907,29354,187935],"score":300,"new_id":249948},{"text":"Расположен","train_count":150,"old_ids":[225381,220495,148942],"score":300,"new_id":249949},{"text":"Численность","train_count":100,"old_ids":[70345,13113,58361,58245],"score":300,"new_id":249950},{"text":"аэропорт","train_count":100,"old_ids":[1424,65131,148283,152933],"score":300,"new_id":249951},{"text":"вынуждены","train_count":75,"old_ids":[148705,2040,54025,6733,75896],"score":300,"new_id":249952},{"text":"государственном","train_count":100,"old_ids":[14942,22078,150968,190759],"score":300,"new_id":249953},{"text":"данные","train_count":300,"old_ids":[6733,151979],"score":300,"new_id":249954},{"text":"использует","train_count":150,"old_ids":[13113,183192,202497],"score":300,"new_id":249955},{"text":"командой","train_count":150,"old_ids":[148558,72884,148293],"score":300,"new_id":249956},{"text":"медицины","train_count":100,"old_ids":[6240,148969,14666,12014],"score":300,"new_id":249957},{"text":"наблюдения","train_count":100,"old_ids":[13785,9629,151305,17941],"score":300,"new_id":249958},{"text":"независимо","train_count":75,"old_ids":[76069,8688,148292,13113,149076],"score":300,"new_id":249959},{"text":"объединены","train_count":100,"old_ids":[148280,32482,152021,75896],"score":300,"new_id":249960},{"text":"победитель","train_count":100,"old_ids":[7754,148280,148276,53062],"score":300,"new_id":249961},{"text":"потерял","train_count":100,"old_ids":[229561,7358,4096,2941],"score":300,"new_id":249962},{"text":"пребывания","train_count":100,"old_ids":[7754,149467,33422,36916],"score":300,"new_id":249963},{"text":"принять","train_count":150,"old_ids":[148489,18372,90638],"score":300,"new_id":249964},{"text":"решением","train_count":150,"old_ids":[2115,44432,149353],"score":300,"new_id":249965},{"text":"сериал","train_count":150,"old_ids":[2175,149098,15481],"score":300,"new_id":249966},{"text":"успеха","train_count":150,"old_ids":[3652,31608,156025],"score":300,"new_id":249967},{"text":"художников","train_count":150,"old_ids":[157951,149244,149668],"score":300,"new_id":249968},{"text":"позиции","train_count":299,"old_ids":[161339,54201],"score":299,"new_id":249969},{"text":"Сергея","train_count":149,"old_ids":[209616,149068,4096],"score":298,"new_id":249970},{"text":"связанные","train_count":149,"old_ids":[2175,163720,151979],"score":298,"new_id":249971},{"text":"состояния","train_count":149,"old_ids":[2175,152544,155012],"score":298,"new_id":249972},{"text":"считают","train_count":149,"old_ids":[2175,87148,148803],"score":298,"new_id":249973},{"text":"учился","train_count":298,"old_ids":[27448,154838],"score":298,"new_id":249974},{"text":"Библиография","train_count":99,"old_ids":[207347,150029,1403,181923],"score":297,"new_id":249975},{"text":"Университет","train_count":99,"old_ids":[39243,150439,152898,7921],"score":297,"new_id":249976},{"text":"двигателей","train_count":99,"old_ids":[6733,5301,66951,154660],"score":297,"new_id":249977},{"text":"дополнительные","train_count":99,"old_ids":[148551,148634,150196,41546],"score":297,"new_id":249978},{"text":"дочерью","train_count":99,"old_ids":[6733,36103,7358,84920],"score":297,"new_id":249979},{"text":"легкоатлет","train_count":99,"old_ids":[156337,45527,7721,149550],"score":297,"new_id":249980},{"text":"секретаря","train_count":99,"old_ids":[241651,166196,16763,4096],"score":297,"new_id":249981},{"text":"семьёй","train_count":99,"old_ids":[159707,4669,42565,11796],"score":297,"new_id":249982},{"text":"художественных","train_count":99,"old_ids":[157951,149244,150198,42240],"score":297,"new_id":249983},{"text":"Мартин","train_count":148,"old_ids":[36736,57383,18372],"score":296,"new_id":249984},{"text":"Начало","train_count":148,"old_ids":[19582,47474,150182],"score":296,"new_id":249985},{"text":"Постановлением","train_count":74,"old_ids":[16362,23514,7105,148275,157264],"score":296,"new_id":249986},{"text":"Ранние","train_count":148,"old_ids":[32392,7105,150528],"score":296,"new_id":249987},{"text":"ассоциации","train_count":148,"old_ids":[27473,1403,210038],"score":296,"new_id":249988},{"text":"итоге","train_count":296,"old_ids":[156189,149068],"score":296,"new_id":249989},{"text":"обеспечение","train_count":148,"old_ids":[148280,149097,155313],"score":296,"new_id":249990},{"text":"представлен","train_count":148,"old_ids":[177917,148448,58361],"score":296,"new_id":249991},{"text":"свободу","train_count":74,"old_ids":[2175,5301,148280,148277,3652],"score":296,"new_id":249992},{"text":"севере","train_count":296,"old_ids":[148938,197375],"score":296,"new_id":249993},{"text":"существуют","train_count":148,"old_ids":[2175,82577,149460],"score":296,"new_id":249994},{"text":"японский","train_count":148,"old_ids":[4096,154767,149800],"score":296,"new_id":249995},{"text":"форме","train_count":295,"old_ids":[54724,1450],"score":295,"new_id":249996},{"text":"Валентин","train_count":98,"old_ids":[15729,15481,18110,18372],"score":294,"new_id":249997},{"text":"Красного","train_count":147,"old_ids":[25497,148761,148325],"score":294,"new_id":249998},{"text":"Несколько","train_count":98,"old_ids":[19582,21785,45527,150535],"score":294,"new_id":249999},{"text":"Строительство","train_count":147,"old_ids":[71000,148283,158663],"score":294,"new_id":250000},{"text":"Умершие","train_count":147,"old_ids":[39243,60909,150366],"score":294,"new_id":250001},{"text":"заболевания","train_count":98,"old_ids":[148397,148421,151186,36916],"score":294,"new_id":250002},{"text":"императором","train_count":98,"old_ids":[15630,149208,55389,148288],"score":294,"new_id":250003},{"text":"медицинских","train_count":98,"old_ids":[6240,148969,159366,150584],"score":294,"new_id":250004},{"text":"медицинской","train_count":98,"old_ids":[6240,148969,159366,149193],"score":294,"new_id":250005},{"text":"ниже","train_count":294,"old_ids":[21918,56090],"score":294,"new_id":250006},{"text":"пехотной","train_count":98,"old_ids":[7754,93834,12666,148317],"score":294,"new_id":250007},{"text":"причине","train_count":147,"old_ids":[148895,5831,149961],"score":294,"new_id":250008},{"text":"продолжает","train_count":98,"old_ids":[148489,148277,148294,155887],"score":294,"new_id":250009},{"text":"сражения","train_count":147,"old_ids":[2175,21441,150282],"score":294,"new_id":250010},{"text":"футбольного","train_count":98,"old_ids":[18768,36183,170946,148325],"score":294,"new_id":250011},{"text":"Впервые","train_count":293,"old_ids":[15729,162206],"score":293,"new_id":250012},{"text":"внимание","train_count":293,"old_ids":[5301,165714],"score":293,"new_id":250013},{"text":"фильмов","train_count":293,"old_ids":[165171,155178],"score":293,"new_id":250014},{"text":"Архитектура","train_count":73,"old_ids":[199593,10161,25135,4640,159142],"score":292,"new_id":250015},{"text":"Достижения","train_count":146,"old_ids":[23355,81839,150282],"score":292,"new_id":250016},{"text":"Казахстан","train_count":146,"old_ids":[172753,152999,49224],"score":292,"new_id":250017},{"text":"Швейцария","train_count":73,"old_ids":[151311,156211,10064,16763,37835],"score":292,"new_id":250018},{"text":"архиепископа","train_count":73,"old_ids":[16763,229751,28696,45527,148414],"score":292,"new_id":250019},{"text":"веществ","train_count":146,"old_ids":[5301,82557,32812],"score":292,"new_id":250020},{"text":"высотой","train_count":146,"old_ids":[148705,217181,148293],"score":292,"new_id":250021},{"text":"гитарист","train_count":146,"old_ids":[14942,167654,32884],"score":292,"new_id":250022},{"text":"древних","train_count":146,"old_ids":[60659,148380,149648],"score":292,"new_id":250023},{"text":"модификации","train_count":73,"old_ids":[6240,148277,59627,71658,54201],"score":292,"new_id":250024},{"text":"назвал","train_count":146,"old_ids":[13785,8688,151019],"score":292,"new_id":250025},{"text":"немецких","train_count":146,"old_ids":[76069,218942,148797],"score":292,"new_id":250026},{"text":"представителем","train_count":73,"old_ids":[177917,148448,9080,77969,6240],"score":292,"new_id":250027},{"text":"престол","train_count":146,"old_ids":[148489,34981,148294],"score":292,"new_id":250028},{"text":"пространства","train_count":146,"old_ids":[155611,33437,89771],"score":292,"new_id":250029},{"text":"развитии","train_count":146,"old_ids":[148464,149734,44633],"score":292,"new_id":250030},{"text":"различными","train_count":146,"old_ids":[148464,87018,149195],"score":292,"new_id":250031},{"text":"режиме","train_count":146,"old_ids":[148279,151970,1450],"score":292,"new_id":250032},{"text":"русском","train_count":292,"old_ids":[152528,153195],"score":292,"new_id":250033},{"text":"физико","train_count":146,"old_ids":[18768,39427,151907],"score":292,"new_id":250034},{"text":"частей","train_count":292,"old_ids":[150813,148298],"score":292,"new_id":250035},{"text":"Организации","train_count":97,"old_ids":[19736,89527,151946,51788],"score":291,"new_id":250036},{"text":"Петрович","train_count":291,"old_ids":[154192,191062],"score":291,"new_id":250037},{"text":"английской","train_count":97,"old_ids":[7105,151203,11796,149193],"score":291,"new_id":250038},{"text":"политику","train_count":97,"old_ids":[148634,9080,1739,34083],"score":291,"new_id":250039},{"text":"сооружений","train_count":97,"old_ids":[148454,8789,54025,59318],"score":291,"new_id":250040},{"text":"сценария","train_count":97,"old_ids":[2175,149147,16763,37835],"score":291,"new_id":250041},{"text":"ВКП","train_count":145,"old_ids":[15729,25497,16362],"score":290,"new_id":250042},{"text":"Изначально","train_count":145,"old_ids":[29189,149340,149749],"score":290,"new_id":250043},{"text":"Павлович","train_count":145,"old_ids":[16362,148292,246235],"score":290,"new_id":250044},{"text":"взял","train_count":145,"old_ids":[5301,152175,2941],"score":290,"new_id":250045},{"text":"друга","train_count":290,"old_ids":[192548,1424],"score":290,"new_id":250046},{"text":"изделий","train_count":145,"old_ids":[39427,151399,11796],"score":290,"new_id":250047},{"text":"исторической","train_count":145,"old_ids":[32884,8789,158585],"score":290,"new_id":250048},{"text":"источников","train_count":145,"old_ids":[32884,36103,149668],"score":290,"new_id":250049},{"text":"муниципалитет","train_count":58,"old_ids":[148405,151744,7754,15481,9080,7921],"score":290,"new_id":250050},{"text":"научной","train_count":145,"old_ids":[13785,27448,148317],"score":290,"new_id":250051},{"text":"публицист","train_count":145,"old_ids":[170852,14666,6389],"score":290,"new_id":250052},{"text":"сообщение","train_count":145,"old_ids":[148454,149350,16669],"score":290,"new_id":250053},{"text":"составил","train_count":145,"old_ids":[2175,150838,61057],"score":290,"new_id":250054},{"text":"пост","train_count":289,"old_ids":[7754,23514],"score":289,"new_id":250055},{"text":"Архангельской","train_count":96,"old_ids":[199593,158702,225639,149193],"score":288,"new_id":250056},{"text":"Киевской","train_count":96,"old_ids":[25497,47552,5301,149193],"score":288,"new_id":250057},{"text":"Соревнования","train_count":72,"old_ids":[18727,8789,148380,148366,36916],"score":288,"new_id":250058},{"text":"вошёл","train_count":288,"old_ids":[148331,229903],"score":288,"new_id":250059},{"text":"выпущено","train_count":96,"old_ids":[148705,7754,40794,69729],"score":288,"new_id":250060},{"text":"выступлений","train_count":144,"old_ids":[148705,148757,161901],"score":288,"new_id":250061},{"text":"газете","train_count":144,"old_ids":[148369,8688,149155],"score":288,"new_id":250062},{"text":"медицине","train_count":96,"old_ids":[6240,148969,14666,76069],"score":288,"new_id":250063},{"text":"методы","train_count":144,"old_ids":[148557,148277,4403],"score":288,"new_id":250064},{"text":"присоединён","train_count":72,"old_ids":[148489,13113,42317,67958,185709],"score":288,"new_id":250065},{"text":"рассматривать","train_count":48,"old_ids":[2115,27473,6240,7721,2115,26125,17460],"score":288,"new_id":250066},{"text":"свободного","train_count":72,"old_ids":[2175,5301,148280,148277,148325],"score":288,"new_id":250067},{"text":"советской","train_count":288,"old_ids":[227012,149193],"score":288,"new_id":250068},{"text":"целях","train_count":144,"old_ids":[10064,41580,10161],"score":288,"new_id":250069},{"text":"экономический","train_count":96,"old_ids":[65131,150192,77436,149537],"score":288,"new_id":250070},{"text":"действий","train_count":287,"old_ids":[149232,156862],"score":287,"new_id":250071},{"text":"карте","train_count":287,"old_ids":[149888,48824],"score":287,"new_id":250072},{"text":"Пушкина","train_count":143,"old_ids":[193623,151487,13785],"score":286,"new_id":250073},{"text":"высоты","train_count":143,"old_ids":[148705,217181,4403],"score":286,"new_id":250074},{"text":"каждом","train_count":143,"old_ids":[12751,148343,148288],"score":286,"new_id":250075},{"text":"общественного","train_count":143,"old_ids":[149350,150198,148325],"score":286,"new_id":250076},{"text":"персонажей","train_count":143,"old_ids":[149208,152590,153703],"score":286,"new_id":250077},{"text":"поколения","train_count":143,"old_ids":[148428,45527,59595],"score":286,"new_id":250078},{"text":"списке","train_count":143,"old_ids":[2175,28696,50800],"score":286,"new_id":250079},{"text":"участии","train_count":143,"old_ids":[27448,80378,44633],"score":286,"new_id":250080},{"text":"численности","train_count":143,"old_ids":[153683,58361,65360],"score":286,"new_id":250081},{"text":"Центральный","train_count":95,"old_ids":[89602,18110,2115,152027],"score":285,"new_id":250082},{"text":"войсками","train_count":285,"old_ids":[159035,196290],"score":285,"new_id":250083},{"text":"наблюдается","train_count":95,"old_ids":[13785,9629,151305,64903],"score":285,"new_id":250084},{"text":"независимых","train_count":57,"old_ids":[76069,8688,148292,13113,15630,60038],"score":285,"new_id":250085},{"text":"переименована","train_count":95,"old_ids":[159357,15630,69729,182032],"score":285,"new_id":250086},{"text":"пехоты","train_count":95,"old_ids":[7754,93834,12666,4403],"score":285,"new_id":250087},{"text":"позволяют","train_count":95,"old_ids":[7754,148303,148331,152984],"score":285,"new_id":250088},{"text":"предположительно","train_count":95,"old_ids":[177917,220495,16467,151269],"score":285,"new_id":250089},{"text":"преобразован","train_count":95,"old_ids":[149790,149586,148275,7105],"score":285,"new_id":250090},{"text":"суда","train_count":285,"old_ids":[2175,151978],"score":285,"new_id":250091},{"text":"волости","train_count":284,"old_ids":[157156,81839],"score":284,"new_id":250092},{"text":"границе","train_count":142,"old_ids":[14942,91872,1450],"score":284,"new_id":250093},{"text":"имела","train_count":284,"old_ids":[15630,150456],"score":284,"new_id":250094},{"text":"капитана","train_count":142,"old_ids":[12751,150478,148841],"score":284,"new_id":250095},{"text":"основателей","train_count":142,"old_ids":[22078,148366,154660],"score":284,"new_id":250096},{"text":"предыдущего","train_count":71,"old_ids":[177917,4403,6733,40794,148702],"score":284,"new_id":250097},{"text":"пяти","train_count":284,"old_ids":[7754,149036],"score":284,"new_id":250098},{"text":"разработке","train_count":142,"old_ids":[148464,149100,50800],"score":284,"new_id":250099},{"text":"средства","train_count":284,"old_ids":[174301,89771],"score":284,"new_id":250100},{"text":"человечества","train_count":71,"old_ids":[5831,11777,148275,53887,150167],"score":284,"new_id":250101},{"text":"певица","train_count":283,"old_ids":[148728,198325],"score":283,"new_id":250102},{"text":"Институте","train_count":94,"old_ids":[161484,158693,36183,1450],"score":282,"new_id":250103},{"text":"Песни","train_count":141,"old_ids":[16362,21785,21918],"score":282,"new_id":250104},{"text":"Подчинён","train_count":141,"old_ids":[174249,155812,185709],"score":282,"new_id":250105},{"text":"Томас","train_count":141,"old_ids":[32868,148288,17417],"score":282,"new_id":250106},{"text":"Филиппа","train_count":94,"old_ids":[52337,149037,74359,1424],"score":282,"new_id":250107},{"text":"дивизия","train_count":141,"old_ids":[150394,39427,37835],"score":282,"new_id":250108},{"text":"информация","train_count":141,"old_ids":[18372,54724,53890],"score":282,"new_id":250109},{"text":"источниках","train_count":141,"old_ids":[32884,36103,191447],"score":282,"new_id":250110},{"text":"обеспечить","train_count":94,"old_ids":[148280,149097,53887,17256],"score":282,"new_id":250111},{"text":"основного","train_count":141,"old_ids":[22078,148366,148325],"score":282,"new_id":250112},{"text":"оставался","train_count":141,"old_ids":[150838,15481,19937],"score":282,"new_id":250113},{"text":"парламента","train_count":141,"old_ids":[151963,179555,84706],"score":282,"new_id":250114},{"text":"получивший","train_count":94,"old_ids":[148634,27448,26125,151642],"score":282,"new_id":250115},{"text":"реакции","train_count":141,"old_ids":[148279,148318,54201],"score":282,"new_id":250116},{"text":"самолёты","train_count":94,"old_ids":[160002,148294,42565,82942],"score":282,"new_id":250117},{"text":"секретарём","train_count":94,"old_ids":[241651,166196,16763,151666],"score":282,"new_id":250118},{"text":"символов","train_count":94,"old_ids":[2175,72326,148378,5301],"score":282,"new_id":250119},{"text":"сказал","train_count":141,"old_ids":[2175,148330,15481],"score":282,"new_id":250120},{"text":"сохранилось","train_count":94,"old_ids":[148454,10161,149926,153174],"score":282,"new_id":250121},{"text":"установлены","train_count":141,"old_ids":[3652,148722,154641],"score":282,"new_id":250122},{"text":"утверждён","train_count":94,"old_ids":[36183,46576,148343,185709],"score":282,"new_id":250123},{"text":"экономист","train_count":141,"old_ids":[65131,150192,32884],"score":282,"new_id":250124},{"text":"линия","train_count":281,"old_ids":[10750,155012],"score":281,"new_id":250125},{"text":"Административно","train_count":70,"old_ids":[210138,150704,32884,77779,167596],"score":280,"new_id":250126},{"text":"Венеции","train_count":140,"old_ids":[15729,150425,54201],"score":280,"new_id":250127},{"text":"Михайловна","train_count":70,"old_ids":[36736,64392,148390,148569,13785],"score":280,"new_id":250128},{"text":"Система","train_count":140,"old_ids":[18727,32884,149721],"score":280,"new_id":250129},{"text":"губернатором","train_count":56,"old_ids":[14942,61324,7358,2040,55389,148288],"score":280,"new_id":250130},{"text":"доме","train_count":280,"old_ids":[159080,1450],"score":280,"new_id":250131},{"text":"лучших","train_count":280,"old_ids":[151408,150047],"score":280,"new_id":250132},{"text":"посвящённый","train_count":70,"old_ids":[7754,22078,152139,155427,171096],"score":280,"new_id":250133},{"text":"потому","train_count":280,"old_ids":[229561,149169],"score":280,"new_id":250134},{"text":"характеризуется","train_count":70,"old_ids":[229103,149463,39427,3652,48046],"score":280,"new_id":250135},{"text":"швейцарский","train_count":70,"old_ids":[11802,156211,10064,16763,149800],"score":280,"new_id":250136},{"text":"Александровна","train_count":93,"old_ids":[34960,221097,148275,13785],"score":279,"new_id":250137},{"text":"Валентина","train_count":93,"old_ids":[15729,15481,18110,66353],"score":279,"new_id":250138},{"text":"Футболисты","train_count":93,"old_ids":[52337,36183,157759,151407],"score":279,"new_id":250139},{"text":"автомобиль","train_count":93,"old_ids":[148292,149127,148280,64257],"score":279,"new_id":250140},{"text":"развитием","train_count":93,"old_ids":[148464,149734,47552,6240],"score":279,"new_id":250141},{"text":"создано","train_count":93,"old_ids":[2175,148303,6733,149600],"score":279,"new_id":250142},{"text":"солдат","train_count":279,"old_ids":[166955,153966],"score":279,"new_id":250143},{"text":"Великий","train_count":139,"old_ids":[15729,58503,149537],"score":278,"new_id":250144},{"text":"Здесь","train_count":278,"old_ids":[34279,170901],"score":278,"new_id":250145},{"text":"Российского","train_count":139,"old_ids":[32392,150405,149280],"score":278,"new_id":250146},{"text":"Северная","train_count":139,"old_ids":[181302,46576,42308],"score":278,"new_id":250147},{"text":"белый","train_count":139,"old_ids":[9629,11777,33477],"score":278,"new_id":250148},{"text":"военной","train_count":278,"old_ids":[224606,148317],"score":278,"new_id":250149},{"text":"клубом","train_count":139,"old_ids":[4640,152887,148288],"score":278,"new_id":250150},{"text":"матчах","train_count":139,"old_ids":[6240,7721,246492],"score":278,"new_id":250151},{"text":"регулярно","train_count":139,"old_ids":[163675,149709,13292],"score":278,"new_id":250152},{"text":"руководил","train_count":139,"old_ids":[19160,150463,61057],"score":278,"new_id":250153},{"text":"самом","train_count":278,"old_ids":[160002,148288],"score":278,"new_id":250154},{"text":"тысячи","train_count":139,"old_ids":[82942,19937,148538],"score":278,"new_id":250155},{"text":"волость","train_count":277,"old_ids":[157156,93309],"score":277,"new_id":250156},{"text":"Ивановна","train_count":92,"old_ids":[29189,149137,148275,13785],"score":276,"new_id":250157},{"text":"Сергеевка","train_count":92,"old_ids":[209616,14942,47745,172637],"score":276,"new_id":250158},{"text":"Советская","train_count":138,"old_ids":[173518,45405,152141],"score":276,"new_id":250159},{"text":"Чикаго","train_count":138,"old_ids":[70345,71658,63046],"score":276,"new_id":250160},{"text":"духовной","train_count":138,"old_ids":[148660,152335,148317],"score":276,"new_id":250161},{"text":"ежегодно","train_count":138,"old_ids":[1450,56090,154486],"score":276,"new_id":250162},{"text":"залива","train_count":138,"old_ids":[8688,45679,90219],"score":276,"new_id":250163},{"text":"здравоохранения","train_count":69,"old_ids":[234021,148331,1403,58805,17941],"score":276,"new_id":250164},{"text":"измерения","train_count":138,"old_ids":[39427,60909,17941],"score":276,"new_id":250165},{"text":"киностудии","train_count":92,"old_ids":[16256,149089,34723,44633],"score":276,"new_id":250166},{"text":"клеток","train_count":138,"old_ids":[4640,149550,148518],"score":276,"new_id":250167},{"text":"литературный","train_count":92,"old_ids":[159329,7358,150849,32401],"score":276,"new_id":250168},{"text":"необходимость","train_count":92,"old_ids":[76069,148280,148334,161757],"score":276,"new_id":250169},{"text":"оборудование","train_count":92,"old_ids":[148280,8789,34723,150187],"score":276,"new_id":250170},{"text":"реализации","train_count":138,"old_ids":[148279,68813,51788],"score":276,"new_id":250171},{"text":"результат","train_count":138,"old_ids":[150564,62709,7721],"score":276,"new_id":250172},{"text":"священник","train_count":138,"old_ids":[183305,149553,67494],"score":276,"new_id":250173},{"text":"французская","train_count":92,"old_ids":[18768,33437,163227,152141],"score":276,"new_id":250174},{"text":"характеристика","train_count":92,"old_ids":[229103,149463,32884,71658],"score":276,"new_id":250175},{"text":"административном","train_count":55,"old_ids":[149099,150704,32884,77779,26125,148524],"score":275,"new_id":250176},{"text":"присоединились","train_count":55,"old_ids":[148489,13113,42317,67958,149037,148685],"score":275,"new_id":250177},{"text":"Песня","train_count":137,"old_ids":[16362,21785,37855],"score":274,"new_id":250178},{"text":"Сити","train_count":137,"old_ids":[18727,9080,1739],"score":274,"new_id":250179},{"text":"Украинской","train_count":137,"old_ids":[208317,18372,149193],"score":274,"new_id":250180},{"text":"газета","train_count":137,"old_ids":[148369,8688,149005],"score":274,"new_id":250181},{"text":"изображение","train_count":137,"old_ids":[39427,150065,150307],"score":274,"new_id":250182},{"text":"получив","train_count":137,"old_ids":[148634,27448,26125],"score":274,"new_id":250183},{"text":"производстве","train_count":137,"old_ids":[161612,148435,151418],"score":274,"new_id":250184},{"text":"следующим","train_count":137,"old_ids":[149161,68608,15630],"score":274,"new_id":250185},{"text":"участке","train_count":137,"old_ids":[27448,80378,50800],"score":274,"new_id":250186},{"text":"Харьковского","train_count":91,"old_ids":[150853,16763,4669,220104],"score":273,"new_id":250187},{"text":"бассейн","train_count":91,"old_ids":[9629,27473,148298,2040],"score":273,"new_id":250188},{"text":"мере","train_count":273,"old_ids":[60909,1450],"score":273,"new_id":250189},{"text":"народов","train_count":273,"old_ids":[151150,148275],"score":273,"new_id":250190},{"text":"прежде","train_count":273,"old_ids":[149790,155034],"score":273,"new_id":250191},{"text":"принцип","train_count":91,"old_ids":[148489,18372,14666,7754],"score":273,"new_id":250192},{"text":"присутствует","train_count":91,"old_ids":[148489,13113,36183,151005],"score":273,"new_id":250193},{"text":"результатом","train_count":91,"old_ids":[150564,62709,7721,148288],"score":273,"new_id":250194},{"text":"соревнований","train_count":91,"old_ids":[154399,148380,148366,149931],"score":273,"new_id":250195},{"text":"стать","train_count":273,"old_ids":[6389,17460],"score":273,"new_id":250196},{"text":"чемпионка","train_count":91,"old_ids":[155356,148686,16806,12751],"score":273,"new_id":250197},{"text":"Возвращение","train_count":68,"old_ids":[15729,148303,5301,50022,16669],"score":272,"new_id":250198},{"text":"Вюртемберга","train_count":68,"old_ids":[15729,154285,170744,150757,148369],"score":272,"new_id":250199},{"text":"Географическое","train_count":68,"old_ids":[35884,151709,152973,77436,153232],"score":272,"new_id":250200},{"text":"Международной","train_count":136,"old_ids":[181834,151896,148317],"score":272,"new_id":250201},{"text":"Металлург","train_count":68,"old_ids":[36736,7921,15481,2941,157647],"score":272,"new_id":250202},{"text":"Правительства","train_count":136,"old_ids":[16362,148381,158269],"score":272,"new_id":250203},{"text":"Топонимы","train_count":68,"old_ids":[32868,148561,16806,15630,4403],"score":272,"new_id":250204},{"text":"бывшая","train_count":136,"old_ids":[9629,33422,171149],"score":272,"new_id":250205},{"text":"выпустил","train_count":136,"old_ids":[148705,153449,2941],"score":272,"new_id":250206},{"text":"выпущенный","train_count":68,"old_ids":[148705,7754,40794,5092,32401],"score":272,"new_id":250207},{"text":"дивизиона","train_count":136,"old_ids":[150394,158668,71221],"score":272,"new_id":250208},{"text":"занималась","train_count":68,"old_ids":[8688,13603,6240,15481,152993],"score":272,"new_id":250209},{"text":"карьере","train_count":136,"old_ids":[12751,149213,148579],"score":272,"new_id":250210},{"text":"матчи","train_count":136,"old_ids":[6240,7721,148538],"score":272,"new_id":250211},{"text":"нередко","train_count":136,"old_ids":[149026,148276,45527],"score":272,"new_id":250212},{"text":"относятся","train_count":272,"old_ids":[152928,161755],"score":272,"new_id":250213},{"text":"погиб","train_count":136,"old_ids":[148428,148388,9629],"score":272,"new_id":250214},{"text":"промышленность","train_count":136,"old_ids":[149490,153886,58245],"score":272,"new_id":250215},{"text":"районах","train_count":136,"old_ids":[180093,71221,10161],"score":272,"new_id":250216},{"text":"сельских","train_count":272,"old_ids":[226181,150584],"score":272,"new_id":250217},{"text":"считал","train_count":136,"old_ids":[2175,87148,15481],"score":272,"new_id":250218},{"text":"штата","train_count":272,"old_ids":[184984,40710],"score":272,"new_id":250219},{"text":"Тогда","train_count":271,"old_ids":[32868,153606],"score":271,"new_id":250220},{"text":"Грузии","train_count":135,"old_ids":[35884,63698,44633],"score":270,"new_id":250221},{"text":"взрослых","train_count":90,"old_ids":[5301,8688,39605,159483],"score":270,"new_id":250222},{"text":"включён","train_count":135,"old_ids":[5301,49689,185709],"score":270,"new_id":250223},{"text":"генеральный","train_count":135,"old_ids":[170802,7358,152027],"score":270,"new_id":250224},{"text":"диссертации","train_count":90,"old_ids":[216775,2175,153922,51788],"score":270,"new_id":250225},{"text":"зрителей","train_count":135,"old_ids":[8688,149454,149074],"score":270,"new_id":250226},{"text":"композитора","train_count":90,"old_ids":[166349,148303,9080,148676],"score":270,"new_id":250227},{"text":"красный","train_count":135,"old_ids":[4640,148761,32401],"score":270,"new_id":250228},{"text":"мозга","train_count":135,"old_ids":[6240,148303,148369],"score":270,"new_id":250229},{"text":"последние","train_count":270,"old_ids":[167014,150528],"score":270,"new_id":250230},{"text":"принадлежала","train_count":90,"old_ids":[148895,149578,148284,240136],"score":270,"new_id":250231},{"text":"разновидность","train_count":90,"old_ids":[148464,148366,45587,58245],"score":270,"new_id":250232},{"text":"следует","train_count":270,"old_ids":[149161,69632],"score":270,"new_id":250233},{"text":"содержат","train_count":90,"old_ids":[2175,148277,55638,7721],"score":270,"new_id":250234},{"text":"условий","train_count":135,"old_ids":[41635,148569,42387],"score":270,"new_id":250235},{"text":"элементами","train_count":90,"old_ids":[65131,151032,84706,61628],"score":270,"new_id":250236},{"text":"систему","train_count":269,"old_ids":[167641,149581],"score":269,"new_id":250237},{"text":"Евровидение","train_count":67,"old_ids":[53241,5301,148327,45587,16669],"score":268,"new_id":250238},{"text":"влияния","train_count":134,"old_ids":[5301,150095,155012],"score":268,"new_id":250239},{"text":"выступать","train_count":134,"old_ids":[148705,148757,17460],"score":268,"new_id":250240},{"text":"заведующим","train_count":67,"old_ids":[8688,148292,148276,68608,15630],"score":268,"new_id":250241},{"text":"зрения","train_count":268,"old_ids":[8688,155209],"score":268,"new_id":250242},{"text":"известно","train_count":268,"old_ids":[244718,13292],"score":268,"new_id":250243},{"text":"использовалось","train_count":67,"old_ids":[13113,183192,148675,15481,95306],"score":268,"new_id":250244},{"text":"методов","train_count":134,"old_ids":[148557,148277,148275],"score":268,"new_id":250245},{"text":"мост","train_count":268,"old_ids":[6240,23514],"score":268,"new_id":250246},{"text":"названа","train_count":134,"old_ids":[13785,8688,182032],"score":268,"new_id":250247},{"text":"немецкая","train_count":134,"old_ids":[76069,218942,150621],"score":268,"new_id":250248},{"text":"опыт","train_count":134,"old_ids":[148561,4403,1729],"score":268,"new_id":250249},{"text":"частоты","train_count":134,"old_ids":[150813,12666,4403],"score":268,"new_id":250250},{"text":"Свердловской","train_count":89,"old_ids":[18727,46576,168512,149193],"score":267,"new_id":250251},{"text":"кандидат","train_count":89,"old_ids":[4640,72884,45587,7721],"score":267,"new_id":250252},{"text":"конфликта","train_count":89,"old_ids":[150270,18768,153864,42074],"score":267,"new_id":250253},{"text":"обслуживания","train_count":89,"old_ids":[148280,150431,26125,36916],"score":267,"new_id":250254},{"text":"писал","train_count":267,"old_ids":[28696,15481],"score":267,"new_id":250255},{"text":"соавторстве","train_count":89,"old_ids":[148454,148292,35458,151418],"score":267,"new_id":250256},{"text":"Длина","train_count":266,"old_ids":[23355,155565],"score":266,"new_id":250257},{"text":"МГУ","train_count":266,"old_ids":[36736,187056],"score":266,"new_id":250258},{"text":"Одновременно","train_count":133,"old_ids":[19736,6733,161523],"score":266,"new_id":250259},{"text":"Филипп","train_count":133,"old_ids":[52337,149037,74359],"score":266,"new_id":250260},{"text":"дерева","train_count":133,"old_ids":[6733,148579,90219],"score":266,"new_id":250261},{"text":"деревьев","train_count":133,"old_ids":[152138,148380,158525],"score":266,"new_id":250262},{"text":"здоровья","train_count":133,"old_ids":[171158,149937,149885],"score":266,"new_id":250263},{"text":"колледж","train_count":133,"old_ids":[180702,149595,16467],"score":266,"new_id":250264},{"text":"международного","train_count":133,"old_ids":[148446,151896,148325],"score":266,"new_id":250265},{"text":"музей","train_count":266,"old_ids":[148405,161809],"score":266,"new_id":250266},{"text":"объём","train_count":133,"old_ids":[148280,32482,151666],"score":266,"new_id":250267},{"text":"потом","train_count":266,"old_ids":[229561,148288],"score":266,"new_id":250268},{"text":"собрание","train_count":133,"old_ids":[2175,149962,25767],"score":266,"new_id":250269},{"text":"столицей","train_count":133,"old_ids":[149487,63890,148298],"score":266,"new_id":250270},{"text":"турнире","train_count":133,"old_ids":[152238,21918,148279],"score":266,"new_id":250271},{"text":"Поэтому","train_count":265,"old_ids":[159265,155084],"score":265,"new_id":250272},{"text":"магистратура","train_count":53,"old_ids":[66050,14942,32884,77779,3652,21441],"score":265,"new_id":250273},{"text":"Антоновка","train_count":88,"old_ids":[34960,242863,148366,12751],"score":264,"new_id":250274},{"text":"Возрождения","train_count":88,"old_ids":[15729,148303,148283,149880],"score":264,"new_id":250275},{"text":"Вьетнама","train_count":88,"old_ids":[15729,210216,13785,66050],"score":264,"new_id":250276},{"text":"Кавалеры","train_count":88,"old_ids":[25497,148292,15481,172094],"score":264,"new_id":250277},{"text":"Ленинградский","train_count":66,"old_ids":[61757,9396,2040,153987,149800],"score":264,"new_id":250278},{"text":"Съёмочная","train_count":66,"old_ids":[18727,32482,151666,36103,42308],"score":264,"new_id":250279},{"text":"возвращении","train_count":88,"old_ids":[150348,5301,50022,80228],"score":264,"new_id":250280},{"text":"дополнительных","train_count":88,"old_ids":[148551,148634,150196,42240],"score":264,"new_id":250281},{"text":"закончил","train_count":88,"old_ids":[8688,148318,16806,161907],"score":264,"new_id":250282},{"text":"команд","train_count":264,"old_ids":[148558,72884],"score":264,"new_id":250283},{"text":"лаборатории","train_count":88,"old_ids":[29354,148913,55389,44633],"score":264,"new_id":250284},{"text":"музыке","train_count":132,"old_ids":[148405,148707,50800],"score":264,"new_id":250285},{"text":"муниципальных","train_count":88,"old_ids":[148405,151744,7754,152994],"score":264,"new_id":250286},{"text":"обозначение","train_count":88,"old_ids":[148280,148303,29196,16669],"score":264,"new_id":250287},{"text":"океана","train_count":132,"old_ids":[1403,50800,148841],"score":264,"new_id":250288},{"text":"оружием","train_count":88,"old_ids":[8789,54025,47552,6240],"score":264,"new_id":250289},{"text":"порядок","train_count":132,"old_ids":[153624,93629,148518],"score":264,"new_id":250290},{"text":"продолжение","train_count":88,"old_ids":[148489,148277,148294,150307],"score":264,"new_id":250291},{"text":"свидетельствует","train_count":66,"old_ids":[2175,152833,7921,24623,151005],"score":264,"new_id":250292},{"text":"своем","train_count":132,"old_ids":[2175,159117,6240],"score":264,"new_id":250293},{"text":"советских","train_count":264,"old_ids":[227012,150584],"score":264,"new_id":250294},{"text":"таблице","train_count":88,"old_ids":[42074,9629,63890,1450],"score":264,"new_id":250295},{"text":"Анна","train_count":263,"old_ids":[34960,222331],"score":263,"new_id":250296},{"text":"Хотя","train_count":263,"old_ids":[150853,156442],"score":263,"new_id":250297},{"text":"активно","train_count":263,"old_ids":[148318,173162],"score":263,"new_id":250298},{"text":"городом","train_count":263,"old_ids":[152689,148288],"score":263,"new_id":250299},{"text":"Каждый","train_count":131,"old_ids":[172753,148343,33477],"score":262,"new_id":250300},{"text":"Чайковского","train_count":131,"old_ids":[70345,148390,220104],"score":262,"new_id":250301},{"text":"бывшей","train_count":131,"old_ids":[9629,33422,150281],"score":262,"new_id":250302},{"text":"внутри","train_count":262,"old_ids":[5301,168561],"score":262,"new_id":250303},{"text":"комедии","train_count":131,"old_ids":[148558,148276,44633],"score":262,"new_id":250304},{"text":"культура","train_count":131,"old_ids":[231736,3652,21441],"score":262,"new_id":250305},{"text":"напряжения","train_count":131,"old_ids":[13785,159388,150282],"score":262,"new_id":250306},{"text":"национальный","train_count":131,"old_ids":[13785,152208,159069],"score":262,"new_id":250307},{"text":"отсутствует","train_count":131,"old_ids":[12666,150338,151005],"score":262,"new_id":250308},{"text":"польский","train_count":262,"old_ids":[183192,149800],"score":262,"new_id":250309},{"text":"предприятия","train_count":262,"old_ids":[177917,171887],"score":262,"new_id":250310},{"text":"приходится","train_count":131,"old_ids":[148895,148334,92907],"score":262,"new_id":250311},{"text":"расположенный","train_count":131,"old_ids":[148761,220495,189530],"score":262,"new_id":250312},{"text":"стиле","train_count":262,"old_ids":[52598,148284],"score":262,"new_id":250313},{"text":"ткани","train_count":131,"old_ids":[1729,4640,13603],"score":262,"new_id":250314},{"text":"тренером","train_count":131,"old_ids":[155339,7358,148288],"score":262,"new_id":250315},{"text":"штаба","train_count":262,"old_ids":[153931,148778],"score":262,"new_id":250316},{"text":"Андерссон","train_count":87,"old_ids":[244015,7358,149663,16806],"score":261,"new_id":250317},{"text":"Василия","train_count":87,"old_ids":[15729,17417,149037,4096],"score":261,"new_id":250318},{"text":"Восточная","train_count":87,"old_ids":[15729,23514,36103,42308],"score":261,"new_id":250319},{"text":"Министров","train_count":87,"old_ids":[36736,18372,32884,148327],"score":261,"new_id":250320},{"text":"главных","train_count":261,"old_ids":[149809,216429],"score":261,"new_id":250321},{"text":"должна","train_count":261,"old_ids":[159172,171831],"score":261,"new_id":250322},{"text":"епископом","train_count":87,"old_ids":[1450,28696,45527,151201],"score":261,"new_id":250323},{"text":"заводов","train_count":87,"old_ids":[8688,148292,148277,148275],"score":261,"new_id":250324},{"text":"императорской","train_count":87,"old_ids":[15630,149208,55389,149193],"score":261,"new_id":250325},{"text":"минут","train_count":261,"old_ids":[150704,36183],"score":261,"new_id":250326},{"text":"отмечается","train_count":87,"old_ids":[12666,6240,53887,64903],"score":261,"new_id":250327},{"text":"работает","train_count":261,"old_ids":[149100,26046],"score":261,"new_id":250328},{"text":"считался","train_count":87,"old_ids":[2175,87148,15481,19937],"score":261,"new_id":250329},{"text":"Берёзовка","train_count":65,"old_ids":[58297,7358,42565,148675,12751],"score":260,"new_id":250330},{"text":"Иосифович","train_count":65,"old_ids":[29189,22078,59627,148275,31889],"score":260,"new_id":250331},{"text":"Карл","train_count":260,"old_ids":[232689,2941],"score":260,"new_id":250332},{"text":"Любовь","train_count":130,"old_ids":[207912,155489,4669],"score":260,"new_id":250333},{"text":"Мухаммад","train_count":65,"old_ids":[36736,149576,148357,66050,6733],"score":260,"new_id":250334},{"text":"Президент","train_count":130,"old_ids":[51844,148377,157621],"score":260,"new_id":250335},{"text":"Творчество","train_count":130,"old_ids":[32868,149086,174558],"score":260,"new_id":250336},{"text":"автомобильных","train_count":65,"old_ids":[148292,149127,148280,64257,42240],"score":260,"new_id":250337},{"text":"вещества","train_count":130,"old_ids":[5301,82557,150167],"score":260,"new_id":250338},{"text":"длиной","train_count":260,"old_ids":[185123,148317],"score":260,"new_id":250339},{"text":"достигает","train_count":130,"old_ids":[230530,66951,26046],"score":260,"new_id":250340},{"text":"еврейского","train_count":130,"old_ids":[148380,171251,149280],"score":260,"new_id":250341},{"text":"играть","train_count":260,"old_ids":[66951,151596],"score":260,"new_id":250342},{"text":"клубе","train_count":130,"old_ids":[4640,152887,1450],"score":260,"new_id":250343},{"text":"командира","train_count":130,"old_ids":[148558,72884,151246],"score":260,"new_id":250344},{"text":"международной","train_count":130,"old_ids":[148446,151896,148317],"score":260,"new_id":250345},{"text":"министром","train_count":130,"old_ids":[150704,32884,149633],"score":260,"new_id":250346},{"text":"отдыха","train_count":130,"old_ids":[12666,218913,1424],"score":260,"new_id":250347},{"text":"популярность","train_count":130,"old_ids":[148428,171447,58245],"score":260,"new_id":250348},{"text":"постановке","train_count":65,"old_ids":[7754,23514,7105,148275,50800],"score":260,"new_id":250349},{"text":"представлены","train_count":130,"old_ids":[177917,148448,154641],"score":260,"new_id":250350},{"text":"применяют","train_count":65,"old_ids":[148489,15630,5092,4096,66694],"score":260,"new_id":250351},{"text":"противника","train_count":260,"old_ids":[200359,84480],"score":260,"new_id":250352},{"text":"прошлом","train_count":130,"old_ids":[149490,11802,150954],"score":260,"new_id":250353},{"text":"статей","train_count":130,"old_ids":[6389,7721,148298],"score":260,"new_id":250354},{"text":"участники","train_count":130,"old_ids":[27448,80378,149665],"score":260,"new_id":250355},{"text":"фронт","train_count":130,"old_ids":[18768,2115,92914],"score":260,"new_id":250356},{"text":"власть","train_count":259,"old_ids":[160966,33348],"score":259,"new_id":250357},{"text":"канала","train_count":259,"old_ids":[151011,92647],"score":259,"new_id":250358},{"text":"орден","train_count":259,"old_ids":[8789,149637],"score":259,"new_id":250359},{"text":"премьер","train_count":259,"old_ids":[149790,189181],"score":259,"new_id":250360},{"text":"сельская","train_count":259,"old_ids":[226181,152141],"score":259,"new_id":250361},{"text":"Высшей","train_count":129,"old_ids":[44093,2175,150281],"score":258,"new_id":250362},{"text":"заболеваний","train_count":86,"old_ids":[148397,148421,151186,149931],"score":258,"new_id":250363},{"text":"медицинского","train_count":86,"old_ids":[6240,148969,159366,149280],"score":258,"new_id":250364},{"text":"механизм","train_count":86,"old_ids":[6240,93834,151946,6240],"score":258,"new_id":250365},{"text":"молодёжи","train_count":129,"old_ids":[209034,207530,148539],"score":258,"new_id":250366},{"text":"попытки","train_count":129,"old_ids":[148428,150258,159082],"score":258,"new_id":250367},{"text":"производству","train_count":129,"old_ids":[161612,148435,151386],"score":258,"new_id":250368},{"text":"раза","train_count":258,"old_ids":[21441,148397],"score":258,"new_id":250369},{"text":"располагался","train_count":86,"old_ids":[217455,151286,15481,19937],"score":258,"new_id":250370},{"text":"символы","train_count":86,"old_ids":[2175,72326,148294,4403],"score":258,"new_id":250371},{"text":"соединение","train_count":86,"old_ids":[2175,42317,67958,16669],"score":258,"new_id":250372},{"text":"столицы","train_count":129,"old_ids":[149487,63890,4403],"score":258,"new_id":250373},{"text":"веков","train_count":257,"old_ids":[148537,148336],"score":257,"new_id":250374},{"text":"меньше","train_count":257,"old_ids":[152475,56916],"score":257,"new_id":250375},{"text":"Африке","train_count":128,"old_ids":[34960,162684,50800],"score":256,"new_id":250376},{"text":"Баден","train_count":128,"old_ids":[58297,149099,5092],"score":256,"new_id":250377},{"text":"Крылья","train_count":128,"old_ids":[25497,36747,155298],"score":256,"new_id":250378},{"text":"библиотека","train_count":128,"old_ids":[148687,157622,12751],"score":256,"new_id":250379},{"text":"восемь","train_count":128,"old_ids":[163934,148281,4669],"score":256,"new_id":250380},{"text":"времена","train_count":256,"old_ids":[199863,1424],"score":256,"new_id":250381},{"text":"дворе","train_count":128,"old_ids":[6733,149086,1450],"score":256,"new_id":250382},{"text":"закон","train_count":128,"old_ids":[8688,148318,16806],"score":256,"new_id":250383},{"text":"колонии","train_count":128,"old_ids":[45527,160115,44633],"score":256,"new_id":250384},{"text":"министров","train_count":128,"old_ids":[150704,32884,148327],"score":256,"new_id":250385},{"text":"множества","train_count":128,"old_ids":[178416,16467,150167],"score":256,"new_id":250386},{"text":"полёта","train_count":128,"old_ids":[148634,42565,42074],"score":256,"new_id":250387},{"text":"правительство","train_count":256,"old_ids":[148607,158663],"score":256,"new_id":250388},{"text":"проживало","train_count":128,"old_ids":[149490,151303,150182],"score":256,"new_id":250389},{"text":"свободной","train_count":64,"old_ids":[2175,5301,148280,148277,148317],"score":256,"new_id":250390},{"text":"свободный","train_count":64,"old_ids":[2175,5301,148280,148277,32401],"score":256,"new_id":250391},{"text":"следующих","train_count":128,"old_ids":[149161,68608,64392],"score":256,"new_id":250392},{"text":"турнир","train_count":128,"old_ids":[152238,21918,2115],"score":256,"new_id":250393},{"text":"университетов","train_count":64,"old_ids":[3652,150439,152898,7921,148275],"score":256,"new_id":250394},{"text":"экономической","train_count":128,"old_ids":[65131,150192,158585],"score":256,"new_id":250395},{"text":"Балтийского","train_count":85,"old_ids":[58297,15481,155198,149280],"score":255,"new_id":250396},{"text":"Зеландии","train_count":85,"old_ids":[34279,11777,72884,44633],"score":255,"new_id":250397},{"text":"Победитель","train_count":85,"old_ids":[16362,148280,148276,53062],"score":255,"new_id":250398},{"text":"Словакии","train_count":85,"old_ids":[18727,160162,16256,1739],"score":255,"new_id":250399},{"text":"Торонто","train_count":85,"old_ids":[32868,8789,16806,23858],"score":255,"new_id":250400},{"text":"английских","train_count":85,"old_ids":[7105,151203,11796,150584],"score":255,"new_id":250401},{"text":"кинофестивале","train_count":51,"old_ids":[16256,13292,18768,34981,26125,150188],"score":255,"new_id":250402},{"text":"многочисленными","train_count":51,"old_ids":[178416,14942,36103,13113,58361,149195],"score":255,"new_id":250403},{"text":"небольшое","train_count":85,"old_ids":[76069,9629,148701,42317],"score":255,"new_id":250404},{"text":"основанная","train_count":85,"old_ids":[22078,148366,7105,42308],"score":255,"new_id":250405},{"text":"праздник","train_count":85,"old_ids":[7754,148464,6733,67494],"score":255,"new_id":250406},{"text":"принимали","train_count":85,"old_ids":[148489,18372,15630,45679],"score":255,"new_id":250407},{"text":"сочинений","train_count":85,"old_ids":[2175,36103,18372,59318],"score":255,"new_id":250408},{"text":"сравнительно","train_count":85,"old_ids":[2175,148381,150196,13292],"score":255,"new_id":250409},{"text":"французских","train_count":85,"old_ids":[18768,33437,163227,150584],"score":255,"new_id":250410},{"text":"чрезвычайно","train_count":85,"old_ids":[5831,150564,158294,13292],"score":255,"new_id":250411},{"text":"Ещё","train_count":254,"old_ids":[53241,155427],"score":254,"new_id":250412},{"text":"СНГ","train_count":127,"old_ids":[18727,19582,35884],"score":254,"new_id":250413},{"text":"Участвовал","train_count":127,"old_ids":[39243,148659,199375],"score":254,"new_id":250414},{"text":"боевых","train_count":254,"old_ids":[203048,155037],"score":254,"new_id":250415},{"text":"буквы","train_count":127,"old_ids":[9629,63855,148705],"score":254,"new_id":250416},{"text":"всем","train_count":254,"old_ids":[5301,159707],"score":254,"new_id":250417},{"text":"князей","train_count":127,"old_ids":[4640,37855,161809],"score":254,"new_id":250418},{"text":"начался","train_count":127,"old_ids":[29196,15481,19937],"score":254,"new_id":250419},{"text":"прозвище","train_count":127,"old_ids":[149490,182867,150006],"score":254,"new_id":250420},{"text":"ребёнка","train_count":127,"old_ids":[149467,42565,155151],"score":254,"new_id":250421},{"text":"сложности","train_count":127,"old_ids":[2175,148608,65360],"score":254,"new_id":250422},{"text":"специальных","train_count":127,"old_ids":[155036,14666,152994],"score":254,"new_id":250423},{"text":"устройства","train_count":254,"old_ids":[3652,164011],"score":254,"new_id":250424},{"text":"участка","train_count":127,"old_ids":[27448,80378,12751],"score":254,"new_id":250425},{"text":"учреждений","train_count":127,"old_ids":[27448,148279,156989],"score":254,"new_id":250426},{"text":"птиц","train_count":253,"old_ids":[160142,10064],"score":253,"new_id":250427},{"text":"Анастасия","train_count":63,"old_ids":[34960,13785,6389,17417,37835],"score":252,"new_id":250428},{"text":"Армения","train_count":126,"old_ids":[199593,6240,17941],"score":252,"new_id":250429},{"text":"Вьетнаме","train_count":84,"old_ids":[15729,210216,13785,148446],"score":252,"new_id":250430},{"text":"Государственная","train_count":84,"old_ids":[35884,22078,150968,175360],"score":252,"new_id":250431},{"text":"Железнодорожные","train_count":63,"old_ids":[154849,77969,8688,166815,159327],"score":252,"new_id":250432},{"text":"Локомотив","train_count":84,"old_ids":[213334,45527,78330,26125],"score":252,"new_id":250433},{"text":"Мэри","train_count":126,"old_ids":[36736,65131,29119],"score":252,"new_id":250434},{"text":"Османской","train_count":84,"old_ids":[19736,2175,155066,149193],"score":252,"new_id":250435},{"text":"Сибирь","train_count":84,"old_ids":[18727,151223,74990,4669],"score":252,"new_id":250436},{"text":"Татьяна","train_count":126,"old_ids":[32868,154764,13785],"score":252,"new_id":250437},{"text":"белорусской","train_count":63,"old_ids":[9629,11777,8789,41635,149193],"score":252,"new_id":250438},{"text":"военно","train_count":252,"old_ids":[224606,13292],"score":252,"new_id":250439},{"text":"вошли","train_count":252,"old_ids":[148331,151120],"score":252,"new_id":250440},{"text":"герцога","train_count":126,"old_ids":[162127,153238,148369],"score":252,"new_id":250441},{"text":"католический","train_count":63,"old_ids":[4640,149900,87018,21785,149537],"score":252,"new_id":250442},{"text":"корреспондентом","train_count":42,"old_ids":[153589,2115,149097,16806,6733,18110,148288],"score":252,"new_id":250443},{"text":"митрополит","train_count":84,"old_ids":[157255,148283,148634,9080],"score":252,"new_id":250444},{"text":"общественных","train_count":126,"old_ids":[149350,150198,42240],"score":252,"new_id":250445},{"text":"переход","train_count":126,"old_ids":[149208,93834,148277],"score":252,"new_id":250446},{"text":"посвящён","train_count":63,"old_ids":[7754,22078,152139,155427,2040],"score":252,"new_id":250447},{"text":"пределами","train_count":126,"old_ids":[149790,55986,150190],"score":252,"new_id":250448},{"text":"предполагалось","train_count":63,"old_ids":[177917,148428,151286,15481,95306],"score":252,"new_id":250449},{"text":"символом","train_count":84,"old_ids":[2175,72326,148378,6240],"score":252,"new_id":250450},{"text":"системах","train_count":126,"old_ids":[167641,149721,10161],"score":252,"new_id":250451},{"text":"содержится","train_count":84,"old_ids":[2175,148277,55638,92907],"score":252,"new_id":250452},{"text":"стихотворение","train_count":63,"old_ids":[52598,10161,12666,149086,16669],"score":252,"new_id":250453},{"text":"требует","train_count":126,"old_ids":[1729,149467,69632],"score":252,"new_id":250454},{"text":"электропоездов","train_count":63,"old_ids":[236536,7754,42317,8688,148716],"score":252,"new_id":250455},{"text":"Лучший","train_count":251,"old_ids":[225635,151642],"score":251,"new_id":250456},{"text":"групп","train_count":251,"old_ids":[14942,83147],"score":251,"new_id":250457},{"text":"самый","train_count":251,"old_ids":[160002,33477],"score":251,"new_id":250458},{"text":"члены","train_count":251,"old_ids":[5831,154641],"score":251,"new_id":250459},{"text":"Памятник","train_count":125,"old_ids":[180371,79142,186469],"score":250,"new_id":250460},{"text":"Партия","train_count":125,"old_ids":[16362,16763,152149],"score":250,"new_id":250461},{"text":"Пермского","train_count":125,"old_ids":[95169,6240,149280],"score":250,"new_id":250462},{"text":"месторождения","train_count":125,"old_ids":[150436,148367,149880],"score":250,"new_id":250463},{"text":"метро","train_count":250,"old_ids":[148557,148283],"score":250,"new_id":250464},{"text":"музыкальной","train_count":125,"old_ids":[148405,168062,159075],"score":250,"new_id":250465},{"text":"музыкальных","train_count":125,"old_ids":[148405,168062,159076],"score":250,"new_id":250466},{"text":"настолько","train_count":125,"old_ids":[13785,6389,157031],"score":250,"new_id":250467},{"text":"общего","train_count":250,"old_ids":[149350,148702],"score":250,"new_id":250468},{"text":"премьера","train_count":125,"old_ids":[149790,6240,183334],"score":250,"new_id":250469},{"text":"скорость","train_count":250,"old_ids":[167465,93309],"score":250,"new_id":250470},{"text":"считает","train_count":125,"old_ids":[2175,87148,26046],"score":250,"new_id":250471},{"text":"техники","train_count":250,"old_ids":[157971,149665],"score":250,"new_id":250472},{"text":"товаров","train_count":125,"old_ids":[148779,16763,148275],"score":250,"new_id":250473},{"text":"футбол","train_count":125,"old_ids":[18768,36183,168460],"score":250,"new_id":250474},{"text":"Аргентины","train_count":83,"old_ids":[34960,89527,151844,12014],"score":249,"new_id":250475},{"text":"Новосёлки","train_count":83,"old_ids":[193897,22078,162292,16256],"score":249,"new_id":250476},{"text":"Уильяма","train_count":83,"old_ids":[39243,64257,4096,66050],"score":249,"new_id":250477},{"text":"актрисы","train_count":83,"old_ids":[148318,153298,13113,4403],"score":249,"new_id":250478},{"text":"гмина","train_count":249,"old_ids":[14942,149717],"score":249,"new_id":250479},{"text":"исследователи","train_count":83,"old_ids":[13113,149161,148275,158140],"score":249,"new_id":250480},{"text":"королевы","train_count":83,"old_ids":[4640,148367,148284,148705],"score":249,"new_id":250481},{"text":"машины","train_count":249,"old_ids":[66050,215958],"score":249,"new_id":250482},{"text":"млекопитающих","train_count":83,"old_ids":[6240,176068,159236,150051],"score":249,"new_id":250483},{"text":"православная","train_count":83,"old_ids":[148607,22078,150800,42308],"score":249,"new_id":250484},{"text":"принесла","train_count":83,"old_ids":[148489,18372,21785,29354],"score":249,"new_id":250485},{"text":"сейчас","train_count":249,"old_ids":[174220,155183],"score":249,"new_id":250486},{"text":"создавать","train_count":83,"old_ids":[2175,148303,38183,150531],"score":249,"new_id":250487},{"text":"составлял","train_count":83,"old_ids":[2175,150838,13302,2941],"score":249,"new_id":250488},{"text":"телесериал","train_count":83,"old_ids":[150141,21785,149098,15481],"score":249,"new_id":250489},{"text":"Арутюнян","train_count":62,"old_ids":[34960,19160,188954,37855,2040],"score":248,"new_id":250490},{"text":"Гарри","train_count":124,"old_ids":[35884,16763,29119],"score":248,"new_id":250491},{"text":"Доктор","train_count":124,"old_ids":[23355,148518,35458],"score":248,"new_id":250492},{"text":"Мазовецкое","train_count":62,"old_ids":[36736,148646,150937,10064,153232],"score":248,"new_id":250493},{"text":"Суперкубка","train_count":62,"old_ids":[18727,47174,7358,34083,175231],"score":248,"new_id":250494},{"text":"архиепископом","train_count":62,"old_ids":[16763,229751,28696,45527,151201],"score":248,"new_id":250495},{"text":"высоту","train_count":124,"old_ids":[148705,217181,3652],"score":248,"new_id":250496},{"text":"исполнение","train_count":124,"old_ids":[13113,148634,150956],"score":248,"new_id":250497},{"text":"начинают","train_count":124,"old_ids":[29196,66353,66694],"score":248,"new_id":250498},{"text":"определённой","train_count":62,"old_ids":[148561,148279,55986,42565,170968],"score":248,"new_id":250499},{"text":"памятника","train_count":124,"old_ids":[148414,79142,223439],"score":248,"new_id":250500},{"text":"появления","train_count":124,"old_ids":[7754,58255,150548],"score":248,"new_id":250501},{"text":"принадлежало","train_count":62,"old_ids":[148895,149578,148284,16467,150182],"score":248,"new_id":250502},{"text":"прямой","train_count":124,"old_ids":[148489,150686,148293],"score":248,"new_id":250503},{"text":"психологии","train_count":124,"old_ids":[7754,151853,158283],"score":248,"new_id":250504},{"text":"труда","train_count":248,"old_ids":[170476,38183],"score":248,"new_id":250505},{"text":"чаще","train_count":248,"old_ids":[148659,150006],"score":248,"new_id":250506},{"text":"дело","train_count":247,"old_ids":[55986,1403],"score":247,"new_id":250507},{"text":"длина","train_count":247,"old_ids":[6733,155565],"score":247,"new_id":250508},{"text":"иметь","train_count":247,"old_ids":[15630,148877],"score":247,"new_id":250509},{"text":"рублей","train_count":247,"old_ids":[181327,152134],"score":247,"new_id":250510},{"text":"Алекса","train_count":123,"old_ids":[34960,148470,148629],"score":246,"new_id":250511},{"text":"Востока","train_count":123,"old_ids":[15729,151290,12751],"score":246,"new_id":250512},{"text":"Иванов","train_count":123,"old_ids":[29189,149137,148275],"score":246,"new_id":250513},{"text":"выпустила","train_count":123,"old_ids":[148705,153449,29354],"score":246,"new_id":250514},{"text":"заведения","train_count":82,"old_ids":[8688,148292,148276,17941],"score":246,"new_id":250515},{"text":"задач","train_count":123,"old_ids":[148397,38183,5831],"score":246,"new_id":250516},{"text":"королевской","train_count":82,"old_ids":[4640,148367,151186,149193],"score":246,"new_id":250517},{"text":"литературной","train_count":82,"old_ids":[159329,7358,150849,148317],"score":246,"new_id":250518},{"text":"островах","train_count":123,"old_ids":[23514,169887,10161],"score":246,"new_id":250519},{"text":"офицера","train_count":82,"old_ids":[1403,148594,10064,87658],"score":246,"new_id":250520},{"text":"перехода","train_count":123,"old_ids":[149208,93834,150028],"score":246,"new_id":250521},{"text":"персонажа","train_count":123,"old_ids":[149208,152590,148458],"score":246,"new_id":250522},{"text":"посетил","train_count":82,"old_ids":[7754,22078,7921,61057],"score":246,"new_id":250523},{"text":"появляются","train_count":123,"old_ids":[7754,58255,155537],"score":246,"new_id":250524},{"text":"процессора","train_count":82,"old_ids":[149490,10064,90202,148676],"score":246,"new_id":250525},{"text":"разрешение","train_count":82,"old_ids":[148464,2115,44432,16669],"score":246,"new_id":250526},{"text":"связанных","train_count":123,"old_ids":[2175,163720,151990],"score":246,"new_id":250527},{"text":"экипажа","train_count":123,"old_ids":[65131,168267,148458],"score":246,"new_id":250528},{"text":"электронной","train_count":82,"old_ids":[65131,149157,151673,148317],"score":246,"new_id":250529},{"text":"Данные","train_count":245,"old_ids":[23355,151979],"score":245,"new_id":250530},{"text":"армия","train_count":245,"old_ids":[16763,174531],"score":245,"new_id":250531},{"text":"вице","train_count":245,"old_ids":[148341,149053],"score":245,"new_id":250532},{"text":"средней","train_count":245,"old_ids":[174301,148820],"score":245,"new_id":250533},{"text":"Владимиром","train_count":61,"old_ids":[15729,148606,15630,74990,148288],"score":244,"new_id":250534},{"text":"Екатеринбург","train_count":61,"old_ids":[53241,150843,7358,18372,163082],"score":244,"new_id":250535},{"text":"Кинокомедии","train_count":61,"old_ids":[25497,158231,148558,148276,44633],"score":244,"new_id":250536},{"text":"академический","train_count":61,"old_ids":[1424,154408,6240,77436,149537],"score":244,"new_id":250537},{"text":"башня","train_count":122,"old_ids":[9629,48989,37855],"score":244,"new_id":250538},{"text":"век","train_count":244,"old_ids":[5301,148301],"score":244,"new_id":250539},{"text":"железнодорожный","train_count":61,"old_ids":[16467,77969,8688,166815,171685],"score":244,"new_id":250540},{"text":"математического","train_count":61,"old_ids":[6240,7721,148281,7721,158638],"score":244,"new_id":250541},{"text":"направлен","train_count":122,"old_ids":[13785,148607,58361],"score":244,"new_id":250542},{"text":"районом","train_count":122,"old_ids":[180093,1403,148524],"score":244,"new_id":250543},{"text":"реконструкции","train_count":122,"old_ids":[148279,150270,206818],"score":244,"new_id":250544},{"text":"символ","train_count":122,"old_ids":[2175,72326,148294],"score":244,"new_id":250545},{"text":"стихи","train_count":244,"old_ids":[52598,150931],"score":244,"new_id":250546},{"text":"учёных","train_count":122,"old_ids":[27448,42565,42240],"score":244,"new_id":250547},{"text":"Альберт","train_count":81,"old_ids":[34960,27112,150757,1729],"score":243,"new_id":250548},{"text":"Анатольевич","train_count":81,"old_ids":[34960,178595,148319,198326],"score":243,"new_id":250549},{"text":"Владислав","train_count":81,"old_ids":[15729,148606,13113,150800],"score":243,"new_id":250550},{"text":"Иоганн","train_count":81,"old_ids":[29189,149095,7105,2040],"score":243,"new_id":250551},{"text":"Одесса","train_count":81,"old_ids":[19736,6733,90202,1424],"score":243,"new_id":250552},{"text":"Современные","train_count":81,"old_ids":[18727,148275,148801,41546],"score":243,"new_id":250553},{"text":"аппарат","train_count":81,"old_ids":[1424,74359,16763,7721],"score":243,"new_id":250554},{"text":"временем","train_count":243,"old_ids":[199863,148281],"score":243,"new_id":250555},{"text":"департамента","train_count":81,"old_ids":[95525,151963,155374,84706],"score":243,"new_id":250556},{"text":"европейской","train_count":81,"old_ids":[148380,148283,152679,149193],"score":243,"new_id":250557},{"text":"завоевания","train_count":81,"old_ids":[8688,148292,42317,155024],"score":243,"new_id":250558},{"text":"инструменты","train_count":81,"old_ids":[18372,150379,89315,4403],"score":243,"new_id":250559},{"text":"народный","train_count":243,"old_ids":[151150,32401],"score":243,"new_id":250560},{"text":"политическая","train_count":81,"old_ids":[148634,9080,77436,150621],"score":243,"new_id":250561},{"text":"полномочия","train_count":81,"old_ids":[148634,148524,36103,37835],"score":243,"new_id":250562},{"text":"посмертно","train_count":81,"old_ids":[7754,22078,60909,159113],"score":243,"new_id":250563},{"text":"теперь","train_count":243,"old_ids":[48824,171404],"score":243,"new_id":250564},{"text":"чемпионатов","train_count":81,"old_ids":[155356,148686,16806,152968],"score":243,"new_id":250565},{"text":"Латвии","train_count":121,"old_ids":[61757,7721,210740],"score":242,"new_id":250566},{"text":"альбоме","train_count":121,"old_ids":[36771,173910,1450],"score":242,"new_id":250567},{"text":"белого","train_count":121,"old_ids":[9629,11777,148449],"score":242,"new_id":250568},{"text":"братом","train_count":121,"old_ids":[9629,77779,148288],"score":242,"new_id":250569},{"text":"искусство","train_count":121,"old_ids":[13113,150352,148593],"score":242,"new_id":250570},{"text":"историко","train_count":121,"old_ids":[32884,8789,151907],"score":242,"new_id":250571},{"text":"мужа","train_count":121,"old_ids":[6240,54025,1424],"score":242,"new_id":250572},{"text":"поезда","train_count":121,"old_ids":[7754,42317,159070],"score":242,"new_id":250573},{"text":"поэзии","train_count":121,"old_ids":[148428,65131,155585],"score":242,"new_id":250574},{"text":"силу","train_count":242,"old_ids":[177419,3652],"score":242,"new_id":250575},{"text":"устройство","train_count":242,"old_ids":[3652,164492],"score":242,"new_id":250576},{"text":"участвовала","train_count":121,"old_ids":[27448,1424,229736],"score":242,"new_id":250577},{"text":"ОАО","train_count":241,"old_ids":[19736,156691],"score":241,"new_id":250578},{"text":"языки","train_count":241,"old_ids":[214432,16256],"score":241,"new_id":250579},{"text":"Васильев","train_count":80,"old_ids":[15729,17417,64257,148380],"score":240,"new_id":250580},{"text":"Византии","train_count":80,"old_ids":[15729,39427,150842,1739],"score":240,"new_id":250581},{"text":"Далее","train_count":120,"old_ids":[23355,15481,47745],"score":240,"new_id":250582},{"text":"Животные","train_count":80,"old_ids":[154849,26125,12666,41546],"score":240,"new_id":250583},{"text":"Красный","train_count":120,"old_ids":[25497,148761,32401],"score":240,"new_id":250584},{"text":"Никола","train_count":120,"old_ids":[19582,151907,29354],"score":240,"new_id":250585},{"text":"Появились","train_count":120,"old_ids":[16362,58255,236525],"score":240,"new_id":250586},{"text":"РККА","train_count":120,"old_ids":[32392,25497,152508],"score":240,"new_id":250587},{"text":"Содержание","train_count":80,"old_ids":[18727,148277,55638,25767],"score":240,"new_id":250588},{"text":"Тверской","train_count":120,"old_ids":[32868,46576,149193],"score":240,"new_id":250589},{"text":"Франция","train_count":240,"old_ids":[221647,68335],"score":240,"new_id":250590},{"text":"боях","train_count":120,"old_ids":[9629,58255,10161],"score":240,"new_id":250591},{"text":"букв","train_count":120,"old_ids":[9629,63855,5301],"score":240,"new_id":250592},{"text":"вагонов","train_count":120,"old_ids":[90219,63046,148366],"score":240,"new_id":250593},{"text":"гораздо","train_count":120,"old_ids":[152974,148646,148551],"score":240,"new_id":250594},{"text":"государственная","train_count":80,"old_ids":[14942,22078,150968,175360],"score":240,"new_id":250595},{"text":"известных","train_count":240,"old_ids":[244718,42240],"score":240,"new_id":250596},{"text":"литературного","train_count":80,"old_ids":[159329,7358,150849,148325],"score":240,"new_id":250597},{"text":"металлов","train_count":120,"old_ids":[148557,15481,148569],"score":240,"new_id":250598},{"text":"миллиона","train_count":80,"old_ids":[6240,61057,10750,71221],"score":240,"new_id":250599},{"text":"направления","train_count":240,"old_ids":[13785,192885],"score":240,"new_id":250600},{"text":"небольшие","train_count":120,"old_ids":[76069,170946,150366],"score":240,"new_id":250601},{"text":"обладателем","train_count":60,"old_ids":[148280,148606,7721,77969,6240],"score":240,"new_id":250602},{"text":"определяет","train_count":80,"old_ids":[148561,148420,41580,7921],"score":240,"new_id":250603},{"text":"памятников","train_count":120,"old_ids":[148414,79142,228756],"score":240,"new_id":250604},{"text":"полк","train_count":240,"old_ids":[148634,4640],"score":240,"new_id":250605},{"text":"появляться","train_count":60,"old_ids":[7754,58255,5301,13302,151381],"score":240,"new_id":250606},{"text":"преподаватель","train_count":120,"old_ids":[149790,161978,173926],"score":240,"new_id":250607},{"text":"рабочего","train_count":120,"old_ids":[21441,151293,148702],"score":240,"new_id":250608},{"text":"развитию","train_count":120,"old_ids":[148464,149734,163855],"score":240,"new_id":250609},{"text":"связана","train_count":120,"old_ids":[2175,163720,148841],"score":240,"new_id":250610},{"text":"составила","train_count":120,"old_ids":[2175,150838,150174],"score":240,"new_id":250611},{"text":"тысячелетия","train_count":48,"old_ids":[82942,19937,5831,11777,7921,37835],"score":240,"new_id":250612},{"text":"удостоен","train_count":120,"old_ids":[34723,151290,5092],"score":240,"new_id":250613},{"text":"вклад","train_count":239,"old_ids":[5301,149450],"score":239,"new_id":250614},{"text":"смог","train_count":239,"old_ids":[170116,14942],"score":239,"new_id":250615},{"text":"старший","train_count":239,"old_ids":[193565,151642],"score":239,"new_id":250616},{"text":"Украина","train_count":238,"old_ids":[208317,66353],"score":238,"new_id":250617},{"text":"Участник","train_count":119,"old_ids":[39243,150813,67494],"score":238,"new_id":250618},{"text":"вошла","train_count":238,"old_ids":[148331,153870],"score":238,"new_id":250619},{"text":"дивизионе","train_count":119,"old_ids":[150394,158668,148917],"score":238,"new_id":250620},{"text":"критик","train_count":119,"old_ids":[4640,149454,36473],"score":238,"new_id":250621},{"text":"нападающий","train_count":119,"old_ids":[13785,159463,155099],"score":238,"new_id":250622},{"text":"наследия","train_count":119,"old_ids":[13785,149161,37835],"score":238,"new_id":250623},{"text":"областной","train_count":119,"old_ids":[148280,29354,181671],"score":238,"new_id":250624},{"text":"орудий","train_count":119,"old_ids":[8789,34723,42387],"score":238,"new_id":250625},{"text":"основы","train_count":119,"old_ids":[22078,148366,4403],"score":238,"new_id":250626},{"text":"оценки","train_count":119,"old_ids":[1403,149147,16256],"score":238,"new_id":250627},{"text":"смерть","train_count":119,"old_ids":[2175,60909,27851],"score":238,"new_id":250628},{"text":"современные","train_count":119,"old_ids":[149297,148801,41546],"score":238,"new_id":250629},{"text":"составляли","train_count":119,"old_ids":[2175,150838,245654],"score":238,"new_id":250630},{"text":"участвовали","train_count":119,"old_ids":[27448,1424,198847],"score":238,"new_id":250631},{"text":"Вологодской","train_count":79,"old_ids":[15729,148378,149224,149193],"score":237,"new_id":250632},{"text":"Новой","train_count":237,"old_ids":[19582,168715],"score":237,"new_id":250633},{"text":"Новый","train_count":237,"old_ids":[193897,33477],"score":237,"new_id":250634},{"text":"Центральная","train_count":79,"old_ids":[89602,18110,157991,42308],"score":237,"new_id":250635},{"text":"Челябинск","train_count":79,"old_ids":[70345,41580,151993,62847],"score":237,"new_id":250636},{"text":"Эндрю","train_count":79,"old_ids":[89030,2040,60659,11577],"score":237,"new_id":250637},{"text":"выхода","train_count":237,"old_ids":[155037,150028],"score":237,"new_id":250638},{"text":"главной","train_count":237,"old_ids":[149809,162489],"score":237,"new_id":250639},{"text":"государством","train_count":79,"old_ids":[14942,22078,150968,152225],"score":237,"new_id":250640},{"text":"жители","train_count":237,"old_ids":[16467,153009],"score":237,"new_id":250641},{"text":"исполнителей","train_count":79,"old_ids":[13113,148634,2040,150471],"score":237,"new_id":250642},{"text":"крепости","train_count":237,"old_ids":[151219,81839],"score":237,"new_id":250643},{"text":"крупнейший","train_count":79,"old_ids":[4640,149568,148820,151642],"score":237,"new_id":250644},{"text":"мероприятия","train_count":79,"old_ids":[60909,148561,29119,154449],"score":237,"new_id":250645},{"text":"назад","train_count":237,"old_ids":[13785,159803],"score":237,"new_id":250646},{"text":"построил","train_count":79,"old_ids":[7754,23514,148283,61057],"score":237,"new_id":250647},{"text":"пулемётов","train_count":79,"old_ids":[148729,151032,42565,148779],"score":237,"new_id":250648},{"text":"церемонии","train_count":79,"old_ids":[180269,148281,16806,44633],"score":237,"new_id":250649},{"text":"Другое","train_count":118,"old_ids":[23355,62215,42317],"score":236,"new_id":250650},{"text":"Советский","train_count":118,"old_ids":[173518,45405,149800],"score":236,"new_id":250651},{"text":"адмирал","train_count":118,"old_ids":[149099,159733,15481],"score":236,"new_id":250652},{"text":"аэродром","train_count":59,"old_ids":[1424,65131,148283,60659,148288],"score":236,"new_id":250653},{"text":"брата","train_count":236,"old_ids":[157471,40710],"score":236,"new_id":250654},{"text":"входила","train_count":236,"old_ids":[5301,238016],"score":236,"new_id":250655},{"text":"входят","train_count":236,"old_ids":[5301,150782],"score":236,"new_id":250656},{"text":"выполнения","train_count":118,"old_ids":[148705,148634,149803],"score":236,"new_id":250657},{"text":"изготовления","train_count":118,"old_ids":[39427,149341,59595],"score":236,"new_id":250658},{"text":"информацию","train_count":118,"old_ids":[18372,54724,150451],"score":236,"new_id":250659},{"text":"исторического","train_count":118,"old_ids":[32884,8789,158638],"score":236,"new_id":250660},{"text":"объект","train_count":118,"old_ids":[148280,152743,1729],"score":236,"new_id":250661},{"text":"определённых","train_count":59,"old_ids":[148561,148279,55986,42565,170741],"score":236,"new_id":250662},{"text":"подразделения","train_count":118,"old_ids":[155908,148464,154426],"score":236,"new_id":250663},{"text":"премия","train_count":236,"old_ids":[149790,174531],"score":236,"new_id":250664},{"text":"присоединения","train_count":59,"old_ids":[148489,13113,42317,67958,17941],"score":236,"new_id":250665},{"text":"социальной","train_count":118,"old_ids":[148454,14666,151291],"score":236,"new_id":250666},{"text":"стадион","train_count":118,"old_ids":[148613,148467,16806],"score":236,"new_id":250667},{"text":"существовало","train_count":118,"old_ids":[2175,154659,177948],"score":236,"new_id":250668},{"text":"сыновей","train_count":118,"old_ids":[54330,148366,148298],"score":236,"new_id":250669},{"text":"факты","train_count":118,"old_ids":[18768,148318,82942],"score":236,"new_id":250670},{"text":"Нидерландах","train_count":47,"old_ids":[19582,45587,7358,2941,158021,10161],"score":235,"new_id":250671},{"text":"концу","train_count":235,"old_ids":[150270,152213],"score":235,"new_id":250672},{"text":"сельсовет","train_count":235,"old_ids":[226181,227012],"score":235,"new_id":250673},{"text":"Австралия","train_count":78,"old_ids":[34960,5301,149389,150095],"score":234,"new_id":250674},{"text":"Георгиевич","train_count":78,"old_ids":[228209,8789,153277,159380],"score":234,"new_id":250675},{"text":"Министры","train_count":78,"old_ids":[36736,18372,32884,36747],"score":234,"new_id":250676},{"text":"Пресвятой","train_count":78,"old_ids":[51844,21785,152139,170627],"score":234,"new_id":250677},{"text":"Считается","train_count":117,"old_ids":[18727,87148,64903],"score":234,"new_id":250678},{"text":"Фридриха","train_count":78,"old_ids":[52337,29119,186983,150532],"score":234,"new_id":250679},{"text":"буквально","train_count":78,"old_ids":[9629,63855,5301,149749],"score":234,"new_id":250680},{"text":"весь","train_count":234,"old_ids":[5301,88434],"score":234,"new_id":250681},{"text":"глава","train_count":234,"old_ids":[149809,90219],"score":234,"new_id":250682},{"text":"заканчивается","train_count":78,"old_ids":[8688,148318,7105,163748],"score":234,"new_id":250683},{"text":"инженером","train_count":78,"old_ids":[18372,148942,7358,148288],"score":234,"new_id":250684},{"text":"кардинал","train_count":117,"old_ids":[149888,153462,2941],"score":234,"new_id":250685},{"text":"королевского","train_count":78,"old_ids":[4640,148367,151186,149280],"score":234,"new_id":250686},{"text":"международный","train_count":117,"old_ids":[148446,151896,32401],"score":234,"new_id":250687},{"text":"небольшая","train_count":78,"old_ids":[76069,9629,148701,34226],"score":234,"new_id":250688},{"text":"последующие","train_count":117,"old_ids":[167014,68608,47552],"score":234,"new_id":250689},{"text":"решает","train_count":117,"old_ids":[2115,44432,26046],"score":234,"new_id":250690},{"text":"серебряный","train_count":78,"old_ids":[2175,148579,151439,32401],"score":234,"new_id":250691},{"text":"социально","train_count":117,"old_ids":[148454,14666,149749],"score":234,"new_id":250692},{"text":"шесть","train_count":234,"old_ids":[56916,33348],"score":234,"new_id":250693},{"text":"больших","train_count":233,"old_ids":[170946,150047],"score":233,"new_id":250694},{"text":"народа","train_count":233,"old_ids":[13785,192044],"score":233,"new_id":250695},{"text":"Владимирской","train_count":58,"old_ids":[15729,148606,15630,74990,149193],"score":232,"new_id":250696},{"text":"Военно","train_count":116,"old_ids":[173647,5092,13292],"score":232,"new_id":250697},{"text":"Георгиевский","train_count":58,"old_ids":[228209,8789,153277,5301,149800],"score":232,"new_id":250698},{"text":"Евгеньевич","train_count":58,"old_ids":[53241,5301,14942,49603,198326],"score":232,"new_id":250699},{"text":"Ленинградская","train_count":58,"old_ids":[61757,9396,2040,153987,152141],"score":232,"new_id":250700},{"text":"Лиги","train_count":116,"old_ids":[61757,66951,1739],"score":232,"new_id":250701},{"text":"Лондон","train_count":116,"old_ids":[61757,16806,182503],"score":232,"new_id":250702},{"text":"Олег","train_count":232,"old_ids":[19736,156337],"score":232,"new_id":250703},{"text":"Степан","train_count":116,"old_ids":[18727,48824,181841],"score":232,"new_id":250704},{"text":"втором","train_count":232,"old_ids":[167396,148288],"score":232,"new_id":250705},{"text":"изменений","train_count":116,"old_ids":[39427,36621,59318],"score":232,"new_id":250706},{"text":"искусстве","train_count":116,"old_ids":[13113,150352,151418],"score":232,"new_id":250707},{"text":"композиций","train_count":116,"old_ids":[148558,161339,150585],"score":232,"new_id":250708},{"text":"конструкции","train_count":232,"old_ids":[150270,206818],"score":232,"new_id":250709},{"text":"красного","train_count":116,"old_ids":[4640,148761,148325],"score":232,"new_id":250710},{"text":"массы","train_count":116,"old_ids":[6240,27473,4403],"score":232,"new_id":250711},{"text":"модификация","train_count":58,"old_ids":[6240,148277,59627,71658,68335],"score":232,"new_id":250712},{"text":"мусульман","train_count":58,"old_ids":[6240,41635,3652,27112,155066],"score":232,"new_id":250713},{"text":"несколькими","train_count":116,"old_ids":[149904,193929,155461],"score":232,"new_id":250714},{"text":"орудия","train_count":116,"old_ids":[8789,34723,37835],"score":232,"new_id":250715},{"text":"певец","train_count":232,"old_ids":[157354,154800],"score":232,"new_id":250716},{"text":"побед","train_count":116,"old_ids":[7754,148280,148276],"score":232,"new_id":250717},{"text":"свидетельствуют","train_count":58,"old_ids":[2175,152833,7921,24623,151766],"score":232,"new_id":250718},{"text":"связаны","train_count":116,"old_ids":[2175,163720,150165],"score":232,"new_id":250719},{"text":"специальные","train_count":116,"old_ids":[155036,14666,158444],"score":232,"new_id":250720},{"text":"Афганистана","train_count":77,"old_ids":[34960,18768,236997,162200],"score":231,"new_id":250721},{"text":"Ленинграда","train_count":77,"old_ids":[61757,9396,2040,196749],"score":231,"new_id":250722},{"text":"Робертс","train_count":77,"old_ids":[32392,148280,153922,2175],"score":231,"new_id":250723},{"text":"академика","train_count":77,"old_ids":[1424,154408,61628,12751],"score":231,"new_id":250724},{"text":"британского","train_count":77,"old_ids":[9629,149454,7105,149280],"score":231,"new_id":250725},{"text":"занимались","train_count":77,"old_ids":[8688,13603,6240,158605],"score":231,"new_id":250726},{"text":"командующий","train_count":77,"old_ids":[148558,72884,68608,42387],"score":231,"new_id":250727},{"text":"мероприятий","train_count":77,"old_ids":[60909,148561,29119,163260],"score":231,"new_id":250728},{"text":"называемых","train_count":77,"old_ids":[13785,152953,153056,60038],"score":231,"new_id":250729},{"text":"опубликована","train_count":77,"old_ids":[148561,149090,148336,148841],"score":231,"new_id":250730},{"text":"созданием","train_count":77,"old_ids":[2175,148303,6733,152952],"score":231,"new_id":250731},{"text":"станций","train_count":231,"old_ids":[49224,150585],"score":231,"new_id":250732},{"text":"французском","train_count":77,"old_ids":[18768,33437,163227,153195],"score":231,"new_id":250733},{"text":"Курской","train_count":115,"old_ids":[25497,86690,148554],"score":230,"new_id":250734},{"text":"Московском","train_count":115,"old_ids":[212095,148336,153195],"score":230,"new_id":250735},{"text":"Парк","train_count":115,"old_ids":[16362,16763,4640],"score":230,"new_id":250736},{"text":"Премьер","train_count":115,"old_ids":[51844,148281,154299],"score":230,"new_id":250737},{"text":"Проект","train_count":115,"old_ids":[51844,42317,150094],"score":230,"new_id":250738},{"text":"Самый","train_count":115,"old_ids":[18727,148357,33477],"score":230,"new_id":250739},{"text":"Сейчас","train_count":115,"old_ids":[18727,148298,155183],"score":230,"new_id":250740},{"text":"лагеря","train_count":115,"old_ids":[151286,7358,4096],"score":230,"new_id":250741},{"text":"лиге","train_count":230,"old_ids":[10750,149068],"score":230,"new_id":250742},{"text":"многом","train_count":115,"old_ids":[6240,148325,6240],"score":230,"new_id":250743},{"text":"охраны","train_count":115,"old_ids":[1403,58805,4403],"score":230,"new_id":250744},{"text":"партийный","train_count":115,"old_ids":[151963,155198,32401],"score":230,"new_id":250745},{"text":"русские","train_count":230,"old_ids":[152528,153218],"score":230,"new_id":250746},{"text":"формирование","train_count":115,"old_ids":[54724,149591,25767],"score":230,"new_id":250747},{"text":"Оскар","train_count":229,"old_ids":[19736,167339],"score":229,"new_id":250748},{"text":"шоу","train_count":229,"old_ids":[155041,3652],"score":229,"new_id":250749},{"text":"Англию","train_count":76,"old_ids":[34960,2040,151203,11577],"score":228,"new_id":250750},{"text":"Иерусалима","train_count":57,"old_ids":[29189,7358,41635,45679,66050],"score":228,"new_id":250751},{"text":"Издательство","train_count":76,"old_ids":[29189,171158,57868,148593],"score":228,"new_id":250752},{"text":"Императорской","train_count":57,"old_ids":[29189,6240,149208,55389,149193],"score":228,"new_id":250753},{"text":"Использование","train_count":114,"old_ids":[29189,151256,166506],"score":228,"new_id":250754},{"text":"Международные","train_count":114,"old_ids":[181834,151896,41546],"score":228,"new_id":250755},{"text":"Музыкальные","train_count":114,"old_ids":[204723,168062,170139],"score":228,"new_id":250756},{"text":"Чемпионата","train_count":57,"old_ids":[70345,148281,148686,16806,40710],"score":228,"new_id":250757},{"text":"авиационной","train_count":57,"old_ids":[148292,1739,28376,16806,148317],"score":228,"new_id":250758},{"text":"алфавита","train_count":76,"old_ids":[15481,18768,148292,148648],"score":228,"new_id":250759},{"text":"архипелага","train_count":57,"old_ids":[16763,10161,45768,150456,148369],"score":228,"new_id":250760},{"text":"аэропорта","train_count":76,"old_ids":[1424,65131,148283,157853],"score":228,"new_id":250761},{"text":"высшее","train_count":114,"old_ids":[148705,2175,155616],"score":228,"new_id":250762},{"text":"высших","train_count":114,"old_ids":[148705,2175,150047],"score":228,"new_id":250763},{"text":"долине","train_count":114,"old_ids":[148551,10750,76069],"score":228,"new_id":250764},{"text":"друзей","train_count":114,"old_ids":[6733,63698,148298],"score":228,"new_id":250765},{"text":"заведений","train_count":76,"old_ids":[8688,148292,148276,59318],"score":228,"new_id":250766},{"text":"интересов","train_count":76,"old_ids":[18372,55055,21785,148275],"score":228,"new_id":250767},{"text":"крупнейшим","train_count":76,"old_ids":[4640,149568,148820,155204],"score":228,"new_id":250768},{"text":"мотивам","train_count":114,"old_ids":[78330,26125,148357],"score":228,"new_id":250769},{"text":"освобождён","train_count":76,"old_ids":[22078,160496,148343,185709],"score":228,"new_id":250770},{"text":"основанный","train_count":76,"old_ids":[22078,148366,26269,11796],"score":228,"new_id":250771},{"text":"очков","train_count":228,"old_ids":[36103,148336],"score":228,"new_id":250772},{"text":"передачи","train_count":228,"old_ids":[159357,154702],"score":228,"new_id":250773},{"text":"посвятил","train_count":57,"old_ids":[7754,22078,5301,149036,2941],"score":228,"new_id":250774},{"text":"посещал","train_count":76,"old_ids":[7754,22078,82557,15481],"score":228,"new_id":250775},{"text":"принадлежат","train_count":76,"old_ids":[148895,149578,148284,162570],"score":228,"new_id":250776},{"text":"радиостанции","train_count":57,"old_ids":[21441,148467,23514,7105,54201],"score":228,"new_id":250777},{"text":"решений","train_count":114,"old_ids":[2115,44432,59318],"score":228,"new_id":250778},{"text":"свободных","train_count":57,"old_ids":[2175,5301,148280,148277,42240],"score":228,"new_id":250779},{"text":"студийный","train_count":76,"old_ids":[6389,34723,42387,32401],"score":228,"new_id":250780},{"text":"учебное","train_count":76,"old_ids":[27448,1450,9629,45480],"score":228,"new_id":250781},{"text":"Там","train_count":227,"old_ids":[32868,148357],"score":227,"new_id":250782},{"text":"востоку","train_count":227,"old_ids":[194731,34083],"score":227,"new_id":250783},{"text":"дважды","train_count":227,"old_ids":[164189,153901],"score":227,"new_id":250784},{"text":"зоне","train_count":227,"old_ids":[8688,148917],"score":227,"new_id":250785},{"text":"общин","train_count":227,"old_ids":[149350,18372],"score":227,"new_id":250786},{"text":"региона","train_count":227,"old_ids":[173007,71221],"score":227,"new_id":250787},{"text":"Виды","train_count":113,"old_ids":[15729,45587,4403],"score":226,"new_id":250788},{"text":"Война","train_count":113,"old_ids":[15729,148293,13785],"score":226,"new_id":250789},{"text":"Красная","train_count":113,"old_ids":[25497,148761,42308],"score":226,"new_id":250790},{"text":"Личная","train_count":113,"old_ids":[61757,31889,42308],"score":226,"new_id":250791},{"text":"Находится","train_count":113,"old_ids":[81573,148334,92907],"score":226,"new_id":250792},{"text":"Сезон","train_count":113,"old_ids":[18727,148377,16806],"score":226,"new_id":250793},{"text":"ведущих","train_count":113,"old_ids":[148700,40794,64392],"score":226,"new_id":250794},{"text":"высокие","train_count":113,"old_ids":[148705,148454,149012],"score":226,"new_id":250795},{"text":"дивизион","train_count":113,"old_ids":[150394,158668,16806],"score":226,"new_id":250796},{"text":"литература","train_count":113,"old_ids":[159329,7358,154182],"score":226,"new_id":250797},{"text":"мощности","train_count":113,"old_ids":[6240,88260,65360],"score":226,"new_id":250798},{"text":"осадков","train_count":113,"old_ids":[22078,149099,148336],"score":226,"new_id":250799},{"text":"писателей","train_count":226,"old_ids":[28696,154660],"score":226,"new_id":250800},{"text":"пространстве","train_count":113,"old_ids":[155611,33437,151418],"score":226,"new_id":250801},{"text":"пространство","train_count":113,"old_ids":[155611,33437,148593],"score":226,"new_id":250802},{"text":"редактором","train_count":113,"old_ids":[148279,38183,202283],"score":226,"new_id":250803},{"text":"установлена","train_count":113,"old_ids":[3652,148722,152703],"score":226,"new_id":250804},{"text":"Азербайджанской","train_count":45,"old_ids":[34960,158401,148390,151828,7105,149193],"score":225,"new_id":250805},{"text":"Альварес","train_count":75,"old_ids":[34960,27112,148972,21785],"score":225,"new_id":250806},{"text":"Главный","train_count":225,"old_ids":[198730,32401],"score":225,"new_id":250807},{"text":"Награждён","train_count":75,"old_ids":[81573,148510,148343,185709],"score":225,"new_id":250808},{"text":"Первое","train_count":225,"old_ids":[95169,159117],"score":225,"new_id":250809},{"text":"водохранилища","train_count":75,"old_ids":[148435,167601,10750,159031],"score":225,"new_id":250810},{"text":"горы","train_count":225,"old_ids":[152974,4403],"score":225,"new_id":250811},{"text":"дистанции","train_count":75,"old_ids":[6733,32884,7105,54201],"score":225,"new_id":250812},{"text":"исследователей","train_count":75,"old_ids":[13113,149161,148275,154660],"score":225,"new_id":250813},{"text":"митрополита","train_count":75,"old_ids":[157255,148283,148634,148648],"score":225,"new_id":250814},{"text":"называемая","train_count":75,"old_ids":[13785,152953,153056,34226],"score":225,"new_id":250815},{"text":"называемые","train_count":75,"old_ids":[13785,152953,153056,48933],"score":225,"new_id":250816},{"text":"официальным","train_count":75,"old_ids":[1403,149243,36771,148540],"score":225,"new_id":250817},{"text":"событий","train_count":225,"old_ids":[224663,155198],"score":225,"new_id":250818},{"text":"сценарий","train_count":75,"old_ids":[2175,149147,16763,42387],"score":225,"new_id":250819},{"text":"транспорта","train_count":225,"old_ids":[157437,182576],"score":225,"new_id":250820},{"text":"Звёздный","train_count":56,"old_ids":[34279,5301,42565,171158,32401],"score":224,"new_id":250821},{"text":"Сын","train_count":224,"old_ids":[18727,178668],"score":224,"new_id":250822},{"text":"Чехии","train_count":112,"old_ids":[70345,93834,44633],"score":224,"new_id":250823},{"text":"вторая","train_count":224,"old_ids":[5301,164256],"score":224,"new_id":250824},{"text":"звезда","train_count":112,"old_ids":[8688,5301,157279],"score":224,"new_id":250825},{"text":"католическая","train_count":56,"old_ids":[4640,149900,87018,21785,150621],"score":224,"new_id":250826},{"text":"крестьян","train_count":112,"old_ids":[149679,156351,2040],"score":224,"new_id":250827},{"text":"крупным","train_count":112,"old_ids":[4640,149568,148540],"score":224,"new_id":250828},{"text":"найти","train_count":224,"old_ids":[13785,150959],"score":224,"new_id":250829},{"text":"ночь","train_count":224,"old_ids":[13292,170479],"score":224,"new_id":250830},{"text":"операция","train_count":112,"old_ids":[148561,7358,53890],"score":224,"new_id":250831},{"text":"открыл","train_count":112,"old_ids":[12666,91201,2941],"score":224,"new_id":250832},{"text":"отличия","train_count":112,"old_ids":[12666,87018,37835],"score":224,"new_id":250833},{"text":"племени","train_count":112,"old_ids":[150469,148281,9396],"score":224,"new_id":250834},{"text":"португальский","train_count":56,"old_ids":[153624,85208,14942,36771,149800],"score":224,"new_id":250835},{"text":"представлена","train_count":112,"old_ids":[177917,148448,152703],"score":224,"new_id":250836},{"text":"природе","train_count":112,"old_ids":[148895,148283,95525],"score":224,"new_id":250837},{"text":"продолжалась","train_count":56,"old_ids":[148489,148277,148294,153002,152993],"score":224,"new_id":250838},{"text":"продюсером","train_count":56,"old_ids":[148489,148277,11577,159541,148288],"score":224,"new_id":250839},{"text":"проиграл","train_count":112,"old_ids":[149490,66951,197362],"score":224,"new_id":250840},{"text":"разряде","train_count":112,"old_ids":[148464,151446,1450],"score":224,"new_id":250841},{"text":"районного","train_count":112,"old_ids":[180093,16806,148325],"score":224,"new_id":250842},{"text":"романе","train_count":112,"old_ids":[149633,7105,1450],"score":224,"new_id":250843},{"text":"серию","train_count":112,"old_ids":[2175,149098,11577],"score":224,"new_id":250844},{"text":"стихотворений","train_count":56,"old_ids":[52598,10161,12666,149086,59318],"score":224,"new_id":250845},{"text":"территорий","train_count":112,"old_ids":[55055,149454,158483],"score":224,"new_id":250846},{"text":"уроженцы","train_count":112,"old_ids":[212624,148942,155055],"score":224,"new_id":250847},{"text":"учеников","train_count":112,"old_ids":[27448,9396,148336],"score":224,"new_id":250848},{"text":"фракции","train_count":112,"old_ids":[18768,155062,54201],"score":224,"new_id":250849},{"text":"чёрный","train_count":112,"old_ids":[5831,156204,32401],"score":224,"new_id":250850},{"text":"японской","train_count":112,"old_ids":[4096,154767,149193],"score":224,"new_id":250851},{"text":"другом","train_count":223,"old_ids":[192548,148288],"score":223,"new_id":250852},{"text":"отделения","train_count":223,"old_ids":[12666,154426],"score":223,"new_id":250853},{"text":"Джеймса","train_count":74,"old_ids":[23355,153703,6240,148629],"score":222,"new_id":250854},{"text":"Кузнецов","train_count":74,"old_ids":[177784,8688,172518,148275],"score":222,"new_id":250855},{"text":"Лоуренс","train_count":74,"old_ids":[213334,53555,5092,2175],"score":222,"new_id":250856},{"text":"Народной","train_count":74,"old_ids":[19582,16763,148277,148317],"score":222,"new_id":250857},{"text":"Семён","train_count":111,"old_ids":[18727,148281,185709],"score":222,"new_id":250858},{"text":"Эдвард","train_count":74,"old_ids":[89030,6733,148972,6733],"score":222,"new_id":250859},{"text":"Этимология","train_count":111,"old_ids":[89030,162151,215816],"score":222,"new_id":250860},{"text":"восстание","train_count":111,"old_ids":[163934,6389,25767],"score":222,"new_id":250861},{"text":"граница","train_count":111,"old_ids":[14942,91872,1424],"score":222,"new_id":250862},{"text":"доктора","train_count":222,"old_ids":[171228,150587],"score":222,"new_id":250863},{"text":"заседании","train_count":74,"old_ids":[8688,17417,148276,149091],"score":222,"new_id":250864},{"text":"ископаемых","train_count":74,"old_ids":[13113,45527,148414,172320],"score":222,"new_id":250865},{"text":"испытания","train_count":111,"old_ids":[13113,165676,36916],"score":222,"new_id":250866},{"text":"музыкального","train_count":111,"old_ids":[148405,168062,155045],"score":222,"new_id":250867},{"text":"новое","train_count":222,"old_ids":[148366,42317],"score":222,"new_id":250868},{"text":"переходит","train_count":74,"old_ids":[149208,93834,148277,9080],"score":222,"new_id":250869},{"text":"погибших","train_count":74,"old_ids":[148428,148388,9629,150047],"score":222,"new_id":250870},{"text":"подполковник","train_count":74,"old_ids":[155908,148634,148336,67494],"score":222,"new_id":250871},{"text":"представителями","train_count":74,"old_ids":[177917,148448,152018,61628],"score":222,"new_id":250872},{"text":"проспекта","train_count":74,"old_ids":[160383,7754,148301,42074],"score":222,"new_id":250873},{"text":"разрешения","train_count":74,"old_ids":[148464,2115,44432,17941],"score":222,"new_id":250874},{"text":"сооружение","train_count":74,"old_ids":[148454,8789,54025,16669],"score":222,"new_id":250875},{"text":"социальных","train_count":111,"old_ids":[148454,14666,152994],"score":222,"new_id":250876},{"text":"сочинения","train_count":74,"old_ids":[2175,36103,18372,17941],"score":222,"new_id":250877},{"text":"учителем","train_count":74,"old_ids":[27448,9080,77969,6240],"score":222,"new_id":250878},{"text":"храм","train_count":222,"old_ids":[150997,6240],"score":222,"new_id":250879},{"text":"художественного","train_count":74,"old_ids":[157951,149244,150198,148325],"score":222,"new_id":250880},{"text":"шотландский","train_count":74,"old_ids":[11802,12666,200641,149800],"score":222,"new_id":250881},{"text":"большей","train_count":221,"old_ids":[170946,150281],"score":221,"new_id":250882},{"text":"лучший","train_count":221,"old_ids":[151408,151642],"score":221,"new_id":250883},{"text":"медаль","train_count":221,"old_ids":[235410,36771],"score":221,"new_id":250884},{"text":"народного","train_count":221,"old_ids":[151150,148325],"score":221,"new_id":250885},{"text":"радио","train_count":221,"old_ids":[21441,160212],"score":221,"new_id":250886},{"text":"серия","train_count":221,"old_ids":[159541,37835],"score":221,"new_id":250887},{"text":"храма","train_count":221,"old_ids":[150997,66050],"score":221,"new_id":250888},{"text":"языках","train_count":221,"old_ids":[214432,150577],"score":221,"new_id":250889},{"text":"Армянской","train_count":110,"old_ids":[199593,167664,149193],"score":220,"new_id":250890},{"text":"Дом","train_count":220,"old_ids":[23355,148288],"score":220,"new_id":250891},{"text":"Ивановича","train_count":55,"old_ids":[29189,149137,148275,31889,1424],"score":220,"new_id":250892},{"text":"Константина","train_count":110,"old_ids":[157349,49224,161288],"score":220,"new_id":250893},{"text":"Промышленность","train_count":44,"old_ids":[51844,148288,4403,11802,58361,58245],"score":220,"new_id":250894},{"text":"богословия","train_count":55,"old_ids":[148421,14942,22078,148569,37835],"score":220,"new_id":250895},{"text":"графа","train_count":220,"old_ids":[148510,152236],"score":220,"new_id":250896},{"text":"изображением","train_count":110,"old_ids":[39427,150065,161161],"score":220,"new_id":250897},{"text":"источники","train_count":110,"old_ids":[32884,36103,149665],"score":220,"new_id":250898},{"text":"итальянского","train_count":110,"old_ids":[9080,157230,176579],"score":220,"new_id":250899},{"text":"неделю","train_count":110,"old_ids":[76069,55986,11577],"score":220,"new_id":250900},{"text":"независимой","train_count":44,"old_ids":[76069,8688,148292,13113,15630,148293],"score":220,"new_id":250901},{"text":"общество","train_count":220,"old_ids":[149350,49903],"score":220,"new_id":250902},{"text":"огонь","train_count":110,"old_ids":[149095,16806,4669],"score":220,"new_id":250903},{"text":"поводу","train_count":110,"old_ids":[150772,148277,3652],"score":220,"new_id":250904},{"text":"рассказы","train_count":110,"old_ids":[2115,27473,151473],"score":220,"new_id":250905},{"text":"рекорд","train_count":110,"old_ids":[148279,153589,6733],"score":220,"new_id":250906},{"text":"человеческой","train_count":44,"old_ids":[5831,11777,148275,53887,21785,148554],"score":220,"new_id":250907},{"text":"Вюртемберг","train_count":73,"old_ids":[15729,154285,170744,241144],"score":219,"new_id":250908},{"text":"Киевский","train_count":73,"old_ids":[25497,47552,5301,149800],"score":219,"new_id":250909},{"text":"Победители","train_count":73,"old_ids":[16362,148280,148276,153009],"score":219,"new_id":250910},{"text":"Предыстория","train_count":73,"old_ids":[51844,166844,152950,37835],"score":219,"new_id":250911},{"text":"Федерального","train_count":73,"old_ids":[52337,148276,7358,151302],"score":219,"new_id":250912},{"text":"Чарльз","train_count":73,"old_ids":[70345,16763,27112,8688],"score":219,"new_id":250913},{"text":"биатлону","train_count":73,"old_ids":[148687,7721,2941,151281],"score":219,"new_id":250914},{"text":"восточном","train_count":73,"old_ids":[5301,23514,36103,148524],"score":219,"new_id":250915},{"text":"интересы","train_count":73,"old_ids":[18372,55055,21785,4403],"score":219,"new_id":250916},{"text":"кавалерии","train_count":73,"old_ids":[12751,151019,7358,44633],"score":219,"new_id":250917},{"text":"насекомых","train_count":73,"old_ids":[13785,148938,148558,60038],"score":219,"new_id":250918},{"text":"оппозиции","train_count":73,"old_ids":[1403,74359,152885,54201],"score":219,"new_id":250919},{"text":"опубликован","train_count":73,"old_ids":[148561,149090,148336,7105],"score":219,"new_id":250920},{"text":"перевозки","train_count":73,"old_ids":[149208,148380,148303,16256],"score":219,"new_id":250921},{"text":"российской","train_count":219,"old_ids":[184874,149193],"score":219,"new_id":250922},{"text":"созданная","train_count":73,"old_ids":[2175,148303,172895,42308],"score":219,"new_id":250923},{"text":"уничтожения","train_count":73,"old_ids":[3652,153001,23858,150282],"score":219,"new_id":250924},{"text":"Всё","train_count":109,"old_ids":[15729,2175,42565],"score":218,"new_id":250925},{"text":"Канада","train_count":109,"old_ids":[25497,148841,38183],"score":218,"new_id":250926},{"text":"Республике","train_count":109,"old_ids":[32392,151869,50800],"score":218,"new_id":250927},{"text":"События","train_count":109,"old_ids":[18727,158689,152149],"score":218,"new_id":250928},{"text":"Спустя","train_count":109,"old_ids":[18727,180808,4096],"score":218,"new_id":250929},{"text":"белых","train_count":109,"old_ids":[9629,11777,60038],"score":218,"new_id":250930},{"text":"вызывает","train_count":109,"old_ids":[148705,8688,151991],"score":218,"new_id":250931},{"text":"выступала","train_count":109,"old_ids":[148705,148757,92647],"score":218,"new_id":250932},{"text":"игроком","train_count":109,"old_ids":[66951,148283,148558],"score":218,"new_id":250933},{"text":"интернет","train_count":109,"old_ids":[18372,55055,151443],"score":218,"new_id":250934},{"text":"наряду","train_count":218,"old_ids":[13785,189994],"score":218,"new_id":250935},{"text":"областного","train_count":109,"old_ids":[148280,29354,197917],"score":218,"new_id":250936},{"text":"плотность","train_count":109,"old_ids":[150469,12666,58245],"score":218,"new_id":250937},{"text":"поддержке","train_count":109,"old_ids":[155908,148605,50800],"score":218,"new_id":250938},{"text":"получило","train_count":109,"old_ids":[148634,27448,150178],"score":218,"new_id":250939},{"text":"признан","train_count":109,"old_ids":[148895,154189,7105],"score":218,"new_id":250940},{"text":"проводились","train_count":109,"old_ids":[177788,149037,148685],"score":218,"new_id":250941},{"text":"проезд","train_count":109,"old_ids":[148489,42317,171158],"score":218,"new_id":250942},{"text":"святых","train_count":109,"old_ids":[2175,180353,60038],"score":218,"new_id":250943},{"text":"среды","train_count":218,"old_ids":[174301,4403],"score":218,"new_id":250944},{"text":"судна","train_count":109,"old_ids":[2175,34723,13785],"score":218,"new_id":250945},{"text":"Впоследствии","train_count":217,"old_ids":[15729,212703],"score":217,"new_id":250946},{"text":"Московский","train_count":217,"old_ids":[212095,223310],"score":217,"new_id":250947},{"text":"Северо","train_count":217,"old_ids":[181302,187942],"score":217,"new_id":250948},{"text":"Вопросы","train_count":72,"old_ids":[15729,148561,39605,4403],"score":216,"new_id":250949},{"text":"Израиль","train_count":108,"old_ids":[29189,156494,64257],"score":216,"new_id":250950},{"text":"Леонидович","train_count":54,"old_ids":[190021,16806,45587,148275,31889],"score":216,"new_id":250951},{"text":"Материалы","train_count":54,"old_ids":[36736,7721,149098,15481,4403],"score":216,"new_id":250952},{"text":"Московская","train_count":108,"old_ids":[212095,148336,152141],"score":216,"new_id":250953},{"text":"Нижний","train_count":108,"old_ids":[19582,151932,149623],"score":216,"new_id":250954},{"text":"Новосибирск","train_count":54,"old_ids":[193897,22078,151223,74990,62847],"score":216,"new_id":250955},{"text":"Павловский","train_count":72,"old_ids":[16362,148292,148569,149800],"score":216,"new_id":250956},{"text":"Почёта","train_count":72,"old_ids":[16362,36103,42565,42074],"score":216,"new_id":250957},{"text":"Рейнджерс","train_count":54,"old_ids":[32392,148298,150559,200530,2175],"score":216,"new_id":250958},{"text":"Человек","train_count":108,"old_ids":[70345,11777,155189],"score":216,"new_id":250959},{"text":"Эрнст","train_count":72,"old_ids":[89030,2115,2040,6389],"score":216,"new_id":250960},{"text":"библиотеки","train_count":108,"old_ids":[148687,157622,16256],"score":216,"new_id":250961},{"text":"вратарь","train_count":72,"old_ids":[5301,77779,16763,4669],"score":216,"new_id":250962},{"text":"входил","train_count":216,"old_ids":[5301,217731],"score":216,"new_id":250963},{"text":"заводы","train_count":72,"old_ids":[8688,148292,148277,4403],"score":216,"new_id":250964},{"text":"излучения","train_count":108,"old_ids":[39427,151408,17941],"score":216,"new_id":250965},{"text":"капитаном","train_count":72,"old_ids":[12751,150478,7105,148288],"score":216,"new_id":250966},{"text":"коллектив","train_count":108,"old_ids":[180702,148470,94227],"score":216,"new_id":250967},{"text":"описанные","train_count":108,"old_ids":[1403,28696,151979],"score":216,"new_id":250968},{"text":"освобождения","train_count":108,"old_ids":[22078,160496,149880],"score":216,"new_id":250969},{"text":"основателем","train_count":54,"old_ids":[22078,148366,7721,77969,6240],"score":216,"new_id":250970},{"text":"переводы","train_count":72,"old_ids":[149208,148380,148277,4403],"score":216,"new_id":250971},{"text":"перешли","train_count":108,"old_ids":[149208,44432,10750],"score":216,"new_id":250972},{"text":"поведение","train_count":108,"old_ids":[150772,148276,16669],"score":216,"new_id":250973},{"text":"постановка","train_count":54,"old_ids":[7754,23514,7105,148275,12751],"score":216,"new_id":250974},{"text":"построенная","train_count":54,"old_ids":[7754,23514,148283,5092,42308],"score":216,"new_id":250975},{"text":"премий","train_count":108,"old_ids":[149790,6240,42387],"score":216,"new_id":250976},{"text":"преподавателем","train_count":54,"old_ids":[149790,161978,150156,77969,6240],"score":216,"new_id":250977},{"text":"статье","train_count":108,"old_ids":[6389,17460,1450],"score":216,"new_id":250978},{"text":"увидеть","train_count":108,"old_ids":[149577,45587,148877],"score":216,"new_id":250979},{"text":"украинская","train_count":108,"old_ids":[241656,18372,152141],"score":216,"new_id":250980},{"text":"управлением","train_count":108,"old_ids":[47174,148381,157264],"score":216,"new_id":250981},{"text":"Могилёвской","train_count":43,"old_ids":[36736,149095,61057,42565,5301,149193],"score":215,"new_id":250982},{"text":"Новая","train_count":215,"old_ids":[193897,34226],"score":215,"new_id":250983},{"text":"административной","train_count":43,"old_ids":[149099,150704,32884,77779,26125,148317],"score":215,"new_id":250984},{"text":"выпуск","train_count":215,"old_ids":[148705,159859],"score":215,"new_id":250985},{"text":"знак","train_count":215,"old_ids":[148852,4640],"score":215,"new_id":250986},{"text":"летом","train_count":215,"old_ids":[149550,148288],"score":215,"new_id":250987},{"text":"Одним","train_count":107,"old_ids":[19736,6733,150067],"score":214,"new_id":250988},{"text":"Термин","train_count":107,"old_ids":[32868,7358,150704],"score":214,"new_id":250989},{"text":"включены","train_count":107,"old_ids":[5301,49689,75896],"score":214,"new_id":250990},{"text":"волны","train_count":214,"old_ids":[157156,12014],"score":214,"new_id":250991},{"text":"единицы","train_count":107,"old_ids":[152021,1739,155055],"score":214,"new_id":250992},{"text":"легион","train_count":107,"old_ids":[148284,148388,16806],"score":214,"new_id":250993},{"text":"лейтенанта","train_count":107,"old_ids":[152134,150947,157146],"score":214,"new_id":250994},{"text":"организация","train_count":214,"old_ids":[167304,53890],"score":214,"new_id":250995},{"text":"повесть","train_count":107,"old_ids":[150772,1450,33348],"score":214,"new_id":250996},{"text":"прошёл","train_count":214,"old_ids":[149490,229903],"score":214,"new_id":250997},{"text":"решил","train_count":214,"old_ids":[148279,194788],"score":214,"new_id":250998},{"text":"соглашение","train_count":107,"old_ids":[148454,149809,150950],"score":214,"new_id":250999},{"text":"стандарта","train_count":107,"old_ids":[49224,152148,42074],"score":214,"new_id":251000},{"text":"характера","train_count":107,"old_ids":[229103,149463,1424],"score":214,"new_id":251001},{"text":"церквей","train_count":107,"old_ids":[180269,4640,156211],"score":214,"new_id":251002},{"text":"Варшаве","train_count":71,"old_ids":[15729,16763,11802,158497],"score":213,"new_id":251003},{"text":"археолог","train_count":71,"old_ids":[16763,159679,148378,14942],"score":213,"new_id":251004},{"text":"заведение","train_count":71,"old_ids":[8688,148292,148276,16669],"score":213,"new_id":251005},{"text":"света","train_count":213,"old_ids":[163385,1424],"score":213,"new_id":251006},{"text":"тонн","train_count":213,"old_ids":[173049,2040],"score":213,"new_id":251007},{"text":"упоминания","train_count":71,"old_ids":[47174,148288,18372,36916],"score":213,"new_id":251008},{"text":"характеристик","train_count":71,"old_ids":[229103,149463,32884,36473],"score":213,"new_id":251009},{"text":"Большое","train_count":106,"old_ids":[58297,148701,42317],"score":212,"new_id":251010},{"text":"Владимировна","train_count":53,"old_ids":[15729,148606,15630,149591,13785],"score":212,"new_id":251011},{"text":"Николаевской","train_count":53,"old_ids":[19582,151907,29354,148380,149193],"score":212,"new_id":251012},{"text":"археологических","train_count":53,"old_ids":[16763,159679,148378,14942,151308],"score":212,"new_id":251013},{"text":"включают","train_count":106,"old_ids":[5301,49689,148803],"score":212,"new_id":251014},{"text":"восстановления","train_count":106,"old_ids":[163934,148722,59595],"score":212,"new_id":251015},{"text":"главную","train_count":106,"old_ids":[149809,5301,73781],"score":212,"new_id":251016},{"text":"границу","train_count":106,"old_ids":[14942,91872,3652],"score":212,"new_id":251017},{"text":"дерево","train_count":106,"old_ids":[6733,148579,148331],"score":212,"new_id":251018},{"text":"идея","train_count":106,"old_ids":[45587,1450,4096],"score":212,"new_id":251019},{"text":"изучал","train_count":106,"old_ids":[39427,27448,15481],"score":212,"new_id":251020},{"text":"императрицы","train_count":53,"old_ids":[15630,149208,7721,29119,155055],"score":212,"new_id":251021},{"text":"информационных","train_count":53,"old_ids":[18372,54724,28376,16806,42240],"score":212,"new_id":251022},{"text":"местности","train_count":212,"old_ids":[150436,65360],"score":212,"new_id":251023},{"text":"мощность","train_count":106,"old_ids":[6240,88260,58245],"score":212,"new_id":251024},{"text":"мощностью","train_count":106,"old_ids":[6240,88260,150616],"score":212,"new_id":251025},{"text":"обладатель","train_count":106,"old_ids":[148280,148606,57868],"score":212,"new_id":251026},{"text":"предназначенный","train_count":53,"old_ids":[177917,13785,149340,5092,32401],"score":212,"new_id":251027},{"text":"регионе","train_count":212,"old_ids":[173007,148917],"score":212,"new_id":251028},{"text":"случай","train_count":106,"old_ids":[38612,27448,148390],"score":212,"new_id":251029},{"text":"состояла","train_count":106,"old_ids":[2175,152544,29354],"score":212,"new_id":251030},{"text":"требования","train_count":106,"old_ids":[1729,149467,149349],"score":212,"new_id":251031},{"text":"устройств","train_count":106,"old_ids":[3652,149228,19587],"score":212,"new_id":251032},{"text":"физической","train_count":106,"old_ids":[18768,39427,158585],"score":212,"new_id":251033},{"text":"художником","train_count":106,"old_ids":[157951,149244,159659],"score":212,"new_id":251034},{"text":"энергии","train_count":212,"old_ids":[154631,151229],"score":212,"new_id":251035},{"text":"конференции","train_count":211,"old_ids":[228191,54201],"score":211,"new_id":251036},{"text":"писателя","train_count":211,"old_ids":[28696,152000],"score":211,"new_id":251037},{"text":"Заслуженные","train_count":42,"old_ids":[34279,17417,2941,54025,5092,41546],"score":210,"new_id":251038},{"text":"Испания","train_count":105,"old_ids":[29189,31608,36916],"score":210,"new_id":251039},{"text":"Калифорнии","train_count":70,"old_ids":[25497,45679,151385,155016],"score":210,"new_id":251040},{"text":"Народная","train_count":70,"old_ids":[19582,16763,148277,42308],"score":210,"new_id":251041},{"text":"Организация","train_count":70,"old_ids":[19736,89527,151946,53890],"score":210,"new_id":251042},{"text":"Петербургской","train_count":210,"old_ids":[182772,149193],"score":210,"new_id":251043},{"text":"Пруссии","train_count":105,"old_ids":[16362,19160,172340],"score":210,"new_id":251044},{"text":"Родилась","train_count":70,"old_ids":[32392,148277,61057,152993],"score":210,"new_id":251045},{"text":"Самарской","train_count":70,"old_ids":[18727,148357,16763,149193],"score":210,"new_id":251046},{"text":"административные","train_count":42,"old_ids":[149099,150704,32884,77779,26125,41546],"score":210,"new_id":251047},{"text":"армий","train_count":105,"old_ids":[16763,6240,42387],"score":210,"new_id":251048},{"text":"будут","train_count":210,"old_ids":[156543,36183],"score":210,"new_id":251049},{"text":"герой","train_count":210,"old_ids":[162127,148293],"score":210,"new_id":251050},{"text":"израильский","train_count":70,"old_ids":[39427,21441,64257,149800],"score":210,"new_id":251051},{"text":"медали","train_count":210,"old_ids":[148446,186671],"score":210,"new_id":251052},{"text":"медицинский","train_count":70,"old_ids":[6240,148969,159366,149800],"score":210,"new_id":251053},{"text":"научного","train_count":105,"old_ids":[13785,27448,148325],"score":210,"new_id":251054},{"text":"небольшим","train_count":70,"old_ids":[76069,9629,148701,15630],"score":210,"new_id":251055},{"text":"областей","train_count":105,"old_ids":[148280,29354,170359],"score":210,"new_id":251056},{"text":"основано","train_count":105,"old_ids":[22078,148366,149600],"score":210,"new_id":251057},{"text":"отмечал","train_count":70,"old_ids":[12666,6240,53887,15481],"score":210,"new_id":251058},{"text":"поддерживал","train_count":70,"old_ids":[155908,148605,26125,15481],"score":210,"new_id":251059},{"text":"показывает","train_count":105,"old_ids":[148428,148330,151991],"score":210,"new_id":251060},{"text":"поступила","train_count":70,"old_ids":[7754,23514,47174,150174],"score":210,"new_id":251061},{"text":"предусматривает","train_count":35,"old_ids":[177917,41635,6240,7721,2115,26125,26046],"score":210,"new_id":251062},{"text":"режиссёров","train_count":42,"old_ids":[148279,16467,13113,2175,42565,148327],"score":210,"new_id":251063},{"text":"ситуации","train_count":105,"old_ids":[2175,149452,51788],"score":210,"new_id":251064},{"text":"согласился","train_count":70,"old_ids":[148454,14942,149827,154838],"score":210,"new_id":251065},{"text":"создаёт","train_count":70,"old_ids":[2175,148303,38183,150396],"score":210,"new_id":251066},{"text":"сопротивление","train_count":105,"old_ids":[148454,200359,53921],"score":210,"new_id":251067},{"text":"супруга","train_count":70,"old_ids":[2175,47174,62215,1424],"score":210,"new_id":251068},{"text":"температуре","train_count":70,"old_ids":[170744,149208,150849,1450],"score":210,"new_id":251069},{"text":"хотел","train_count":105,"old_ids":[10161,12666,11777],"score":210,"new_id":251070},{"text":"экспериментальной","train_count":42,"old_ids":[192809,31608,7358,15630,18110,151291],"score":210,"new_id":251071},{"text":"Марк","train_count":209,"old_ids":[225978,4640],"score":209,"new_id":251072},{"text":"морской","train_count":209,"old_ids":[176874,149193],"score":209,"new_id":251073},{"text":"руки","train_count":209,"old_ids":[19160,16256],"score":209,"new_id":251074},{"text":"Белорусской","train_count":52,"old_ids":[58297,11777,8789,41635,149193],"score":208,"new_id":251075},{"text":"Дмитрия","train_count":104,"old_ids":[23355,157255,155075],"score":208,"new_id":251076},{"text":"Карла","train_count":208,"old_ids":[232689,29354],"score":208,"new_id":251077},{"text":"Киева","train_count":104,"old_ids":[25497,47552,90219],"score":208,"new_id":251078},{"text":"Кирилл","train_count":104,"old_ids":[188729,175740,2941],"score":208,"new_id":251079},{"text":"Ленинский","train_count":104,"old_ids":[61757,9396,176321],"score":208,"new_id":251080},{"text":"Флоренции","train_count":52,"old_ids":[52337,2941,8789,5092,54201],"score":208,"new_id":251081},{"text":"авиационного","train_count":52,"old_ids":[148292,1739,28376,16806,148325],"score":208,"new_id":251082},{"text":"архитекторы","train_count":52,"old_ids":[16763,10161,25135,151676,4403],"score":208,"new_id":251083},{"text":"героев","train_count":104,"old_ids":[162127,42317,5301],"score":208,"new_id":251084},{"text":"дороге","train_count":208,"old_ids":[152578,149068],"score":208,"new_id":251085},{"text":"защита","train_count":104,"old_ids":[148397,13739,148648],"score":208,"new_id":251086},{"text":"исполнении","train_count":104,"old_ids":[13113,148634,155739],"score":208,"new_id":251087},{"text":"легиона","train_count":104,"old_ids":[148284,148388,71221],"score":208,"new_id":251088},{"text":"машиностроения","train_count":52,"old_ids":[66050,28420,13292,148943,17941],"score":208,"new_id":251089},{"text":"моделей","train_count":104,"old_ids":[148324,55986,148298],"score":208,"new_id":251090},{"text":"наконец","train_count":104,"old_ids":[13785,4640,158418],"score":208,"new_id":251091},{"text":"немецком","train_count":104,"old_ids":[76069,218942,148558],"score":208,"new_id":251092},{"text":"открыто","train_count":104,"old_ids":[12666,91201,23858],"score":208,"new_id":251093},{"text":"политического","train_count":104,"old_ids":[148634,9080,158638],"score":208,"new_id":251094},{"text":"предприятий","train_count":208,"old_ids":[177917,174819],"score":208,"new_id":251095},{"text":"сельском","train_count":208,"old_ids":[226181,153195],"score":208,"new_id":251096},{"text":"сыграла","train_count":104,"old_ids":[54330,154731,92647],"score":208,"new_id":251097},{"text":"центральной","train_count":208,"old_ids":[214191,151291],"score":208,"new_id":251098},{"text":"Август","train_count":69,"old_ids":[34960,5301,14942,58475],"score":207,"new_id":251099},{"text":"Александро","train_count":69,"old_ids":[34960,151205,7105,159503],"score":207,"new_id":251100},{"text":"Йозеф","train_count":69,"old_ids":[150882,148303,1450,18768],"score":207,"new_id":251101},{"text":"Поселения","train_count":69,"old_ids":[16362,22078,11777,17941],"score":207,"new_id":251102},{"text":"автоматически","train_count":69,"old_ids":[148292,149127,7721,165455],"score":207,"new_id":251103},{"text":"бас","train_count":207,"old_ids":[9629,17417],"score":207,"new_id":251104},{"text":"войну","train_count":207,"old_ids":[159035,148776],"score":207,"new_id":251105},{"text":"действительности","train_count":69,"old_ids":[149232,19587,53062,65360],"score":207,"new_id":251106},{"text":"дорога","train_count":207,"old_ids":[152578,148369],"score":207,"new_id":251107},{"text":"знаменитой","train_count":69,"old_ids":[148852,36621,9080,148293],"score":207,"new_id":251108},{"text":"любви","train_count":207,"old_ids":[190228,148341],"score":207,"new_id":251109},{"text":"называемой","train_count":69,"old_ids":[13785,152953,153056,148293],"score":207,"new_id":251110},{"text":"остановки","train_count":69,"old_ids":[23514,7105,148275,16256],"score":207,"new_id":251111},{"text":"отделение","train_count":207,"old_ids":[12666,157182],"score":207,"new_id":251112},{"text":"путешествие","train_count":69,"old_ids":[198375,44432,32812,47552],"score":207,"new_id":251113},{"text":"свободно","train_count":69,"old_ids":[2175,5301,148280,164551],"score":207,"new_id":251114},{"text":"средневековой","train_count":69,"old_ids":[174301,76069,148537,176784],"score":207,"new_id":251115},{"text":"федерального","train_count":69,"old_ids":[18768,148276,7358,151302],"score":207,"new_id":251116},{"text":"характер","train_count":207,"old_ids":[229103,149463],"score":207,"new_id":251117},{"text":"Память","train_count":103,"old_ids":[180371,79142,27851],"score":206,"new_id":251118},{"text":"Работы","train_count":103,"old_ids":[173031,148453,4403],"score":206,"new_id":251119},{"text":"Смерть","train_count":103,"old_ids":[18727,60909,27851],"score":206,"new_id":251120},{"text":"бригады","train_count":103,"old_ids":[159499,148369,149055],"score":206,"new_id":251121},{"text":"глубина","train_count":103,"old_ids":[14942,152887,66353],"score":206,"new_id":251122},{"text":"говорит","train_count":206,"old_ids":[160776,9080],"score":206,"new_id":251123},{"text":"завершения","train_count":103,"old_ids":[148397,46576,149656],"score":206,"new_id":251124},{"text":"конфликт","train_count":103,"old_ids":[150270,18768,167688],"score":206,"new_id":251125},{"text":"корабль","train_count":206,"old_ids":[228309,212601],"score":206,"new_id":251126},{"text":"малых","train_count":103,"old_ids":[6240,15481,60038],"score":206,"new_id":251127},{"text":"некоторым","train_count":103,"old_ids":[76069,151549,149957],"score":206,"new_id":251128},{"text":"прибыл","train_count":103,"old_ids":[148895,148812,2941],"score":206,"new_id":251129},{"text":"провинция","train_count":103,"old_ids":[161800,18372,68335],"score":206,"new_id":251130},{"text":"сестры","train_count":103,"old_ids":[2175,34981,36747],"score":206,"new_id":251131},{"text":"сумел","train_count":103,"old_ids":[2175,37815,11777],"score":206,"new_id":251132},{"text":"тоже","train_count":206,"old_ids":[23858,56090],"score":206,"new_id":251133},{"text":"турецкой","train_count":103,"old_ids":[152238,154800,148554],"score":206,"new_id":251134},{"text":"указывает","train_count":103,"old_ids":[3652,148330,151991],"score":206,"new_id":251135},{"text":"учреждения","train_count":103,"old_ids":[27448,148279,149880],"score":206,"new_id":251136},{"text":"домов","train_count":205,"old_ids":[159080,148275],"score":205,"new_id":251137},{"text":"картины","train_count":205,"old_ids":[149888,244623],"score":205,"new_id":251138},{"text":"компаний","train_count":205,"old_ids":[166349,149931],"score":205,"new_id":251139},{"text":"проекту","train_count":205,"old_ids":[162947,85208],"score":205,"new_id":251140},{"text":"романа","train_count":205,"old_ids":[149633,148841],"score":205,"new_id":251141},{"text":"российских","train_count":205,"old_ids":[184874,150584],"score":205,"new_id":251142},{"text":"такого","train_count":205,"old_ids":[42074,149075],"score":205,"new_id":251143},{"text":"Биатлонисты","train_count":51,"old_ids":[207347,7721,160115,32884,4403],"score":204,"new_id":251144},{"text":"Западная","train_count":102,"old_ids":[172194,177720,42308],"score":204,"new_id":251145},{"text":"Западного","train_count":102,"old_ids":[172194,148414,176362],"score":204,"new_id":251146},{"text":"Композитор","train_count":68,"old_ids":[191297,7754,148303,158559],"score":204,"new_id":251147},{"text":"Король","train_count":102,"old_ids":[25497,8789,148319],"score":204,"new_id":251148},{"text":"Орехово","train_count":51,"old_ids":[19736,2115,93834,148275,1403],"score":204,"new_id":251149},{"text":"Польша","train_count":102,"old_ids":[16362,148701,1424],"score":204,"new_id":251150},{"text":"Рейн","train_count":102,"old_ids":[32392,148298,2040],"score":204,"new_id":251151},{"text":"Современное","train_count":68,"old_ids":[18727,148275,148801,45480],"score":204,"new_id":251152},{"text":"Стэнли","train_count":102,"old_ids":[71000,156454,10750],"score":204,"new_id":251153},{"text":"автомобильной","train_count":51,"old_ids":[148292,149127,148280,64257,148317],"score":204,"new_id":251154},{"text":"актёров","train_count":68,"old_ids":[148318,1729,42565,148327],"score":204,"new_id":251155},{"text":"борьбу","train_count":102,"old_ids":[148913,4669,148853],"score":204,"new_id":251156},{"text":"варианты","train_count":102,"old_ids":[155281,7105,82942],"score":204,"new_id":251157},{"text":"внутреннего","train_count":68,"old_ids":[5301,154175,148516,150674],"score":204,"new_id":251158},{"text":"водохранилище","train_count":68,"old_ids":[148435,167601,10750,150006],"score":204,"new_id":251159},{"text":"возраст","train_count":102,"old_ids":[150348,21441,6389],"score":204,"new_id":251160},{"text":"деревянные","train_count":68,"old_ids":[152138,148380,4096,155130],"score":204,"new_id":251161},{"text":"залив","train_count":102,"old_ids":[8688,45679,5301],"score":204,"new_id":251162},{"text":"зарубежных","train_count":68,"old_ids":[216459,61324,1450,155228],"score":204,"new_id":251163},{"text":"зоны","train_count":204,"old_ids":[148890,12014],"score":204,"new_id":251164},{"text":"обнаружены","train_count":102,"old_ids":[148280,151974,163352],"score":204,"new_id":251165},{"text":"организовал","train_count":102,"old_ids":[210793,148675,15481],"score":204,"new_id":251166},{"text":"остатки","train_count":102,"old_ids":[23514,7721,16256],"score":204,"new_id":251167},{"text":"постановки","train_count":51,"old_ids":[7754,23514,7105,148275,16256],"score":204,"new_id":251168},{"text":"потерпел","train_count":68,"old_ids":[229561,7358,7754,11777],"score":204,"new_id":251169},{"text":"превышает","train_count":102,"old_ids":[149790,148705,153642],"score":204,"new_id":251170},{"text":"прозаик","train_count":102,"old_ids":[149490,148397,36473],"score":204,"new_id":251171},{"text":"пулемёт","train_count":102,"old_ids":[148729,151032,150396],"score":204,"new_id":251172},{"text":"связан","train_count":102,"old_ids":[2175,163720,7105],"score":204,"new_id":251173},{"text":"современном","train_count":102,"old_ids":[149297,148801,148524],"score":204,"new_id":251174},{"text":"согласия","train_count":68,"old_ids":[148454,14942,149827,37835],"score":204,"new_id":251175},{"text":"сотрудничал","train_count":68,"old_ids":[217181,149035,153001,15481],"score":204,"new_id":251176},{"text":"философских","train_count":51,"old_ids":[18768,61057,152406,18768,150584],"score":204,"new_id":251177},{"text":"чешский","train_count":102,"old_ids":[5831,44432,149800],"score":204,"new_id":251178},{"text":"являлось","train_count":68,"old_ids":[4096,5301,13302,153174],"score":204,"new_id":251179},{"text":"Марии","train_count":203,"old_ids":[225978,44633],"score":203,"new_id":251180},{"text":"врач","train_count":203,"old_ids":[152221,5831],"score":203,"new_id":251181},{"text":"игрока","train_count":203,"old_ids":[66951,234579],"score":203,"new_id":251182},{"text":"Берлин","train_count":101,"old_ids":[58297,7358,155421],"score":202,"new_id":251183},{"text":"Исторические","train_count":101,"old_ids":[29189,152950,168329],"score":202,"new_id":251184},{"text":"величине","train_count":101,"old_ids":[160849,5831,149961],"score":202,"new_id":251185},{"text":"любого","train_count":101,"old_ids":[35587,148421,63046],"score":202,"new_id":251186},{"text":"лёгкой","train_count":101,"old_ids":[172384,14942,148554],"score":202,"new_id":251187},{"text":"новым","train_count":202,"old_ids":[148366,149957],"score":202,"new_id":251188},{"text":"певицы","train_count":101,"old_ids":[148728,148341,155055],"score":202,"new_id":251189},{"text":"пытался","train_count":101,"old_ids":[150258,165444,19937],"score":202,"new_id":251190},{"text":"сериале","train_count":101,"old_ids":[2175,149098,150188],"score":202,"new_id":251191},{"text":"сообщества","train_count":101,"old_ids":[148454,149350,150167],"score":202,"new_id":251192},{"text":"хоккею","train_count":101,"old_ids":[148491,201229,11577],"score":202,"new_id":251193},{"text":"Джонни","train_count":67,"old_ids":[23355,16467,16806,21918],"score":201,"new_id":251194},{"text":"Казанской","train_count":67,"old_ids":[25497,148646,7105,149193],"score":201,"new_id":251195},{"text":"Одесской","train_count":67,"old_ids":[19736,6733,90202,148554],"score":201,"new_id":251196},{"text":"Университете","train_count":67,"old_ids":[39243,150439,152898,149155],"score":201,"new_id":251197},{"text":"взгляды","train_count":67,"old_ids":[5301,8688,150336,149055],"score":201,"new_id":251198},{"text":"восточнее","train_count":67,"old_ids":[5301,23514,36103,149499],"score":201,"new_id":251199},{"text":"государственные","train_count":67,"old_ids":[14942,22078,150968,172545],"score":201,"new_id":251200},{"text":"действительный","train_count":67,"old_ids":[149232,19587,53062,32401],"score":201,"new_id":251201},{"text":"историческая","train_count":67,"old_ids":[32884,8789,77436,150621],"score":201,"new_id":251202},{"text":"местных","train_count":201,"old_ids":[150436,42240],"score":201,"new_id":251203},{"text":"небольшими","train_count":67,"old_ids":[76069,9629,148701,148623],"score":201,"new_id":251204},{"text":"парламентских","train_count":67,"old_ids":[151963,29354,89315,150584],"score":201,"new_id":251205},{"text":"прекратил","train_count":67,"old_ids":[149790,4640,77779,61057],"score":201,"new_id":251206},{"text":"принципу","train_count":67,"old_ids":[148489,18372,14666,148729],"score":201,"new_id":251207},{"text":"свыше","train_count":201,"old_ids":[2175,197954],"score":201,"new_id":251208},{"text":"скрипач","train_count":67,"old_ids":[62847,29119,7754,47474],"score":201,"new_id":251209},{"text":"сохранилась","train_count":67,"old_ids":[148454,10161,149926,155063],"score":201,"new_id":251210},{"text":"театрального","train_count":67,"old_ids":[48824,7721,2115,151302],"score":201,"new_id":251211},{"text":"федерации","train_count":67,"old_ids":[18768,148276,7358,51788],"score":201,"new_id":251212},{"text":"французов","train_count":67,"old_ids":[18768,33437,152213,148675],"score":201,"new_id":251213},{"text":"Андреев","train_count":50,"old_ids":[34960,2040,60659,47745,5301],"score":200,"new_id":251214},{"text":"Виктора","train_count":100,"old_ids":[15729,36473,150587],"score":200,"new_id":251215},{"text":"Возможно","train_count":100,"old_ids":[15729,148303,153165],"score":200,"new_id":251216},{"text":"Исчезнувшие","train_count":40,"old_ids":[29189,2175,5831,148377,148776,160359],"score":200,"new_id":251217},{"text":"Кубке","train_count":100,"old_ids":[25497,61324,50800],"score":200,"new_id":251218},{"text":"Мендоса","train_count":50,"old_ids":[36736,5092,6733,22078,1424],"score":200,"new_id":251219},{"text":"Первым","train_count":200,"old_ids":[95169,159147],"score":200,"new_id":251220},{"text":"Финляндия","train_count":50,"old_ids":[52337,18372,13302,150559,37835],"score":200,"new_id":251221},{"text":"Франсуа","train_count":100,"old_ids":[221647,148811,1424],"score":200,"new_id":251222},{"text":"Чемпионаты","train_count":50,"old_ids":[70345,148281,148686,16806,149351],"score":200,"new_id":251223},{"text":"Является","train_count":200,"old_ids":[82440,153190],"score":200,"new_id":251224},{"text":"академия","train_count":100,"old_ids":[1424,154408,174531],"score":200,"new_id":251225},{"text":"архитектуре","train_count":50,"old_ids":[16763,10161,25135,150094,201647],"score":200,"new_id":251226},{"text":"войн","train_count":200,"old_ids":[159035,2040],"score":200,"new_id":251227},{"text":"высоких","train_count":100,"old_ids":[148705,148454,148797],"score":200,"new_id":251228},{"text":"городах","train_count":200,"old_ids":[162121,178396],"score":200,"new_id":251229},{"text":"коллектива","train_count":100,"old_ids":[180702,148470,160193],"score":200,"new_id":251230},{"text":"координат","train_count":100,"old_ids":[45527,84475,7721],"score":200,"new_id":251231},{"text":"летию","train_count":100,"old_ids":[2941,149519,11577],"score":200,"new_id":251232},{"text":"математике","train_count":50,"old_ids":[6240,7721,148281,7721,168861],"score":200,"new_id":251233},{"text":"мелких","train_count":100,"old_ids":[6240,11777,148797],"score":200,"new_id":251234},{"text":"местного","train_count":200,"old_ids":[150436,148325],"score":200,"new_id":251235},{"text":"моста","train_count":200,"old_ids":[6240,151196],"score":200,"new_id":251236},{"text":"национальных","train_count":100,"old_ids":[13785,152208,159076],"score":200,"new_id":251237},{"text":"обсерватории","train_count":50,"old_ids":[148280,159541,5301,55389,44633],"score":200,"new_id":251238},{"text":"оказались","train_count":100,"old_ids":[1403,148330,158605],"score":200,"new_id":251239},{"text":"определённые","train_count":50,"old_ids":[148561,148279,55986,42565,155130],"score":200,"new_id":251240},{"text":"постоянной","train_count":100,"old_ids":[7754,152544,170968],"score":200,"new_id":251241},{"text":"предметов","train_count":100,"old_ids":[177917,148557,148275],"score":200,"new_id":251242},{"text":"принятия","train_count":100,"old_ids":[148489,18372,154449],"score":200,"new_id":251243},{"text":"приобрёл","train_count":100,"old_ids":[148895,149962,162292],"score":200,"new_id":251244},{"text":"программирования","train_count":100,"old_ids":[224012,61628,159253],"score":200,"new_id":251245},{"text":"ресурсов","train_count":100,"old_ids":[171408,86690,148275],"score":200,"new_id":251246},{"text":"семейство","train_count":100,"old_ids":[159707,148298,148593],"score":200,"new_id":251247},{"text":"телевизионный","train_count":50,"old_ids":[159593,5301,158668,16806,32401],"score":200,"new_id":251248},{"text":"урождённая","train_count":100,"old_ids":[212624,241242,172453],"score":200,"new_id":251249},{"text":"эмиграции","train_count":100,"old_ids":[65131,205175,54201],"score":200,"new_id":251250},{"text":"ООО","train_count":199,"old_ids":[159765,19736],"score":199,"new_id":251251},{"text":"второе","train_count":199,"old_ids":[167396,42317],"score":199,"new_id":251252},{"text":"известна","train_count":199,"old_ids":[244718,13785],"score":199,"new_id":251253},{"text":"крови","train_count":199,"old_ids":[166232,1739],"score":199,"new_id":251254},{"text":"реж","train_count":199,"old_ids":[148279,16467],"score":199,"new_id":251255},{"text":"южной","train_count":199,"old_ids":[11577,159484],"score":199,"new_id":251256},{"text":"Авторы","train_count":99,"old_ids":[34960,5301,153187],"score":198,"new_id":251257},{"text":"Верхней","train_count":99,"old_ids":[238301,10161,148820],"score":198,"new_id":251258},{"text":"Екатерины","train_count":99,"old_ids":[53241,178712,12014],"score":198,"new_id":251259},{"text":"Ивановская","train_count":66,"old_ids":[29189,149137,148275,152141],"score":198,"new_id":251260},{"text":"Иосиф","train_count":99,"old_ids":[29189,22078,59627],"score":198,"new_id":251261},{"text":"Молдавской","train_count":66,"old_ids":[36736,148294,223911,149193],"score":198,"new_id":251262},{"text":"Получил","train_count":66,"old_ids":[16362,148294,27448,61057],"score":198,"new_id":251263},{"text":"Последние","train_count":99,"old_ids":[16362,158537,150528],"score":198,"new_id":251264},{"text":"Ростов","train_count":99,"old_ids":[32392,23514,148275],"score":198,"new_id":251265},{"text":"Формуле","train_count":66,"old_ids":[52337,30206,3652,148284],"score":198,"new_id":251266},{"text":"Харькове","train_count":66,"old_ids":[150853,16763,4669,212658],"score":198,"new_id":251267},{"text":"Шахтёр","train_count":66,"old_ids":[204269,10161,1729,156204],"score":198,"new_id":251268},{"text":"Яковлево","train_count":66,"old_ids":[82440,148336,148284,148331],"score":198,"new_id":251269},{"text":"аргентинский","train_count":66,"old_ids":[16763,160675,18372,149800],"score":198,"new_id":251270},{"text":"архитектором","train_count":66,"old_ids":[16763,10161,25135,202283],"score":198,"new_id":251271},{"text":"большие","train_count":198,"old_ids":[170946,150366],"score":198,"new_id":251272},{"text":"британских","train_count":66,"old_ids":[9629,149454,7105,150584],"score":198,"new_id":251273},{"text":"вооружении","train_count":66,"old_ids":[148331,8789,54025,80228],"score":198,"new_id":251274},{"text":"еврейской","train_count":99,"old_ids":[148380,171251,149193],"score":198,"new_id":251275},{"text":"единственный","train_count":198,"old_ids":[152021,159759],"score":198,"new_id":251276},{"text":"защиту","train_count":99,"old_ids":[148397,13739,149452],"score":198,"new_id":251277},{"text":"культуре","train_count":198,"old_ids":[231736,201647],"score":198,"new_id":251278},{"text":"маршрут","train_count":99,"old_ids":[173719,157013,1729],"score":198,"new_id":251279},{"text":"называет","train_count":99,"old_ids":[13785,8688,151991],"score":198,"new_id":251280},{"text":"нацистской","train_count":66,"old_ids":[13785,14666,6389,149193],"score":198,"new_id":251281},{"text":"немецкие","train_count":99,"old_ids":[76069,218942,149012],"score":198,"new_id":251282},{"text":"океане","train_count":66,"old_ids":[1403,50800,7105,1450],"score":198,"new_id":251283},{"text":"пгт","train_count":99,"old_ids":[7754,14942,1729],"score":198,"new_id":251284},{"text":"педагогического","train_count":66,"old_ids":[148728,154155,14942,158638],"score":198,"new_id":251285},{"text":"платформа","train_count":99,"old_ids":[7754,75774,151406],"score":198,"new_id":251286},{"text":"попытался","train_count":66,"old_ids":[148428,150258,165444,19937],"score":198,"new_id":251287},{"text":"принёс","train_count":66,"old_ids":[148489,18372,42565,2175],"score":198,"new_id":251288},{"text":"программе","train_count":198,"old_ids":[224012,148446],"score":198,"new_id":251289},{"text":"рассматривают","train_count":33,"old_ids":[2115,27473,6240,7721,2115,26125,148803],"score":198,"new_id":251290},{"text":"расстоянии","train_count":99,"old_ids":[148761,148855,155016],"score":198,"new_id":251291},{"text":"результатов","train_count":99,"old_ids":[150564,62709,152968],"score":198,"new_id":251292},{"text":"решению","train_count":99,"old_ids":[2115,44432,148922],"score":198,"new_id":251293},{"text":"связей","train_count":99,"old_ids":[2175,163720,148298],"score":198,"new_id":251294},{"text":"сражение","train_count":99,"old_ids":[2175,21441,150307],"score":198,"new_id":251295},{"text":"степенью","train_count":66,"old_ids":[6389,148445,49603,11577],"score":198,"new_id":251296},{"text":"теоретик","train_count":66,"old_ids":[48824,8789,7921,36473],"score":198,"new_id":251297},{"text":"тёмно","train_count":99,"old_ids":[1729,151666,13292],"score":198,"new_id":251298},{"text":"умерла","train_count":99,"old_ids":[37815,7358,29354],"score":198,"new_id":251299},{"text":"энциклопедический","train_count":33,"old_ids":[156454,14666,151639,7754,148276,77436,149537],"score":198,"new_id":251300},{"text":"момента","train_count":197,"old_ids":[159668,84706],"score":197,"new_id":251301},{"text":"Городок","train_count":98,"old_ids":[35884,149345,148518],"score":196,"new_id":251302},{"text":"Иерусалиме","train_count":49,"old_ids":[29189,7358,41635,45679,148446],"score":196,"new_id":251303},{"text":"Михаилом","train_count":49,"old_ids":[36736,64392,1424,150178,6240],"score":196,"new_id":251304},{"text":"Наполеона","train_count":98,"old_ids":[81573,220149,71221],"score":196,"new_id":251305},{"text":"Новосибирской","train_count":49,"old_ids":[193897,22078,151223,74990,149193],"score":196,"new_id":251306},{"text":"Писатели","train_count":98,"old_ids":[16362,13113,158140],"score":196,"new_id":251307},{"text":"Соединённых","train_count":49,"old_ids":[18727,42317,67958,42565,170741],"score":196,"new_id":251308},{"text":"адъютант","train_count":49,"old_ids":[149099,32482,66694,7105,1729],"score":196,"new_id":251309},{"text":"ведущий","train_count":98,"old_ids":[148700,40794,42387],"score":196,"new_id":251310},{"text":"воздух","train_count":98,"old_ids":[150348,148660,10161],"score":196,"new_id":251311},{"text":"возникает","train_count":98,"old_ids":[150348,84480,7921],"score":196,"new_id":251312},{"text":"встречи","train_count":98,"old_ids":[5301,48941,165512],"score":196,"new_id":251313},{"text":"значительной","train_count":98,"old_ids":[149340,53062,148317],"score":196,"new_id":251314},{"text":"источник","train_count":98,"old_ids":[32884,36103,67494],"score":196,"new_id":251315},{"text":"легкоатлетка","train_count":49,"old_ids":[156337,45527,7721,149550,12751],"score":196,"new_id":251316},{"text":"материнской","train_count":49,"old_ids":[6240,7721,7358,18372,149193],"score":196,"new_id":251317},{"text":"музее","train_count":98,"old_ids":[148405,8688,47745],"score":196,"new_id":251318},{"text":"названий","train_count":98,"old_ids":[13785,8688,171245],"score":196,"new_id":251319},{"text":"отправлен","train_count":98,"old_ids":[12666,148607,58361],"score":196,"new_id":251320},{"text":"пианист","train_count":98,"old_ids":[148686,13603,6389],"score":196,"new_id":251321},{"text":"просвещения","train_count":98,"old_ids":[160383,5301,158442],"score":196,"new_id":251322},{"text":"руководства","train_count":98,"old_ids":[19160,150463,89771],"score":196,"new_id":251323},{"text":"секунд","train_count":98,"old_ids":[241651,33065,6733],"score":196,"new_id":251324},{"text":"сентябрь","train_count":98,"old_ids":[2175,156113,151874],"score":196,"new_id":251325},{"text":"словами","train_count":98,"old_ids":[2175,160162,61628],"score":196,"new_id":251326},{"text":"современный","train_count":98,"old_ids":[149297,148801,32401],"score":196,"new_id":251327},{"text":"традиционно","train_count":98,"old_ids":[150246,148467,162472],"score":196,"new_id":251328},{"text":"установлено","train_count":98,"old_ids":[3652,148722,162836],"score":196,"new_id":251329},{"text":"шасси","train_count":98,"old_ids":[11802,27473,1739],"score":196,"new_id":251330},{"text":"Августа","train_count":65,"old_ids":[34960,5301,14942,154271],"score":195,"new_id":251331},{"text":"Григория","train_count":65,"old_ids":[35884,29119,152974,37835],"score":195,"new_id":251332},{"text":"Ивана","train_count":195,"old_ids":[29189,182032],"score":195,"new_id":251333},{"text":"Кавказской","train_count":65,"old_ids":[25497,148292,148330,149193],"score":195,"new_id":251334},{"text":"Лос","train_count":195,"old_ids":[61757,22078],"score":195,"new_id":251335},{"text":"Средневековья","train_count":39,"old_ids":[18727,148420,76069,148537,148336,149885],"score":195,"new_id":251336},{"text":"Телесериалы","train_count":39,"old_ids":[32868,11777,21785,149098,15481,4403],"score":195,"new_id":251337},{"text":"Юго","train_count":195,"old_ids":[168959,63046],"score":195,"new_id":251338},{"text":"встретился","train_count":65,"old_ids":[5301,48941,7921,154838],"score":195,"new_id":251339},{"text":"двадцать","train_count":65,"old_ids":[164189,6733,10064,17460],"score":195,"new_id":251340},{"text":"деревянный","train_count":65,"old_ids":[152138,148380,4096,171096],"score":195,"new_id":251341},{"text":"дизайн","train_count":65,"old_ids":[6733,39427,148390,2040],"score":195,"new_id":251342},{"text":"журналистов","train_count":65,"old_ids":[16467,53555,150981,153305],"score":195,"new_id":251343},{"text":"забил","train_count":195,"old_ids":[148397,194662],"score":195,"new_id":251344},{"text":"занимала","train_count":65,"old_ids":[8688,13603,6240,92647],"score":195,"new_id":251345},{"text":"знаменитый","train_count":65,"old_ids":[148852,36621,9080,33477],"score":195,"new_id":251346},{"text":"зона","train_count":195,"old_ids":[8688,71221],"score":195,"new_id":251347},{"text":"издания","train_count":195,"old_ids":[39427,158529],"score":195,"new_id":251348},{"text":"компьютерной","train_count":65,"old_ids":[166349,84920,55055,148317],"score":195,"new_id":251349},{"text":"мультфильм","train_count":65,"old_ids":[6240,62709,165171,6240],"score":195,"new_id":251350},{"text":"отличался","train_count":65,"old_ids":[12666,87018,15481,19937],"score":195,"new_id":251351},{"text":"посёлков","train_count":65,"old_ids":[7754,22078,162292,148336],"score":195,"new_id":251352},{"text":"правом","train_count":195,"old_ids":[148607,148288],"score":195,"new_id":251353},{"text":"преступления","train_count":65,"old_ids":[148489,34981,47174,59595],"score":195,"new_id":251354},{"text":"применением","train_count":65,"old_ids":[148489,15630,5092,149353],"score":195,"new_id":251355},{"text":"происхождении","train_count":65,"old_ids":[149490,13113,148491,156413],"score":195,"new_id":251356},{"text":"сделать","train_count":195,"old_ids":[2175,168771],"score":195,"new_id":251357},{"text":"серьёзно","train_count":65,"old_ids":[159541,4669,42565,213287],"score":195,"new_id":251358},{"text":"сознания","train_count":65,"old_ids":[2175,148303,2040,36916],"score":195,"new_id":251359},{"text":"сохраняется","train_count":65,"old_ids":[148454,58805,4096,48046],"score":195,"new_id":251360},{"text":"электронных","train_count":65,"old_ids":[65131,149157,151673,42240],"score":195,"new_id":251361},{"text":"Управление","train_count":194,"old_ids":[39243,166674],"score":194,"new_id":251362},{"text":"боя","train_count":194,"old_ids":[9629,58255],"score":194,"new_id":251363},{"text":"бывших","train_count":97,"old_ids":[9629,33422,150047],"score":194,"new_id":251364},{"text":"даёт","train_count":194,"old_ids":[38183,150396],"score":194,"new_id":251365},{"text":"звук","train_count":97,"old_ids":[8688,5301,63855],"score":194,"new_id":251366},{"text":"командование","train_count":97,"old_ids":[148558,72884,150187],"score":194,"new_id":251367},{"text":"компонент","train_count":97,"old_ids":[148558,154767,18110],"score":194,"new_id":251368},{"text":"лошадей","train_count":97,"old_ids":[37483,150011,149232],"score":194,"new_id":251369},{"text":"любой","train_count":194,"old_ids":[35587,155070],"score":194,"new_id":251370},{"text":"мифологии","train_count":97,"old_ids":[6240,59627,158283],"score":194,"new_id":251371},{"text":"называться","train_count":97,"old_ids":[13785,152953,94309],"score":194,"new_id":251372},{"text":"образовании","train_count":97,"old_ids":[149586,148275,149091],"score":194,"new_id":251373},{"text":"общественной","train_count":97,"old_ids":[149350,150198,148317],"score":194,"new_id":251374},{"text":"объявлено","train_count":97,"old_ids":[148280,156545,172331],"score":194,"new_id":251375},{"text":"отношении","train_count":194,"old_ids":[152928,155496],"score":194,"new_id":251376},{"text":"подготовке","train_count":97,"old_ids":[155908,149341,50800],"score":194,"new_id":251377},{"text":"получают","train_count":97,"old_ids":[148634,27448,148803],"score":194,"new_id":251378},{"text":"пользователей","train_count":97,"old_ids":[183192,148675,154660],"score":194,"new_id":251379},{"text":"разработана","train_count":97,"old_ids":[148464,149100,148841],"score":194,"new_id":251380},{"text":"тяжёлой","train_count":97,"old_ids":[149636,197306,155669],"score":194,"new_id":251381},{"text":"физических","train_count":97,"old_ids":[18768,39427,151308],"score":194,"new_id":251382},{"text":"финала","train_count":194,"old_ids":[218085,29354],"score":194,"new_id":251383},{"text":"Мать","train_count":193,"old_ids":[36736,17460],"score":193,"new_id":251384},{"text":"описание","train_count":193,"old_ids":[1403,90397],"score":193,"new_id":251385},{"text":"продукции","train_count":193,"old_ids":[184810,54201],"score":193,"new_id":251386},{"text":"этап","train_count":193,"old_ids":[161869,7754],"score":193,"new_id":251387},{"text":"Васильевна","train_count":64,"old_ids":[15729,17417,64257,187935],"score":192,"new_id":251388},{"text":"Войска","train_count":64,"old_ids":[15729,148293,2175,12751],"score":192,"new_id":251389},{"text":"Гутьеррес","train_count":64,"old_ids":[35884,79860,7358,171408],"score":192,"new_id":251390},{"text":"Евфимий","train_count":48,"old_ids":[53241,5301,18768,15630,42387],"score":192,"new_id":251391},{"text":"Екатеринбурге","train_count":32,"old_ids":[53241,150843,7358,18372,9629,53555,149068],"score":192,"new_id":251392},{"text":"Елизавета","train_count":96,"old_ids":[53241,178537,200154],"score":192,"new_id":251393},{"text":"Журнал","train_count":96,"old_ids":[154849,53555,155371],"score":192,"new_id":251394},{"text":"Казанского","train_count":64,"old_ids":[25497,148646,7105,149280],"score":192,"new_id":251395},{"text":"Королевской","train_count":64,"old_ids":[25497,148367,151186,149193],"score":192,"new_id":251396},{"text":"Лев","train_count":192,"old_ids":[61757,148380],"score":192,"new_id":251397},{"text":"Правители","train_count":96,"old_ids":[16362,148381,153009],"score":192,"new_id":251398},{"text":"Ростовской","train_count":64,"old_ids":[32392,23514,148275,149193],"score":192,"new_id":251399},{"text":"Священной","train_count":96,"old_ids":[18727,152139,205103],"score":192,"new_id":251400},{"text":"Современный","train_count":64,"old_ids":[18727,148275,148801,32401],"score":192,"new_id":251401},{"text":"Текст","train_count":96,"old_ids":[32868,148301,6389],"score":192,"new_id":251402},{"text":"Улицы","train_count":96,"old_ids":[39243,63890,4403],"score":192,"new_id":251403},{"text":"Франциско","train_count":64,"old_ids":[221647,10064,13113,45527],"score":192,"new_id":251404},{"text":"Харькова","train_count":64,"old_ids":[150853,16763,4669,155223],"score":192,"new_id":251405},{"text":"Цезаря","train_count":64,"old_ids":[89602,148377,16763,4096],"score":192,"new_id":251406},{"text":"Чёрное","train_count":96,"old_ids":[70345,156204,45480],"score":192,"new_id":251407},{"text":"Швеция","train_count":96,"old_ids":[151311,148537,68335],"score":192,"new_id":251408},{"text":"Шекспира","train_count":64,"old_ids":[151311,148301,31608,151246],"score":192,"new_id":251409},{"text":"Экспресс","train_count":48,"old_ids":[89030,4640,31608,2115,90202],"score":192,"new_id":251410},{"text":"автобусов","train_count":96,"old_ids":[192627,245644,148275],"score":192,"new_id":251411},{"text":"бакалавриат","train_count":48,"old_ids":[148778,12751,150800,29119,7721],"score":192,"new_id":251412},{"text":"барабанщик","train_count":64,"old_ids":[9629,148622,154749,161782],"score":192,"new_id":251413},{"text":"бассейне","train_count":64,"old_ids":[9629,27473,148298,76069],"score":192,"new_id":251414},{"text":"восточная","train_count":64,"old_ids":[5301,23514,36103,42308],"score":192,"new_id":251415},{"text":"долины","train_count":96,"old_ids":[148551,10750,12014],"score":192,"new_id":251416},{"text":"заключается","train_count":96,"old_ids":[148397,49689,64903],"score":192,"new_id":251417},{"text":"звания","train_count":192,"old_ids":[8688,155024],"score":192,"new_id":251418},{"text":"источником","train_count":96,"old_ids":[32884,36103,159659],"score":192,"new_id":251419},{"text":"кантоне","train_count":96,"old_ids":[151011,1729,148917],"score":192,"new_id":251420},{"text":"количеством","train_count":96,"old_ids":[45527,161317,6240],"score":192,"new_id":251421},{"text":"компьютерная","train_count":64,"old_ids":[166349,84920,55055,42308],"score":192,"new_id":251422},{"text":"легко","train_count":192,"old_ids":[156337,45527],"score":192,"new_id":251423},{"text":"лидером","train_count":96,"old_ids":[10750,152138,148288],"score":192,"new_id":251424},{"text":"металлических","train_count":48,"old_ids":[148557,15481,87018,21785,148797],"score":192,"new_id":251425},{"text":"написана","train_count":96,"old_ids":[13785,28696,148841],"score":192,"new_id":251426},{"text":"областях","train_count":96,"old_ids":[148280,29354,187648],"score":192,"new_id":251427},{"text":"операционной","train_count":48,"old_ids":[148561,7358,28376,16806,148317],"score":192,"new_id":251428},{"text":"определённого","train_count":48,"old_ids":[148561,148279,55986,42565,171935],"score":192,"new_id":251429},{"text":"отношений","train_count":192,"old_ids":[152928,155561],"score":192,"new_id":251430},{"text":"планировалось","train_count":96,"old_ids":[167592,177702,95306],"score":192,"new_id":251431},{"text":"пользовался","train_count":64,"old_ids":[183192,148675,15481,19937],"score":192,"new_id":251432},{"text":"появившиеся","train_count":48,"old_ids":[7754,58255,154575,150366,19937],"score":192,"new_id":251433},{"text":"правительством","train_count":96,"old_ids":[148607,53062,152225],"score":192,"new_id":251434},{"text":"применялись","train_count":48,"old_ids":[148489,15630,5092,4096,150938],"score":192,"new_id":251435},{"text":"присутствуют","train_count":64,"old_ids":[148489,13113,36183,151766],"score":192,"new_id":251436},{"text":"проблема","train_count":64,"old_ids":[148489,148280,148284,66050],"score":192,"new_id":251437},{"text":"проектирования","train_count":64,"old_ids":[148489,42317,149818,159253],"score":192,"new_id":251438},{"text":"протяжённость","train_count":48,"old_ids":[148489,156442,16467,185709,58245],"score":192,"new_id":251439},{"text":"районами","train_count":96,"old_ids":[180093,71221,61628],"score":192,"new_id":251440},{"text":"религиозных","train_count":64,"old_ids":[2115,58503,167130,42240],"score":192,"new_id":251441},{"text":"решено","train_count":96,"old_ids":[2115,44432,69729],"score":192,"new_id":251442},{"text":"стихотворения","train_count":48,"old_ids":[52598,10161,12666,149086,17941],"score":192,"new_id":251443},{"text":"участки","train_count":96,"old_ids":[27448,80378,16256],"score":192,"new_id":251444},{"text":"учебного","train_count":64,"old_ids":[27448,1450,9629,148325],"score":192,"new_id":251445},{"text":"хочет","train_count":96,"old_ids":[10161,36103,7921],"score":192,"new_id":251446},{"text":"эпизодов","train_count":48,"old_ids":[65131,7754,39427,148277,148275],"score":192,"new_id":251447},{"text":"Вторая","train_count":191,"old_ids":[15729,164256],"score":191,"new_id":251448},{"text":"ДНК","train_count":191,"old_ids":[23355,201104],"score":191,"new_id":251449},{"text":"акций","train_count":191,"old_ids":[148318,150585],"score":191,"new_id":251450},{"text":"миру","train_count":191,"old_ids":[61628,19160],"score":191,"new_id":251451},{"text":"поражение","train_count":191,"old_ids":[161739,150307],"score":191,"new_id":251452},{"text":"поражения","train_count":191,"old_ids":[161739,150282],"score":191,"new_id":251453},{"text":"предприятие","train_count":191,"old_ids":[177917,179060],"score":191,"new_id":251454},{"text":"разные","train_count":191,"old_ids":[148464,41546],"score":191,"new_id":251455},{"text":"слишком","train_count":191,"old_ids":[31140,159923],"score":191,"new_id":251456},{"text":"Андре","train_count":95,"old_ids":[34960,2040,156329],"score":190,"new_id":251457},{"text":"Бельгии","train_count":95,"old_ids":[58297,24623,151229],"score":190,"new_id":251458},{"text":"Бранденбурга","train_count":38,"old_ids":[58297,33437,149637,9629,53555,148369],"score":190,"new_id":251459},{"text":"Иерусалимский","train_count":38,"old_ids":[29189,7358,41635,45679,6240,149800],"score":190,"new_id":251460},{"text":"Именно","train_count":190,"old_ids":[29189,155079],"score":190,"new_id":251461},{"text":"Победителем","train_count":38,"old_ids":[16362,148280,148276,9080,77969,6240],"score":190,"new_id":251462},{"text":"Средние","train_count":95,"old_ids":[18727,148420,150528],"score":190,"new_id":251463},{"text":"Царскосельского","train_count":38,"old_ids":[89602,16763,62847,22078,24623,149280],"score":190,"new_id":251464},{"text":"бизнеса","train_count":95,"old_ids":[9629,165683,1424],"score":190,"new_id":251465},{"text":"борьба","train_count":95,"old_ids":[148913,4669,148778],"score":190,"new_id":251466},{"text":"взять","train_count":95,"old_ids":[5301,8688,90638],"score":190,"new_id":251467},{"text":"всём","train_count":95,"old_ids":[5301,2175,151666],"score":190,"new_id":251468},{"text":"группами","train_count":95,"old_ids":[14942,83147,150190],"score":190,"new_id":251469},{"text":"закрыт","train_count":95,"old_ids":[148397,91201,1729],"score":190,"new_id":251470},{"text":"имеющие","train_count":95,"old_ids":[15630,1450,153177],"score":190,"new_id":251471},{"text":"компонентов","train_count":95,"old_ids":[148558,154767,151282],"score":190,"new_id":251472},{"text":"мастер","train_count":190,"old_ids":[66050,170759],"score":190,"new_id":251473},{"text":"оккупации","train_count":95,"old_ids":[148518,149741,51788],"score":190,"new_id":251474},{"text":"песней","train_count":95,"old_ids":[7754,21785,148820],"score":190,"new_id":251475},{"text":"победа","train_count":95,"old_ids":[7754,157723,38183],"score":190,"new_id":251476},{"text":"разведки","train_count":95,"old_ids":[148464,148700,16256],"score":190,"new_id":251477},{"text":"рассказ","train_count":95,"old_ids":[2115,27473,148330],"score":190,"new_id":251478},{"text":"редакции","train_count":95,"old_ids":[148279,38183,156986],"score":190,"new_id":251479},{"text":"сельсовете","train_count":95,"old_ids":[226181,148454,181997],"score":190,"new_id":251480},{"text":"собора","train_count":190,"old_ids":[169983,148676],"score":190,"new_id":251481},{"text":"справа","train_count":95,"old_ids":[31608,21441,90219],"score":190,"new_id":251482},{"text":"титула","train_count":95,"old_ids":[1729,149452,29354],"score":190,"new_id":251483},{"text":"факультете","train_count":95,"old_ids":[152236,231736,149155],"score":190,"new_id":251484},{"text":"человеческого","train_count":38,"old_ids":[5831,11777,148275,53887,21785,149075],"score":190,"new_id":251485},{"text":"экипаж","train_count":95,"old_ids":[65131,168267,16467],"score":190,"new_id":251486},{"text":"Германией","train_count":63,"old_ids":[240053,6240,25767,11796],"score":189,"new_id":251487},{"text":"Джорджа","train_count":63,"old_ids":[23355,16467,8789,183062],"score":189,"new_id":251488},{"text":"Елизаветы","train_count":63,"old_ids":[53241,178537,45405,4403],"score":189,"new_id":251489},{"text":"Иосифа","train_count":63,"old_ids":[29189,22078,59627,1424],"score":189,"new_id":251490},{"text":"Максимальная","train_count":63,"old_ids":[226568,2175,87857,42308],"score":189,"new_id":251491},{"text":"Народы","train_count":63,"old_ids":[19582,16763,148277,4403],"score":189,"new_id":251492},{"text":"Общая","train_count":189,"old_ids":[70749,159175],"score":189,"new_id":251493},{"text":"Попович","train_count":63,"old_ids":[16362,148561,148275,31889],"score":189,"new_id":251494},{"text":"Постепенно","train_count":63,"old_ids":[16362,23514,157515,13292],"score":189,"new_id":251495},{"text":"Уильямс","train_count":63,"old_ids":[39243,64257,150686,2175],"score":189,"new_id":251496},{"text":"Флорида","train_count":63,"old_ids":[52337,2941,150140,38183],"score":189,"new_id":251497},{"text":"договору","train_count":63,"old_ids":[6733,148449,149086,3652],"score":189,"new_id":251498},{"text":"изначально","train_count":189,"old_ids":[39427,167193],"score":189,"new_id":251499},{"text":"конец","train_count":189,"old_ids":[4640,158418],"score":189,"new_id":251500},{"text":"королевство","train_count":63,"old_ids":[4640,148367,151186,148593],"score":189,"new_id":251501},{"text":"месяца","train_count":189,"old_ids":[164065,83895],"score":189,"new_id":251502},{"text":"написанная","train_count":63,"old_ids":[13785,28696,7105,42308],"score":189,"new_id":251503},{"text":"отмечает","train_count":63,"old_ids":[12666,6240,53887,26046],"score":189,"new_id":251504},{"text":"правителем","train_count":63,"old_ids":[148607,9080,77969,6240],"score":189,"new_id":251505},{"text":"события","train_count":189,"old_ids":[224663,152149],"score":189,"new_id":251506},{"text":"совокупность","train_count":63,"old_ids":[149297,1403,149741,58245],"score":189,"new_id":251507},{"text":"сотрудничества","train_count":63,"old_ids":[217181,149035,153001,150167],"score":189,"new_id":251508},{"text":"СПб","train_count":188,"old_ids":[176156,9629],"score":188,"new_id":251509},{"text":"Славы","train_count":94,"old_ids":[18727,29354,148705],"score":188,"new_id":251510},{"text":"Указом","train_count":94,"old_ids":[39243,148330,148288],"score":188,"new_id":251511},{"text":"белорусского","train_count":47,"old_ids":[9629,11777,8789,41635,149280],"score":188,"new_id":251512},{"text":"внимания","train_count":94,"old_ids":[5301,150067,36916],"score":188,"new_id":251513},{"text":"возвращения","train_count":94,"old_ids":[150348,5301,158733],"score":188,"new_id":251514},{"text":"железнодорожные","train_count":47,"old_ids":[16467,77969,8688,166815,159327],"score":188,"new_id":251515},{"text":"заведовал","train_count":47,"old_ids":[8688,148292,148276,148275,15481],"score":188,"new_id":251516},{"text":"заведующего","train_count":47,"old_ids":[8688,148292,148276,68608,148702],"score":188,"new_id":251517},{"text":"известность","train_count":188,"old_ids":[244718,58245],"score":188,"new_id":251518},{"text":"изучение","train_count":94,"old_ids":[39427,27448,16669],"score":188,"new_id":251519},{"text":"испытаний","train_count":94,"old_ids":[13113,165676,149931],"score":188,"new_id":251520},{"text":"количестве","train_count":94,"old_ids":[45527,87018,153535],"score":188,"new_id":251521},{"text":"комитет","train_count":94,"old_ids":[148558,9080,7921],"score":188,"new_id":251522},{"text":"консерваторию","train_count":47,"old_ids":[150270,159541,5301,55389,163855],"score":188,"new_id":251523},{"text":"край","train_count":188,"old_ids":[148998,11796],"score":188,"new_id":251524},{"text":"критиков","train_count":94,"old_ids":[4640,149454,152863],"score":188,"new_id":251525},{"text":"материалам","train_count":47,"old_ids":[6240,7721,149098,92647,6240],"score":188,"new_id":251526},{"text":"музыкой","train_count":94,"old_ids":[148405,148707,148554],"score":188,"new_id":251527},{"text":"невозможно","train_count":94,"old_ids":[76069,150348,153165],"score":188,"new_id":251528},{"text":"перешла","train_count":94,"old_ids":[149208,44432,29354],"score":188,"new_id":251529},{"text":"предложил","train_count":188,"old_ids":[177917,245958],"score":188,"new_id":251530},{"text":"рукоположён","train_count":47,"old_ids":[19160,45527,220495,16467,185709],"score":188,"new_id":251531},{"text":"станицы","train_count":94,"old_ids":[6389,45144,4403],"score":188,"new_id":251532},{"text":"судебных","train_count":47,"old_ids":[2175,34723,1450,9629,42240],"score":188,"new_id":251533},{"text":"судей","train_count":94,"old_ids":[2175,34723,148298],"score":188,"new_id":251534},{"text":"близ","train_count":187,"old_ids":[9629,149466],"score":187,"new_id":251535},{"text":"уезд","train_count":187,"old_ids":[3652,151919],"score":187,"new_id":251536},{"text":"Афганистане","train_count":62,"old_ids":[34960,18768,236997,193538],"score":186,"new_id":251537},{"text":"Вадим","train_count":93,"old_ids":[15729,149099,15630],"score":186,"new_id":251538},{"text":"Восточного","train_count":62,"old_ids":[15729,23514,36103,148325],"score":186,"new_id":251539},{"text":"Генеральный","train_count":62,"old_ids":[35884,5092,7358,152027],"score":186,"new_id":251540},{"text":"Давида","train_count":62,"old_ids":[23355,148292,1739,38183],"score":186,"new_id":251541},{"text":"Земля","train_count":93,"old_ids":[34279,148281,13302],"score":186,"new_id":251542},{"text":"Знак","train_count":93,"old_ids":[34279,13785,4640],"score":186,"new_id":251543},{"text":"Израиле","train_count":62,"old_ids":[29189,156494,61057,1450],"score":186,"new_id":251544},{"text":"ООН","train_count":186,"old_ids":[19736,156877],"score":186,"new_id":251545},{"text":"Орловской","train_count":62,"old_ids":[19736,2115,148569,149193],"score":186,"new_id":251546},{"text":"Уже","train_count":186,"old_ids":[39243,56090],"score":186,"new_id":251547},{"text":"Узбекистана","train_count":62,"old_ids":[39243,8688,167691,162200],"score":186,"new_id":251548},{"text":"алфавит","train_count":62,"old_ids":[15481,18768,148292,9080],"score":186,"new_id":251549},{"text":"взаимодействие","train_count":62,"old_ids":[5301,148397,149076,203732],"score":186,"new_id":251550},{"text":"вопрос","train_count":186,"old_ids":[148331,160383],"score":186,"new_id":251551},{"text":"демократической","train_count":62,"old_ids":[95525,187677,77779,158585],"score":186,"new_id":251552},{"text":"друзьями","train_count":62,"old_ids":[6733,63698,149885,61628],"score":186,"new_id":251553},{"text":"журналов","train_count":62,"old_ids":[16467,53555,13785,148569],"score":186,"new_id":251554},{"text":"инструмента","train_count":62,"old_ids":[18372,150379,6240,84706],"score":186,"new_id":251555},{"text":"исполнения","train_count":93,"old_ids":[13113,148634,149803],"score":186,"new_id":251556},{"text":"итальянских","train_count":62,"old_ids":[9080,157230,2040,150584],"score":186,"new_id":251557},{"text":"кислота","train_count":62,"old_ids":[4640,13113,2941,150475],"score":186,"new_id":251558},{"text":"князья","train_count":62,"old_ids":[4640,37855,8688,149885],"score":186,"new_id":251559},{"text":"кризиса","train_count":62,"old_ids":[155869,8688,13113,1424],"score":186,"new_id":251560},{"text":"лауреатом","train_count":62,"old_ids":[29354,201647,7721,148288],"score":186,"new_id":251561},{"text":"леса","train_count":186,"old_ids":[2941,158634],"score":186,"new_id":251562},{"text":"литературных","train_count":62,"old_ids":[159329,7358,150849,42240],"score":186,"new_id":251563},{"text":"лётчик","train_count":93,"old_ids":[2941,150396,159796],"score":186,"new_id":251564},{"text":"массового","train_count":62,"old_ids":[6240,27473,148275,148449],"score":186,"new_id":251565},{"text":"небольшого","train_count":62,"old_ids":[76069,9629,148701,148449],"score":186,"new_id":251566},{"text":"одежды","train_count":93,"old_ids":[1403,95525,153901],"score":186,"new_id":251567},{"text":"отряд","train_count":186,"old_ids":[12666,151446],"score":186,"new_id":251568},{"text":"переводов","train_count":62,"old_ids":[149208,148380,148277,148275],"score":186,"new_id":251569},{"text":"пистолет","train_count":62,"old_ids":[7754,32884,148294,7921],"score":186,"new_id":251570},{"text":"позднего","train_count":62,"old_ids":[7754,148303,6733,150674],"score":186,"new_id":251571},{"text":"посвящена","train_count":62,"old_ids":[7754,22078,152139,182656],"score":186,"new_id":251572},{"text":"православных","train_count":62,"old_ids":[148607,22078,150800,42240],"score":186,"new_id":251573},{"text":"прибыли","train_count":93,"old_ids":[148895,148812,10750],"score":186,"new_id":251574},{"text":"принята","train_count":93,"old_ids":[148489,18372,158267],"score":186,"new_id":251575},{"text":"промышленных","train_count":93,"old_ids":[149490,153886,42240],"score":186,"new_id":251576},{"text":"путешествия","train_count":62,"old_ids":[198375,44432,32812,37835],"score":186,"new_id":251577},{"text":"располагалась","train_count":62,"old_ids":[217455,151286,15481,152993],"score":186,"new_id":251578},{"text":"рассматриваются","train_count":31,"old_ids":[2115,27473,6240,7721,2115,26125,152998],"score":186,"new_id":251579},{"text":"редко","train_count":186,"old_ids":[148420,45527],"score":186,"new_id":251580},{"text":"специальной","train_count":93,"old_ids":[155036,14666,151291],"score":186,"new_id":251581},{"text":"стадионе","train_count":93,"old_ids":[148613,148467,148917],"score":186,"new_id":251582},{"text":"сумму","train_count":93,"old_ids":[2175,37815,148405],"score":186,"new_id":251583},{"text":"территориальной","train_count":62,"old_ids":[55055,149454,150140,151291],"score":186,"new_id":251584},{"text":"топлива","train_count":93,"old_ids":[23858,149907,90219],"score":186,"new_id":251585},{"text":"трамвая","train_count":93,"old_ids":[150246,6240,155044],"score":186,"new_id":251586},{"text":"турнирах","train_count":93,"old_ids":[152238,21918,172199],"score":186,"new_id":251587},{"text":"фигуры","train_count":62,"old_ids":[18768,66951,3652,36747],"score":186,"new_id":251588},{"text":"финансовых","train_count":93,"old_ids":[221987,7105,213612],"score":186,"new_id":251589},{"text":"футбольных","train_count":62,"old_ids":[18768,36183,170946,42240],"score":186,"new_id":251590},{"text":"деньги","train_count":185,"old_ids":[198047,148388],"score":185,"new_id":251591},{"text":"женщины","train_count":185,"old_ids":[148942,182004],"score":185,"new_id":251592},{"text":"металлургический","train_count":37,"old_ids":[148557,15481,2941,157647,77436,149537],"score":185,"new_id":251593},{"text":"музыка","train_count":185,"old_ids":[148405,168062],"score":185,"new_id":251594},{"text":"Донецкой","train_count":92,"old_ids":[23355,158418,148554],"score":184,"new_id":251595},{"text":"Знамени","train_count":92,"old_ids":[34279,13785,152168],"score":184,"new_id":251596},{"text":"Иерусалим","train_count":46,"old_ids":[29189,7358,41635,45679,6240],"score":184,"new_id":251597},{"text":"Иногда","train_count":184,"old_ids":[29189,155874],"score":184,"new_id":251598},{"text":"Киселёв","train_count":46,"old_ids":[25497,13113,11777,42565,5301],"score":184,"new_id":251599},{"text":"Константинович","train_count":92,"old_ids":[157349,166493,192611],"score":184,"new_id":251600},{"text":"Средней","train_count":92,"old_ids":[18727,148420,148820],"score":184,"new_id":251601},{"text":"короткометражный","train_count":46,"old_ids":[153589,12666,45527,198295,171685],"score":184,"new_id":251602},{"text":"методом","train_count":92,"old_ids":[148557,148277,148288],"score":184,"new_id":251603},{"text":"моменту","train_count":92,"old_ids":[148324,89315,3652],"score":184,"new_id":251604},{"text":"мужской","train_count":92,"old_ids":[6240,54025,149193],"score":184,"new_id":251605},{"text":"отдельный","train_count":92,"old_ids":[12666,159194,32401],"score":184,"new_id":251606},{"text":"позволила","train_count":46,"old_ids":[7754,148303,148331,10750,29354],"score":184,"new_id":251607},{"text":"политические","train_count":92,"old_ids":[148634,9080,168329],"score":184,"new_id":251608},{"text":"популярных","train_count":92,"old_ids":[148428,171447,42240],"score":184,"new_id":251609},{"text":"постановление","train_count":46,"old_ids":[7754,23514,7105,148275,53921],"score":184,"new_id":251610},{"text":"сборная","train_count":92,"old_ids":[2175,148913,42308],"score":184,"new_id":251611},{"text":"свободное","train_count":46,"old_ids":[2175,5301,148280,148277,45480],"score":184,"new_id":251612},{"text":"сердце","train_count":92,"old_ids":[159541,6733,149053],"score":184,"new_id":251613},{"text":"сыном","train_count":184,"old_ids":[54330,148524],"score":184,"new_id":251614},{"text":"факт","train_count":92,"old_ids":[18768,148318,1729],"score":184,"new_id":251615},{"text":"художница","train_count":92,"old_ids":[157951,149244,153313],"score":184,"new_id":251616},{"text":"шириной","train_count":92,"old_ids":[28420,29119,148317],"score":184,"new_id":251617},{"text":"Брестского","train_count":61,"old_ids":[58297,2115,34981,149280],"score":183,"new_id":251618},{"text":"Мазовецкого","train_count":61,"old_ids":[36736,148646,150937,238492],"score":183,"new_id":251619},{"text":"военнослужащих","train_count":61,"old_ids":[224606,13292,167185,153309],"score":183,"new_id":251620},{"text":"духовенства","train_count":61,"old_ids":[148660,152335,5092,89771],"score":183,"new_id":251621},{"text":"использованы","train_count":61,"old_ids":[13113,183192,148675,150165],"score":183,"new_id":251622},{"text":"началу","train_count":183,"old_ids":[29196,168385],"score":183,"new_id":251623},{"text":"объёму","train_count":61,"old_ids":[148280,32482,42565,148405],"score":183,"new_id":251624},{"text":"победой","train_count":61,"old_ids":[7754,148280,148276,148293],"score":183,"new_id":251625},{"text":"подразделений","train_count":61,"old_ids":[155908,148464,55986,59318],"score":183,"new_id":251626},{"text":"политическим","train_count":61,"old_ids":[148634,9080,77436,153199],"score":183,"new_id":251627},{"text":"предполагается","train_count":61,"old_ids":[177917,148428,151286,64903],"score":183,"new_id":251628},{"text":"пригласил","train_count":61,"old_ids":[148895,14942,149827,61057],"score":183,"new_id":251629},{"text":"принадлежали","train_count":61,"old_ids":[148895,149578,148284,206018],"score":183,"new_id":251630},{"text":"редакцией","train_count":61,"old_ids":[148279,38183,4640,171120],"score":183,"new_id":251631},{"text":"считалось","train_count":61,"old_ids":[2175,87148,15481,95306],"score":183,"new_id":251632},{"text":"штаб","train_count":183,"old_ids":[153931,9629],"score":183,"new_id":251633},{"text":"Внешний","train_count":91,"old_ids":[15729,157540,149623],"score":182,"new_id":251634},{"text":"Ларри","train_count":91,"old_ids":[61757,16763,29119],"score":182,"new_id":251635},{"text":"Плотность","train_count":91,"old_ids":[16362,154713,58245],"score":182,"new_id":251636},{"text":"батальона","train_count":91,"old_ids":[168937,36771,71221],"score":182,"new_id":251637},{"text":"возникла","train_count":91,"old_ids":[150348,67494,29354],"score":182,"new_id":251638},{"text":"данного","train_count":182,"old_ids":[172895,148325],"score":182,"new_id":251639},{"text":"инженера","train_count":91,"old_ids":[18372,148942,87658],"score":182,"new_id":251640},{"text":"исполнил","train_count":91,"old_ids":[13113,159196,2941],"score":182,"new_id":251641},{"text":"массовой","train_count":91,"old_ids":[6240,27473,168715],"score":182,"new_id":251642},{"text":"осенью","train_count":91,"old_ids":[22078,49603,11577],"score":182,"new_id":251643},{"text":"предлагает","train_count":91,"old_ids":[177917,151286,26046],"score":182,"new_id":251644},{"text":"представления","train_count":91,"old_ids":[177917,148448,59595],"score":182,"new_id":251645},{"text":"приговорён","train_count":91,"old_ids":[148895,160776,185709],"score":182,"new_id":251646},{"text":"произошёл","train_count":91,"old_ids":[149490,152476,229903],"score":182,"new_id":251647},{"text":"расположенных","train_count":91,"old_ids":[148761,220495,187434],"score":182,"new_id":251648},{"text":"российского","train_count":182,"old_ids":[184874,149280],"score":182,"new_id":251649},{"text":"сельское","train_count":182,"old_ids":[226181,170391],"score":182,"new_id":251650},{"text":"сообщество","train_count":91,"old_ids":[148454,149350,49903],"score":182,"new_id":251651},{"text":"физике","train_count":91,"old_ids":[18768,39427,168861],"score":182,"new_id":251652},{"text":"штате","train_count":182,"old_ids":[184984,152657],"score":182,"new_id":251653},{"text":"Руси","train_count":181,"old_ids":[221073,1739],"score":181,"new_id":251654},{"text":"издание","train_count":181,"old_ids":[39427,158351],"score":181,"new_id":251655},{"text":"какой","train_count":181,"old_ids":[12751,148554],"score":181,"new_id":251656},{"text":"метод","train_count":181,"old_ids":[148557,148277],"score":181,"new_id":251657},{"text":"последнего","train_count":181,"old_ids":[167014,150674],"score":181,"new_id":251658},{"text":"Александром","train_count":90,"old_ids":[34960,221097,148288],"score":180,"new_id":251659},{"text":"Бейкер","train_count":90,"old_ids":[58297,148298,178259],"score":180,"new_id":251660},{"text":"Белого","train_count":90,"old_ids":[58297,11777,148449],"score":180,"new_id":251661},{"text":"Бразилия","train_count":60,"old_ids":[58297,148464,149037,4096],"score":180,"new_id":251662},{"text":"Восточный","train_count":60,"old_ids":[15729,23514,36103,32401],"score":180,"new_id":251663},{"text":"Гусейнов","train_count":60,"old_ids":[35884,41635,148298,148366],"score":180,"new_id":251664},{"text":"Дебютировал","train_count":45,"old_ids":[185698,9629,11577,21797,177702],"score":180,"new_id":251665},{"text":"Джерси","train_count":90,"old_ids":[23355,200530,148407],"score":180,"new_id":251666},{"text":"Жан","train_count":180,"old_ids":[154849,7105],"score":180,"new_id":251667},{"text":"Коммунистической","train_count":45,"old_ids":[191297,148405,21918,6389,158585],"score":180,"new_id":251668},{"text":"Михалёво","train_count":45,"old_ids":[36736,64392,15481,42565,148331],"score":180,"new_id":251669},{"text":"Православной","train_count":45,"old_ids":[16362,148381,22078,150800,148317],"score":180,"new_id":251670},{"text":"Принцип","train_count":60,"old_ids":[51844,18372,14666,7754],"score":180,"new_id":251671},{"text":"Районы","train_count":90,"old_ids":[32392,148390,154479],"score":180,"new_id":251672},{"text":"Романович","train_count":45,"old_ids":[32392,148288,7105,148275,31889],"score":180,"new_id":251673},{"text":"Топографической","train_count":60,"old_ids":[32868,154346,152973,158585],"score":180,"new_id":251674},{"text":"Третий","train_count":90,"old_ids":[32868,166196,42387],"score":180,"new_id":251675},{"text":"Третья","train_count":60,"old_ids":[32868,2115,148877,4096],"score":180,"new_id":251676},{"text":"Часть","train_count":180,"old_ids":[213878,33348],"score":180,"new_id":251677},{"text":"Экранизации","train_count":60,"old_ids":[89030,4640,149926,170369],"score":180,"new_id":251678},{"text":"Юсупов","train_count":60,"old_ids":[168959,2175,47174,148275],"score":180,"new_id":251679},{"text":"аппаратов","train_count":60,"old_ids":[1424,74359,16763,152968],"score":180,"new_id":251680},{"text":"базы","train_count":90,"old_ids":[9629,148646,4403],"score":180,"new_id":251681},{"text":"венгерский","train_count":90,"old_ids":[171477,162127,149800],"score":180,"new_id":251682},{"text":"высокая","train_count":90,"old_ids":[148705,148454,150621],"score":180,"new_id":251683},{"text":"высокий","train_count":90,"old_ids":[148705,148454,149537],"score":180,"new_id":251684},{"text":"достопримечательности","train_count":30,"old_ids":[6733,151290,148489,15630,53887,57868,65360],"score":180,"new_id":251685},{"text":"заключённых","train_count":60,"old_ids":[148397,49689,42565,170741],"score":180,"new_id":251686},{"text":"закрытия","train_count":90,"old_ids":[148397,91201,152149],"score":180,"new_id":251687},{"text":"замок","train_count":180,"old_ids":[148397,187677],"score":180,"new_id":251688},{"text":"знаменитого","train_count":60,"old_ids":[148852,36621,9080,148449],"score":180,"new_id":251689},{"text":"количеству","train_count":90,"old_ids":[45527,87018,158730],"score":180,"new_id":251690},{"text":"композиторов","train_count":60,"old_ids":[166349,148303,9080,149937],"score":180,"new_id":251691},{"text":"крепость","train_count":180,"old_ids":[151219,93309],"score":180,"new_id":251692},{"text":"маленький","train_count":60,"old_ids":[6240,15481,49603,149537],"score":180,"new_id":251693},{"text":"математический","train_count":36,"old_ids":[6240,7721,148281,7721,77436,149537],"score":180,"new_id":251694},{"text":"микрорайона","train_count":45,"old_ids":[201286,2115,8789,148390,71221],"score":180,"new_id":251695},{"text":"монастыре","train_count":90,"old_ids":[179659,151407,148279],"score":180,"new_id":251696},{"text":"независимого","train_count":36,"old_ids":[76069,8688,148292,13113,15630,148449],"score":180,"new_id":251697},{"text":"ноябрь","train_count":90,"old_ids":[13292,4096,151874],"score":180,"new_id":251698},{"text":"обозначает","train_count":60,"old_ids":[148280,148303,29196,26046],"score":180,"new_id":251699},{"text":"опубликованы","train_count":60,"old_ids":[148561,149090,148336,150165],"score":180,"new_id":251700},{"text":"оригинальной","train_count":60,"old_ids":[8789,66951,66353,159075],"score":180,"new_id":251701},{"text":"официальное","train_count":60,"old_ids":[1403,149243,36771,45480],"score":180,"new_id":251702},{"text":"погибли","train_count":90,"old_ids":[148428,148388,150029],"score":180,"new_id":251703},{"text":"поддерживает","train_count":60,"old_ids":[155908,148605,26125,26046],"score":180,"new_id":251704},{"text":"позволили","train_count":60,"old_ids":[7754,148303,148331,232220],"score":180,"new_id":251705},{"text":"показал","train_count":90,"old_ids":[148428,148330,15481],"score":180,"new_id":251706},{"text":"полковника","train_count":90,"old_ids":[148634,148336,84480],"score":180,"new_id":251707},{"text":"предыдущей","train_count":45,"old_ids":[177917,4403,6733,40794,148298],"score":180,"new_id":251708},{"text":"расширения","train_count":90,"old_ids":[148761,28420,155209],"score":180,"new_id":251709},{"text":"самостоятельно","train_count":90,"old_ids":[160002,152544,159028],"score":180,"new_id":251710},{"text":"семинарии","train_count":90,"old_ids":[148938,149717,159079],"score":180,"new_id":251711},{"text":"сменил","train_count":90,"old_ids":[2175,152168,2941],"score":180,"new_id":251712},{"text":"сотрудников","train_count":90,"old_ids":[217181,149035,149668],"score":180,"new_id":251713},{"text":"социального","train_count":90,"old_ids":[148454,14666,151302],"score":180,"new_id":251714},{"text":"территориях","train_count":60,"old_ids":[55055,149454,154617,10161],"score":180,"new_id":251715},{"text":"убийства","train_count":90,"old_ids":[61324,42387,89771],"score":180,"new_id":251716},{"text":"указом","train_count":90,"old_ids":[3652,148330,148288],"score":180,"new_id":251717},{"text":"употребление","train_count":60,"old_ids":[47174,12666,149467,53921],"score":180,"new_id":251718},{"text":"членами","train_count":90,"old_ids":[5831,152703,61628],"score":180,"new_id":251719},{"text":"игру","train_count":179,"old_ids":[66951,19160],"score":179,"new_id":251720},{"text":"Звезда","train_count":89,"old_ids":[34279,5301,157279],"score":178,"new_id":251721},{"text":"Расстояние","train_count":178,"old_ids":[225381,177369],"score":178,"new_id":251722},{"text":"Рядом","train_count":89,"old_ids":[32392,93629,148288],"score":178,"new_id":251723},{"text":"Станции","train_count":89,"old_ids":[71000,7105,54201],"score":178,"new_id":251724},{"text":"белой","train_count":89,"old_ids":[9629,11777,148293],"score":178,"new_id":251725},{"text":"будущем","train_count":89,"old_ids":[156543,40794,148281],"score":178,"new_id":251726},{"text":"восьми","train_count":89,"old_ids":[5301,95306,61628],"score":178,"new_id":251727},{"text":"голоса","train_count":89,"old_ids":[229476,22078,1424],"score":178,"new_id":251728},{"text":"дальше","train_count":89,"old_ids":[6733,36771,56916],"score":178,"new_id":251729},{"text":"единственным","train_count":178,"old_ids":[152021,161299],"score":178,"new_id":251730},{"text":"концепции","train_count":89,"old_ids":[150270,150368,54201],"score":178,"new_id":251731},{"text":"мужчины","train_count":89,"old_ids":[6240,54025,154141],"score":178,"new_id":251732},{"text":"населением","train_count":89,"old_ids":[13785,189921,149353],"score":178,"new_id":251733},{"text":"находилось","train_count":89,"old_ids":[13785,148334,158606],"score":178,"new_id":251734},{"text":"отличаются","train_count":89,"old_ids":[12666,87018,152998],"score":178,"new_id":251735},{"text":"отсутствуют","train_count":89,"old_ids":[12666,150338,151766],"score":178,"new_id":251736},{"text":"помощником","train_count":89,"old_ids":[151201,88260,159659],"score":178,"new_id":251737},{"text":"посту","train_count":89,"old_ids":[7754,23514,3652],"score":178,"new_id":251738},{"text":"представление","train_count":89,"old_ids":[177917,148448,53921],"score":178,"new_id":251739},{"text":"представлял","train_count":89,"old_ids":[177917,149814,2941],"score":178,"new_id":251740},{"text":"профессионального","train_count":89,"old_ids":[149490,165373,155045],"score":178,"new_id":251741},{"text":"религии","train_count":89,"old_ids":[2115,58503,151229],"score":178,"new_id":251742},{"text":"скорости","train_count":178,"old_ids":[167465,81839],"score":178,"new_id":251743},{"text":"специального","train_count":89,"old_ids":[155036,14666,151302],"score":178,"new_id":251744},{"text":"ссылки","train_count":89,"old_ids":[2175,63469,16256],"score":178,"new_id":251745},{"text":"утверждает","train_count":89,"old_ids":[36183,46576,162443],"score":178,"new_id":251746},{"text":"Венгрия","train_count":59,"old_ids":[15729,5092,14942,155075],"score":177,"new_id":251747},{"text":"Конгрегации","train_count":59,"old_ids":[157349,150612,14942,51788],"score":177,"new_id":251748},{"text":"Одессы","train_count":59,"old_ids":[19736,6733,21785,54330],"score":177,"new_id":251749},{"text":"Принимал","train_count":59,"old_ids":[51844,18372,15630,15481],"score":177,"new_id":251750},{"text":"Ричарда","train_count":59,"old_ids":[32392,31889,16763,38183],"score":177,"new_id":251751},{"text":"Улица","train_count":177,"old_ids":[39243,168314],"score":177,"new_id":251752},{"text":"английским","train_count":59,"old_ids":[7105,151203,11796,171949],"score":177,"new_id":251753},{"text":"вселенной","train_count":59,"old_ids":[5301,189921,5092,148317],"score":177,"new_id":251754},{"text":"замуж","train_count":177,"old_ids":[201250,54025],"score":177,"new_id":251755},{"text":"известные","train_count":177,"old_ids":[244718,41546],"score":177,"new_id":251756},{"text":"интерфейс","train_count":59,"old_ids":[18372,55055,157213,2175],"score":177,"new_id":251757},{"text":"историческое","train_count":59,"old_ids":[32884,8789,77436,153232],"score":177,"new_id":251758},{"text":"кандидатов","train_count":59,"old_ids":[4640,72884,45587,152968],"score":177,"new_id":251759},{"text":"крестьянских","train_count":59,"old_ids":[149679,156351,2040,150584],"score":177,"new_id":251760},{"text":"обозначается","train_count":59,"old_ids":[148280,148303,29196,64903],"score":177,"new_id":251761},{"text":"парка","train_count":177,"old_ids":[151963,12751],"score":177,"new_id":251762},{"text":"перебрался","train_count":59,"old_ids":[159357,157471,15481,19937],"score":177,"new_id":251763},{"text":"пересечении","train_count":59,"old_ids":[149208,21785,53887,80228],"score":177,"new_id":251764},{"text":"платформе","train_count":59,"old_ids":[7754,75774,54724,1450],"score":177,"new_id":251765},{"text":"политехнического","train_count":59,"old_ids":[148634,25135,172533,170462],"score":177,"new_id":251766},{"text":"полуострове","train_count":59,"old_ids":[164724,23514,148327,1450],"score":177,"new_id":251767},{"text":"праздника","train_count":59,"old_ids":[7754,148464,6733,84480],"score":177,"new_id":251768},{"text":"присутствии","train_count":59,"old_ids":[148489,13113,36183,152394],"score":177,"new_id":251769},{"text":"продолжить","train_count":59,"old_ids":[148489,148277,148294,156477],"score":177,"new_id":251770},{"text":"создателей","train_count":59,"old_ids":[2175,148303,153966,149074],"score":177,"new_id":251771},{"text":"супруги","train_count":59,"old_ids":[2175,47174,62215,1739],"score":177,"new_id":251772},{"text":"французским","train_count":59,"old_ids":[18768,33437,163227,171949],"score":177,"new_id":251773},{"text":"футболистов","train_count":59,"old_ids":[18768,36183,157759,153305],"score":177,"new_id":251774},{"text":"Бургенланда","train_count":44,"old_ids":[58297,157647,5092,2941,158021],"score":176,"new_id":251775},{"text":"Джузеппе","train_count":44,"old_ids":[23355,150939,152165,74359,1450],"score":176,"new_id":251776},{"text":"Западно","train_count":88,"old_ids":[172194,148414,153244],"score":176,"new_id":251777},{"text":"Информация","train_count":88,"old_ids":[161484,54724,53890],"score":176,"new_id":251778},{"text":"Канаде","train_count":88,"old_ids":[25497,148841,95525],"score":176,"new_id":251779},{"text":"Композиторы","train_count":44,"old_ids":[191297,7754,148303,158559,4403],"score":176,"new_id":251780},{"text":"Парижа","train_count":88,"old_ids":[16362,148699,148458],"score":176,"new_id":251781},{"text":"Франческо","train_count":88,"old_ids":[221647,148614,45527],"score":176,"new_id":251782},{"text":"артиллерийских","train_count":44,"old_ids":[16763,21797,165328,42387,150584],"score":176,"new_id":251783},{"text":"видимому","train_count":88,"old_ids":[152833,15630,149169],"score":176,"new_id":251784},{"text":"воздушных","train_count":88,"old_ids":[150348,203308,42240],"score":176,"new_id":251785},{"text":"восстановлен","train_count":88,"old_ids":[163934,148722,58361],"score":176,"new_id":251786},{"text":"должно","train_count":176,"old_ids":[159172,155015],"score":176,"new_id":251787},{"text":"западном","train_count":88,"old_ids":[148397,177720,148524],"score":176,"new_id":251788},{"text":"значительную","train_count":88,"old_ids":[149340,53062,73781],"score":176,"new_id":251789},{"text":"итальянской","train_count":88,"old_ids":[9080,157230,176771],"score":176,"new_id":251790},{"text":"кандидатскую","train_count":44,"old_ids":[4640,72884,45587,7721,171123],"score":176,"new_id":251791},{"text":"королева","train_count":88,"old_ids":[4640,148367,208787],"score":176,"new_id":251792},{"text":"лучшую","train_count":176,"old_ids":[151408,172268],"score":176,"new_id":251793},{"text":"мелкие","train_count":88,"old_ids":[6240,11777,149012],"score":176,"new_id":251794},{"text":"микрорайон","train_count":44,"old_ids":[201286,2115,8789,148390,16806],"score":176,"new_id":251795},{"text":"основная","train_count":88,"old_ids":[22078,148366,42308],"score":176,"new_id":251796},{"text":"официальный","train_count":88,"old_ids":[1403,149243,152027],"score":176,"new_id":251797},{"text":"пассажирского","train_count":44,"old_ids":[7754,27473,149096,74990,149280],"score":176,"new_id":251798},{"text":"первенстве","train_count":88,"old_ids":[149208,171477,151418],"score":176,"new_id":251799},{"text":"последующих","train_count":88,"old_ids":[167014,68608,64392],"score":176,"new_id":251800},{"text":"прошла","train_count":176,"old_ids":[149490,153870],"score":176,"new_id":251801},{"text":"пьесы","train_count":88,"old_ids":[187715,21785,4403],"score":176,"new_id":251802},{"text":"сигнала","train_count":88,"old_ids":[2175,66951,156045],"score":176,"new_id":251803},{"text":"следующего","train_count":88,"old_ids":[149161,68608,148702],"score":176,"new_id":251804},{"text":"срок","train_count":176,"old_ids":[2175,155110],"score":176,"new_id":251805},{"text":"сцены","train_count":88,"old_ids":[2175,10064,75896],"score":176,"new_id":251806},{"text":"считаются","train_count":88,"old_ids":[2175,87148,152998],"score":176,"new_id":251807},{"text":"трассе","train_count":88,"old_ids":[153298,27473,1450],"score":176,"new_id":251808},{"text":"участвует","train_count":88,"old_ids":[27448,1424,151005],"score":176,"new_id":251809},{"text":"учреждение","train_count":88,"old_ids":[27448,148279,155368],"score":176,"new_id":251810},{"text":"экономике","train_count":88,"old_ids":[65131,150192,168861],"score":176,"new_id":251811},{"text":"Массачусетс","train_count":35,"old_ids":[36736,27473,47474,41635,7921,2175],"score":175,"new_id":251812},{"text":"достижения","train_count":175,"old_ids":[239082,150282],"score":175,"new_id":251813},{"text":"земель","train_count":175,"old_ids":[154161,24623],"score":175,"new_id":251814},{"text":"подготовки","train_count":175,"old_ids":[155908,207586],"score":175,"new_id":251815},{"text":"реабилитирован","train_count":35,"old_ids":[148279,153051,61057,9080,149591,7105],"score":175,"new_id":251816},{"text":"среднем","train_count":175,"old_ids":[174301,152217],"score":175,"new_id":251817},{"text":"Исландии","train_count":58,"old_ids":[29189,38612,72884,44633],"score":174,"new_id":251818},{"text":"Казань","train_count":58,"old_ids":[25497,148646,7105,4669],"score":174,"new_id":251819},{"text":"Калгари","train_count":58,"old_ids":[25497,15481,14942,148699],"score":174,"new_id":251820},{"text":"Коломенского","train_count":58,"old_ids":[25497,148378,36621,149280],"score":174,"new_id":251821},{"text":"Колумбии","train_count":58,"old_ids":[25497,148294,37815,223125],"score":174,"new_id":251822},{"text":"Комитет","train_count":87,"old_ids":[191297,9080,7921],"score":174,"new_id":251823},{"text":"Мексике","train_count":58,"old_ids":[36736,148301,2175,168861],"score":174,"new_id":251824},{"text":"Михаэль","train_count":58,"old_ids":[36736,64392,1424,220452],"score":174,"new_id":251825},{"text":"Наряду","train_count":58,"old_ids":[19582,16763,93629,3652],"score":174,"new_id":251826},{"text":"Нормандии","train_count":58,"old_ids":[19582,30206,72884,44633],"score":174,"new_id":251827},{"text":"Орсини","train_count":58,"old_ids":[19736,159344,18372,1739],"score":174,"new_id":251828},{"text":"Реки","train_count":174,"old_ids":[175129,16256],"score":174,"new_id":251829},{"text":"Рязанской","train_count":58,"old_ids":[32392,60860,7105,149193],"score":174,"new_id":251830},{"text":"Фрэнк","train_count":58,"old_ids":[52337,2115,156454,4640],"score":174,"new_id":251831},{"text":"вертолёта","train_count":58,"old_ids":[46576,23858,172384,42074],"score":174,"new_id":251832},{"text":"возвращение","train_count":58,"old_ids":[150348,5301,50022,16669],"score":174,"new_id":251833},{"text":"воздухе","train_count":87,"old_ids":[150348,148660,159679],"score":174,"new_id":251834},{"text":"вокалист","train_count":87,"old_ids":[148331,162547,6389],"score":174,"new_id":251835},{"text":"входили","train_count":174,"old_ids":[5301,205240],"score":174,"new_id":251836},{"text":"другого","train_count":174,"old_ids":[192548,148449],"score":174,"new_id":251837},{"text":"записал","train_count":87,"old_ids":[148397,28696,15481],"score":174,"new_id":251838},{"text":"звучание","train_count":58,"old_ids":[8688,5301,27448,25767],"score":174,"new_id":251839},{"text":"звучит","train_count":58,"old_ids":[8688,5301,27448,9080],"score":174,"new_id":251840},{"text":"значительное","train_count":87,"old_ids":[149340,53062,45480],"score":174,"new_id":251841},{"text":"исполнял","train_count":87,"old_ids":[13113,148634,238117],"score":174,"new_id":251842},{"text":"командующим","train_count":58,"old_ids":[148558,72884,68608,15630],"score":174,"new_id":251843},{"text":"концертов","train_count":87,"old_ids":[150270,180269,148779],"score":174,"new_id":251844},{"text":"критики","train_count":87,"old_ids":[4640,149454,151278],"score":174,"new_id":251845},{"text":"массовых","train_count":58,"old_ids":[6240,27473,148275,60038],"score":174,"new_id":251846},{"text":"механики","train_count":58,"old_ids":[6240,93834,13603,16256],"score":174,"new_id":251847},{"text":"окрестностей","train_count":58,"old_ids":[148518,2115,34981,151273],"score":174,"new_id":251848},{"text":"оперы","train_count":174,"old_ids":[148561,172094],"score":174,"new_id":251849},{"text":"отметить","train_count":87,"old_ids":[12666,148557,17256],"score":174,"new_id":251850},{"text":"отсутствия","train_count":87,"old_ids":[12666,150338,151172],"score":174,"new_id":251851},{"text":"покинуть","train_count":87,"old_ids":[148428,16256,150367],"score":174,"new_id":251852},{"text":"построили","train_count":58,"old_ids":[7754,23514,148283,149037],"score":174,"new_id":251853},{"text":"поэтесса","train_count":58,"old_ids":[148428,184088,90202,1424],"score":174,"new_id":251854},{"text":"привели","train_count":87,"old_ids":[148489,26125,58503],"score":174,"new_id":251855},{"text":"продолжается","train_count":58,"old_ids":[148489,148277,148294,156687],"score":174,"new_id":251856},{"text":"разработанный","train_count":58,"old_ids":[148464,149100,26269,11796],"score":174,"new_id":251857},{"text":"романов","train_count":87,"old_ids":[149633,7105,148275],"score":174,"new_id":251858},{"text":"советского","train_count":174,"old_ids":[227012,149280],"score":174,"new_id":251859},{"text":"сообщает","train_count":87,"old_ids":[148454,149350,26046],"score":174,"new_id":251860},{"text":"уровней","train_count":87,"old_ids":[53555,148275,148820],"score":174,"new_id":251861},{"text":"установить","train_count":87,"old_ids":[3652,148722,17256],"score":174,"new_id":251862},{"text":"шести","train_count":174,"old_ids":[11802,85198],"score":174,"new_id":251863},{"text":"яйца","train_count":87,"old_ids":[4096,11796,83895],"score":174,"new_id":251864},{"text":"вторым","train_count":173,"old_ids":[167396,149957],"score":173,"new_id":251865},{"text":"основу","train_count":173,"old_ids":[22078,223292],"score":173,"new_id":251866},{"text":"собор","train_count":173,"old_ids":[169983,8789],"score":173,"new_id":251867},{"text":"тремя","train_count":173,"old_ids":[1729,156579],"score":173,"new_id":251868},{"text":"Антон","train_count":86,"old_ids":[34960,155092,16806],"score":172,"new_id":251869},{"text":"Археологические","train_count":43,"old_ids":[199593,159679,148378,14942,168329],"score":172,"new_id":251870},{"text":"Белый","train_count":86,"old_ids":[58297,11777,33477],"score":172,"new_id":251871},{"text":"Вестфалия","train_count":43,"old_ids":[15729,34981,18768,45679,4096],"score":172,"new_id":251872},{"text":"Воспоминания","train_count":43,"old_ids":[15729,22078,151201,18372,36916],"score":172,"new_id":251873},{"text":"КГБ","train_count":86,"old_ids":[25497,35884,58297],"score":172,"new_id":251874},{"text":"Литературные","train_count":43,"old_ids":[61757,9080,7358,150849,41546],"score":172,"new_id":251875},{"text":"Невского","train_count":86,"old_ids":[60508,5301,149280],"score":172,"new_id":251876},{"text":"Перед","train_count":172,"old_ids":[95169,148276],"score":172,"new_id":251877},{"text":"Тимофеевич","train_count":43,"old_ids":[32868,149076,18768,47745,159380],"score":172,"new_id":251878},{"text":"Хонъимбо","train_count":43,"old_ids":[150853,16806,32482,15630,148421],"score":172,"new_id":251879},{"text":"Хосе","train_count":86,"old_ids":[150853,22078,1450],"score":172,"new_id":251880},{"text":"адмирала","train_count":86,"old_ids":[149099,159733,92647],"score":172,"new_id":251881},{"text":"архитектура","train_count":43,"old_ids":[16763,10161,25135,4640,159142],"score":172,"new_id":251882},{"text":"высоким","train_count":86,"old_ids":[148705,148454,153199],"score":172,"new_id":251883},{"text":"высокого","train_count":86,"old_ids":[148705,148454,149075],"score":172,"new_id":251884},{"text":"генерального","train_count":86,"old_ids":[170802,7358,151302],"score":172,"new_id":251885},{"text":"государствами","train_count":43,"old_ids":[14942,22078,150968,89771,61628],"score":172,"new_id":251886},{"text":"далеко","train_count":86,"old_ids":[39596,1450,45527],"score":172,"new_id":251887},{"text":"информационной","train_count":43,"old_ids":[18372,54724,28376,16806,148317],"score":172,"new_id":251888},{"text":"компьютеров","train_count":86,"old_ids":[166349,84920,201158],"score":172,"new_id":251889},{"text":"концерты","train_count":86,"old_ids":[150270,180269,82942],"score":172,"new_id":251890},{"text":"крупного","train_count":86,"old_ids":[4640,149568,148325],"score":172,"new_id":251891},{"text":"металлический","train_count":43,"old_ids":[148557,15481,87018,21785,149537],"score":172,"new_id":251892},{"text":"мечети","train_count":86,"old_ids":[6240,53887,149519],"score":172,"new_id":251893},{"text":"мирового","train_count":86,"old_ids":[61628,148327,148449],"score":172,"new_id":251894},{"text":"открыты","train_count":86,"old_ids":[12666,91201,82942],"score":172,"new_id":251895},{"text":"открытый","train_count":86,"old_ids":[12666,91201,159557],"score":172,"new_id":251896},{"text":"плана","train_count":172,"old_ids":[150469,148841],"score":172,"new_id":251897},{"text":"поведения","train_count":86,"old_ids":[150772,148276,17941],"score":172,"new_id":251898},{"text":"помощник","train_count":86,"old_ids":[151201,88260,67494],"score":172,"new_id":251899},{"text":"рабочей","train_count":86,"old_ids":[21441,151293,148298],"score":172,"new_id":251900},{"text":"сведения","train_count":172,"old_ids":[2175,151536],"score":172,"new_id":251901},{"text":"семь","train_count":172,"old_ids":[159707,4669],"score":172,"new_id":251902},{"text":"стрельбы","train_count":86,"old_ids":[48941,24623,148812],"score":172,"new_id":251903},{"text":"суд","train_count":172,"old_ids":[2175,34723],"score":172,"new_id":251904},{"text":"судебной","train_count":43,"old_ids":[2175,34723,1450,9629,148317],"score":172,"new_id":251905},{"text":"существовал","train_count":86,"old_ids":[2175,154659,151019],"score":172,"new_id":251906},{"text":"участвовать","train_count":86,"old_ids":[27448,1424,153630],"score":172,"new_id":251907},{"text":"фантастический","train_count":43,"old_ids":[18768,157146,6389,77436,149537],"score":172,"new_id":251908},{"text":"царство","train_count":86,"old_ids":[10064,16763,148593],"score":172,"new_id":251909},{"text":"якобы","train_count":86,"old_ids":[4096,45527,148812],"score":172,"new_id":251910},{"text":"Волгоградской","train_count":57,"old_ids":[15729,148294,63046,167220],"score":171,"new_id":251911},{"text":"Египте","train_count":57,"old_ids":[53241,14942,45768,48824],"score":171,"new_id":251912},{"text":"Королевского","train_count":57,"old_ids":[25497,148367,151186,149280],"score":171,"new_id":251913},{"text":"Палаццо","train_count":57,"old_ids":[16362,92647,10064,153238],"score":171,"new_id":251914},{"text":"Парижской","train_count":57,"old_ids":[16362,148699,16467,149193],"score":171,"new_id":251915},{"text":"Сам","train_count":171,"old_ids":[18727,148357],"score":171,"new_id":251916},{"text":"Саратовской","train_count":57,"old_ids":[18727,16763,152968,149193],"score":171,"new_id":251917},{"text":"Стефана","train_count":57,"old_ids":[18727,48824,18768,148841],"score":171,"new_id":251918},{"text":"Шуберта","train_count":57,"old_ids":[151311,61324,7358,42074],"score":171,"new_id":251919},{"text":"выступавший","train_count":57,"old_ids":[148705,148757,148292,151642],"score":171,"new_id":251920},{"text":"демократов","train_count":57,"old_ids":[95525,187677,77779,148275],"score":171,"new_id":251921},{"text":"документальный","train_count":57,"old_ids":[148551,34083,89315,152027],"score":171,"new_id":251922},{"text":"идёт","train_count":171,"old_ids":[45587,150396],"score":171,"new_id":251923},{"text":"казахстанский","train_count":57,"old_ids":[149387,10161,49224,149800],"score":171,"new_id":251924},{"text":"максимальная","train_count":57,"old_ids":[66050,170821,87857,42308],"score":171,"new_id":251925},{"text":"необходимые","train_count":57,"old_ids":[76069,148280,227708,48933],"score":171,"new_id":251926},{"text":"объясняется","train_count":57,"old_ids":[148280,32482,151665,173065],"score":171,"new_id":251927},{"text":"отечественной","train_count":57,"old_ids":[12666,53887,150198,148317],"score":171,"new_id":251928},{"text":"продолжения","train_count":57,"old_ids":[148489,148277,148294,150282],"score":171,"new_id":251929},{"text":"статусом","train_count":57,"old_ids":[6389,7721,41635,148288],"score":171,"new_id":251930},{"text":"теоретической","train_count":57,"old_ids":[48824,8789,7921,158585],"score":171,"new_id":251931},{"text":"Великобританию","train_count":34,"old_ids":[15729,58503,45527,9629,149454,87252],"score":170,"new_id":251932},{"text":"Людовика","train_count":85,"old_ids":[207912,148716,71658],"score":170,"new_id":251933},{"text":"Махтумкули","train_count":34,"old_ids":[173345,10161,1729,37815,34083,10750],"score":170,"new_id":251934},{"text":"Олимпийские","train_count":85,"old_ids":[19736,204086,153218],"score":170,"new_id":251935},{"text":"Сент","train_count":170,"old_ids":[18727,18110],"score":170,"new_id":251936},{"text":"Существуют","train_count":85,"old_ids":[18727,82577,149460],"score":170,"new_id":251937},{"text":"включена","train_count":85,"old_ids":[5301,49689,48623],"score":170,"new_id":251938},{"text":"грекокатолическая","train_count":34,"old_ids":[150612,210215,149900,87018,21785,150621],"score":170,"new_id":251939},{"text":"заняли","train_count":85,"old_ids":[193134,4096,10750],"score":170,"new_id":251940},{"text":"запад","train_count":170,"old_ids":[148397,177720],"score":170,"new_id":251941},{"text":"западнее","train_count":85,"old_ids":[148397,177720,149499],"score":170,"new_id":251942},{"text":"записей","train_count":85,"old_ids":[148397,28696,148298],"score":170,"new_id":251943},{"text":"кинофестиваль","train_count":34,"old_ids":[16256,13292,18768,34981,26125,36771],"score":170,"new_id":251944},{"text":"коллективы","train_count":85,"old_ids":[180702,148470,164476],"score":170,"new_id":251945},{"text":"переводе","train_count":170,"old_ids":[159357,202031],"score":170,"new_id":251946},{"text":"полуострова","train_count":85,"old_ids":[164724,23514,169887],"score":170,"new_id":251947},{"text":"популярности","train_count":85,"old_ids":[148428,171447,65360],"score":170,"new_id":251948},{"text":"почётный","train_count":85,"old_ids":[152988,150396,32401],"score":170,"new_id":251949},{"text":"проведена","train_count":85,"old_ids":[161800,148276,48623],"score":170,"new_id":251950},{"text":"сезонов","train_count":85,"old_ids":[2175,148377,152740],"score":170,"new_id":251951},{"text":"чёрные","train_count":85,"old_ids":[5831,156204,41546],"score":170,"new_id":251952},{"text":"запись","train_count":169,"old_ids":[148397,154662],"score":169,"new_id":251953},{"text":"известного","train_count":169,"old_ids":[244718,148325],"score":169,"new_id":251954},{"text":"мне","train_count":169,"old_ids":[6240,76069],"score":169,"new_id":251955},{"text":"отношению","train_count":169,"old_ids":[152928,160366],"score":169,"new_id":251956},{"text":"раньше","train_count":169,"old_ids":[33437,161954],"score":169,"new_id":251957},{"text":"ВВС","train_count":168,"old_ids":[15729,198772],"score":168,"new_id":251958},{"text":"Городские","train_count":84,"old_ids":[35884,149345,153218],"score":168,"new_id":251959},{"text":"Занимался","train_count":42,"old_ids":[34279,13603,6240,15481,19937],"score":168,"new_id":251960},{"text":"Кандидат","train_count":56,"old_ids":[25497,72884,45587,7721],"score":168,"new_id":251961},{"text":"Конституции","train_count":84,"old_ids":[157349,152608,54201],"score":168,"new_id":251962},{"text":"Красноярск","train_count":42,"old_ids":[25497,148761,13292,163466,62847],"score":168,"new_id":251963},{"text":"Льва","train_count":84,"old_ids":[61757,4669,90219],"score":168,"new_id":251964},{"text":"Манчестер","train_count":56,"old_ids":[227647,5831,34981,7358],"score":168,"new_id":251965},{"text":"Мурманской","train_count":56,"old_ids":[36736,53555,155066,149193],"score":168,"new_id":251966},{"text":"Нойштадт","train_count":42,"old_ids":[19582,148293,153931,6733,1729],"score":168,"new_id":251967},{"text":"Посёлки","train_count":56,"old_ids":[16362,22078,162292,16256],"score":168,"new_id":251968},{"text":"Романов","train_count":56,"old_ids":[32392,148288,7105,148275],"score":168,"new_id":251969},{"text":"Спортивные","train_count":84,"old_ids":[18727,153624,160124],"score":168,"new_id":251970},{"text":"Уэбб","train_count":56,"old_ids":[39243,65131,9629,9629],"score":168,"new_id":251971},{"text":"Штирии","train_count":84,"old_ids":[151311,21797,159079],"score":168,"new_id":251972},{"text":"Штирия","train_count":84,"old_ids":[151311,21797,155075],"score":168,"new_id":251973},{"text":"Штутгарт","train_count":42,"old_ids":[151311,1729,36183,14942,57383],"score":168,"new_id":251974},{"text":"авиационный","train_count":42,"old_ids":[148292,1739,28376,16806,32401],"score":168,"new_id":251975},{"text":"автомобили","train_count":56,"old_ids":[148292,149127,148280,149037],"score":168,"new_id":251976},{"text":"вариантов","train_count":84,"old_ids":[155281,7105,148779],"score":168,"new_id":251977},{"text":"внутренние","train_count":56,"old_ids":[5301,154175,148516,150528],"score":168,"new_id":251978},{"text":"возвышенности","train_count":56,"old_ids":[150348,148705,148964,65360],"score":168,"new_id":251979},{"text":"возле","train_count":168,"old_ids":[150348,148284],"score":168,"new_id":251980},{"text":"волосы","train_count":84,"old_ids":[157156,22078,4403],"score":168,"new_id":251981},{"text":"встречает","train_count":56,"old_ids":[5301,48941,53887,26046],"score":168,"new_id":251982},{"text":"выражение","train_count":84,"old_ids":[148705,21441,150307],"score":168,"new_id":251983},{"text":"газетах","train_count":56,"old_ids":[148369,8688,149005,10161],"score":168,"new_id":251984},{"text":"государственное","train_count":56,"old_ids":[14942,22078,150968,173992],"score":168,"new_id":251985},{"text":"государственную","train_count":56,"old_ids":[14942,22078,150968,186380],"score":168,"new_id":251986},{"text":"греческий","train_count":56,"old_ids":[154731,53887,21785,149537],"score":168,"new_id":251987},{"text":"диапазоне","train_count":56,"old_ids":[176661,148414,8688,148917],"score":168,"new_id":251988},{"text":"достопримечательностей","train_count":28,"old_ids":[6733,151290,148489,15630,53887,57868,151273],"score":168,"new_id":251989},{"text":"европейского","train_count":56,"old_ids":[148380,148283,152679,149280],"score":168,"new_id":251990},{"text":"железных","train_count":84,"old_ids":[16467,77969,217560],"score":168,"new_id":251991},{"text":"заводом","train_count":56,"old_ids":[8688,148292,148277,148288],"score":168,"new_id":251992},{"text":"запущен","train_count":56,"old_ids":[148397,7754,40794,5092],"score":168,"new_id":251993},{"text":"играла","train_count":84,"old_ids":[66951,2115,92647],"score":168,"new_id":251994},{"text":"издательства","train_count":84,"old_ids":[39427,6733,168548],"score":168,"new_id":251995},{"text":"изучению","train_count":84,"old_ids":[39427,27448,148922],"score":168,"new_id":251996},{"text":"контроль","train_count":168,"old_ids":[197078,27112],"score":168,"new_id":251997},{"text":"многозначный","train_count":56,"old_ids":[6240,148325,149340,32401],"score":168,"new_id":251998},{"text":"наибольшее","train_count":56,"old_ids":[13785,151223,148701,47745],"score":168,"new_id":251999},{"text":"научные","train_count":84,"old_ids":[13785,27448,41546],"score":168,"new_id":252000},{"text":"национальности","train_count":84,"old_ids":[13785,152208,173415],"score":168,"new_id":252001},{"text":"ничего","train_count":168,"old_ids":[153001,148702],"score":168,"new_id":252002},{"text":"одержал","train_count":84,"old_ids":[148277,55638,15481],"score":168,"new_id":252003},{"text":"операционных","train_count":42,"old_ids":[148561,7358,28376,16806,42240],"score":168,"new_id":252004},{"text":"переговоры","train_count":84,"old_ids":[159357,160776,4403],"score":168,"new_id":252005},{"text":"позволяло","train_count":42,"old_ids":[7754,148303,148331,13302,37483],"score":168,"new_id":252006},{"text":"поисках","train_count":84,"old_ids":[148428,13113,150577],"score":168,"new_id":252007},{"text":"поколение","train_count":84,"old_ids":[148428,45527,53921],"score":168,"new_id":252008},{"text":"политика","train_count":84,"old_ids":[148634,9080,71658],"score":168,"new_id":252009},{"text":"посередине","train_count":42,"old_ids":[7754,22078,148579,67958,1450],"score":168,"new_id":252010},{"text":"построения","train_count":56,"old_ids":[7754,23514,148283,17941],"score":168,"new_id":252011},{"text":"появление","train_count":84,"old_ids":[7754,58255,150560],"score":168,"new_id":252012},{"text":"приглашён","train_count":56,"old_ids":[148895,149809,11802,185709],"score":168,"new_id":252013},{"text":"принадлежности","train_count":56,"old_ids":[148895,149578,148284,153925],"score":168,"new_id":252014},{"text":"прототип","train_count":56,"old_ids":[148489,12666,12666,45768],"score":168,"new_id":252015},{"text":"протяжённостью","train_count":42,"old_ids":[148489,156442,16467,185709,150616],"score":168,"new_id":252016},{"text":"путей","train_count":168,"old_ids":[198375,148298],"score":168,"new_id":252017},{"text":"разработки","train_count":168,"old_ids":[148464,194980],"score":168,"new_id":252018},{"text":"рассматривает","train_count":28,"old_ids":[2115,27473,6240,7721,2115,26125,26046],"score":168,"new_id":252019},{"text":"руководителей","train_count":84,"old_ids":[19160,150463,150471],"score":168,"new_id":252020},{"text":"свидетельства","train_count":42,"old_ids":[2175,152833,7921,24623,89771],"score":168,"new_id":252021},{"text":"слияния","train_count":84,"old_ids":[31140,4096,155012],"score":168,"new_id":252022},{"text":"содержащий","train_count":42,"old_ids":[2175,148277,55638,1424,152311],"score":168,"new_id":252023},{"text":"сопротивления","train_count":84,"old_ids":[148454,200359,59595],"score":168,"new_id":252024},{"text":"сохранить","train_count":84,"old_ids":[148454,58805,17256],"score":168,"new_id":252025},{"text":"улицу","train_count":84,"old_ids":[3652,63890,3652],"score":168,"new_id":252026},{"text":"условиям","train_count":56,"old_ids":[41635,148569,37835,6240],"score":168,"new_id":252027},{"text":"успешным","train_count":56,"old_ids":[3652,31608,44432,148540],"score":168,"new_id":252028},{"text":"человеку","train_count":56,"old_ids":[5831,11777,150937,34083],"score":168,"new_id":252029},{"text":"чёрно","train_count":84,"old_ids":[5831,156204,13292],"score":168,"new_id":252030},{"text":"Более","train_count":167,"old_ids":[58297,152156],"score":167,"new_id":252031},{"text":"выход","train_count":167,"old_ids":[155037,148277],"score":167,"new_id":252032},{"text":"третьей","train_count":167,"old_ids":[200024,148298],"score":167,"new_id":252033},{"text":"Востоке","train_count":83,"old_ids":[15729,151290,50800],"score":166,"new_id":252034},{"text":"Всеволод","train_count":83,"old_ids":[188786,227700,6733],"score":166,"new_id":252035},{"text":"Иисуса","train_count":83,"old_ids":[29189,13113,165383],"score":166,"new_id":252036},{"text":"Людмила","train_count":83,"old_ids":[61757,171640,212220],"score":166,"new_id":252037},{"text":"Рейна","train_count":83,"old_ids":[32392,148298,13785],"score":166,"new_id":252038},{"text":"Русские","train_count":166,"old_ids":[221073,153218],"score":166,"new_id":252039},{"text":"Слово","train_count":83,"old_ids":[18727,37483,148331],"score":166,"new_id":252040},{"text":"ЦСКА","train_count":166,"old_ids":[89602,166874],"score":166,"new_id":252041},{"text":"Чарли","train_count":83,"old_ids":[70345,16763,10750],"score":166,"new_id":252042},{"text":"Юрьевич","train_count":83,"old_ids":[168959,149213,198326],"score":166,"new_id":252043},{"text":"Япония","train_count":83,"old_ids":[82440,154767,37835],"score":166,"new_id":252044},{"text":"верхней","train_count":166,"old_ids":[150652,148820],"score":166,"new_id":252045},{"text":"высокую","train_count":83,"old_ids":[148705,148454,155297],"score":166,"new_id":252046},{"text":"географии","train_count":83,"old_ids":[149068,1403,163617],"score":166,"new_id":252047},{"text":"границей","train_count":83,"old_ids":[14942,91872,148298],"score":166,"new_id":252048},{"text":"дворце","train_count":83,"old_ids":[6733,149086,149053],"score":166,"new_id":252049},{"text":"делам","train_count":166,"old_ids":[55986,148357],"score":166,"new_id":252050},{"text":"записан","train_count":83,"old_ids":[148397,28696,7105],"score":166,"new_id":252051},{"text":"изделия","train_count":83,"old_ids":[39427,151399,4096],"score":166,"new_id":252052},{"text":"молодых","train_count":166,"old_ids":[209034,218913],"score":166,"new_id":252053},{"text":"называть","train_count":83,"old_ids":[13785,8688,151297],"score":166,"new_id":252054},{"text":"неизвестно","train_count":83,"old_ids":[76069,244718,13292],"score":166,"new_id":252055},{"text":"отрасли","train_count":83,"old_ids":[12666,148761,10750],"score":166,"new_id":252056},{"text":"официальных","train_count":83,"old_ids":[1403,149243,152994],"score":166,"new_id":252057},{"text":"приготовления","train_count":83,"old_ids":[148895,149341,59595],"score":166,"new_id":252058},{"text":"приказу","train_count":83,"old_ids":[148895,148330,3652],"score":166,"new_id":252059},{"text":"произошла","train_count":83,"old_ids":[149490,152476,153870],"score":166,"new_id":252060},{"text":"птицы","train_count":166,"old_ids":[160142,155055],"score":166,"new_id":252061},{"text":"смысле","train_count":166,"old_ids":[2175,222870],"score":166,"new_id":252062},{"text":"стадиона","train_count":83,"old_ids":[148613,148467,71221],"score":166,"new_id":252063},{"text":"танка","train_count":166,"old_ids":[171780,12751],"score":166,"new_id":252064},{"text":"торговля","train_count":83,"old_ids":[35458,148762,13302],"score":166,"new_id":252065},{"text":"трамвай","train_count":83,"old_ids":[150246,6240,176249],"score":166,"new_id":252066},{"text":"фронте","train_count":83,"old_ids":[18768,151673,48824],"score":166,"new_id":252067},{"text":"церковью","train_count":83,"old_ids":[180269,148336,84920],"score":166,"new_id":252068},{"text":"Министерство","train_count":55,"old_ids":[36736,18372,154507,148593],"score":165,"new_id":252069},{"text":"Нижнесилезского","train_count":33,"old_ids":[19582,151932,149904,61057,148377,149280],"score":165,"new_id":252070},{"text":"Центральном","train_count":55,"old_ids":[89602,18110,157991,148524],"score":165,"new_id":252071},{"text":"Ярославской","train_count":55,"old_ids":[82440,39605,150800,149193],"score":165,"new_id":252072},{"text":"авиакомпании","train_count":55,"old_ids":[148292,148517,148558,164420],"score":165,"new_id":252073},{"text":"актёром","train_count":55,"old_ids":[148318,1729,42565,149633],"score":165,"new_id":252074},{"text":"биологических","train_count":55,"old_ids":[148687,148378,14942,151308],"score":165,"new_id":252075},{"text":"встретил","train_count":55,"old_ids":[5301,48941,7921,61057],"score":165,"new_id":252076},{"text":"губернаторство","train_count":33,"old_ids":[14942,61324,7358,2040,55389,148593],"score":165,"new_id":252077},{"text":"зависимость","train_count":55,"old_ids":[8688,148292,13113,161757],"score":165,"new_id":252078},{"text":"имена","train_count":165,"old_ids":[15630,48623],"score":165,"new_id":252079},{"text":"используемый","train_count":55,"old_ids":[13113,183192,148995,173556],"score":165,"new_id":252080},{"text":"историческом","train_count":55,"old_ids":[32884,8789,77436,148558],"score":165,"new_id":252081},{"text":"командующего","train_count":55,"old_ids":[148558,72884,68608,148702],"score":165,"new_id":252082},{"text":"литературу","train_count":55,"old_ids":[159329,7358,149959,19160],"score":165,"new_id":252083},{"text":"мало","train_count":165,"old_ids":[6240,150182],"score":165,"new_id":252084},{"text":"отдела","train_count":165,"old_ids":[12666,151400],"score":165,"new_id":252085},{"text":"партию","train_count":165,"old_ids":[151963,174325],"score":165,"new_id":252086},{"text":"периодически","train_count":55,"old_ids":[7754,149098,148277,165455],"score":165,"new_id":252087},{"text":"политическую","train_count":55,"old_ids":[148634,9080,77436,155297],"score":165,"new_id":252088},{"text":"полномочий","train_count":55,"old_ids":[148634,148524,36103,42387],"score":165,"new_id":252089},{"text":"поселился","train_count":55,"old_ids":[7754,22078,58503,151397],"score":165,"new_id":252090},{"text":"принадлежность","train_count":55,"old_ids":[148895,149578,148284,156767],"score":165,"new_id":252091},{"text":"приобретает","train_count":55,"old_ids":[148895,149962,7921,26046],"score":165,"new_id":252092},{"text":"проведённых","train_count":55,"old_ids":[161800,148276,42565,170741],"score":165,"new_id":252093},{"text":"реставрации","train_count":55,"old_ids":[2115,34981,148292,191514],"score":165,"new_id":252094},{"text":"сохранился","train_count":55,"old_ids":[148454,10161,149926,151397],"score":165,"new_id":252095},{"text":"сочинение","train_count":55,"old_ids":[2175,36103,18372,16669],"score":165,"new_id":252096},{"text":"столкновения","train_count":55,"old_ids":[6389,148294,4640,164261],"score":165,"new_id":252097},{"text":"увеличилось","train_count":55,"old_ids":[149577,58503,5831,158606],"score":165,"new_id":252098},{"text":"уничтожить","train_count":55,"old_ids":[3652,153001,23858,156477],"score":165,"new_id":252099},{"text":"Анджелес","train_count":82,"old_ids":[244015,158731,21785],"score":164,"new_id":252100},{"text":"Армии","train_count":164,"old_ids":[199593,159999],"score":164,"new_id":252101},{"text":"Виктория","train_count":82,"old_ids":[15729,36473,204238],"score":164,"new_id":252102},{"text":"Всесоюзного","train_count":41,"old_ids":[15729,2175,158518,157348,148325],"score":164,"new_id":252103},{"text":"Демократической","train_count":41,"old_ids":[23355,148281,148518,77779,158585],"score":164,"new_id":252104},{"text":"Крыма","train_count":82,"old_ids":[25497,36747,66050],"score":164,"new_id":252105},{"text":"Победы","train_count":82,"old_ids":[16362,148280,166844],"score":164,"new_id":252106},{"text":"Состав","train_count":164,"old_ids":[18727,150838],"score":164,"new_id":252107},{"text":"Фернандо","train_count":41,"old_ids":[52337,7358,2040,72884,1403],"score":164,"new_id":252108},{"text":"аналогично","train_count":41,"old_ids":[7105,15481,149095,31889,13292],"score":164,"new_id":252109},{"text":"апрель","train_count":82,"old_ids":[1424,148489,24623],"score":164,"new_id":252110},{"text":"армянского","train_count":82,"old_ids":[16763,167664,149280],"score":164,"new_id":252111},{"text":"барокко","train_count":82,"old_ids":[163045,148518,45527],"score":164,"new_id":252112},{"text":"батальон","train_count":82,"old_ids":[168937,36771,16806],"score":164,"new_id":252113},{"text":"белорусская","train_count":41,"old_ids":[9629,11777,8789,41635,152141],"score":164,"new_id":252114},{"text":"воинов","train_count":82,"old_ids":[148331,1739,148366],"score":164,"new_id":252115},{"text":"вызвало","train_count":82,"old_ids":[148705,8688,177948],"score":164,"new_id":252116},{"text":"гостиницы","train_count":41,"old_ids":[14942,23514,18372,1739,155055],"score":164,"new_id":252117},{"text":"действительно","train_count":82,"old_ids":[149232,19587,151269],"score":164,"new_id":252118},{"text":"древности","train_count":82,"old_ids":[60659,148380,65360],"score":164,"new_id":252119},{"text":"золотые","train_count":82,"old_ids":[227597,12666,48933],"score":164,"new_id":252120},{"text":"известны","train_count":164,"old_ids":[244718,12014],"score":164,"new_id":252121},{"text":"имеющий","train_count":82,"old_ids":[15630,1450,155099],"score":164,"new_id":252122},{"text":"кафедре","train_count":82,"old_ids":[12751,151398,156329],"score":164,"new_id":252123},{"text":"качества","train_count":164,"old_ids":[12751,160642],"score":164,"new_id":252124},{"text":"комбинат","train_count":82,"old_ids":[148558,151993,7721],"score":164,"new_id":252125},{"text":"корпусе","train_count":82,"old_ids":[153589,149447,1450],"score":164,"new_id":252126},{"text":"крайней","train_count":82,"old_ids":[148998,11796,148820],"score":164,"new_id":252127},{"text":"менеджмента","train_count":41,"old_ids":[36621,148276,16467,6240,84706],"score":164,"new_id":252128},{"text":"минуты","train_count":82,"old_ids":[150704,36183,4403],"score":164,"new_id":252129},{"text":"модернизации","train_count":41,"old_ids":[6240,148277,7358,155451,51788],"score":164,"new_id":252130},{"text":"молодого","train_count":82,"old_ids":[209034,6733,148449],"score":164,"new_id":252131},{"text":"описания","train_count":82,"old_ids":[1403,28696,36916],"score":164,"new_id":252132},{"text":"осталось","train_count":82,"old_ids":[23514,15481,95306],"score":164,"new_id":252133},{"text":"остановочный","train_count":41,"old_ids":[23514,7105,148275,36103,32401],"score":164,"new_id":252134},{"text":"открытым","train_count":82,"old_ids":[12666,91201,175440],"score":164,"new_id":252135},{"text":"открытых","train_count":82,"old_ids":[12666,91201,159928],"score":164,"new_id":252136},{"text":"параметров","train_count":82,"old_ids":[151963,150832,148327],"score":164,"new_id":252137},{"text":"педагогический","train_count":41,"old_ids":[148728,154155,14942,77436,149537],"score":164,"new_id":252138},{"text":"персонажи","train_count":82,"old_ids":[149208,152590,148539],"score":164,"new_id":252139},{"text":"песне","train_count":82,"old_ids":[7754,21785,76069],"score":164,"new_id":252140},{"text":"подразделение","train_count":82,"old_ids":[155908,148464,157182],"score":164,"new_id":252141},{"text":"подчинения","train_count":82,"old_ids":[155908,155812,17941],"score":164,"new_id":252142},{"text":"показатель","train_count":82,"old_ids":[148428,148330,57868],"score":164,"new_id":252143},{"text":"помимо","train_count":164,"old_ids":[151201,149076],"score":164,"new_id":252144},{"text":"предметы","train_count":82,"old_ids":[177917,148557,4403],"score":164,"new_id":252145},{"text":"прошли","train_count":164,"old_ids":[149490,151120],"score":164,"new_id":252146},{"text":"реже","train_count":164,"old_ids":[148279,56090],"score":164,"new_id":252147},{"text":"сигнал","train_count":82,"old_ids":[2175,66951,155371],"score":164,"new_id":252148},{"text":"символизирует","train_count":41,"old_ids":[2175,72326,1403,10750,202647],"score":164,"new_id":252149},{"text":"тканей","train_count":82,"old_ids":[1729,151011,148298],"score":164,"new_id":252150},{"text":"указано","train_count":82,"old_ids":[3652,148330,149600],"score":164,"new_id":252151},{"text":"филологический","train_count":41,"old_ids":[148594,37483,149572,77436,149537],"score":164,"new_id":252152},{"text":"финансов","train_count":82,"old_ids":[221987,7105,149297],"score":164,"new_id":252153},{"text":"Чтобы","train_count":163,"old_ids":[186639,148812],"score":163,"new_id":252154},{"text":"авторов","train_count":163,"old_ids":[148292,155610],"score":163,"new_id":252155},{"text":"выпуска","train_count":163,"old_ids":[148705,150693],"score":163,"new_id":252156},{"text":"другое","train_count":163,"old_ids":[192548,42317],"score":163,"new_id":252157},{"text":"продуктов","train_count":163,"old_ids":[184810,148779],"score":163,"new_id":252158},{"text":"структуры","train_count":163,"old_ids":[75586,153281],"score":163,"new_id":252159},{"text":"Болгария","train_count":54,"old_ids":[58297,148294,151937,37835],"score":162,"new_id":252160},{"text":"Высшая","train_count":81,"old_ids":[44093,2175,171149],"score":162,"new_id":252161},{"text":"Два","train_count":162,"old_ids":[23355,90219],"score":162,"new_id":252162},{"text":"Действие","train_count":81,"old_ids":[23355,148298,149168],"score":162,"new_id":252163},{"text":"Джонс","train_count":54,"old_ids":[23355,16467,16806,2175],"score":162,"new_id":252164},{"text":"Женщины","train_count":81,"old_ids":[154849,5092,182004],"score":162,"new_id":252165},{"text":"Зелёные","train_count":54,"old_ids":[34279,11777,42565,41546],"score":162,"new_id":252166},{"text":"Кишинёв","train_count":54,"old_ids":[188729,159400,42565,5301],"score":162,"new_id":252167},{"text":"Липецкой","train_count":54,"old_ids":[61757,45768,154800,148554],"score":162,"new_id":252168},{"text":"Минск","train_count":81,"old_ids":[36736,18372,62847],"score":162,"new_id":252169},{"text":"Примеры","train_count":81,"old_ids":[51844,77579,4403],"score":162,"new_id":252170},{"text":"Страница","train_count":81,"old_ids":[71000,91872,1424],"score":162,"new_id":252171},{"text":"Структура","train_count":81,"old_ids":[71000,61238,159142],"score":162,"new_id":252172}]'''
candidates = json.loads(NEW_TOKENS_JSON)

print(f"Всего кандидатов: {len(candidates)}")
print(f"{'слово':<15}{'частота на train':>18}{'токенов в оригинале':>22}")
for item in candidates[:8]:
    print(f"{item['text']:<15}{item['train_count']:>18}"
          f"{len(item['old_ids']):>22}")


Всего кандидатов: 4096
слово            частота на train   токенов в оригинале
году                        20126                     2
Примечания                   4459                     4
был                         11933                     2
также                        8591                     2
после                        4266                     3
Ссылки                       4167                     3
России                       3789                     3
СССР                         3559                     3


In [5]:
from tokenizers import AddedToken

extended = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
added = extended.add_tokens([
    AddedToken(item["text"], single_word=True, normalized=False)
    for item in candidates
])
print(f"Добавлено новых токенов: {added}")

word = "году"
orig_ids = original.encode(word, add_special_tokens=False)
ext_ids = extended.encode(word, add_special_tokens=False)
print(f"Слово: {word!r}")
print(f"Оригинальный токенизатор: {len(orig_ids)} токен(ов) -> {orig_ids}")
print(f"  по кусочкам: {[original.decode([i]) for i in orig_ids]}")
print(f"Расширенный токенизатор:  {len(ext_ids)} токен(ов) -> {ext_ids}")
print(f"  по кусочкам: {[extended.decode([i]) for i in ext_ids]}")


Добавлено новых токенов: 4096
Слово: 'году'
Оригинальный токенизатор: 2 токен(ов) -> [63046, 148660]
  по кусочкам: ['го', 'ду']
Расширенный токенизатор:  1 токен(ов) -> [248077]
  по кусочкам: ['году']


Отдельное слово сжимается — как и было обещано. Но в реальном тексте слова окружены пробелами и другими словами, а не стоят изолированно. Проверим тот же токен, но в естественном контексте (с ведущим пробелом, как он встречается в середине предложения):

In [6]:
spaced = " году"
orig_ids2 = original.encode(spaced, add_special_tokens=False)
ext_ids2 = extended.encode(spaced, add_special_tokens=False)

print(f"{spaced!r}: оригинал {len(orig_ids2)} ток. {orig_ids2}")
print(f"  -> расширенный {len(ext_ids2)} ток. {ext_ids2}")
orig_pieces = [original.decode([i]) for i in orig_ids2]
ext_pieces = [extended.decode([i]) for i in ext_ids2]
print(f"  оригинал по кусочкам:    {orig_pieces}")
print(f"  расширенный по кусочкам: {ext_pieces}")


' году': оригинал 1 ток. [170486]
  -> расширенный 2 ток. [220, 248077]
  оригинал по кусочкам:    [' году']
  расширенный по кусочкам: [' ', 'году']


Вот и причина провала: у оригинального токенизатора " году" (с пробелом)
уже был **одним** эффективным токеном — старые BPE-объединения умеют
склеивать пробел со словом. Новый токен добавили только для "году" без
пробела, и в реальном тексте пробел пришлось откусывать отдельно —
1 токен превратился в 2. Выигрыш на изолированном слове не пережил
встречу с реальным текстом.

## Результат

Агрегированные числа — это результат уже проведённого измерения на
полном корпусе (10 000 train + 1 000 dev документов Wikipedia RU и
70 реальных вопросах MMLU-ProX-Lite RU validation). Здесь эти числа не
пересчитываются заново, чтобы не тянуть весь корпус повторно — но
причина эффекта выше показана вживую, на настоящем токенизаторе.


In [7]:
report = {
    "train": {"token_reduction": -0.039374595530415224},
    "dev": {"token_reduction": -0.041269459999099256},
    "validation_prompts": {"token_reduction": -0.0573983636709583},
    "timing": {"median_slowdown": 0.9504441717452191},
    "passed": False,
}

rows = [
    ("train (10 000 док.)", report["train"]["token_reduction"]),
    ("dev (1 000 док.)", report["dev"]["token_reduction"]),
    ("validation MCQ (70 промптов)",
     report["validation_prompts"]["token_reduction"]),
]
print(f"{'набор':<32}{'изменение числа токенов':>26}")
for name, reduction in rows:
    sign = "+" if reduction < 0 else "-"
    print(f"{name:<32}{sign}{abs(reduction) * 100:>21.2f}%")

print()
print("Порог перехода к GPU: сокращение >= 5% на dev, без роста на "
      "validation, замедление кодирования <= 1.5x")
print(f"Итог gate: passed = {report['passed']}")


набор                              изменение числа токенов
train (10 000 док.)             +                 3.94%
dev (1 000 док.)                +                 4.13%
validation MCQ (70 промптов)    +                 5.74%

Порог перехода к GPU: сокращение >= 5% на dev, без роста на validation, замедление кодирования <= 1.5x
Итог gate: passed = False


Число токенов **выросло**, а не сократилось — на всех трёх наборах.
CPU-кодирование стало быстрее на ~5% (`median_slowdown` ≈ 0.95), но это
не влияет на GPU/E2E время.

## Вывод

Кандидат не отправлялся на GPU — порог сокращения токенов не пройден
ещё на этапе локальной проверки.
